# Scene-array caching — frozen implementation v1

**Deliverable:** a source-qualified scene block for every one of the existing 4,000 CrisisMMD images.
This notebook does not relabel SPI, train a scheduler, or interrupt the current SPI notebook.

```
Pinned image → EXIF/RGB → YOLO-World with 32 explicit prompts
             → per-class counts/areas/masks → 170-byte scene block
             → atomic SQLite cache → JSONL + 130-feature scheduler arrays
```

The vocabulary and method are frozen for this implementation. New domain accuracy and scheduling benefit
remain unmeasured. `fixture` mode validates software contracts only; `real` mode performs actual detection.
Do not interpret successful caching, larger counts, or detector confidence as urgency validation.


## 0. Environment and execution identity

Canonical repository: `SemanticSchedulerEnd2End`. Local interpreter: its `.venv`.
Kaggle uses the managed runtime, not the laptop environment. Keep managed torch/CUDA unchanged.
Readable source snapshots below are generated from the canonical modules; no Base64 or hidden source archive.

Default execution is real on Kaggle, fixture locally. Set `SCENE_ARRAY_MODE=real` explicitly for local
inference. Local image downloads remain disabled. All generated files stay in this experiment's run directory.


In [1]:
print('ENVIRONMENT START', flush=True)
import os, sys, json, time, subprocess, hashlib, importlib.metadata
from pathlib import Path
IS_KAGGLE = Path('/kaggle/working').exists()
MODE = os.environ.get('SCENE_ARRAY_MODE', 'real' if IS_KAGGLE else 'fixture')
if MODE not in ('real', 'fixture'): raise ValueError('MODE must be real or fixture')
if IS_KAGGLE:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '--no-deps', '-q',
        'ultralytics==8.3.237', 'git+https://github.com/openai/CLIP.git@a1d071733d7111c9c014f024669f959182114e33'])
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
        'ftfy', 'regex', 'tqdm', 'psutil==7.2.2'])
    PROJECT = Path('/kaggle/working')
else:
    candidates = [Path.cwd(), *Path.cwd().parents]
    PROJECT = next((p for p in candidates if (p/'parameters/scene_arrays.py').is_file()), None)
    if PROJECT is None: raise RuntimeError('run locally from canonical repository or a child directory')
RUN_ID = os.environ.get('SCENE_ARRAY_RUN_ID', time.strftime('%Y%m%dT%H%M%SZ', time.gmtime()))
if Path(RUN_ID).name != RUN_ID: raise ValueError('run ID must be one path component')
RUN = Path(os.environ.get('SCENE_ARRAY_RESUME_DIR', str(PROJECT/'experiments/scene_array_cache/runs'/RUN_ID)))
RUN.mkdir(parents=True, exist_ok=True)
BOOTSTRAP = RUN/'source_snapshot'
(BOOTSTRAP/'parameters').mkdir(parents=True, exist_ok=True)
(BOOTSTRAP/'experiments/scene_array_cache').mkdir(parents=True, exist_ok=True)
for package in ('parameters', 'experiments', 'experiments/scene_array_cache'):
    (BOOTSTRAP/package/'__init__.py').write_text('# Generated runtime package; canonical source is in the repository.\n')
os.chdir(BOOTSTRAP)
sys.path.insert(0, str(BOOTSTRAP))
print('Actual interpreter', sys.executable, sys.version, 'mode', MODE, 'run', RUN, flush=True)
print('ENVIRONMENT END', flush=True)


ENVIRONMENT START


Actual interpreter D:\Bunker\BaseCamp\WirelessCommunication\SemanticCommunication\SemanticSchedulerEnd2End\.venv\Scripts\python.exe 3.12.14 (main, Aug 25 2026, 14:01:42) [MSC v.1944 64 bit (AMD64)] mode fixture run D:\Bunker\BaseCamp\WirelessCommunication\SemanticCommunication\SemanticSchedulerEnd2End\experiments\scene_array_cache\runs\fixture_validation_20261030T063912Z


ENVIRONMENT END


## 1. Readable shared implementation

These cells materialize a verified runtime snapshot under this run directory. They never overwrite
the working repository. The canonical source hashes are checked immediately afterward.


In [2]:
%%writefile parameters/scene_arrays.py
"""Versioned detector evidence arrays. Geometry is not urgency or physical depth."""
from dataclasses import dataclass
import hashlib
import json
import math
import struct


SCHEMA = 'scene_arrays_v1'
# IDs are positions, never reordered. Regions have no defensible instance count.
OBJECTS = ('person', 'car', 'truck', 'bus', 'motorcycle', 'bicycle', 'boat',
           'airplane', 'train', 'building', 'bridge', 'utility pole',
           'traffic light', 'traffic sign', 'tree', 'animal', 'chair', 'couch',
           'table', 'bed', 'television', 'laptop', 'mobile phone', 'refrigerator')
REGIONS = ('road', 'water', 'debris', 'smoke', 'flame', 'snow', 'ground', 'grass')
CLASSES = OBJECTS + REGIONS
REGISTRY = {'schema': SCHEMA, 'classes': [
    {'id': i, 'name': label, 'prompt': label,
     'count_applicable': label in OBJECTS} for i, label in enumerate(CLASSES)],
    'area': 'largest clipped xyxy rectangle divided by oriented frame area',
    'empty_detection_area': 0.0, 'counts': 'retained boxes, not verified assets'}


def digest(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(',', ':'),
                                     allow_nan=False).encode()).hexdigest()


REGISTRY_SHA256 = digest(REGISTRY)
PREFIX = struct.Struct('<BB32s32s')
WIRE_BYTES = PREFIX.size + 32 * 2 + 32 + 4 + 4  # 170 bytes; scene block only


@dataclass(frozen=True)
class SceneArray:
    counts: tuple
    largest_area: tuple
    count_available: tuple
    area_available: tuple
    detector_ok: bool
    saturated: bool
    producer_sha256: str
    registry_sha256: str = REGISTRY_SHA256

    def __post_init__(self):
        for v in (self.counts, self.largest_area, self.count_available, self.area_available):
            if len(v) != 32:
                raise ValueError('exactly 32 ordered slots required')
        if self.registry_sha256 != REGISTRY_SHA256:
            raise ValueError('incompatible vocabulary/schema')
        if len(bytes.fromhex(self.producer_sha256)) != 32:
            raise ValueError('full producer SHA256 required')
        if type(self.detector_ok) is not bool or type(self.saturated) is not bool:
            raise ValueError('boolean status flags required')
        for i, (n, a, cm, am) in enumerate(zip(self.counts, self.largest_area,
                                              self.count_available, self.area_available)):
            if type(n) is not int or not 0 <= n <= 65535:
                raise ValueError('uint16 detection counts required')
            if not isinstance(a, (int, float)) or not math.isfinite(a) or not 0 <= a <= 1:
                raise ValueError('finite normalized area required')
            if type(cm) is not bool or type(am) is not bool:
                raise ValueError('boolean masks required')
            if not self.detector_ok and (cm or am or n or a):
                raise ValueError('failed detector cannot expose available measurements')
            if cm and CLASSES[i] not in OBJECTS:
                raise ValueError('region instance counts are inapplicable')
            if not cm and n != 0 or not am and a != 0:
                raise ValueError('unavailable values must have canonical zero storage')

    def record(self):
        return {'schema': SCHEMA, 'registry_sha256': self.registry_sha256,
                'producer_sha256': self.producer_sha256,
                'counts': list(self.counts), 'largest_area': list(self.largest_area),
                'count_available': list(self.count_available),
                'area_available': list(self.area_available),
                'detector_ok': self.detector_ok, 'saturated': self.saturated}

    @classmethod
    def from_record(cls, row):
        if row['schema'] != SCHEMA:
            raise ValueError('incompatible scene schema')
        return cls(*(tuple(row[k]) for k in ('counts', 'largest_area',
                                            'count_available', 'area_available')),
                   row['detector_ok'], row['saturated'], row['producer_sha256'],
                   row['registry_sha256'])

    def features(self, expected_producer_sha256):
        if expected_producer_sha256 != self.producer_sha256:
            raise ValueError('scheduler and extractor producer contracts differ')
        return ([math.log1p(n) for n in self.counts] + list(self.largest_area)
                + list(map(float, self.count_available))
                + list(map(float, self.area_available))
                + [float(self.detector_ok), float(self.saturated)])

    def serialize(self):
        def mask(values):
            return sum(int(v) << i for i, v in enumerate(values))
        flags = int(self.detector_ok) | (int(self.saturated) << 1)
        return (PREFIX.pack(1, flags, bytes.fromhex(self.registry_sha256),
                            bytes.fromhex(self.producer_sha256))
                + struct.pack('<32H', *self.counts)
                + bytes(round(v * 255) for v in self.largest_area)
                + struct.pack('<II', mask(self.count_available), mask(self.area_available)))

    @classmethod
    def deserialize(cls, data, expected_producer_sha256):
        if len(data) != WIRE_BYTES:
            raise ValueError('scene block must be exactly 170 bytes')
        version, flags, registry, producer = PREFIX.unpack_from(data)
        if version != 1 or flags & ~3 or registry.hex() != REGISTRY_SHA256:
            raise ValueError('invalid version, flags or vocabulary')
        if producer.hex() != expected_producer_sha256:
            raise ValueError('incompatible detector configuration/weights')
        counts = struct.unpack_from('<32H', data, PREFIX.size)
        start = PREFIX.size + 64
        area = tuple(v / 255 for v in data[start:start + 32])
        cm, am = struct.unpack_from('<II', data, start + 32)
        return cls(counts, area, tuple(bool(cm & (1 << i)) for i in range(32)),
                   tuple(bool(am & (1 << i)) for i in range(32)),
                   bool(flags & 1), bool(flags & 2), producer.hex())


def summarize(detections, width, height, producer_sha256, max_detections=300):
    """One box list -> arrays plus audit. No category importance weights."""
    if type(width) is not int or type(height) is not int or width <= 0 or height <= 0:
        raise ValueError('positive oriented image dimensions required')
    if type(max_detections) is not int or max_detections <= 0:
        raise ValueError('positive detection cap required')
    counts = [0] * 32
    areas = [0.0] * 32
    rejected = []
    kept = []
    for number, row in enumerate(detections):
        # A wrong dictionary is an integration failure, not a discarded object.
        label = row.get('label')
        if label not in CLASSES:
            raise ValueError('class outside configured vocabulary: ' + str(label))
        i = CLASSES.index(label)
        try:
            box = tuple(float(x) for x in row['box'])
            if len(box) != 4 or not all(math.isfinite(x) for x in box):
                raise ValueError('invalid box')
            x1, y1, x2, y2 = box
            if x2 <= x1 or y2 <= y1:
                raise ValueError('reversed/empty box')
            x1, x2 = max(0, min(width, x1)), max(0, min(width, x2))
            y1, y2 = max(0, min(height, y1)), max(0, min(height, y2))
            if x2 <= x1 or y2 <= y1:
                raise ValueError('box outside image')
            score = float(row['score'])
            if not math.isfinite(score) or not 0 <= score <= 1:
                raise ValueError('invalid detector score')
        except (ValueError, KeyError, TypeError) as error:
            rejected.append({'index': number, 'reason': str(error)})
            continue
        if label in OBJECTS:
            counts[i] += 1
        areas[i] = max(areas[i], (x2 - x1) * (y2 - y1) / (width * height))
        kept.append({'class_id': i, 'label': label, 'box': [x1, y1, x2, y2], 'score': score})
    result = SceneArray(tuple(counts), tuple(areas), tuple(c in OBJECTS for c in CLASSES),
                        (True,) * 32, True, len(detections) >= max_detections,
                        producer_sha256)
    return result, {'detections': kept, 'rejected_boxes': rejected,
                    'width': width, 'height': height}


def unavailable(producer_sha256):
    return SceneArray((0,) * 32, (0.0,) * 32, (False,) * 32, (False,) * 32,
                      False, False, producer_sha256)


def class_slot(label):
    """Outside vocabulary is distinct from an available slot containing zero."""
    if label not in CLASSES:
        raise ValueError('outside_vocabulary: ' + str(label))
    return CLASSES.index(label)


Writing parameters/scene_arrays.py


In [3]:
%%writefile experiments/scene_array_cache/cache.py
"""Bounded scene-array cache; evaluator labels and SPI targets never enter detection."""
from contextlib import contextmanager
from dataclasses import asdict
from pathlib import Path
import hashlib
import importlib.metadata
import json
import os
import platform
import sqlite3
import sys
import time
import urllib.request

from parameters.scene_arrays import (CLASSES, OBJECTS, REGISTRY, REGISTRY_SHA256,
    WIRE_BYTES, SceneArray, summarize, unavailable, digest)

SETTINGS = {'conf': .25, 'iou': .5, 'imgsz': 640, 'max_det': 300,
            'agnostic_nms': False, 'half': False, 'augment': False, 'verbose': False}
DATA_REVISION = '10a5626ba112ab9c50369c473a2062bcf205d708'
WEIGHTS_SHA256 = '9b2c17ab6124a913e9b3a5c170617920d91b0f01111a8479da69f00e2cf27792'


def file_hash(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()


def write_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + '.partial')
    with tmp.open('w', encoding='utf8') as stream:
        json.dump(value, stream, indent=2, allow_nan=False)
        stream.flush()
        os.fsync(stream.fileno())
    tmp.replace(path)


@contextmanager
def stage(run, name):
    """Persist stdout/stderr alongside normal notebook output, including failures."""
    from contextlib import redirect_stdout, redirect_stderr
    run = Path(run)
    (run / 'logs').mkdir(parents=True, exist_ok=True)
    class Tee:
        def __init__(self, parent, log): self.parent, self.log = parent, log
        def write(self, text):
            self.parent.write(text); self.log.write(text); self.log.flush()
            return len(text)
        def flush(self): self.parent.flush(); self.log.flush()
        def isatty(self): return False
    start = time.monotonic()
    status = 'complete'
    with (run / 'logs' / (name + '.log')).open('a', encoding='utf8') as log:
        with redirect_stdout(Tee(sys.stdout, log)), redirect_stderr(Tee(sys.stderr, log)):
            print('START', name, 'output', run, flush=True)
            try:
                yield
            except BaseException:
                import traceback
                status = 'failed'
                traceback.print_exc()
                raise
            finally:
                receipt = {'stage': name, 'status': status, 'seconds': time.monotonic() - start}
                write_json(run / 'logs' / (name + '.json'), receipt)
                print('END', receipt, flush=True)


def environment():
    values = {'python': sys.version, 'executable': sys.executable,
              'host': platform.node(), 'platform': platform.platform()}
    for name in ('torch', 'ultralytics', 'numpy', 'Pillow', 'clip', 'psutil'):
        try: values[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError: values[name] = 'not installed'
    try:
        import torch
        values['cuda_available'] = torch.cuda.is_available()
        values['gpus'] = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
        values['torch_threads'] = torch.get_num_threads()
    except ImportError:
        values['cuda_available'] = False
    return values


class Detector:
    def __init__(self, weights, device):
        import torch
        from ultralytics import YOLOWorld
        start = time.perf_counter()
        weights = Path(weights)
        if file_hash(weights) != WEIGHTS_SHA256:
            raise ValueError('unapproved YOLO-World weights')
        self.device = device
        self.model = YOLOWorld(str(weights)).to(device)
        self.load_seconds = time.perf_counter() - start
        start = time.perf_counter()
        self.model.set_classes(list(CLASSES))
        self.sync()
        self.vocabulary_setup_seconds = time.perf_counter() - start
        features = self.model.model.txt_feats.detach().cpu().contiguous()
        self.signature = {'schema': 'scene_arrays_v1', 'registry_sha256': REGISTRY_SHA256,
            'weights_sha256': WEIGHTS_SHA256, 'settings': SETTINGS,
            'text_features_sha256': hashlib.sha256(features.numpy().tobytes()).hexdigest(),
            'ultralytics': importlib.metadata.version('ultralytics'),
            'torch': torch.__version__, 'device': device, 'dtype': 'float32',
            'input_convention': 'EXIF-upright RGB PIL, Ultralytics native letterbox',
            'postprocessing_source_sha256': file_hash(Path(__file__).parents[2] / 'parameters/scene_arrays.py'),
            'runner_source_sha256': file_hash(__file__)}
        self.producer_sha256 = digest(self.signature)
        # Text features are cached; the text encoder is not a per-image cost.
        self.model.model.clip_model = None
        import gc
        gc.collect()
        if device.startswith('cuda'): torch.cuda.empty_cache()
        self.parameters = sum(p.numel() for p in self.model.model.parameters())

    def sync(self):
        if self.device.startswith('cuda'):
            import torch
            torch.cuda.synchronize(self.device)

    def predict(self, image):
        result = self.model.predict(image, device=self.device, **SETTINGS)[0]
        names = [result.names[i] for i in range(len(result.names))]
        if tuple(names) != CLASSES:
            raise ValueError('detector vocabulary order differs from scheduler registry')
        rows = [{'label': result.names[int(i)], 'box': b, 'score': float(s)}
                for b, i, s in zip(result.boxes.xyxy.detach().cpu().tolist(),
                                   result.boxes.cls.detach().cpu().tolist(),
                                   result.boxes.conf.detach().cpu().tolist())]
        return rows


def get_image(row, directory, allow_download=False):
    """Bound network retries and check expected raw-image hashes when supplied."""
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    source = Path(row['image_path'])
    if source.is_file():
        path = source
    else:
        path = directory / (hashlib.sha256(row['image_id'].encode()).hexdigest() + '.image')
        if not path.exists():
            if not allow_download:
                raise FileNotFoundError('image missing; local bulk downloading is disabled')
            relative = row['image_path']
            if not relative.startswith('data_image/') or '..' in relative.split('/'):
                raise ValueError('not a pinned CrisisMMD image path')
            url = 'https://huggingface.co/datasets/QCRI/CrisisMMD/resolve/' + DATA_REVISION + '/' + relative
            for attempt in range(5):
                try:
                    suffix = '?download=true&attempt=' + str(attempt)
                    with urllib.request.urlopen(url + suffix, timeout=60) as response:
                        raw = response.read(20 * 1024 * 1024 + 1)
                    if len(raw) > 20 * 1024 * 1024:
                        raise ValueError('image exceeds declared 20MiB cap')
                    part = path.with_suffix('.partial')
                    part.write_bytes(raw)
                    part.replace(path)
                    break
                except Exception:
                    if attempt == 4: raise
                    time.sleep(min(16, 2 ** attempt))
    actual = file_hash(path)
    expected = row.get('image_sha256')
    if expected and expected != actual:
        raise ValueError('raw image SHA256 mismatch: ' + row['image_id'])
    return path, actual


def read_image(path):
    from PIL import Image, ImageOps
    with Image.open(path) as image:
        return ImageOps.exif_transpose(image).convert('RGB')


def extract(path, image_id, sha, detector):
    """Executed loading, predict, array and wire costs; cache reads use another path."""
    total = time.perf_counter()
    times = {}
    try:
        begin = time.perf_counter()
        image = read_image(path)
        times['image_load_exif_rgb_ms'] = (time.perf_counter() - begin) * 1000
        detector.sync(); begin = time.perf_counter()
        detections = detector.predict(image)
        detector.sync()
        times['detector_preprocess_forward_postprocess_transfer_ms'] = (time.perf_counter() - begin) * 1000
        begin = time.perf_counter()
        arrays, audit = summarize(detections, image.width, image.height,
                                  detector.producer_sha256, SETTINGS['max_det'])
        times['array_calculation_ms'] = (time.perf_counter() - begin) * 1000
        begin = time.perf_counter()
        wire = arrays.serialize()
        decoded = SceneArray.deserialize(wire, detector.producer_sha256)
        error = max(abs(a-b) for a, b in zip(arrays.largest_area, decoded.largest_area))
        if error > .5 / 255 + 1e-12: raise ValueError('area quantization bound violated')
        times['serialization_validation_ms'] = (time.perf_counter() - begin) * 1000
        status, reason = 'ok', None
    except Exception as error:
        arrays = unavailable(detector.producer_sha256)
        audit = {}
        wire = arrays.serialize()
        status, reason = 'failed', type(error).__name__ + ': ' + str(error)
    times['total_extraction_ms'] = (time.perf_counter() - total) * 1000
    return {'image_id': image_id, 'image_sha256': sha, 'scene': arrays.record(),
            'audit': audit, 'status': status, 'failure_reason': reason,
            'timing': times, 'measurement': 'fresh inference', 'wire_bytes': len(wire),
            'wire_hex': wire.hex()}, wire


def cache_all(rows, detector, run, allow_download=False, seconds_limit=14400):
    run = Path(run)
    run.mkdir(parents=True, exist_ok=True)
    ids = [r['image_id'] for r in rows]
    if len(set(ids)) != len(ids): raise ValueError('duplicate input IDs')
    identity = {'producer': detector.signature,
                'inputs': [{'image_id': r['image_id'], 'image_path': r['image_path'],
                            'image_sha256': r.get('image_sha256'), 'split': r.get('split'),
                            'group': r.get('group')} for r in rows]}
    expected = digest(identity)
    db = sqlite3.connect(run / 'scene_cache.sqlite')
    db.executescript('CREATE TABLE IF NOT EXISTS meta(key TEXT PRIMARY KEY,value TEXT);'
                     'CREATE TABLE IF NOT EXISTS outputs(id TEXT PRIMARY KEY,record TEXT,wire BLOB,checksum TEXT);')
    prior = db.execute("SELECT value FROM meta WHERE key='identity'").fetchone()
    if prior and prior[0] != expected:
        db.close(); raise ValueError('cache source, model, vocabulary or configuration mismatch')
    with db: db.execute('INSERT OR IGNORE INTO meta VALUES(?,?)', ('identity', expected))
    write_json(run / 'cache_identity.json', identity)
    write_json(run / 'registry.json', REGISTRY)
    write_json(run / 'environment.json', environment())
    started = time.monotonic()
    fresh = hits = 0
    try:
        for number, row in enumerate(rows):
            if time.monotonic() - started > seconds_limit:
                print('SESSION BUDGET: preserving checkpoint', flush=True); break
            prior = db.execute('SELECT record,wire,checksum FROM outputs WHERE id=?', (row['image_id'],)).fetchone()
            if prior:
                if hashlib.sha256(prior[0].encode() + prior[1]).hexdigest() != prior[2]:
                    raise ValueError('stored row integrity failure')
                SceneArray.deserialize(prior[1], detector.producer_sha256)
                hits += 1
                continue
            download_begin = time.perf_counter()
            try:
                path, sha = get_image(row, run / 'images', allow_download)
                fetch_ms = (time.perf_counter() - download_begin) * 1000
                record, wire = extract(path, row['image_id'], sha, detector)
                record['fetch_verify_ms'] = fetch_ms
            except Exception as error:
                array = unavailable(detector.producer_sha256); wire = array.serialize()
                record = {'image_id': row['image_id'], 'image_sha256': row.get('image_sha256'),
                          'scene': array.record(), 'status': 'failed',
                          'failure_reason': type(error).__name__ + ': ' + str(error),
                          'timing': {}, 'measurement': 'no inference: input failed',
                          'wire_bytes': len(wire), 'wire_hex': wire.hex()}
            # Routing/provenance fields are carried, never fed to scene extraction.
            record['source_group'] = row.get('group', 'tweet:' + row['image_id'].rsplit('_', 1)[0])
            record['source_split'] = row.get('split')
            text = json.dumps(record, sort_keys=True, allow_nan=False)
            checksum = hashlib.sha256(text.encode() + wire).hexdigest()
            with db: db.execute('INSERT INTO outputs VALUES(?,?,?,?)', (row['image_id'], text, wire, checksum))
            fresh += 1
            if fresh % 25 == 0:
                print('SCENE CACHE', number+1, '/', len(rows), 'fresh', fresh, 'cache_hits', hits, flush=True)
                export(db, run, len(rows))
        return export(db, run, len(rows))
    finally:
        try:
            if sys.exc_info()[0] is None:
                export(db, run, len(rows))
            else:
                try:
                    export(db, run, len(rows))
                except Exception as error:
                    print('Partial export failed; preserving original error and SQLite:', str(error), flush=True)
        finally:
            db.close()


def export(db, run, expected_count):
    """One row per image, atomic flat JSONL and scheduler-ready numeric matrix."""
    import numpy as np
    run = Path(run)
    features, wire_features, ids, successful = [], [], [], []
    parts = run / 'scene_records.jsonl.partial'
    failures = []
    with parts.open('w', encoding='utf8') as stream:
        for i, text, wire, checksum in db.execute('SELECT id,record,wire,checksum FROM outputs ORDER BY id'):
            if hashlib.sha256(text.encode() + wire).hexdigest() != checksum:
                raise ValueError('cannot export corrupt cache row')
            record = json.loads(text); array = SceneArray.from_record(record['scene'])
            decoded = SceneArray.deserialize(wire, array.producer_sha256)
            stream.write(text + '\n')
            ids.append(i); successful.append(record['status'] == 'ok')
            features.append(array.features(array.producer_sha256))
            wire_features.append(decoded.features(array.producer_sha256))
            if record['status'] != 'ok': failures.append({'image_id': i, 'reason': record['failure_reason']})
        stream.flush(); os.fsync(stream.fileno())
    parts.replace(run / 'scene_records.jsonl')
    with (run / 'scheduler_arrays.npz.partial').open('wb') as stream:
        np.savez_compressed(stream, image_ids=np.array(ids),
                            float_features=np.asarray(features, dtype=np.float32).reshape(-1, 130),
                            wire_features=np.asarray(wire_features, dtype=np.float32).reshape(-1, 130),
                            successful=np.array(successful, dtype=bool), registry_sha256=REGISTRY_SHA256,
                            producer_sha256=(json.loads(db.execute('SELECT record FROM outputs LIMIT 1').fetchone()[0])['scene']['producer_sha256'] if ids else ''),
                            feature_layout='log1p_counts32,areas32,count_masks32,area_masks32,ok,saturated')
    (run / 'scheduler_arrays.npz.partial').replace(run / 'scheduler_arrays.npz')
    progress = {'stored': len(ids), 'expected': expected_count, 'successful': sum(successful),
                'failed': len(failures), 'coverage_complete': len(ids) == expected_count,
                'all_successful': len(ids) == expected_count and not failures,
                'wire_bytes_per_scene_block': WIRE_BYTES,
                'accuracy_status': 'not measured; no reference boxes supplied'}
    write_json(run / 'progress.json', progress)
    write_json(run / 'failures.json', failures)
    return progress


def profile(rows, weights, run, devices=('cpu', 'cuda:0')):
    """Ten fixed development images, 10 warmups, 30 CPU/100 GPU fresh calls."""
    import gc
    import numpy as np
    import psutil
    import torch
    run = Path(run)
    fixed = [r for r in rows if r.get('split') == 'train'][:10]
    if len(fixed) != 10: raise ValueError('ten declared development images required')
    paths = [get_image(r, run / 'images', False) for r in fixed]
    for device in devices:
        print('PROFILE', device, '10 fixed development images', flush=True)
        detector = Detector(weights, device)
        warm_path, warm_sha = paths[0]
        for _ in range(10):
            result, _ = extract(warm_path, fixed[0]['image_id'], warm_sha, detector)
            if result['status'] != 'ok': raise ValueError('profiling warmup failed: ' + result['failure_reason'])
        measured = []
        repeats = 3 if device == 'cpu' else 10
        for row, (path, sha) in zip(fixed, paths):
            for _ in range(repeats):
                result, _ = extract(path, row['image_id'], sha, detector)
                if result['status'] != 'ok': raise ValueError('profile invocation failed')
                measured.append({'image_id': row['image_id'], **result['timing']})
        detector.sync(); start = time.perf_counter()
        for row, (path, sha) in zip(fixed, paths): extract(path, row['image_id'], sha, detector)
        detector.sync(); elapsed = time.perf_counter() - start
        if device.startswith('cuda'): torch.cuda.reset_peak_memory_stats(device)
        extract(warm_path, fixed[0]['image_id'], warm_sha, detector); detector.sync()
        memory = {'rss_bytes': psutil.Process().memory_info().rss,
                  'cuda_peak_allocated_bytes': torch.cuda.max_memory_allocated(device) if device.startswith('cuda') else None,
                  'cuda_peak_reserved_bytes': torch.cuda.max_memory_reserved(device) if device.startswith('cuda') else None,
                  'scope': 'absolute process memory, separate pass from timing'}
        try:
            image = read_image(warm_path)
            with torch.inference_mode(), torch.profiler.profile(
                    activities=[torch.profiler.ProfilerActivity.CPU], with_flops=True) as trace:
                detector.predict(image)
            events = trace.key_averages()
            flops = {'status': 'partial', 'supported_flops': sum(e.flops for e in events),
                     'uncounted_operators': [e.key for e in events if not e.flops],
                     'convention': 'two FLOPs per MAC', 'vocabulary_size': 32,
                     'image_size': 640, 'batch_size': 1, 'dtype': 'float32',
                     'scope': 'detector invocation including supported preprocess/postprocess operators; not full pipeline arithmetic'}
        except Exception as error:
            flops = {'status': 'failed', 'supported_flops': None, 'reason': str(error)}
        summary = {key: {'p50_ms': float(np.median([r[key] for r in measured])),
                         'p95_ms': float(np.percentile([r[key] for r in measured], 95))}
                   for key in measured[0] if key.endswith('_ms')}
        report = {'device': device, 'host': platform.node(), 'environment': environment(),
                  'producer': detector.signature, 'samples': measured, 'timing_summary': summary,
                  'warmup_calls': 10, 'measured_calls': len(measured),
                  'batch_one_sequential_throughput': 10 / elapsed,
                  'throughput_scope': '10 sequential batch-one full extractions; not batched inference',
                  'memory': memory, 'flops': flops, 'parameters': detector.parameters,
                  'weights_bytes': Path(weights).stat().st_size,
                  'model_object_load_seconds': detector.load_seconds,
                  'vocabulary_setup_seconds': detector.vocabulary_setup_seconds,
                  'load_scope': 'fresh object; OS/model download caches may be warm',
                  'scene_block_bytes': WIRE_BYTES}
        write_json(run / 'profiles' / (device.replace(':', '_') + '.json'), report)
        del detector; gc.collect()
        if device.startswith('cuda'): torch.cuda.empty_cache()


def report(run):
    """Cost/availability summary; never claims detector accuracy or ranking benefit."""
    import numpy as np
    run = Path(run)
    records = [json.loads(line) for line in (run / 'scene_records.jsonl').read_text().splitlines()]
    records = [r for r in records if r['status'] == 'ok']
    if not records: raise ValueError('no successful records for summary')
    arrays = [SceneArray.from_record(r['scene']) for r in records]
    write_json(run / 'summary.json', {
        'successful': len(records), 'saturated': sum(a.saturated for a in arrays),
        'all_zero_detections': sum(not any(a.largest_area) for a in arrays),
        'per_class_detected_images': {c: sum(a.largest_area[i] > 0 for a in arrays) for i, c in enumerate(CLASSES)},
        'actual_fresh_extraction_p50_ms': float(np.median([r['timing']['total_extraction_ms'] for r in records])),
        'actual_fresh_extraction_p95_ms': float(np.percentile([r['timing']['total_extraction_ms'] for r in records], 95)),
        'area_quantization_max_error_bound': .5 / 255,
        'retained_evidence': 'labels/boxes/scores in audit; arrays contain counts and largest rectangles',
        'inference_scope': 'detector-dependent scene composition; not urgency, depth or complete scene semantics',
        'supervision_used': False, 'detector_accuracy_measured': False,
        'scheduler_benefit_measured': False})
    artifacts = [{'path': str(p.relative_to(run)), 'bytes': p.stat().st_size, 'sha256': file_hash(p)}
                 for p in sorted(run.rglob('*')) if p.is_file() and 'images' not in p.relative_to(run).parts
                 and not p.name.endswith(('.partial', '.sqlite-journal')) and p.name != 'artifact_manifest.json']
    write_json(run / 'artifact_manifest.json', artifacts)
    return json.loads((run / 'summary.json').read_text())


Writing experiments/scene_array_cache/cache.py


In [4]:
from experiments.scene_array_cache.cache import (Detector, SETTINGS, WEIGHTS_SHA256, stage,
    write_json, file_hash, environment, cache_all, report, profile, get_image, read_image, extract)
from parameters.scene_arrays import (CLASSES, OBJECTS, REGIONS, REGISTRY, REGISTRY_SHA256,
    WIRE_BYTES, SceneArray, summarize, unavailable, digest)
SOURCE_RECEIPT = {'parameters/scene_arrays.py': '6f6707cae96776c6cc7facbc8f525b810c8ab59d8408c80f47a6435de5e72afe', 'experiments/scene_array_cache/cache.py': '51aac63af08aeb44f1f85b0b0385a5c7e088981cdefcf8ea9a070d1b893c94b3'}
SOURCE_COMMIT = '032c23a3dafc7bfeb39a84ba55193b6b6f3a22a0'
with stage(RUN, '01_source_environment'):
    for relative, expected in SOURCE_RECEIPT.items():
        if file_hash(BOOTSTRAP/relative) != expected: raise ValueError('source snapshot hash mismatch')
    write_json(RUN/'source.json', {'commit': SOURCE_COMMIT, 'files': SOURCE_RECEIPT,
        'scope': 'exact selected files from dirty checkout; commit alone does not identify these edits'})
    write_json(RUN/'environment.json', environment())
    print(environment(), flush=True)


START 01_source_environment output D:\Bunker\BaseCamp\WirelessCommunication\SemanticCommunication\SemanticSchedulerEnd2End\experiments\scene_array_cache\runs\fixture_validation_20261030T063912Z


END {'stage': '01_source_environment', 'status': 'failed', 'seconds': 0.0}


Traceback (most recent call last):
  File "D:\Bunker\BaseCamp\WirelessCommunication\SemanticCommunication\SemanticSchedulerEnd2End\experiments\scene_array_cache\runs\fixture_validation_20261030T063912Z\source_snapshot\experiments\scene_array_cache\cache.py", line 62, in stage
    yield
  File "C:\Users\satya\AppData\Local\Temp\ipykernel_23724\2074804555.py", line 9, in <module>
    if file_hash(BOOTSTRAP/relative) != expected: raise ValueError('source snapshot hash mismatch')
                                                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
ValueError: source snapshot hash mismatch


ValueError: source snapshot hash mismatch

## 2. Frozen vocabulary, representation and settings

24 object categories have count + area. Eight region categories have area only; their count masks
are false. Areas are rectangles, not segmentation or distance. Default COCO classes are replaced
explicitly. Zero detections is different from detector failure. Outside-vocabulary queries are errors,
not zero-valued observations. A saturated result can undercount and carries its cap flag.

Counts use uint16; areas use uint8 with rounding. Full vocabulary and producer SHA256 values,
status, and masks produce an exact **170-byte standalone scene block**. This is not the old 35-byte
header and does not include SPI, relevance, novelty or routing metadata. Float and wire-derived features
are exported separately so the scheduler can train on the representation it will receive.


In [ ]:
with stage(RUN, '02_contract'):
    print('SETTINGS', SETTINGS, flush=True)
    for entry in REGISTRY['classes']: print(entry, flush=True)
    write_json(RUN/'registry.json', REGISTRY)
    write_json(RUN/'configuration.json', {'mode': MODE, 'settings': SETTINGS,
        'registry_sha256': REGISTRY_SHA256, 'scene_block_bytes': WIRE_BYTES,
        'feature_layout': ['log1p(counts)[32]', 'areas[32]', 'count masks[32]',
                           'area masks[32]', 'detector_ok', 'saturated'],
        'detector_accuracy_status': 'not measured for this expanded vocabulary'})


## 3. Exact existing 4,000-image cohort

This table contains identities and source paths only. Native labels, SPI labels, mission paragraphs
and future observations do not enter detection. The source dataset revision is pinned. Existing raw
image hashes are checked; new hashes are measured and saved, not claimed known in advance.
Data are fetched only on Kaggle or read from supplied local image paths. No full archive is downloaded.


In [ ]:
COHORT_ROWS = [('919975227953762304_0', 'data_image/california_wildfires/16_10_2017/919975227953762304_0.jpg', 'test', '0bdc6b0604407cd5739f8f37551a87b8ee1809fc2e4d439f0f27ccb2c345f145'), ('917935359039868928_0', 'data_image/california_wildfires/11_10_2017/917935359039868928_0.jpg', 'dev', 'f0ecdc7f596cd8e40c789a59d15b12b88c2eeab17ea74f049c5725df8a915080'), ('920765218216660992_0', 'data_image/california_wildfires/18_10_2017/920765218216660992_0.jpg', 'dev', 'ffcdfeb73a9c58958adf5c116d58e15e0a647732d7be39e00be957bc150d8afd'), ('917862029465026560_0', 'data_image/california_wildfires/10_10_2017/917862029465026560_0.jpg', 'test', '8dafa16c177e3ad26a9f396a7eb70d0b5cf55b218ee19620296b223276811362'), ('920016499473047552_1', 'data_image/california_wildfires/16_10_2017/920016499473047552_1.jpg', 'train', 'd334ae15121f7a18230f4783ef0fd25dfb95ef5662da36447d9b8927fca8267d'), ('917993008456519681_0', 'data_image/california_wildfires/11_10_2017/917993008456519681_0.jpg', 'train', '7b55edce3d9066e6ffa6569f3b98e27662aa1c7970b5ae6a729accf1aa44f87a'), ('918108362973335555_0', 'data_image/california_wildfires/11_10_2017/918108362973335555_0.jpg', 'train', '325f2abbfc8d634e89650545ba6a4f9cc03bad3e00ae1993224ad29fe2724a97'), ('918075669048938496_0', 'data_image/california_wildfires/11_10_2017/918075669048938496_0.jpg', 'test', '276ff277979db3a4343b236bb8d237086ee16c3f36706d2842dc3d6ea071203c'), ('917828415373189120_0', 'data_image/california_wildfires/10_10_2017/917828415373189120_0.jpg', 'dev', 'f48884d38896559dbaa4f2c83d9c6b1d96ad5a1e4812c7def0e22f4417dd3e07'), ('922547024725184512_0', 'data_image/california_wildfires/23_10_2017/922547024725184512_0.jpg', 'train', '4daab1e271a246c9646bed41ddfcc65efd19cf9e1586a480e2510b590a34bae7'), ('917848117084430336_0', 'data_image/california_wildfires/10_10_2017/917848117084430336_0.jpg', 'test', '07630a7465b0ea3c2f650a9bfc96a01f7bd7f3cfabb3ae66fe56abe91c90bcc9'), ('917829663552045057_0', 'data_image/california_wildfires/10_10_2017/917829663552045057_0.jpg', 'dev', '88feae2bb09df85db22e68f3868a9b962df7b48f01dab20190cd95956eaef88d'), ('920319688348901376_0', 'data_image/california_wildfires/17_10_2017/920319688348901376_0.jpg', 'dev', '9745061b161d48a661b1403218ff0015ef28a30a5d7aeed2dbcb4cd3d8532fed'), ('919708605200310272_0', 'data_image/california_wildfires/15_10_2017/919708605200310272_0.jpg', 'train', 'a4460607fc3a2e518bc73629f33ba64879afe6d75049a0a2147dc4a8f5726473'), ('918051144454344704_0', 'data_image/california_wildfires/11_10_2017/918051144454344704_0.jpg', 'train', '7b094e67a02270c168716792c24a3c7b0832b600f00aea600940142500c5a149'), ('919715666264965121_0', 'data_image/california_wildfires/16_10_2017/919715666264965121_0.jpg', 'train', '701652c658c15fb681af6f30b6a2d20158e0ecc749beb43bcea04fd54f293b2e'), ('922468220388429824_0', 'data_image/california_wildfires/23_10_2017/922468220388429824_0.jpg', 'dev', '167c8d89267a7230c81419372c5a3eef6c4c916bc5895534cf5b4c7fabd360b2'), ('920016499473047552_0', 'data_image/california_wildfires/16_10_2017/920016499473047552_0.jpg', 'train', '1cc1cca9e45b2cc1fa804bbe33c4cc87aa534411513432dd1bc7638ad00afac0'), ('918104972268535810_0', 'data_image/california_wildfires/11_10_2017/918104972268535810_0.jpg', 'train', 'ce6c95eaba9052f1da9228d71f549360ad25ec86e10c167bf14db211aacb40c1'), ('922569327034769408_1', 'data_image/california_wildfires/23_10_2017/922569327034769408_1.jpg', 'train', '4712962e3bffb779d52071780db4fc7dc5ef61be882eae425c8cb0bb28e06b10'), ('922595496245460998_0', 'data_image/california_wildfires/23_10_2017/922595496245460998_0.jpg', 'train', '8b881719bbd81936b6e8dea6ab9a20a97e0b91c995a18da42f3542d8737ea06a'), ('922456424415940608_0', 'data_image/california_wildfires/23_10_2017/922456424415940608_0.jpg', 'train', 'a0ed22b928a44116e87c809768d8730173c4231313247fa5316701df6aa4412c'), ('918087716050530304_0', 'data_image/california_wildfires/11_10_2017/918087716050530304_0.jpg', 'train', '23a6776c237456aba33d09a246d7a938490f2b058eb86364ba6e6aade10ca4c0'), ('921176414446960640_0', 'data_image/california_wildfires/20_10_2017/921176414446960640_0.jpg', 'test', 'e3bf5b9c8b38c1d3c084e817da8704a18b6c6ebbaa2e421922a9810a993562da'), ('920340725266497536_0', 'data_image/california_wildfires/17_10_2017/920340725266497536_0.jpg', 'test', '232873003b777d77d1aeaadc4ef9d79f8a818a1b998e99b0abe667d6be6a9d07'), ('919971112892628994_0', 'data_image/california_wildfires/16_10_2017/919971112892628994_0.jpg', 'test', '04da33319922dd493f47d9e5a249e3f1d80653f7b85cf21066c13906c5a63653'), ('919965129898205190_1', 'data_image/california_wildfires/16_10_2017/919965129898205190_1.jpg', 'train', '84fa561f3e45ae35c3d31253a38a1a45228abf10e618abba705e0a53dc8f03d7'), ('919869455928238081_0', 'data_image/california_wildfires/16_10_2017/919869455928238081_0.jpg', 'test', '8a3e8fc8e875b045a0f8057b3f5d600549d82a02a3ec15f938702c1421edd2eb'), ('919674860175060992_0', 'data_image/california_wildfires/15_10_2017/919674860175060992_0.jpg', 'train', '371ebf7865a580f16f2fe5e87c5291eacb62615548a491ec0037170484128b83'), ('920621738932649985_0', 'data_image/california_wildfires/18_10_2017/920621738932649985_0.jpg', 'test', 'bc81655a6d8855495a224c0ea452e307d073e18edcb6a06aede393f00f363742'), ('919596537566523398_0', 'data_image/california_wildfires/15_10_2017/919596537566523398_0.jpg', 'test', '5af21fa5d34d57e5e192bde6362fac26bfc618bdfe601dd93ad9f888667adc24'), ('920042438181269504_0', 'data_image/california_wildfires/16_10_2017/920042438181269504_0.jpg', 'test', '970882e5a06e053c1ae2323d96ccc545b061a700caacc9ac94567dfb36777fe4'), ('917793881533571073_0', 'data_image/california_wildfires/10_10_2017/917793881533571073_0.jpg', 'dev', '5300d0317d45e535c20d80549e9f8fbdef5d654557832fd11d4a894b9d10210d'), ('922116615050739715_0', 'data_image/california_wildfires/22_10_2017/922116615050739715_0.jpg', 'dev', 'a260500f3566be6a4853aa0954c4e0b2c0f6e7d844375a7402ee04b93b5ee902'), ('907393512630611969_0', 'data_image/hurricane_harvey/12_9_2017/907393512630611969_0.jpg', 'train', '7981fa97725c0fcc53d40c6fedd25ebcfd9ce30a0bb13ec4b9d919495f7785e6'), ('904692950780846082_0', 'data_image/hurricane_harvey/4_9_2017/904692950780846082_0.jpg', 'dev', 'd80ab09e742941c390cc23557bf9a5dc900812e06c4d32fc6690d4adf178915b'), ('908021007612555270_0', 'data_image/hurricane_harvey/13_9_2017/908021007612555270_0.jpg', 'test', '59f3975be39cc19ad0a7bb3277db0dc47015fa0edb5362a2521c260cd4bd20a9'), ('904837965440053248_0', 'data_image/hurricane_harvey/4_9_2017/904837965440053248_0.jpg', 'train', '703d7e9fb34692d191de107ec46bd0f9a4ac9abf639b4f93b4296605596b4a89'), ('908446167545126912_0', 'data_image/hurricane_harvey/14_9_2017/908446167545126912_0.jpg', 'train', '268c5ff06f26ea1af662a6e6329cd373a666ac89025beb93af8508ad6ff2c397'), ('906801256127254529_0', 'data_image/hurricane_harvey/10_9_2017/906801256127254529_0.jpg', 'train', '2bfcbb37cf63b90074291915b58b72dbc9cb9741cc6f760771449159060ba9a0'), ('905351240715866112_0', 'data_image/hurricane_harvey/6_9_2017/905351240715866112_0.jpg', 'train', 'cd0bf9f2012b577d0a5e71576c13774fef8cd82eb2a7f94efa9542af1e2e23a0'), ('908496398898925569_0', 'data_image/hurricane_harvey/15_9_2017/908496398898925569_0.jpg', 'dev', '0bf04561da961e1b9b20784c33e7941a2c867d1931883dcad58290036e89c9dd'), ('907259489728024577_0', 'data_image/hurricane_harvey/11_9_2017/907259489728024577_0.jpg', 'train', '233d61cfc704909ff3c455432a7ea7dbd3089851eb37df13cac5fbcb9bc8269b'), ('905310236646322176_0', 'data_image/hurricane_harvey/6_9_2017/905310236646322176_0.jpg', 'test', '7d5d162b45a34246b9084c3aec20a11cac00a08b060c9e460b9d30038b4acbba'), ('901743093493489664_0', 'data_image/hurricane_harvey/27_8_2017/901743093493489664_0.jpg', 'dev', 'b484353c968a673e8aab8bfdbbd99499e4807061e55b02f58b6b8d44fbe43d35'), ('904533693120659456_0', 'data_image/hurricane_harvey/4_9_2017/904533693120659456_0.jpg', 'train', '05272731e22a1f88792d7cf9d81489214a850b710c80f18025c0e8b1d2462823'), ('909177170576252928_0', 'data_image/hurricane_harvey/16_9_2017/909177170576252928_0.jpg', 'dev', '711ea4f0738395c00e64cae64f7dce723e51a76a2edb9073b29f5084994ba27a'), ('901697659613966336_0', 'data_image/hurricane_harvey/27_8_2017/901697659613966336_0.jpg', 'test', 'bc481f900bcca5bd8fa6e47b3a87a0c8db1e92e89a03463c8b9f2877ce8b0378'), ('901771298937982976_0', 'data_image/hurricane_harvey/27_8_2017/901771298937982976_0.jpg', 'dev', 'c46c79fd5d199d10d23ecb017a63763c72877c2a50e376c6f91cb83c1d03f134'), ('904291469409878016_0', 'data_image/hurricane_harvey/3_9_2017/904291469409878016_0.jpg', 'train', '2c6d1fb324723a7c0c678786f34f1da87ebd18d74005835f879c1c917c37cf5d'), ('904741376159567873_0', 'data_image/hurricane_harvey/4_9_2017/904741376159567873_0.jpg', 'dev', '8c46b11a21fb2ee802790ddb33fac220df4658040bc8ee8eba3585d93de663c6'), ('904221556938825729_3', 'data_image/hurricane_harvey/3_9_2017/904221556938825729_3.jpg', 'train', 'cacca173296cf8dfdf72f5d4d5772bd564637a2982796db28137ac85b5e42ecc'), ('904842215620583425_0', 'data_image/hurricane_harvey/4_9_2017/904842215620583425_0.jpg', 'train', 'd857ddfac42e7fe108eb45c5e6d3c2465a061dfdbf444f8252dd050596232ccc'), ('908510389981368321_0', 'data_image/hurricane_harvey/15_9_2017/908510389981368321_0.jpg', 'test', '6ffea52f388a973db875c10c65a9b7a965660667cfe4e75ca1c07c7f5ced4020'), ('905852739716075521_0', 'data_image/hurricane_harvey/7_9_2017/905852739716075521_0.jpg', 'test', '32d784d9754c35748a3f8e62554b6d1f04a2b14b5d359b1f92d0208dec8730bc'), ('906772360002523136_0', 'data_image/hurricane_harvey/10_9_2017/906772360002523136_0.jpg', 'dev', 'b9e1db18b0fdc444c55e587f612ddc9975dc00bab6eabbb1b5d9c0faaf819fb8'), ('905114566081867781_0', 'data_image/hurricane_harvey/5_9_2017/905114566081867781_0.jpg', 'train', 'b994d17364ce862f452a5fcef1cfc65917efda7a4b477469defb72969e21356e'), ('901813742933073921_1', 'data_image/hurricane_harvey/27_8_2017/901813742933073921_1.jpg', 'dev', '376937613744d34fdd1d03f8547f13bb520d2a391a78dae2bbe3c2fbd7a886d0'), ('906916099152891904_0', 'data_image/hurricane_harvey/10_9_2017/906916099152891904_0.jpg', 'test', '8eea4758f8828f80ce35f7898a03c8aea41bb12b70578d8ce1563280bd4acc4b'), ('907354855899123714_2', 'data_image/hurricane_harvey/11_9_2017/907354855899123714_2.jpg', 'train', '18c5d320f58601ae44388a47c6797a794d02938085ce5f7e9f8cc088d3917c57'), ('907153066864136193_0', 'data_image/hurricane_harvey/11_9_2017/907153066864136193_0.jpg', 'train', 'a57af33b6b2081bb6318aacdbe7b64b517b409d4e22cb542891cf37e39c06abd'), ('907737364750929920_0', 'data_image/hurricane_harvey/12_9_2017/907737364750929920_0.jpg', 'train', 'fb0e626574a5e8298f1ba0192cc63aaf67c80525b43f8fe673adb43e5f32c1e5'), ('906199050206068737_0', 'data_image/hurricane_harvey/8_9_2017/906199050206068737_0.jpg', 'test', 'c29c0d1f2b1ab7eff854704032980038c4c7ca7674a654ac6c4b4ff579568d08'), ('907672880535805952_0', 'data_image/hurricane_harvey/12_9_2017/907672880535805952_0.jpg', 'train', 'd2451c6445a781b50ce597029972c42af3de989c7677c37f0573c73a6fa6904f'), ('905781342201368576_0', 'data_image/hurricane_harvey/7_9_2017/905781342201368576_0.jpg', 'train', '51a825039206b1b0a6d95476849bafb833cd9b7da52fe941240e7a12e7abf700'), ('908723853798641664_0', 'data_image/hurricane_harvey/15_9_2017/908723853798641664_0.jpg', 'dev', '32413ac33f350c5f536c43afec9c0a18041f919fb100a2870a0fbf0c49d4998e'), ('904410044304908288_0', 'data_image/hurricane_harvey/3_9_2017/904410044304908288_0.jpg', 'train', '3c08d17f1e169939a3be8cb4823a144ecff09e26d73607c6b41aa26cf2125af9'), ('907756518535294976_1', 'data_image/hurricane_harvey/13_9_2017/907756518535294976_1.jpg', 'dev', 'f23f8ca993b52c2fc649b6475513d7d756821d4407c18c72612cc3135dd34239'), ('905445939573252097_0', 'data_image/hurricane_harvey/6_9_2017/905445939573252097_0.jpg', 'test', 'f0b5c41eb053500369a441aaf6a382057ff47eb5ad1917fb6745c01123df2dae'), ('907336771435982848_0', 'data_image/hurricane_harvey/11_9_2017/907336771435982848_0.jpg', 'dev', '9509498b35cce89fa4695f28679d11828db8e5134095a144ca6405aa32cbae37'), ('901697605851336704_1', 'data_image/hurricane_harvey/27_8_2017/901697605851336704_1.jpg', 'train', '05c37ab3581285117f416fff711796688347ff59ca8839b28958df26b8174e9a'), ('901704875926544384_0', 'data_image/hurricane_harvey/27_8_2017/901704875926544384_0.jpg', 'test', 'aa3a6828193d3ce392c6228db69eb3e915054d29e73e08763a9b335f13894b49'), ('907148756780433408_0', 'data_image/hurricane_harvey/11_9_2017/907148756780433408_0.jpg', 'train', '0ee2c1b24714dc7f67123fcb680f25110865fcfa786a487dde993fd24d52780d'), ('905143518569615361_0', 'data_image/hurricane_harvey/5_9_2017/905143518569615361_0.jpg', 'train', '6229ebad97a5f2d59b8e3070083ca86804f4de1b88e678ab637ae8f13e06d4fa'), ('910195055180234753_0', 'data_image/hurricane_irma/19_9_2017/910195055180234753_0.jpg', 'train', 'b7e6f1f59a4b985902f94433021224b45a8f828f7e85b04a019783ba041847dd'), ('910198450347810817_0', 'data_image/hurricane_irma/19_9_2017/910198450347810817_0.jpg', 'train', '49a6e0396628752e7bf22f42dca9a9256ea86130cc5c7d764251e3131ba5650a'), ('909423323229990914_0', 'data_image/hurricane_irma/17_9_2017/909423323229990914_0.jpg', 'train', '5afdb0b814278c573f2b13ff29eaefbf3e1c92dd7b77bb112779d7a86261b5e6'), ('909389092294660096_0', 'data_image/hurricane_irma/17_9_2017/909389092294660096_0.jpg', 'test', '9c89fd111223ddf6b37a48d50d81bcb54b48b21d8c7d9bc0486bddae595dd02d'), ('909413901665734656_0', 'data_image/hurricane_irma/17_9_2017/909413901665734656_0.jpg', 'dev', '177da135869a92e9aec391044e6eda4939b8ee01f2b95c14936759e70c071d65'), ('910260755131285504_0', 'data_image/hurricane_irma/19_9_2017/910260755131285504_0.jpg', 'train', 'd5ba1c45fc03d74b3c53b4d1d667a826ded00e06bb3df6d25adbc62bd73a3f35'), ('909781597569847296_0', 'data_image/hurricane_irma/18_9_2017/909781597569847296_0.jpg', 'dev', 'f70d64ddf6f1463d9f00dabc6ab6720768d519a84c96e95bb947f21c15c97546'), ('909942148665040897_0', 'data_image/hurricane_irma/19_9_2017/909942148665040897_0.jpg', 'train', '81a1c8f2000ce4134ce4e850ccad302c5feffdebcaee9e56aec2a393cc5b35fe'), ('910202655733112833_0', 'data_image/hurricane_irma/19_9_2017/910202655733112833_0.jpg', 'test', 'f2a5ba28b9dceeb49851b5e0e9079fe558f0c5e39ada8d25e9560945e31c00de'), ('909864629030944768_0', 'data_image/hurricane_irma/18_9_2017/909864629030944768_0.jpg', 'train', '1ba279d3f4881f04f812931c683ddc85be3056781217c5d3ed8de5f119ec9f82'), ('910179107568324609_0', 'data_image/hurricane_irma/19_9_2017/910179107568324609_0.jpg', 'test', '60e71e2fa58e2e3517707de033ca14678a936495798b693d72627e8dbd7893c0'), ('906927917862723586_0', 'data_image/hurricane_irma/10_9_2017/906927917862723586_0.jpg', 'train', '8c1e22870b030bf27e47c11f2228dde17c004fdaa82027e928fbd8b23209a150'), ('909428750084800513_0', 'data_image/hurricane_irma/17_9_2017/909428750084800513_0.jpg', 'train', 'e825cbb6046a608200633ccd8e28853ffb251222ca501a8e8b718781ceee7996'), ('909748121651089408_0', 'data_image/hurricane_irma/18_9_2017/909748121651089408_0.jpg', 'train', '415d98f43caa09b0f46ac60f5ac53e4637912afb74dc5b8275560a420fa4d91a'), ('909833497640095746_0', 'data_image/hurricane_irma/18_9_2017/909833497640095746_0.jpg', 'test', '9902128c6622c36e90bab5fb1dfbfdbf52e94f110bfa501acca8f62578a08749'), ('905700070972116992_0', 'data_image/hurricane_irma/7_9_2017/905700070972116992_0.jpg', 'dev', 'ab45c21bd65abe7cad15218bb6a2715bbb3fccf8f891562aabbe4cd4cdb4474c'), ('905702043058393088_0', 'data_image/hurricane_irma/7_9_2017/905702043058393088_0.jpg', 'test', '32b6b7872c056657072661521b8a763ac079dde03e6224f31fd734e4485381bd'), ('910057725186445313_0', 'data_image/hurricane_irma/19_9_2017/910057725186445313_0.jpg', 'train', 'd30fb6367a85fc980cc54bf734c3d10bb322929275bb37f704cb32c9c000eed9'), ('910151285793488897_0', 'data_image/hurricane_irma/19_9_2017/910151285793488897_0.jpg', 'train', 'd6a5c07398a1427db17ba4bd2983deaae309539618a0e8f4999ecdf5a8517093'), ('910177644444110848_1', 'data_image/hurricane_irma/19_9_2017/910177644444110848_1.jpg', 'test', '2e4622aa894ab3691a3363feffc50cdef65a88596efb4e2f56a03737a6ce70b1'), ('909494205390651397_0', 'data_image/hurricane_irma/17_9_2017/909494205390651397_0.jpg', 'test', '0b856c5a3255ec274325016aec5722a31d07765f881f3c97e6977b11263baf4a'), ('905636651447328768_0', 'data_image/hurricane_irma/7_9_2017/905636651447328768_0.jpg', 'test', 'e0e53a04246f0850152557964aaaa387bccc43e97f368f0069a52d5268e11c20'), ('909734318716616704_0', 'data_image/hurricane_irma/18_9_2017/909734318716616704_0.jpg', 'test', '89c537ef0782156491eae742c815f3a79d05eb0d73e080fb6dbaad709683abb8'), ('909394600007274501_0', 'data_image/hurricane_irma/17_9_2017/909394600007274501_0.jpg', 'train', 'f2cd13b5aa0fba3c810df753c5f53d863211356403ae1d4896ca8ce7e084e93a'), ('909443799079931904_0', 'data_image/hurricane_irma/17_9_2017/909443799079931904_0.jpg', 'dev', '2530442913727040b8562b8856b7000fa0f9c35c0176f83b6d978ebda0d1fbce'), ('909840043178643456_1', 'data_image/hurricane_irma/18_9_2017/909840043178643456_1.jpg', 'train', 'ef6478bca855807c201b2cee7d879ff6db8b723224348c6484fa0148328ac98c'), ('910028552585797632_0', 'data_image/hurricane_irma/19_9_2017/910028552585797632_0.jpg', 'dev', '656f98d7aad017f021f185f00e952197473048d1fa07e3b45b327497502ae4c4'), ('909890549318144005_0', 'data_image/hurricane_irma/18_9_2017/909890549318144005_0.jpg', 'test', '45ef97c2ba76bc2bb1530fbbc0341b1bef45b91183086d5faa5b19ea1d539a84'), ('909843612309102592_1', 'data_image/hurricane_irma/18_9_2017/909843612309102592_1.jpg', 'train', '5e812de4e9a67bbd4815b276ce8e0556b6f3b139dafe2e81466576cb6a0c98a6'), ('910258479784431619_0', 'data_image/hurricane_irma/19_9_2017/910258479784431619_0.jpg', 'dev', 'a28e51231bd5d2ed16f607dda844a8f1eec8e59aaac44a3bd518d932d49d2ade'), ('910147218841014272_0', 'data_image/hurricane_irma/19_9_2017/910147218841014272_0.jpg', 'train', '683bd01a778aa90c4fb6c3f9a0ffad34827c80b9b01c44475c8dac12e79d691b'), ('909844074655514625_0', 'data_image/hurricane_irma/18_9_2017/909844074655514625_0.jpg', 'test', 'ee17573cba31793d2d84cd904bc77796a43a18d8b43039a3d0640eb9ab3a0df1'), ('909839832066686978_1', 'data_image/hurricane_irma/18_9_2017/909839832066686978_1.jpg', 'train', 'ec8f278f3f36feec1bcc6f18af4f40fcfd37ff9bac8025914d81ef2f89ba8730'), ('910187552266743808_0', 'data_image/hurricane_irma/19_9_2017/910187552266743808_0.jpg', 'train', '90b816a3bf269da164ed092b23a033055158813cc0b4d47ecd15e64c87c41c15'), ('909746257970761728_0', 'data_image/hurricane_irma/18_9_2017/909746257970761728_0.jpg', 'test', 'a16f7bb06258cddbe4eeb8979f460393f9e62fc5aadec63e1f775e1e2225ec71'), ('910042181741793281_1', 'data_image/hurricane_irma/19_9_2017/910042181741793281_1.jpg', 'train', '2bbb0a6b2b6c46186fe53bee48fc669535afa507f080570ef0bf0b401d88a70c'), ('909949382329602048_0', 'data_image/hurricane_irma/19_9_2017/909949382329602048_0.jpg', 'train', '03b80b11a206db57595dab051e72b2d276b2e46325ff286fe76a75d3c6ed74a7'), ('909409487030771712_0', 'data_image/hurricane_irma/17_9_2017/909409487030771712_0.jpg', 'test', '9460fb272af8dc86be930840195ecc8ab439dc6c2a823dfd2e0bb9e1d042a92a'), ('909740728909205504_0', 'data_image/hurricane_irma/18_9_2017/909740728909205504_0.jpg', 'train', '353b8761071654e3497c0c331c6ed8aa93b5daae2d46ab002daad3daa845c7c5'), ('909403767736619008_0', 'data_image/hurricane_irma/17_9_2017/909403767736619008_0.jpg', 'train', '3a4d35d0cd5a0eb10015c40e75f81ae779e64c57aff4446582b7d56b352e4ad4'), ('909891281341505536_0', 'data_image/hurricane_irma/18_9_2017/909891281341505536_0.jpg', 'dev', 'a3cb22f9eea64421b6f69e9b5bec466f5409ce530335fc1bef598c0e51a7121a'), ('911567186719801344_0', 'data_image/hurricane_maria/23_9_2017/911567186719801344_0.jpg', 'train', 'd180485f23361454583f835429147fe7b56d6acd20f804734238ca6fc8cce190'), ('912393267387478016_0', 'data_image/hurricane_maria/25_9_2017/912393267387478016_0.jpg', 'train', '9446a40c8f14b62080ba9b3a635ed24b1ad8d2d9b9a9b4fdda838fad0dfc31b4'), ('922560865144012803_0', 'data_image/hurricane_maria/23_10_2017/922560865144012803_0.jpg', 'test', '2967be61ab15ba793ccd0c0d2b111ff4b5bcf07e1db01121dab904a989c67a80'), ('923222945761316864_0', 'data_image/hurricane_maria/25_10_2017/923222945761316864_0.jpg', 'train', '6d8cdf6e61c1a08e0ab3d3f4b8d9849bfea4f4d7b03f7c5fda372e78a48e5717'), ('922812556045144064_0', 'data_image/hurricane_maria/24_10_2017/922812556045144064_0.jpg', 'test', 'a3e26e2a61d3992d25295aff91beeb6b0e3ffa8cfc6a9f36935ee333e9e7b92a'), ('928651782946947072_0', 'data_image/hurricane_maria/9_11_2017/928651782946947072_0.jpg', 'dev', '5aea8167fcd65fc79c56a1357ab1953da4c4c110d8d132784d10dd64838f275d'), ('923262755246878720_0', 'data_image/hurricane_maria/25_10_2017/923262755246878720_0.jpg', 'train', '850fc7b2f966b1df1995b1ca7450db68cdebdfe62ff9bea7837f491feefca2e2'), ('911907090112798720_0', 'data_image/hurricane_maria/24_9_2017/911907090112798720_0.jpg', 'test', '00786c9cc5bff70992ddfbc0f245e0f256b7e68db3735b517d6a23e03da738ab'), ('913418300893392896_0', 'data_image/hurricane_maria/28_9_2017/913418300893392896_0.jpg', 'test', 'f929c03a6379b2ec725edf2fb39c3c1b08a138ed2dfe0dde6f82a55653ba082c'), ('922364261153853446_0', 'data_image/hurricane_maria/23_10_2017/922364261153853446_0.jpg', 'test', 'dbd9fa4f8ad20e7a1dfb24302e7024a9c102d430bd09bdc93e3460532186725e'), ('923577268965912577_0', 'data_image/hurricane_maria/26_10_2017/923577268965912577_0.jpg', 'train', '8d90715cbd2b37829bf7b86bf12f1c4006df5e27c521e9f9d79cbe1e03b59531'), ('913934423573368832_0', 'data_image/hurricane_maria/30_9_2017/913934423573368832_0.jpg', 'train', '6f69bfe7db710b393a35601b54cc2ed4771ec585d1938105df63c7d92e04df35'), ('914244636142600197_0', 'data_image/hurricane_maria/30_9_2017/914244636142600197_0.jpg', 'train', '7691321da29b6df56f026f8d71c3a6f7217c391f52d19c974336719f82283ba3'), ('911892515321733120_0', 'data_image/hurricane_maria/24_9_2017/911892515321733120_0.jpg', 'train', '2d9b3ca1ad4be01c113808425d5e2f155a7be6392b4960099aadc119720b12a0'), ('912952239471382529_0', 'data_image/hurricane_maria/27_9_2017/912952239471382529_0.jpg', 'dev', '72c04da25e502014286ab5ba17261ccd8d8c9c87fa1af1930ea6bebf2428a527'), ('917782042473041920_0', 'data_image/hurricane_maria/10_10_2017/917782042473041920_0.jpg', 'train', '1bccfc6c1b9281d5e7c82a28aab03924b99bc529881c4346925d973e7e539cca'), ('910723784474349569_1', 'data_image/hurricane_maria/21_9_2017/910723784474349569_1.jpg', 'train', 'b74b321ea56fcbd731c9424dd8eda5eb464576de0bbd75ad6cff6e5f42e63425'), ('914873754039099392_0', 'data_image/hurricane_maria/2_10_2017/914873754039099392_0.jpg', 'test', 'accd427cd075e3ad844b67e24b7cdaac89dd3839301f7a8702e9d38fe49a2a88'), ('921542832334262274_0', 'data_image/hurricane_maria/21_10_2017/921542832334262274_0.jpg', 'train', '8f477629859c72c295eed72954e1143855bf63d221a5686962ee8a033b212922'), ('916211513601511424_0', 'data_image/hurricane_maria/6_10_2017/916211513601511424_0.jpg', 'dev', 'beb68eefdfa9103478712b14a6b0f0742b3d9ab187286b9d47fd12d4d11be992'), ('922857782453338112_0', 'data_image/hurricane_maria/24_10_2017/922857782453338112_0.jpg', 'dev', '04ab3495f8400ffd25f4d16658a5383ba94f21f26ad202de5d2608ab25c07041'), ('916310521661591552_0', 'data_image/hurricane_maria/6_10_2017/916310521661591552_0.jpg', 'train', 'd6a6334f844f8ed0b8144b72be6b9cb63e9d9b57ce2fc9bfef53302aca8911ac'), ('928128643733139456_0', 'data_image/hurricane_maria/8_11_2017/928128643733139456_0.jpg', 'test', '9914b1036bdd2ebf7035f6f19fbb5ca8efaba314befc4c9e06598e518172c8ff'), ('922501358640615424_0', 'data_image/hurricane_maria/23_10_2017/922501358640615424_0.jpg', 'test', 'f775ded4a98008aa47ee1188f4773b1c78b3ecd1880d023864e9c5f9fb307250'), ('913560802459873280_0', 'data_image/hurricane_maria/29_9_2017/913560802459873280_0.jpg', 'train', '5a65e6de97cc443da50709244c529b085c0f8eaed393759fe22b0d1f8a97e4a4'), ('914683563169259520_0', 'data_image/hurricane_maria/2_10_2017/914683563169259520_0.jpg', 'train', '18b92ae4dde529bf13d4613aa11bd2f20086c482eaecb736515b842b6083a6fd'), ('913853461107810305_0', 'data_image/hurricane_maria/29_9_2017/913853461107810305_0.jpg', 'train', '6f9bd5bf803b60f4bd8930df572ccd96ee8b98bdfe4b2fbf488de4aed204b23e'), ('923383471183052800_0', 'data_image/hurricane_maria/26_10_2017/923383471183052800_0.jpg', 'test', '230637b57fc73cb0e32f73c501c4789330ffaa4d04900f6c9395a0557dd1b4f3'), ('914871738722390016_1', 'data_image/hurricane_maria/2_10_2017/914871738722390016_1.jpg', 'train', '94cce41a3d953b2ff0d6ab31dd4cc8d2e8b40a75e7d7f50f158f5f95401ca340'), ('910534114922647553_0', 'data_image/hurricane_maria/20_9_2017/910534114922647553_0.jpg', 'test', '2f85d5608547fb54935d61753537c3c293c0a1af528250264c0c1c878e108b86'), ('914135376138092544_0', 'data_image/hurricane_maria/30_9_2017/914135376138092544_0.jpg', 'dev', '7ae9d5d26d409724b66ca38ab0c5823c572338aefd884ce5bfdb62011339ad0b'), ('914527783543549952_0', 'data_image/hurricane_maria/1_10_2017/914527783543549952_0.jpg', 'train', '6755ee520a78483a911b12757ff7f7c371ef310161e3b379de04104483f94cf0'), ('923612046054297601_0', 'data_image/hurricane_maria/26_10_2017/923612046054297601_0.jpg', 'dev', 'e04e85b31a4e592e23a34cb8e8ce6bf77461f0b1d7732eb9145f4bd570f1067a'), ('914555402737668097_1', 'data_image/hurricane_maria/1_10_2017/914555402737668097_1.jpg', 'dev', '4b4eb84be53feecb18b2c9e57491b9c1acfbaac2262e20d0f54d91694c7b4d26'), ('912417707605114880_0', 'data_image/hurricane_maria/25_9_2017/912417707605114880_0.jpg', 'dev', '34236de09e7c789223cff63ffa007df0b56fa9f90bbd8bccfd8ca3e8828eb1d8'), ('922492911337517056_0', 'data_image/hurricane_maria/23_10_2017/922492911337517056_0.jpg', 'train', '996f5ed086c478582902b4ecb249adbb8cf505b490e2d574f97b285ea7b2ed28'), ('928673310052962304_0', 'data_image/hurricane_maria/9_11_2017/928673310052962304_0.jpg', 'test', '6c73970653011423d47226782921fbd046134c63342faec19d20c274b3c7cf04'), ('928454474355658752_0', 'data_image/hurricane_maria/9_11_2017/928454474355658752_0.jpg', 'dev', '4a2827d2c25e668dc9321c241755a57c8532c0d06608613b1368f4476010570d'), ('912944540226609152_1', 'data_image/hurricane_maria/27_9_2017/912944540226609152_1.jpg', 'dev', '53db7ee9b6fd5217bb9cb89ce23c9bffc2636c03d7f673666c67dbff26941e8a'), ('921338135375745024_0', 'data_image/hurricane_maria/20_10_2017/921338135375745024_0.jpg', 'test', '0a48327743110fc83503d32f9bb089c5a643b9d91a51054720c9a39a639da528'), ('930762101836124161_0', 'data_image/iraq_iran_earthquake/15_11_2017/930762101836124161_0.jpg', 'test', 'c7df94fd423d0a1655c0c90ba53424a73965e5467bb9c4df1618fc58eb3a17e2'), ('930308604200026112_0', 'data_image/iraq_iran_earthquake/14_11_2017/930308604200026112_0.jpg', 'test', '9d449a7238cd61e09b28fbe94d25ac9ef90c6686fa0211259c3495f20b177f30'), ('930122324510105600_0', 'data_image/iraq_iran_earthquake/13_11_2017/930122324510105600_0.jpg', 'train', '78cdb3d11eba2fa4c416955301f4d8888cafdfa4b44d2fdea2702c58a41fae2e'), ('930007227506704385_0', 'data_image/iraq_iran_earthquake/13_11_2017/930007227506704385_0.jpg', 'train', 'bac4deee224f9b607488b84447d058fbfee5ebeec7c236ba284a90d4da7067a3'), ('930046640827777027_0', 'data_image/iraq_iran_earthquake/13_11_2017/930046640827777027_0.jpg', 'train', '7a13f7d85f35d58c08046f40d7324318e3c74568f8d4697a832ee48f2dfceeeb'), ('930347490263666688_1', 'data_image/iraq_iran_earthquake/14_11_2017/930347490263666688_1.jpg', 'train', '740ef09025298930ff310d58b4ebca84733234329946cd6dc2e2cf9cf2a4bb33'), ('931603001499881472_0', 'data_image/iraq_iran_earthquake/17_11_2017/931603001499881472_0.jpg', 'train', '472c9c6be6ad14b569ced9fa2fe553c2de0715004cc7c5ad493580258484c247'), ('930864228411457536_0', 'data_image/iraq_iran_earthquake/15_11_2017/930864228411457536_0.jpg', 'test', 'be19000eb32e2ff874a6876056ada66ee113bedfced2696559f02a1c8444298c'), ('930484110358974464_0', 'data_image/iraq_iran_earthquake/14_11_2017/930484110358974464_0.jpg', 'test', '8e42e606b72be1b3b7d7f6108330bfae2182ad967c387457dd73fe5625c83463'), ('930437861287055362_0', 'data_image/iraq_iran_earthquake/14_11_2017/930437861287055362_0.jpg', 'train', 'a71ad2394ad1b41d0b049e923b47b44353ff561addbf623bde5fac5dbc2a3a15'), ('930181623043239938_0', 'data_image/iraq_iran_earthquake/13_11_2017/930181623043239938_0.jpg', 'train', 'ef0cc384951dcf7d471683d5f30d4ccb5bf6f024701f28d6b6ef9d8e9343c209'), ('930025819975618560_0', 'data_image/iraq_iran_earthquake/13_11_2017/930025819975618560_0.jpg', 'train', '1b9563bf6e02d6a4dc8f6f8d3366344364881abcbd9d0d3daf04e5e7abb2427d'), ('930559522397196290_0', 'data_image/iraq_iran_earthquake/14_11_2017/930559522397196290_0.jpg', 'test', '8e42e606b72be1b3b7d7f6108330bfae2182ad967c387457dd73fe5625c83463'), ('931493766816559104_0', 'data_image/iraq_iran_earthquake/17_11_2017/931493766816559104_0.jpg', 'train', '1f5fea4af348ee9458395aecb56489c272e1b84a44838dd0fdff23aa902dded3'), ('931263251664404480_0', 'data_image/iraq_iran_earthquake/16_11_2017/931263251664404480_0.jpg', 'train', 'e4073ee6eb29d347debf657fadd9a06935ab4b28f36d342aead931ae6017bbde'), ('930001778115985408_0', 'data_image/iraq_iran_earthquake/13_11_2017/930001778115985408_0.jpg', 'test', 'a21d01cb2182ad5fa5641f4d274f8b981c30ea64259398a246d527cb1e327fdc'), ('930191344609394688_0', 'data_image/iraq_iran_earthquake/13_11_2017/930191344609394688_0.jpg', 'test', 'e15b96079a6ca55b0f682864bef8c58d50a9f7b922b68af3bde7f89488fdcae5'), ('930391498998378496_0', 'data_image/iraq_iran_earthquake/14_11_2017/930391498998378496_0.jpg', 'train', '08928c2aa6c499a73335a7d5ae43d26833869d3df17d6a5834712bd6ced79e0e'), ('930039379996938242_2', 'data_image/iraq_iran_earthquake/13_11_2017/930039379996938242_2.jpg', 'train', '303650b62a8ea651ed1578a7f38d675541203fd938a5d81ffe943d7b0580dd86'), ('930457005550112770_0', 'data_image/iraq_iran_earthquake/14_11_2017/930457005550112770_0.jpg', 'test', '0d7567d4ddc5f7a0411c5a39460505c9b9ba92d8f1ac0ca46786b5f73dd59045'), ('930380460726865921_1', 'data_image/iraq_iran_earthquake/14_11_2017/930380460726865921_1.jpg', 'train', '5394b1f09c98d0587d9487af1d335ca3a7a22247609ba5525892da02fc54a62b'), ('930843480657600512_0', 'data_image/iraq_iran_earthquake/15_11_2017/930843480657600512_0.jpg', 'train', '118217781c2b3b307d685b42d38ba93b55516dd972eaec321241511d442cbed5'), ('930001999155945472_0', 'data_image/iraq_iran_earthquake/13_11_2017/930001999155945472_0.jpg', 'train', '661612ba2e309664e45ed78b6e12b20e11b1201ecc4bec1f001b8b543144220e'), ('910658635851644928_0', 'data_image/mexico_earthquake/21_9_2017/910658635851644928_0.jpg', 'dev', '93cc967ebbca284c12898046e7bbad71611da37c0021121df4650185b5c17e65'), ('911636927383666688_0', 'data_image/mexico_earthquake/23_9_2017/911636927383666688_0.jpg', 'train', '52b5ced40ed283e80eb896e684425a0057961d8f545f940193f8e9fd90f1ed64'), ('910716949843337216_1', 'data_image/mexico_earthquake/21_9_2017/910716949843337216_1.jpg', 'train', 'baa1591dc00d49d69eebc73ffb4fbeae65f656df46fb688f807d122be04924b2'), ('912301115257053184_2', 'data_image/mexico_earthquake/25_9_2017/912301115257053184_2.jpg', 'dev', '529e8a3109fa1b8dc2a779475c7883f358e7441c6829fb784f73fe1d4306a4c4'), ('910543087566454785_0', 'data_image/mexico_earthquake/20_9_2017/910543087566454785_0.jpg', 'train', '8d655cd7d86ffd67a28374dac823a93067dc5486b435273dd17c24e5b95766a8'), ('911773444978675712_0', 'data_image/mexico_earthquake/24_9_2017/911773444978675712_0.jpg', 'train', '04b55770938a62d7764439b99697fc027e2d3f4040a812f26fc6b368265fa5b0'), ('910728261457502209_0', 'data_image/mexico_earthquake/21_9_2017/910728261457502209_0.jpg', 'train', 'aa3b1fff9b881a9ce62442f3063ae5dbd718b618dc58a7f9e84678ef0568b362'), ('910559238795051008_0', 'data_image/mexico_earthquake/20_9_2017/910559238795051008_0.jpg', 'train', 'b61f2b8758353298fd4eaa9cbadd75fb0bef4f5345b5c92282b941ea50a39a59'), ('912301115257053184_0', 'data_image/mexico_earthquake/25_9_2017/912301115257053184_0.jpg', 'dev', 'aafdf525ed2cc659e611abf52944e881a0f6984d12db47ef95ed928bd041c5a0'), ('913102526383144961_2', 'data_image/mexico_earthquake/27_9_2017/913102526383144961_2.jpg', 'test', 'd90a37171897631e5a8d722fa2a42322fc22a649f523582345b16673e53a000e'), ('910541745007427587_0', 'data_image/mexico_earthquake/20_9_2017/910541745007427587_0.jpg', 'test', '91440e189ce5127385a064a33987bb6f2ceb0673268a7964f979a6f09ace2182'), ('910678493419200514_0', 'data_image/mexico_earthquake/21_9_2017/910678493419200514_0.jpg', 'train', '6982526d0f6c1a11dbe95f4f1ece4e68fea4b861a50454d408e81a380e470c96'), ('912111866595471360_2', 'data_image/mexico_earthquake/25_9_2017/912111866595471360_2.jpg', 'dev', '5df2f731a358b07af86cfed5ff6d04c747027354f77961702a5288e102ec4193'), ('911908841675264002_0', 'data_image/mexico_earthquake/24_9_2017/911908841675264002_0.jpg', 'test', '6186904ba4c3e92e02411feef49e3af2bbb14911474e39fd05b172815704e545'), ('911688552936726529_2', 'data_image/mexico_earthquake/23_9_2017/911688552936726529_2.jpg', 'train', 'a58339ccb609ed64e77d3e90cc0aa80948a3b8ace4482912ce8c5c5accc7170e'), ('911844414573735936_0', 'data_image/mexico_earthquake/24_9_2017/911844414573735936_0.jpg', 'dev', 'b2fa3412b92d5da28099bbf251418c5d1d0ef992a02365ad4da785423aa11dd9'), ('910533037212893185_0', 'data_image/mexico_earthquake/20_9_2017/910533037212893185_0.jpg', 'train', '4f6e5f34deae01df2f1320607793e9f08aa4e08518fa528933d80d09a437d001'), ('910614931262681089_0', 'data_image/mexico_earthquake/20_9_2017/910614931262681089_0.jpg', 'test', '0bd1c628dc748f1f469a83b2cfa15c6f1f90855a6d3f691a33c677e24da09e74'), ('910698946514771969_0', 'data_image/mexico_earthquake/21_9_2017/910698946514771969_0.jpg', 'train', '6e1dc86664df0f89cd13f46a055e61239801031d15500ae5e42861ca23257de8'), ('911621930926579712_0', 'data_image/mexico_earthquake/23_9_2017/911621930926579712_0.jpg', 'train', '6a0cf50b1e9dbac9b179fabf4146373bff9bffd484ad9917961e9040686ca326'), ('911625123417018368_0', 'data_image/mexico_earthquake/23_9_2017/911625123417018368_0.png', 'test', '5018e2dee2ea4b90cb412cd77de8d18424028e6d30311c9318b3bfa63a2d7e6f'), ('910702443977089029_0', 'data_image/mexico_earthquake/21_9_2017/910702443977089029_0.jpg', 'dev', '6e1dc86664df0f89cd13f46a055e61239801031d15500ae5e42861ca23257de8'), ('910708899098980352_0', 'data_image/mexico_earthquake/21_9_2017/910708899098980352_0.jpg', 'dev', 'fd9d3d5deba8fc123889b7459133b43453986239904d78c10e62d2f365bb5892'), ('910713914182324224_0', 'data_image/mexico_earthquake/21_9_2017/910713914182324224_0.jpg', 'dev', 'c7aec5b15167904cf4a937055baa25c84e301c0db37f183739a7b0de5f8449e6'), ('910585470127501312_0', 'data_image/mexico_earthquake/20_9_2017/910585470127501312_0.jpg', 'train', '61736ac59b334a88fb0da66e33f21fb6a532e42c8491bd3bd9889c3c50ff6021'), ('910523546442575873_0', 'data_image/mexico_earthquake/20_9_2017/910523546442575873_0.jpg', 'train', '4c4b7b433f4b9ebb6dcb2252013d3ae8cd9756683fd66df53c9f2c9de1817c14'), ('911803150503362561_0', 'data_image/mexico_earthquake/24_9_2017/911803150503362561_0.jpg', 'train', 'bc2de5a0e21720bac014967dd7a1baa6b8ef2b6c2ebc6165f5085f8a5206e3a8'), ('870029636750630912_0', 'data_image/srilanka_floods/31_5_2017/870029636750630912_0.jpg', 'train', 'd1033f5aa255c4aab1dc0aba642ff0e23ba0e48e2ef256cd291e29bc3e646940'), ('879733127677136897_1', 'data_image/srilanka_floods/27_6_2017/879733127677136897_1.jpg', 'train', 'e6cabd4caefdb516497ce50661323b1311aec9009aa2afc3fd2cca08a929cc72'), ('870548008256978944_0', 'data_image/srilanka_floods/2_6_2017/870548008256978944_0.jpg', 'test', '154a88d46606829a147d7df60626c1b9f5e3ca45b29d483dced51bf0572343f9'), ('870130736069656577_0', 'data_image/srilanka_floods/1_6_2017/870130736069656577_0.jpg', 'test', 'dbcf577e88478e53b23b500bdb2a7ecf7f756a27b636ec433ee347c510ca7df0'), ('870552370744315905_0', 'data_image/srilanka_floods/2_6_2017/870552370744315905_0.jpg', 'test', 'a4f03a062b27d181d6f168b102868308cd1930ce5ac634f9353ebd955071030b'), ('870230068156448769_0', 'data_image/srilanka_floods/1_6_2017/870230068156448769_0.jpg', 'train', '8ee3261f97319021ac1d9c51dcfc9f8959cdbb70ffeddb48b130c9f65c8fb07e'), ('870668890564567040_3', 'data_image/srilanka_floods/2_6_2017/870668890564567040_3.jpg', 'train', 'f644b4fe9fa87d17a43483b646af0afe2bf4ac9a0458f0b7eeea84d189405b98'), ('870357094737817600_0', 'data_image/srilanka_floods/1_6_2017/870357094737817600_0.jpg', 'test', '068034e1317b114f3d8cff24e81406f2f01b04a97e3ea2524401a81078d91eb2'), ('870052558928138240_2', 'data_image/srilanka_floods/31_5_2017/870052558928138240_2.jpg', 'train', '97e708e4d07f44b553d01f90a1d357c2c1161eea2cb4384b35219062010072cd'), ('870182877706715139_0', 'data_image/srilanka_floods/1_6_2017/870182877706715139_0.jpg', 'test', '368065faa3e2d5247851af1bc4b56247a794daaf62346b4ba30e52b85c4a9e27'), ('870182877706715139_1', 'data_image/srilanka_floods/1_6_2017/870182877706715139_1.jpg', 'test', '514f3fa55b0a74eda79b8f4707c73f5b6034b4921a309e8e07a0eb0505eecd20'), ('870678906940129280_0', 'data_image/srilanka_floods/2_6_2017/870678906940129280_0.jpg', 'dev', 'b7e1bab87b7e8afa3845acf6465d526d13d63b7d0f53f40399b79c57c82fde80'), ('877812373251178496_0', 'data_image/srilanka_floods/22_6_2017/877812373251178496_0.jpg', 'train', '3e1cf02e99b6f967a3a424042b3d3ee8bb6c7483e41ff5184f1d036e850d5397'), ('876592481214881795_0', 'data_image/srilanka_floods/19_6_2017/876592481214881795_0.jpg', 'test', '0facb4720a4d4206f845550f6fed2f497f03bcc53d4568138a5d356937efec99'), ('875116643453632512_2', 'data_image/srilanka_floods/14_6_2017/875116643453632512_2.jpg', 'dev', '0f3a559e0a2796477eda7bfc8961913a9832897fb0fb2e5bcd1523e976f3f75e'), ('869982155270750208_0', 'data_image/srilanka_floods/31_5_2017/869982155270750208_0.jpg', 'test', '22ead70a83f30c74833768d06ffc0e11164be957a554199d7051e0d490471c6e'), ('870635993661091840_0', 'data_image/srilanka_floods/2_6_2017/870635993661091840_0.jpg', 'train', '41c59a9755978502dffd02197fd1fd0c1574073df197379ea0a4df4d5cdb425d'), ('873426944896884737_0', 'data_image/srilanka_floods/10_6_2017/873426944896884737_0.jpg', 'train', 'f984c3ed9eb43f20c62e618b84a712df9897b118be69445daee5cefff9fbaf58'), ('874374138021912576_1', 'data_image/srilanka_floods/12_6_2017/874374138021912576_1.jpg', 'train', '29007a1b3138ee09df3127eb4114149ef9724e940c09fd19208fc8565d0590a2'), ('870176000109182977_1', 'data_image/srilanka_floods/1_6_2017/870176000109182977_1.jpg', 'train', '683d144d18854c722f2acc1b9cfbbc30f9b59c10a37f143e207c5890d6dacf5d'), ('870570338723086337_0', 'data_image/srilanka_floods/2_6_2017/870570338723086337_0.jpg', 'dev', '1470bacf133f38984e56ac3089b4d16b2c4950a7a83dc4aa5d80377d81a391db'), ('870106961584508929_0', 'data_image/srilanka_floods/1_6_2017/870106961584508929_0.jpg', 'train', '54c8124aeb2654e95d87ac8c703fd2e153b8d60d23d22ab5a44ba9bb03149d5e'), ('878326464734609408_0', 'data_image/srilanka_floods/23_6_2017/878326464734609408_0.jpg', 'train', '122514c00dd817c6014bf0a2436f7368350f0f7ee62884e35bd95ba17bf30742'), ('870884328258940928_0', 'data_image/srilanka_floods/3_6_2017/870884328258940928_0.jpg', 'dev', 'a6e80fbdf828dfa16ab9b2555968ce59263862923878a61c3e44ab1b51cf914d'), ('870539388001722368_0', 'data_image/srilanka_floods/2_6_2017/870539388001722368_0.jpg', 'dev', '39465ff14f21fb3cd9599f3739a0ef2347aee5df908c5effff2a050df2c8ede9'), ('870977227923406848_0', 'data_image/srilanka_floods/3_6_2017/870977227923406848_0.jpg', 'test', '662830ec2ad5c2e94fd00fd61e63be30cf984c10a268816e004b049c8d893866'), ('870990463238406144_1', 'data_image/srilanka_floods/3_6_2017/870990463238406144_1.jpg', 'train', '23c6d420b593eee42ad0cbd7490fb35147b7b22d95f72cbd6444b85eb9795532'), ('870559758008418304_0', 'data_image/srilanka_floods/2_6_2017/870559758008418304_0.jpg', 'train', '4b3b5f6d7d8e29e0fe9eef1937d8270fb52ea1818669206881f9abb7ce25b194'), ('870559283880112128_2', 'data_image/srilanka_floods/2_6_2017/870559283880112128_2.jpg', 'test', 'f0658048b49849d44935b136bff6fb63054b58b6a5cd416ecdfc1b0551ebc698'), ('917956700149227520_0', 'data_image/california_wildfires/11_10_2017/917956700149227520_0.jpg', 'test', '2eb70ded82fa7342ceac160204d800ebf17f759522e17d347d9b8253505c70ac'), ('920853531942297602_0', 'data_image/california_wildfires/19_10_2017/920853531942297602_0.jpg', 'train', 'ec9064c633e1ec64f29a9b69a010e8cde177163e3a4db736b7f2a63b932e3f80'), ('917960728610594817_0', 'data_image/california_wildfires/11_10_2017/917960728610594817_0.jpg', 'train', '993d77f82ae8eba247eab21410c3fcc56c9aef23e7a446d61d5f707ebb2a13c6'), ('917965776384077825_0', 'data_image/california_wildfires/11_10_2017/917965776384077825_0.jpg', 'train', '020251d059d3af0ff989117c57d29823d52e7c21c78f9686c621fc366e3f3db3'), ('920202485926637568_0', 'data_image/california_wildfires/17_10_2017/920202485926637568_0.jpg', 'dev', 'dc3f0bfbef9290c11b19d81cb26bbbdd2042154c7d0b7273f5468bc4a62e29d2'), ('920550861343948800_0', 'data_image/california_wildfires/18_10_2017/920550861343948800_0.jpg', 'test', '803020044ceffa63cf048ce355c750ebe667224e5ca22d88a875456341c8d21f'), ('917828952533716994_0', 'data_image/california_wildfires/10_10_2017/917828952533716994_0.jpg', 'test', '21d496ffe364ad89a87409efa52bf0486a3d6abee03ff1878f5c81695df5fe70'), ('917831909111222272_0', 'data_image/california_wildfires/10_10_2017/917831909111222272_0.jpg', 'test', 'b78218574e6a2c7f0378cf9bc77f5ee4364a11fe9ba38c2108a2d1678f0461cf'), ('918279224946982913_0', 'data_image/california_wildfires/12_10_2017/918279224946982913_0.jpg', 'test', '3a3ef8aa20c5e08bbb38d76398ca9b6ad0467891906852770e892d6406533d35'), ('923232141856604161_0', 'data_image/california_wildfires/25_10_2017/923232141856604161_0.png', 'dev', '3e2b29d160758ae9b3d8aadc95c58043ef2e4e23bdb8329decada67a4fa18121'), ('917993642052280320_0', 'data_image/california_wildfires/11_10_2017/917993642052280320_0.jpg', 'test', 'bd465f6ac7d4c31e0cae849d3ba39f95110e1f69854a24f966f7790f1ebeaa20'), ('920419311843532801_0', 'data_image/california_wildfires/17_10_2017/920419311843532801_0.jpg', 'test', '79206a6fd2211a38d0e6c7b3845212578d09164fe7c89c744190b541f8ef4302'), ('917951614589263872_0', 'data_image/california_wildfires/11_10_2017/917951614589263872_0.jpg', 'train', '6488aaabb1dc5457cb87cea51e061a8c7f85253a980aefe4d8b35fdfe25ce2d6'), ('917794580728295424_0', 'data_image/california_wildfires/10_10_2017/917794580728295424_0.jpg', 'dev', '0048a254d17c94f349fd7a2ab493868adf3e2e4c20d302270472427a6371bb4e'), ('918394712842735616_0', 'data_image/california_wildfires/12_10_2017/918394712842735616_0.jpg', 'dev', '1de317214e84aeb49d1b6d5506edca83f213cf6f919fb8b9f1c0dcb5f5695e64'), ('920569455759056896_0', 'data_image/california_wildfires/18_10_2017/920569455759056896_0.jpg', 'test', '88adbacc25183589ca15d89ce0a7a33333f69bbd64b2c2f7856e94a8d9ed4016'), ('918021142815952897_0', 'data_image/california_wildfires/11_10_2017/918021142815952897_0.jpg', 'train', '8d16d5b19d1f125716ae2f2e2f012f5a9720c1b49e38ca799385783eb124e54a'), ('922569327034769408_0', 'data_image/california_wildfires/23_10_2017/922569327034769408_0.jpg', 'train', '7f98c9e48be76e2d5041c8400f7927ac413031f796d225d066b1bbe776e8883b'), ('922490387683577856_0', 'data_image/california_wildfires/23_10_2017/922490387683577856_0.jpg', 'train', 'da4581dc23800cbbb8aeae3cc18be577964406d4de8cd54ae4046374249cdefc'), ('917852446898102272_0', 'data_image/california_wildfires/10_10_2017/917852446898102272_0.jpg', 'test', 'a96b6877294e91991077fc50658afc50138203557eb5d9141873a0f720ce7045'), ('918108612869754883_0', 'data_image/california_wildfires/11_10_2017/918108612869754883_0.jpg', 'train', 'c483f928270e463d9f0112da698079205475d142341dc012ca0245ebd5f7213a'), ('922184601069608961_0', 'data_image/california_wildfires/22_10_2017/922184601069608961_0.jpg', 'train', 'd016adbf073907944474114da128c89232b452653de3d40b655f7257da5cf8ab'), ('917796307191951360_0', 'data_image/california_wildfires/10_10_2017/917796307191951360_0.jpg', 'dev', '59a4d074d51bf0d2dd3511e3e8357468205393fc3df72cda6271208b25614e89'), ('917957391957848066_0', 'data_image/california_wildfires/11_10_2017/917957391957848066_0.jpg', 'train', 'd04198c8ca723c7490156f14f659a570976b81f8a84fd304ce2dd704d30b2ec4'), ('919601346025627648_0', 'data_image/california_wildfires/15_10_2017/919601346025627648_0.jpg', 'test', 'a47e95faef0d5a2b75bc354829829dabd925f61402abbddaf4f119a0c8c7b2ac'), ('919962898054746112_0', 'data_image/california_wildfires/16_10_2017/919962898054746112_0.jpg', 'train', '6ac43b0da9b754ec65886ef77adff3b59a6c751ae766e2754db11b97a8f74c08'), ('917800657796595715_0', 'data_image/california_wildfires/10_10_2017/917800657796595715_0.jpg', 'train', '78224878a9a99bed960a52ba5e67d4abd168b13b50ba1ab0f877aaf73701c32d'), ('920523825506324480_0', 'data_image/california_wildfires/18_10_2017/920523825506324480_0.jpg', 'train', '6aa9dd75eff7bd4612eecdb91ee9b01de932d43e4a35284fb232100a4e7dc3c0'), ('919578737653448704_0', 'data_image/california_wildfires/15_10_2017/919578737653448704_0.jpg', 'dev', '4cfe036d023ef9d7557923490dfee78f2bd9fd3a7926d4809554c12fc16deccf'), ('917828788473552896_0', 'data_image/california_wildfires/10_10_2017/917828788473552896_0.jpg', 'dev', '64ffaadeffbf3e6622f483c3653427dcdfeb110898feed66c33d543903879909'), ('921137048601448448_0', 'data_image/california_wildfires/19_10_2017/921137048601448448_0.jpg', 'train', '8d21b0baac8753a87c012f96cf21cbc353bb9c822ddd57f629253d8f28f9e564'), ('905930890735439873_2', 'data_image/hurricane_harvey/7_9_2017/905930890735439873_2.jpg', 'test', 'e360ee8e8f99b489737d2404bb39896d678d13d43ffd8df8e2ae4bb9a25e4be1'), ('909922362589454336_0', 'data_image/hurricane_harvey/18_9_2017/909922362589454336_0.jpg', 'test', 'de0918b1eb23d25a2daf4473bf9bc49e69797d531b31fed9d189cfd1b402d12d'), ('907150052250058752_0', 'data_image/hurricane_harvey/11_9_2017/907150052250058752_0.jpg', 'train', 'e6f19d01fa7e3ac70742b68ab738a9d3f4d592d46f36df1ef176eec2d22682f4'), ('901701669196529669_0', 'data_image/hurricane_harvey/27_8_2017/901701669196529669_0.jpg', 'dev', '5fbfc230e754ce232d3767e1a8c68fb87980dee6512b43931ad75d28d96e1e4f'), ('908894358660960257_0', 'data_image/hurricane_harvey/16_9_2017/908894358660960257_0.jpg', 'test', '276a48c1bc3f8ecb93ee889bf61b27623dd679d3d90da4d2a9a0a128e720dcfc'), ('908862255512657920_0', 'data_image/hurricane_harvey/16_9_2017/908862255512657920_0.jpg', 'test', 'e20ceb67a74bb9afac42605ed46d8f9afd5bfa57b5dc56c777719e6ddd661505'), ('905427798466207746_1', 'data_image/hurricane_harvey/6_9_2017/905427798466207746_1.jpg', 'train', '850298aa8d6d7c6e3a07b9b2898baf708b4aebf4bd6b3800c00d3458d07cb162'), ('907772978116653056_0', 'data_image/hurricane_harvey/13_9_2017/907772978116653056_0.jpg', 'test', 'f5f4423b962e88ebe0bd0cf91660e93610463f700627dd9566cea3471dbac6a8'), ('904826427291271168_0', 'data_image/hurricane_harvey/4_9_2017/904826427291271168_0.jpg', 'dev', '12c961b175cf8406bcad6cfd36e60f5cb0c74740e5a432e3a5a27d9d2fc8a770'), ('901805288717787136_0', 'data_image/hurricane_harvey/27_8_2017/901805288717787136_0.jpg', 'test', 'a84a70b742707a5d079246ab646e96abd3f60d679ffffc7f4579782218098d8d'), ('904779342953279489_0', 'data_image/hurricane_harvey/4_9_2017/904779342953279489_0.jpg', 'test', '388bb55c0ba879cce720cf30637f9ac447fba8d028de18c205918390de1fc299'), ('905269220837138434_0', 'data_image/hurricane_harvey/6_9_2017/905269220837138434_0.jpg', 'train', 'b429a96bef85eeeb5661bb851072fb593f7ab512ccec15f97b5a2e3bd4092013'), ('906604898434744320_0', 'data_image/hurricane_harvey/9_9_2017/906604898434744320_0.jpg', 'train', '2caa7f2b6f50b7ba34cdabe5966643350140a74dcccaf6f0012427e7fa91c589'), ('904237843857498112_0', 'data_image/hurricane_harvey/3_9_2017/904237843857498112_0.jpg', 'train', '5b87e676c2fc26bb63d4738e1c050b6a6ebfb56c31dd67d2d4776dd4ac450f65'), ('904874268382527488_0', 'data_image/hurricane_harvey/5_9_2017/904874268382527488_0.jpg', 'train', '2bd742cc1e7e1e8786075854a6e4eb703f2011da66d4455f49bbf3c70f4b1c5f'), ('904968873933365248_0', 'data_image/hurricane_harvey/5_9_2017/904968873933365248_0.jpg', 'dev', '3a712dd77b402325fc0572bd6d4b38c97bc8961f3e2f9490b1c5d0aba671e07c'), ('908584190446264320_0', 'data_image/hurricane_harvey/15_9_2017/908584190446264320_0.jpg', 'test', '56dbf579ba658cd58e3e6f7dafbce7d5594001394e9caae6a9a8b20cf8551eda'), ('901783778602962944_0', 'data_image/hurricane_harvey/27_8_2017/901783778602962944_0.jpg', 'dev', 'fbf1b5dcfb53b5288351edae86111e1ccf78ea8c718838b771d18f86ec075905'), ('909135905063026688_1', 'data_image/hurricane_harvey/16_9_2017/909135905063026688_1.jpg', 'test', '3ba9fde1d567b8b9cff52ee28e5d2bd30463b1c9bb358e8bffd850347c49a7d7'), ('901814219191943168_0', 'data_image/hurricane_harvey/27_8_2017/901814219191943168_0.jpg', 'train', '36fd45d144af7578b639a15118e07541f9f42fff5246c874b55a5931f1de5730'), ('904606118525239296_0', 'data_image/hurricane_harvey/4_9_2017/904606118525239296_0.jpg', 'dev', '83615a4f60b5b39ca3b632f04bc3105ca2cbfa3b3afba7c4114d1c04e3947cb5'), ('904706043690745856_0', 'data_image/hurricane_harvey/4_9_2017/904706043690745856_0.jpg', 'test', '347bbd77b3ec0cf5b612679f10d86bec48f3893d3a46c6532690b6ea3e5adb8c'), ('907079883775528960_0', 'data_image/hurricane_harvey/11_9_2017/907079883775528960_0.jpg', 'dev', '9f1e333c987f2fdfb619aabd96202f0bb35aa3deb4b1f673c83a2a9d6fefbebc'), ('906344925234593792_0', 'data_image/hurricane_harvey/9_9_2017/906344925234593792_0.jpg', 'dev', 'bdfc28b9417a9e3c05691b5c8c21a965c441e703845033617b807dcf06b99d98'), ('905400128864968706_0', 'data_image/hurricane_harvey/6_9_2017/905400128864968706_0.jpg', 'train', '8609f9b9e1bafea188754964bccc975c023f8876d6fd43f1857f1c756e20de7a'), ('908037953204695047_0', 'data_image/hurricane_harvey/13_9_2017/908037953204695047_0.jpg', 'test', '5f304c1c587d0ac9b1832b18e8b2935aeb911c359b5899ef5cb23055601cea73'), ('907948019043196929_0', 'data_image/hurricane_harvey/13_9_2017/907948019043196929_0.jpg', 'dev', 'af1e1e171f4ff4252b236833c6da7ea1e42c2f0b77e5bd82cab5a44823ad84bf'), ('904221934124130304_0', 'data_image/hurricane_harvey/3_9_2017/904221934124130304_0.jpg', 'dev', '144d9421debf89323adc53433645575673b6f1fadc92c1811570a64d6e97e850'), ('901768014583840768_0', 'data_image/hurricane_harvey/27_8_2017/901768014583840768_0.jpg', 'train', '63167f4f7b94fd870646bfa33f647b6e6bf0e8e57d11becf2ead130564ae415f'), ('904933387114340352_0', 'data_image/hurricane_harvey/5_9_2017/904933387114340352_0.jpg', 'dev', 'bde3d71e9f47f91f0fda4d7d757c5b4eb06f300d273b152a01b28e8a9b1925cf'), ('905405070526873600_0', 'data_image/hurricane_harvey/6_9_2017/905405070526873600_0.jpg', 'test', 'ed7db9fcdc38520ea13afc06d4f3c9a5d16c09077f193062094163021b9e0182'), ('905019523937443842_0', 'data_image/hurricane_harvey/5_9_2017/905019523937443842_0.jpg', 'test', '02881961cd9389823823752e876235de67d3b33ee0f0263aa9fad3223ce89955'), ('907551747836485633_0', 'data_image/hurricane_harvey/12_9_2017/907551747836485633_0.jpg', 'test', '9af796c70166e18fcf0a17b34586c161ed6f98f623ceaa252fff10c62d28c881'), ('905458734775889923_0', 'data_image/hurricane_harvey/6_9_2017/905458734775889923_0.jpg', 'train', '38feaa09ce6f9755d62e9335f7ebe5f33263acf60cd7a4f1ac9c6fd55e6eaea6'), ('904366441952034820_0', 'data_image/hurricane_harvey/3_9_2017/904366441952034820_0.jpg', 'test', 'db95c492280a5e6a9c7451e907c539e176b4dd6945d747745d038e4b25828dee'), ('904285405486428160_0', 'data_image/hurricane_harvey/3_9_2017/904285405486428160_0.jpg', 'test', '34feb31ddb297d154be9139b47094f4e398b2d682eaa73dcdd3b0daac5fc8437'), ('909130349741006848_0', 'data_image/hurricane_harvey/16_9_2017/909130349741006848_0.jpg', 'train', '06aa8230cff8d4be63b835cdefa68adb8aeed78e955343412e3a707d6ca54d75'), ('909695665562902528_0', 'data_image/hurricane_irma/18_9_2017/909695665562902528_0.jpg', 'test', 'eceb0c895d8c27786429bd4b7df35699fc1e8397a4339d93fc2bebbdb3be3b41'), ('905653422820491265_0', 'data_image/hurricane_irma/7_9_2017/905653422820491265_0.jpg', 'train', '79ac8d71a011872ac911eaf121c4b1328d32e49c54571405273f8501c61d7959'), ('909447240959262720_3', 'data_image/hurricane_irma/17_9_2017/909447240959262720_3.jpg', 'train', 'ed8d7467099e91fbeca03912532202f5e191819797a02592e9e4376af63be0f3'), ('910270769653075969_0', 'data_image/hurricane_irma/19_9_2017/910270769653075969_0.jpg', 'train', 'a096edfdde7b7c160abbf8e3074963d947195e564023f298b64935a67f312f1b'), ('910100679771000833_0', 'data_image/hurricane_irma/19_9_2017/910100679771000833_0.jpg', 'train', '221ad550b42b2b4ffc555cb630b669054d116099aa94a9bd208b3c66ae1ffc8f'), ('905715102258569216_0', 'data_image/hurricane_irma/7_9_2017/905715102258569216_0.jpg', 'dev', '337f0379291d858f4c160a20fe2974fb59b8319b01338fa93fcffd7d400ec9b7'), ('909428988719886337_0', 'data_image/hurricane_irma/17_9_2017/909428988719886337_0.jpg', 'dev', 'a580b89f9125f057ef6935026d59c7d0730edeb9fb986d486df1ea974c628059'), ('909996543385665537_0', 'data_image/hurricane_irma/19_9_2017/909996543385665537_0.jpg', 'train', 'aca41ab4caaf6600a61812bbd46324262c017dd2b0f6872a9f2d64e0c88aabe8'), ('909417984124506112_0', 'data_image/hurricane_irma/17_9_2017/909417984124506112_0.jpg', 'dev', '6afd6408906c237eb75aa50c01389ea7025c137b9bc661a1715559f7714390b4'), ('910217243514036224_0', 'data_image/hurricane_irma/19_9_2017/910217243514036224_0.jpg', 'test', 'e1a99f88bc97a15d635fcfc1afedb6721f9cc9528caf867c39c5e14b5ecc7f53'), ('909824750071042049_0', 'data_image/hurricane_irma/18_9_2017/909824750071042049_0.jpg', 'train', '8fb26a205420e1fa1b73795a22f76afb3ea6dc8864bbdb9863c313bab9d8b527'), ('909698273551831041_0', 'data_image/hurricane_irma/18_9_2017/909698273551831041_0.jpg', 'train', '59eb695ae135a96118c22db1b6e19472ea89ad9f8041423c9ce81d842ff3c9ac'), ('909916947567054848_0', 'data_image/hurricane_irma/18_9_2017/909916947567054848_0.jpg', 'train', 'fc0a1aa97ab2c483e8a1524b1d7af35e57f2e45de4cd5e2b4d2854dc2fa3395c'), ('909813776236650496_0', 'data_image/hurricane_irma/18_9_2017/909813776236650496_0.jpg', 'train', 'b3120135c39a5e246ee529c7cbd30eec35d835d2348266327802ff17ab1a6bb4'), ('909885589817843714_0', 'data_image/hurricane_irma/18_9_2017/909885589817843714_0.jpg', 'test', 'b09da222bde6f76aa77297a4d45c9c9018e46eab3a2b01c086cf44383636b1a6'), ('909405458049445889_0', 'data_image/hurricane_irma/17_9_2017/909405458049445889_0.jpg', 'train', '1db394b4df47629823281c16144d39547fe3c385b09ba5ccf0f8e8ab37878ca6'), ('905715037506961408_0', 'data_image/hurricane_irma/7_9_2017/905715037506961408_0.jpg', 'test', '9233cb53111655f173ef2df4428d3500758c07425687119801fbe1b024fade93'), ('909765635902705665_0', 'data_image/hurricane_irma/18_9_2017/909765635902705665_0.jpg', 'test', 'dfab427414471a1c67c04b0255fdb6833c3064f5fe46c452ff6d25094b930b62'), ('909379184274714625_0', 'data_image/hurricane_irma/17_9_2017/909379184274714625_0.jpg', 'train', '7adb5a6418870ba47cc1a2f414491337fb6a9875ddc358ea6077830d7f6116c2'), ('905669053879951361_3', 'data_image/hurricane_irma/7_9_2017/905669053879951361_3.jpg', 'train', 'ec6122576476754697176903a51166484e564a893c2ea9017dabff269d900ae0'), ('910266304476667906_1', 'data_image/hurricane_irma/19_9_2017/910266304476667906_1.jpg', 'train', '66d0f745ddd26dd4748b58d953dd3be717c65267ff42409537d808aea2a601a5'), ('909739206293049344_0', 'data_image/hurricane_irma/18_9_2017/909739206293049344_0.jpg', 'dev', '10e282e1f60d0b57cd9e016e8e7a2b418aaf66e66d9f4848234358aabc677eea'), ('909748987862888448_0', 'data_image/hurricane_irma/18_9_2017/909748987862888448_0.jpg', 'dev', '8c14058ce1e05273a9e23a31607a30dc7c008f6da808b5f8c9360eb1ca9d76b8'), ('910038385703735296_0', 'data_image/hurricane_irma/19_9_2017/910038385703735296_0.jpg', 'test', 'bdea9e3e2acbaad3fa6ecea44f8f8db07e600a58a14864160ddf52fdf4116dc9'), ('909706987088334848_0', 'data_image/hurricane_irma/18_9_2017/909706987088334848_0.jpg', 'train', '9e9f1586c40a0e4fad0baa3d9160edccc5f4dee4fb41cc8fb5009d7295a8768d'), ('909386138376441856_0', 'data_image/hurricane_irma/17_9_2017/909386138376441856_0.jpg', 'train', 'df09261e7d3fb5fe5a4d556fb5f328d76210e81545952d588ccd99bd6e84ae2c'), ('909835536327745536_0', 'data_image/hurricane_irma/18_9_2017/909835536327745536_0.jpg', 'train', 'a83ad78ba6beaf265d9a648bce77b1d2a681958ff75b58f50800ab53e893bf07'), ('910156911403782144_0', 'data_image/hurricane_irma/19_9_2017/910156911403782144_0.jpg', 'test', '3d39825be271e1c5c4489b258da5604737843946d1c6dd9be1bbe58958d999d2'), ('910136946147573761_0', 'data_image/hurricane_irma/19_9_2017/910136946147573761_0.jpg', 'train', '9d2ea5cce485a24e7bbf5c6bc0a1e8d5078c636ae39705df751c10a1ffeae2a8'), ('909974100419911680_0', 'data_image/hurricane_irma/19_9_2017/909974100419911680_0.jpg', 'dev', '15268cad5dc3c7c5a1f423b11505a2621846b7d5298111fd8ed559b54423363d'), ('909841313289383936_0', 'data_image/hurricane_irma/18_9_2017/909841313289383936_0.jpg', 'train', '63e80ab439c11d8f6f53eb629696c9d6c28e19df276125735b09c5140dadf257'), ('910024622309068800_0', 'data_image/hurricane_irma/19_9_2017/910024622309068800_0.jpg', 'train', '6f368ef0b3787851dc82656bde305f0cf4a4a05fac04b1054a978008ca1310df'), ('909838007397740545_0', 'data_image/hurricane_irma/18_9_2017/909838007397740545_0.png', 'test', 'd46eb5bac3c79a599114fc14712511154a212f98523402ffdeb8c8e2cd6865fb'), ('910002553634607104_0', 'data_image/hurricane_irma/19_9_2017/910002553634607104_0.jpg', 'dev', '1a72cf9461c3e15007d1e77af6a6d4b95349956e3da8f11f70e2292ac7137842'), ('909434968652251136_0', 'data_image/hurricane_irma/17_9_2017/909434968652251136_0.jpg', 'train', '672ddd64235e04d3ad9b8da64f86feb0c86ea77bfef0698c7dbb8dd8625df8e6'), ('910098383834099712_0', 'data_image/hurricane_irma/19_9_2017/910098383834099712_0.jpg', 'train', 'e5d66fe0b6e9c4e1e641da61f7084fce70405e298e08aebc211c605b43948f36'), ('910021992195268608_0', 'data_image/hurricane_irma/19_9_2017/910021992195268608_0.jpg', 'train', 'b8c69c816281cbfafb1c739c27f0f65e5f7d70989c1c7d00fe9f3bcaead3085e'), ('910044117035601921_0', 'data_image/hurricane_irma/19_9_2017/910044117035601921_0.jpg', 'test', '0f4451a1f90e8c859cebc390054d9eb4fb7c2252a052939c7f70bb2c3eb3a066'), ('922322835896287232_0', 'data_image/hurricane_maria/23_10_2017/922322835896287232_0.jpg', 'train', '2064da46d0a10a2157e89ed01f71fe74dc946a1946f88ca081be781ce38d4f9b'), ('916070325019987969_0', 'data_image/hurricane_maria/5_10_2017/916070325019987969_0.jpg', 'dev', 'a043b195836e98dfbfe18134a655b4125989b8546e26ebfd27da142c7aa2da51'), ('916249991244505088_0', 'data_image/hurricane_maria/6_10_2017/916249991244505088_0.jpg', 'test', 'c0abd6c64429104408605173bdbed2e4354c00a39c7c2fd558377bd1ccd06d6b'), ('923358056758632448_1', 'data_image/hurricane_maria/26_10_2017/923358056758632448_1.jpg', 'train', '171473aece74cd1439318d6294256907ec3d2ccc245ab4bf82900d82ada73f2c'), ('922474905538564096_0', 'data_image/hurricane_maria/23_10_2017/922474905538564096_0.jpg', 'test', '413558f0bbb74afdddb864c1f840832dd60cf70457c0d7d89962a06c198ea616'), ('911646715073171456_0', 'data_image/hurricane_maria/23_9_2017/911646715073171456_0.jpg', 'train', '789039d46913cf739f5cc8ea967f692b18e63334cb2d69fc818967b2e0315cfc'), ('910538720117391360_1', 'data_image/hurricane_maria/20_9_2017/910538720117391360_1.jpg', 'dev', '9ae870592c7afaee15fa515bb26e3a8009aa5025c41bcf4136b2a691b87541d5'), ('912279176895451136_0', 'data_image/hurricane_maria/25_9_2017/912279176895451136_0.jpg', 'test', '845a69bef84539f498b3f56fa041fab252d8c59a8f997e487156fdb5b86400f7'), ('912292673490472960_0', 'data_image/hurricane_maria/25_9_2017/912292673490472960_0.jpg', 'dev', '516d4d9e13ec5a4a1b365675f2489209105427cd5ae34b9c368c7c380f90a316'), ('927385055349030912_0', 'data_image/hurricane_maria/6_11_2017/927385055349030912_0.jpg', 'train', 'f972e2784b23a94792927a4b6411b45f3ca0ed7b63e0aedcbd90a6d6493c886a'), ('910726407076098048_0', 'data_image/hurricane_maria/21_9_2017/910726407076098048_0.jpg', 'train', 'cd78639915ad3f543d37084c1abed21cae4ea8060fe7f100723e27b3a46ec9ef'), ('910754701234274304_0', 'data_image/hurricane_maria/21_9_2017/910754701234274304_0.jpg', 'train', '98c4438aed83f238e5537fa410573f9d1cc7dd9930ddfc9af0bbc8a6bdbdbbed'), ('910534805565071361_0', 'data_image/hurricane_maria/20_9_2017/910534805565071361_0.jpg', 'dev', '44997e7b0dabc786d9c1c85f9dc0ec8ae59a42ba53aa2033e03c3290400c9aeb'), ('921427706880954369_0', 'data_image/hurricane_maria/20_10_2017/921427706880954369_0.jpg', 'train', '5dad325227181355f3fca18a5293dc7ad9655ccf9b2ff766ab3bba95681200fa'), ('910533136286650368_0', 'data_image/hurricane_maria/20_9_2017/910533136286650368_0.jpg', 'dev', '0a68d262ebbd75030fb55413f7c9739379dff31512e3045607b296e050c11b41'), ('921322602693525506_0', 'data_image/hurricane_maria/20_10_2017/921322602693525506_0.jpg', 'dev', '528bf1b8799078be605c94cbcc9dedbb954cfbd088f052285ecbf1e34efba62b'), ('922512802362937345_0', 'data_image/hurricane_maria/23_10_2017/922512802362937345_0.jpg', 'train', '925f54148bfe3a9302f1b16ab44a30510df63af0a24b260eaa9ace5f63560304'), ('911795195368415232_0', 'data_image/hurricane_maria/24_9_2017/911795195368415232_0.jpg', 'train', '1fde0b3bc5383c4f0aee4d3d8f88599397dc79524b9b2f29cdd282c7ed944202'), ('912065374929264640_1', 'data_image/hurricane_maria/24_9_2017/912065374929264640_1.jpg', 'test', 'd1a8bdf411eb644961ceb9586dcdf26ea5c1fea95e13c9811df769d5952add24'), ('922889146997211136_0', 'data_image/hurricane_maria/24_10_2017/922889146997211136_0.jpg', 'test', 'f2c9ab62dd60f50f53659883f2ab7067a537496953532f1e22755e479fb71bf8'), ('921461077086482432_0', 'data_image/hurricane_maria/20_10_2017/921461077086482432_0.jpg', 'train', '6897a8a36c749a8d07f7dfbd78e93fbdf38f7808fdc47affe9c38b4665ec0ddd'), ('921390909031354368_0', 'data_image/hurricane_maria/20_10_2017/921390909031354368_0.jpg', 'dev', 'd1900ca01cdc182772dc033cef13af82a29025efb7d49ca6d7ab6e0fd3eaa863'), ('922983668238422017_0', 'data_image/hurricane_maria/25_10_2017/922983668238422017_0.jpg', 'train', 'a523e66b870b2a30efe2877475a937388809e2620acb62bb05fd43681b7bb0fc'), ('928308259491942400_0', 'data_image/hurricane_maria/8_11_2017/928308259491942400_0.jpg', 'dev', '8debaf54e5133befb7574f1f8d11e63b717f65c9701448de189af0fbf7300ff5'), ('922800994169315329_0', 'data_image/hurricane_maria/24_10_2017/922800994169315329_0.jpg', 'dev', 'c865ce7b48796e4bf0a69766fed964306266fb5d07138b04b0e4f84c827f0c55'), ('910717248708345857_0', 'data_image/hurricane_maria/21_9_2017/910717248708345857_0.jpg', 'train', '814ab606b712d97ded085bf1ad451fc7718b997e18186c5a11bd41072cfe593e'), ('910704791550136320_0', 'data_image/hurricane_maria/21_9_2017/910704791550136320_0.jpg', 'train', 'c770258f44436cf7c785481015357e80a007c8ceb7c59a3ca853add70bb67ece'), ('913892408621981696_0', 'data_image/hurricane_maria/29_9_2017/913892408621981696_0.jpg', 'dev', '45116b931ef6d9a96080f42858825d9db09240e952c74808614a518013ae2cdc'), ('910532027107172353_0', 'data_image/hurricane_maria/20_9_2017/910532027107172353_0.jpg', 'train', '2bea32c38605ea7b0b0d2ffbf415dc60be16353c0e5b4653047b81734f173a29'), ('913065602251714560_0', 'data_image/hurricane_maria/27_9_2017/913065602251714560_0.png', 'train', '1089614389780d2f22c76abc667a5bf09b3769ba2a95e209542b64381a58dbed'), ('923209934501584896_0', 'data_image/hurricane_maria/25_10_2017/923209934501584896_0.jpg', 'train', '75b4c109344b64aadeb9c3c1747b367c27f24beb65d636cc3f358a84eeec9730'), ('922601042533007362_0', 'data_image/hurricane_maria/23_10_2017/922601042533007362_0.jpg', 'test', '3584e0e874493c537e4bec9119de290485c5676b1653b7f40edd332bd7209762'), ('923365901138833409_1', 'data_image/hurricane_maria/26_10_2017/923365901138833409_1.jpg', 'dev', '340619da0328328a63d616d200a71e8452064bc4d4500b69fd4870b29b9cda11'), ('914357340341768192_0', 'data_image/hurricane_maria/1_10_2017/914357340341768192_0.jpg', 'train', '7168dd3c0a4d54343664af40f94648e64640610ceccd7422f7fe4196fe46c6e5'), ('922591169586003968_2', 'data_image/hurricane_maria/23_10_2017/922591169586003968_2.jpg', 'train', '491bd225828059c8a8b399adcde6e23ee58a66af2be64421ed7e8b1e864bc55b'), ('913935418235641856_0', 'data_image/hurricane_maria/30_9_2017/913935418235641856_0.jpg', 'train', '9be0d864625c6bb641ced698df64c0b11ef73062314be73b84279ef6cfa4b617'), ('927943725132722183_0', 'data_image/hurricane_maria/7_11_2017/927943725132722183_0.jpg', 'dev', 'e269ca34dad1666fc9f1e49855d9f4cb9173813d1b465c63053d609feaaa1f0c'), ('913141954287931393_0', 'data_image/hurricane_maria/27_9_2017/913141954287931393_0.jpg', 'dev', 'c5f1f8883fb81a055781647f36d30c491455450d8f80d0a9f5148790b92addc0'), ('930308604200026112_1', 'data_image/iraq_iran_earthquake/14_11_2017/930308604200026112_1.jpg', 'test', '3411adc2658a129042c65b1af9c3453da51256f1e4e528aba51ef3a61d3c6c32'), ('930186148705599489_0', 'data_image/iraq_iran_earthquake/13_11_2017/930186148705599489_0.jpg', 'train', '3912a01036b7dfa184a5b4fb82a1fd1ec846e1bcefd804f3ec65534254472cc4'), ('930573880619659264_0', 'data_image/iraq_iran_earthquake/14_11_2017/930573880619659264_0.jpg', 'test', '8e42e606b72be1b3b7d7f6108330bfae2182ad967c387457dd73fe5625c83463'), ('929996080498003968_0', 'data_image/iraq_iran_earthquake/13_11_2017/929996080498003968_0.jpg', 'dev', '0603f48c483467b8594bcf041b826bacc1942be79924dad971241add3ffdd204'), ('930347490263666688_2', 'data_image/iraq_iran_earthquake/14_11_2017/930347490263666688_2.jpg', 'train', 'ad5c6f309d4b10483884c7d7e9e622964e6184f85e41137b598ec56e6954bc05'), ('930015597135585280_0', 'data_image/iraq_iran_earthquake/13_11_2017/930015597135585280_0.jpg', 'test', 'eb38b7edb242ec2e63736d5b6832f38fe8a85cb1bcbd7e49731b14863c7a3cda'), ('930609791713665025_0', 'data_image/iraq_iran_earthquake/15_11_2017/930609791713665025_0.jpg', 'train', '364815eceee69f2d21af6524a16effb2fb48d864c849f5e579f18566ab0eec18'), ('930272493947850752_0', 'data_image/iraq_iran_earthquake/14_11_2017/930272493947850752_0.jpg', 'test', 'af141ad39a57831297b5303b7b4e80a6a452d40c63c52711305558f0eed2322a'), ('930114382268334080_0', 'data_image/iraq_iran_earthquake/13_11_2017/930114382268334080_0.jpg', 'train', 'fc92bc119634b32678f914a9ac4e4beaf2ec82cfc857e36768bb60be61744205'), ('929989414826024960_0', 'data_image/iraq_iran_earthquake/13_11_2017/929989414826024960_0.jpg', 'train', 'c4431841fae065ce680822b0df3b8d25590c7359fc8cb3dd9f9f0a9bcc7a897e'), ('930463119100358656_0', 'data_image/iraq_iran_earthquake/14_11_2017/930463119100358656_0.jpg', 'test', 'faeab3b78b291f6a4830fc7e3693afce00acd112700a0c1339065ef9f4716c19'), ('930069080173723648_0', 'data_image/iraq_iran_earthquake/13_11_2017/930069080173723648_0.jpg', 'test', '44548d24503e9c96c5db494ab421c44d5423434db0123cd00b201113685ad997'), ('930389337895776256_0', 'data_image/iraq_iran_earthquake/14_11_2017/930389337895776256_0.jpg', 'test', '9bfbfda59c74d1cedda8c4436ad02bb2389486a098efe949817d71b001d34f33'), ('930057942426181632_0', 'data_image/iraq_iran_earthquake/13_11_2017/930057942426181632_0.jpg', 'train', 'd8b773af986aeb7b34326b9c628fa81ad16adfafae5439093036a32b62d18fe8'), ('930015716903862272_0', 'data_image/iraq_iran_earthquake/13_11_2017/930015716903862272_0.jpg', 'train', 'b36e6bd158209dfd394ad481de1c1af498a0f429307707e2d1682f3833e4ffa9'), ('929990644466110469_0', 'data_image/iraq_iran_earthquake/13_11_2017/929990644466110469_0.jpg', 'dev', '582699b86bcead5630409fbb7829c6504a64c9e1ca33bc12fcfc32ddcbe0530b'), ('930116757938954248_0', 'data_image/iraq_iran_earthquake/13_11_2017/930116757938954248_0.jpg', 'train', '4d2c0b6db6dd36e89a9bbbfc9b79cc5233eace220289b827fc69cfc8591cd371'), ('931771549685899264_0', 'data_image/iraq_iran_earthquake/18_11_2017/931771549685899264_0.jpg', 'train', 'd7abddccf82c4c6042d736ff7ba8963ab5b2d0cb9c68044aede4df28954285c9'), ('930051455276519424_0', 'data_image/iraq_iran_earthquake/13_11_2017/930051455276519424_0.png', 'dev', '99a0693c1249c8bf896944309dfac479eee75b8f3ffdc8ef57b5918fb819e407'), ('930161349212196864_0', 'data_image/iraq_iran_earthquake/13_11_2017/930161349212196864_0.jpg', 'dev', '0ce067dbb889cac55c86b46f772c4cae14ce26ca1b535123de137a3a43c933e4'), ('911655630460669953_0', 'data_image/mexico_earthquake/23_9_2017/911655630460669953_0.jpg', 'train', '967ca11e156de2a0acf3c29060d233bb0326557af578c61bca12484727e65982'), ('910559231916560385_0', 'data_image/mexico_earthquake/20_9_2017/910559231916560385_0.jpg', 'train', '4501f50bf42d71cfec55d901308709f82d2743e1067f7592920e4d005b5099d2'), ('910707946983706624_0', 'data_image/mexico_earthquake/21_9_2017/910707946983706624_0.jpg', 'test', 'e028cd70909fb0f9b98274f47e6df28502da96bbfe6cd4256966b573741d0811'), ('912048104651788289_2', 'data_image/mexico_earthquake/24_9_2017/912048104651788289_2.jpg', 'train', '721ed0964c81efd9d963f838c1b669129ef4d731e6dc91270d78465b3d5bbbfe'), ('911623362388090880_0', 'data_image/mexico_earthquake/23_9_2017/911623362388090880_0.jpg', 'test', 'eeaa3a0de71e73c5974ed37db0948e9599593bab65f8b5964eab096412cccfa3'), ('910764633811816449_0', 'data_image/mexico_earthquake/21_9_2017/910764633811816449_0.jpg', 'train', 'a6941d3dab169848f3ccb6d2773bbd966d3810c7417f94541b867b398b37c255'), ('912966692241907712_0', 'data_image/mexico_earthquake/27_9_2017/912966692241907712_0.jpg', 'test', '375da2531ae670084bd3d3287ef9e3ebc32625015627db2d3f2ecca22b3e710f'), ('910624371533766656_0', 'data_image/mexico_earthquake/20_9_2017/910624371533766656_0.jpg', 'test', 'c4fe0bfcdd807c7133e35da5691c69aa43039f692161a7745b95c9a921f511b1'), ('910537562132901889_0', 'data_image/mexico_earthquake/20_9_2017/910537562132901889_0.jpg', 'train', '21216db4ea58c887b6db6fc47e060ab6d98b2efee47a7daca7c658cd538c9a79'), ('910624850703773696_0', 'data_image/mexico_earthquake/20_9_2017/910624850703773696_0.jpg', 'dev', 'd7a2368a2d12edd043c3d773570c94bd1167050e354f86968510e77392a4f4a0'), ('912054112283439104_0', 'data_image/mexico_earthquake/24_9_2017/912054112283439104_0.jpg', 'test', '83055f8265b94640f2859d50c80122db83288a8bda59f62b42d1e20286e0bbeb'), ('914238423250083841_0', 'data_image/mexico_earthquake/30_9_2017/914238423250083841_0.jpg', 'train', '41d8c5310d7f64ceef23dff5afc837aaef8e76c3897c265e127e53f20fcd978b'), ('913697454788358145_0', 'data_image/mexico_earthquake/29_9_2017/913697454788358145_0.jpg', 'dev', 'da4a13bd22dd6c7d9814877d8e4a9889cbac1ae9f6023cf86d9ac05e50fb5155'), ('910696403248103424_0', 'data_image/mexico_earthquake/21_9_2017/910696403248103424_0.jpg', 'test', 'fd9d3d5deba8fc123889b7459133b43453986239904d78c10e62d2f365bb5892'), ('914885862386814981_0', 'data_image/mexico_earthquake/2_10_2017/914885862386814981_0.jpg', 'test', '95efaa684dab8adf8815fe1b04933c3ea284c7533e81114f88b8c92606cccb2c'), ('911688552936726529_3', 'data_image/mexico_earthquake/23_9_2017/911688552936726529_3.jpg', 'train', 'b20f79ef08482deb2905872dd3248fc62300d4584aca9b3a9ea414941f496dec'), ('911680036578762756_0', 'data_image/mexico_earthquake/23_9_2017/911680036578762756_0.jpg', 'dev', 'ced4338f6ba1224dfcfffe395ff17384f2fac356a92dac09ec94ddda011d7a17'), ('910525324856823813_0', 'data_image/mexico_earthquake/20_9_2017/910525324856823813_0.jpg', 'dev', 'bb7dbf0b436d178e6bb545e9a47afb3224960b14094ff0e0474f4832da1eeb35'), ('910564618334035968_0', 'data_image/mexico_earthquake/20_9_2017/910564618334035968_0.jpg', 'train', 'b4f3c317d095d6a8e8cef7a8fa45ffb44452a7ef49c074e074f762c0d97e0c3b'), ('910636262997901312_0', 'data_image/mexico_earthquake/20_9_2017/910636262997901312_0.jpg', 'test', '1e4ca41dfd1b24a7820c389d9568c219462045c0f1732d33e7f7168cab5243ed'), ('910655089211142145_0', 'data_image/mexico_earthquake/21_9_2017/910655089211142145_0.jpg', 'dev', '81d948e5b6eb43a54d244c88275ca4156c23707a77f513b8d1c196f4c3975a10'), ('910525151719997440_0', 'data_image/mexico_earthquake/20_9_2017/910525151719997440_0.jpg', 'test', 'ae035d4496d3f3c189f05dc14f55083856d2112403862d35a81ed682d6b07e49'), ('911682931743010817_0', 'data_image/mexico_earthquake/23_9_2017/911682931743010817_0.jpg', 'train', '6472c4e1ebae2c986bc127ff8e4bbe2ecdc88fb9ffee3b3f8e9b0716fc36eb65'), ('911688552936726529_1', 'data_image/mexico_earthquake/23_9_2017/911688552936726529_1.jpg', 'train', 'b3410a636741d5c125ac62aa4c8876c273fcc2b9e24bf767107e86c177014247'), ('910525286009102336_2', 'data_image/mexico_earthquake/20_9_2017/910525286009102336_2.jpg', 'train', 'e1b0e046e95819a4abb58440c1a8a53a769a9b17d2e10e77914e29b211924bb4'), ('870942570704019456_3', 'data_image/srilanka_floods/3_6_2017/870942570704019456_3.jpg', 'test', '8f35a74f000b29080d53b39bed0aaa1d4b1c1d55e8e3b863baa357f8885a636e'), ('870114879541137408_0', 'data_image/srilanka_floods/1_6_2017/870114879541137408_0.jpg', 'test', 'b7050df6fb1c22ff1b6775de5408851f732bd66c966f3ff90715b9f98c209f34'), ('871220595928834049_0', 'data_image/srilanka_floods/4_6_2017/871220595928834049_0.jpg', 'dev', '78e51a76eb0776d6d3e18fa9bab337944fc7156d71bc877e7a8f6cee1b93b841'), ('870140216656502784_0', 'data_image/srilanka_floods/1_6_2017/870140216656502784_0.jpg', 'test', '17754497525054caed8772b9dfacb6818d8230af710454661c3f1a584c517c0b'), ('873306605122703360_0', 'data_image/srilanka_floods/9_6_2017/873306605122703360_0.jpg', 'train', '86ec95fb1f7f1acb91453c5d8a8ee038a47df0c29f2f1505d01b0399036e1ba7'), ('875665628354314240_0', 'data_image/srilanka_floods/16_6_2017/875665628354314240_0.jpg', 'dev', '958289c2762a9b69aef146049f551a44bbb10b901ca0e69f2ad27e4bb184fd6b'), ('870052558928138240_1', 'data_image/srilanka_floods/31_5_2017/870052558928138240_1.jpg', 'train', 'c8df01cc4af7c9205e1168ca2e1661b07aefe040d1d8d225c7600f8c6bc55312'), ('870032047145746432_0', 'data_image/srilanka_floods/31_5_2017/870032047145746432_0.jpg', 'test', '7504528852b585f897ddeeae120cbf8bc493610fb57c386284f4bed61d93d270'), ('870255568384016384_0', 'data_image/srilanka_floods/1_6_2017/870255568384016384_0.jpg', 'test', '6d1c753b3a3ab180539e84c9bbf75918b16a959e7d03575e29cbc16052004ca5'), ('870532701509500928_0', 'data_image/srilanka_floods/2_6_2017/870532701509500928_0.jpg', 'test', '330ccc9d7fa2b23f10ac8ab1743cd608b7d7e1607fe119668b3c2214e69720e8'), ('870574893280092160_0', 'data_image/srilanka_floods/2_6_2017/870574893280092160_0.jpg', 'train', '874bc8ed6eeeee3d485eb2c3684101f576a3e6c1c09ddd35f0aa157f638512a8'), ('870079994306871296_0', 'data_image/srilanka_floods/1_6_2017/870079994306871296_0.jpg', 'dev', '869d11a3d5afd1580a1293a52e2559ca40c7fc1f4bd80ef4bfe0f1d9db71e7cf'), ('870450100342063104_0', 'data_image/srilanka_floods/2_6_2017/870450100342063104_0.jpg', 'train', 'c250b62a6682b0f3e56bb800faa4112b2fafa9a5a0531ff591dfca440314583d'), ('870522593056509953_0', 'data_image/srilanka_floods/2_6_2017/870522593056509953_0.jpg', 'dev', 'e875a1e242f6f384b7e2524953a5d398bb0ac3bfa7279856269f220f19f9adee'), ('873004234425286657_0', 'data_image/srilanka_floods/9_6_2017/873004234425286657_0.jpg', 'train', '75b8ba5c9ed7b469559bc5a3145f14436af06ba5b219ea919c4e8e8cd18758a0'), ('870479677789360129_0', 'data_image/srilanka_floods/2_6_2017/870479677789360129_0.jpg', 'dev', '040d8839501765de153fd1b1dd8bcb7e42a95cdf5cfe8aee0a178b46f2a5176a'), ('874237986245947392_0', 'data_image/srilanka_floods/12_6_2017/874237986245947392_0.jpg', 'train', 'a7e236bf6a410faf2180a797b85b291dd026cb7ae2913d3161f76de2996e2f11'), ('870138088156086272_0', 'data_image/srilanka_floods/1_6_2017/870138088156086272_0.jpg', 'train', '4ca4de36e2f1e5bea5515c90155af0d66fe436420bc500261b78a60ff0d318d0'), ('870309263503114241_0', 'data_image/srilanka_floods/1_6_2017/870309263503114241_0.jpg', 'train', 'd1b0f23d5c130a2761c1b6c47df02c5f0ec3aa9e7b06d24fc4915485755423e1'), ('870035816990425089_1', 'data_image/srilanka_floods/31_5_2017/870035816990425089_1.jpg', 'train', 'afb4d79a3dec37e62b4d4fe724cf0806e700cedd223db808897d442f2903aca6'), ('870568320969453572_0', 'data_image/srilanka_floods/2_6_2017/870568320969453572_0.jpg', 'test', '65d4686fd52f413ab83d3a164737d80126e9702a48359acd61b9c74108fa6ad4'), ('870110587661021185_0', 'data_image/srilanka_floods/1_6_2017/870110587661021185_0.jpg', 'train', 'ea4d3a8e1a4dea25c078582aef5c0be2821830c90b9469ee8cc1bb36501407e3'), ('873491045442441216_1', 'data_image/srilanka_floods/10_6_2017/873491045442441216_1.jpg', 'train', 'f147d9858c336d2a0c963df1048da0a2e7a840910a0860a106d95918427396a7'), ('871545956159950849_0', 'data_image/srilanka_floods/5_6_2017/871545956159950849_0.jpg', 'test', '3604ddd8e6f7b519442222de7c9d4eef12e162261c6805963ffa0394d89bc323'), ('870283105151668224_2', 'data_image/srilanka_floods/1_6_2017/870283105151668224_2.jpg', 'dev', '470deb4a02247459ffb8c3c6ca1ee26c2f0d97bd25d2d77c3e943ba961a10ae5'), ('917933078969966592_0', 'data_image/california_wildfires/11_10_2017/917933078969966592_0.jpg', 'train', '6c5aa532172ff285c0438340907f4352cf9e6b2b5ddddd79618d9746e2bc926a'), ('917975588685664257_0', 'data_image/california_wildfires/11_10_2017/917975588685664257_0.jpg', 'train', '3581db6250b5fbda7a4593e0361c06270755fa02998882633ca0931a11daca14'), ('919729915926159362_0', 'data_image/california_wildfires/16_10_2017/919729915926159362_0.jpg', 'test', '7c5cd955de58880a3af12be12d45579bff768e181fadb9d9339b3a622756275e'), ('921766663254659072_0', 'data_image/california_wildfires/21_10_2017/921766663254659072_0.png', 'train', '3e2b29d160758ae9b3d8aadc95c58043ef2e4e23bdb8329decada67a4fa18121'), ('918237370536906753_0', 'data_image/california_wildfires/11_10_2017/918237370536906753_0.jpg', 'test', '6db95ef3632e36d083af8c395f2c5d288448e6ef9b227cba73ac6e0271014f08'), ('919731334368694274_0', 'data_image/california_wildfires/16_10_2017/919731334368694274_0.jpg', 'test', 'ae884542d2a3b98fc0bdbcbf1a0f1c398b2284d5172838f67a2a84723c011ebc'), ('921736430963101696_0', 'data_image/california_wildfires/21_10_2017/921736430963101696_0.jpg', 'dev', '64a4a9824548e9d19356e2e4a3cae1edac7f7b75503d468aa18e1b7b687974a7'), ('920989205970522112_0', 'data_image/california_wildfires/19_10_2017/920989205970522112_0.jpg', 'test', 'ff32949314ff68c950f7b425f32ded879b2861cdec0c574605d4f4cfcae870be'), ('923577783367946240_2', 'data_image/california_wildfires/26_10_2017/923577783367946240_2.jpg', 'train', 'b54c587161cf8a5972bd065e0e590c65dd007ff05d7971d60636d47a5a1491ad'), ('921095476455321600_1', 'data_image/california_wildfires/19_10_2017/921095476455321600_1.jpg', 'train', 'b9c6ada7b695a695897681281e2f728ec182909891a4ffe68295cb0ea2bafb29'), ('918355835549601792_0', 'data_image/california_wildfires/12_10_2017/918355835549601792_0.jpg', 'dev', 'ba7e98005872f1afb439b2de51eabbdb8e047c817a7c3b5dc07beb63ce7658a6'), ('918076997036924928_0', 'data_image/california_wildfires/11_10_2017/918076997036924928_0.jpg', 'test', '383a5fee4e1e87644301a3f862b35591d9f8aff254d60eb64e1891265a2d42c6'), ('919929232012095488_0', 'data_image/california_wildfires/16_10_2017/919929232012095488_0.jpg', 'train', 'c33654dac9257a8beb68446e1c48d95b9e5eee5c808a04d0aec81dd7f308e6fa'), ('920430848398069760_0', 'data_image/california_wildfires/17_10_2017/920430848398069760_0.jpg', 'train', 'c1851329395d8769130b5247a37be653352f008833ddb3465ebaa4891eb055a0'), ('920553611045801985_0', 'data_image/california_wildfires/18_10_2017/920553611045801985_0.jpg', 'dev', '803020044ceffa63cf048ce355c750ebe667224e5ca22d88a875456341c8d21f'), ('923547784736673792_0', 'data_image/california_wildfires/26_10_2017/923547784736673792_0.jpg', 'test', '6bbd1576eec39468457cff900be37be9c0a569a836d3e84b7962a6e15bdd1972'), ('920667555941011457_0', 'data_image/california_wildfires/18_10_2017/920667555941011457_0.jpg', 'test', 'f46288bb9767555df78389665e579f5b4fa7bc559ea8d48494e62c83b1b53574'), ('920424292227141633_0', 'data_image/california_wildfires/17_10_2017/920424292227141633_0.jpg', 'train', '2af0541100a5cef49ea56bc8ce21b37240ebb23187edbb31d45af7266475f5a1'), ('923054562188988416_0', 'data_image/california_wildfires/25_10_2017/923054562188988416_0.jpg', 'test', '199e83a32073738c1c9fd689fb9a52532e28a91237fe733fce7a03c40c4f566a'), ('919927912769339392_0', 'data_image/california_wildfires/16_10_2017/919927912769339392_0.jpg', 'train', '21b4064c0e780c4481d7e50405714800fad8ef0f34028cc96ea7deb76795f8f7'), ('921425360373329920_0', 'data_image/california_wildfires/20_10_2017/921425360373329920_0.jpg', 'train', '426073ef37c37b6d170064aa60c4867c7300dd329f6c9dbee7a7b7fea17d0957'), ('918008803530039296_0', 'data_image/california_wildfires/11_10_2017/918008803530039296_0.jpg', 'dev', '7c995eef05690349d11b12c4c3389f0c1c01567ba0a2cc7aff4c46eee4655c9e'), ('917839185779806208_0', 'data_image/california_wildfires/10_10_2017/917839185779806208_0.jpg', 'train', 'bb2821a9565a3d5638947ba748b6329bd23c965d765e644c91ee565b80c7d384'), ('921743270434177024_0', 'data_image/california_wildfires/21_10_2017/921743270434177024_0.jpg', 'test', '51652e27a9bb8343162d51017cc689f2e9d75914a9aaed6ac742160d3e0f02b8'), ('920831780629782528_2', 'data_image/california_wildfires/19_10_2017/920831780629782528_2.jpg', 'train', '9fbea1b72a15ef8708a18e1c6758eced52955f78ee5d7713ec99fc75aea90cc2'), ('917863283419357185_0', 'data_image/california_wildfires/10_10_2017/917863283419357185_0.jpg', 'test', '11cd806888cc420b6d2ef126a61a0fb6da758dcf9a862e39f9916a7dc4def2bc'), ('919603064587223041_0', 'data_image/california_wildfires/15_10_2017/919603064587223041_0.jpg', 'train', '49e2c851bcf3efe426f02e09c902291a546b23c4549beb36d17d2b8a07541b96'), ('919971985844785159_0', 'data_image/california_wildfires/16_10_2017/919971985844785159_0.jpg', 'train', '42c76f72dfca3f3078c4bdffaddf0cf64f86f699b5fc3867ea4e9053998bc462'), ('922206138246029312_0', 'data_image/california_wildfires/22_10_2017/922206138246029312_0.jpg', 'test', '00f2c276d3c2dd143800c3ba534692b70bc90314504ac4601b223dfd730f755f'), ('921259503928176641_0', 'data_image/california_wildfires/20_10_2017/921259503928176641_0.jpg', 'train', '1507f8cb8aebeae63b19560810d9ebc0f4a4092fab98c173791d2d8fde2b0249'), ('905230326359891968_0', 'data_image/hurricane_harvey/6_9_2017/905230326359891968_0.jpg', 'train', '5a0a0b49e4dfc8c987bc7ab88278e4a28f6f7bae8a340418725b725e16e78332'), ('901705240419938305_0', 'data_image/hurricane_harvey/27_8_2017/901705240419938305_0.jpg', 'dev', '151af56f2a5294a752d040286284524325d4dcad6732d9c5a8eb0a97bb0cc279'), ('908912388002459648_0', 'data_image/hurricane_harvey/16_9_2017/908912388002459648_0.jpg', 'dev', '7983c1fd0128ddf9d9536d338eb3f196e31527034feba4a6f5748f91d67180e6'), ('905859989335556096_0', 'data_image/hurricane_harvey/7_9_2017/905859989335556096_0.jpg', 'train', 'bca6d952159bcd742f88643d9d58c69bf88f093eaae7ad9d75962f134cc85b64'), ('905310580369711105_1', 'data_image/hurricane_harvey/6_9_2017/905310580369711105_1.jpg', 'train', 'c783c473c4e313e901f9e63599b0969f520076f6985cc074a84406292b3f6edd'), ('904240884241326081_0', 'data_image/hurricane_harvey/3_9_2017/904240884241326081_0.jpg', 'test', '254d6b7858a1d04f9a3473383624fa675884245d15944a5496e3a97422dc1ba1'), ('904623462077140993_0', 'data_image/hurricane_harvey/4_9_2017/904623462077140993_0.jpg', 'test', '0868486c980946d3b7679a897f5d9aa088a3a6807ef6bc175743ce075ce99a8c'), ('906792024094986240_0', 'data_image/hurricane_harvey/10_9_2017/906792024094986240_0.jpg', 'dev', 'fcdf35a4c2d30f784a967e2cc2629a5e0a13b1c209f1fffa886d8a166e4b15a4'), ('901650454832185344_0', 'data_image/hurricane_harvey/27_8_2017/901650454832185344_0.jpg', 'test', 'ccb126766cc2a0df44665090cb33e57f93852b69894715bcd6f85fbc084d5c6c'), ('905407906757181440_0', 'data_image/hurricane_harvey/6_9_2017/905407906757181440_0.jpg', 'dev', '3ad334dcef7e552e7765640b9210bb9ef46daafe0ace8ae1f0f3713265821768'), ('907413526175784960_0', 'data_image/hurricane_harvey/12_9_2017/907413526175784960_0.jpg', 'train', 'dcfd66966bffb11e249c7232138def8c03753140b486d5ed18a4c6f2def1d94f'), ('908100686273024000_1', 'data_image/hurricane_harvey/13_9_2017/908100686273024000_1.jpg', 'train', '0ee6b30ff3c3f32e81a0e70efea0cb360da2c87e7b9e937edd8f5ab4ffd7b0ea'), ('904525755270483969_0', 'data_image/hurricane_harvey/4_9_2017/904525755270483969_0.jpg', 'test', '2a1404484d19a2600f5a449056d69b7d2024f42e33606587c647ee31963ee84a'), ('909228114554867712_0', 'data_image/hurricane_harvey/17_9_2017/909228114554867712_0.jpg', 'train', 'd7970a22349818fc303e43382c9134ab535532e96b1a27906fb2d33f57697e2b'), ('904480383395352577_0', 'data_image/hurricane_harvey/3_9_2017/904480383395352577_0.jpg', 'test', '1c7374c840dd87eb38b55ed4f6460588f311127b3f2f1d3782c72d6eb067d052'), ('905548507343454208_0', 'data_image/hurricane_harvey/6_9_2017/905548507343454208_0.jpg', 'dev', 'cb8de632286457ded24469f8120f895d994ecd7210e3b97502a637fc58428ab7'), ('909135905063026688_2', 'data_image/hurricane_harvey/16_9_2017/909135905063026688_2.jpg', 'test', '31541742ef689bdc95dd207b9a61e26a83c982dd639b32f549f57da0e336997f'), ('901650574785093632_0', 'data_image/hurricane_harvey/27_8_2017/901650574785093632_0.jpg', 'test', '82391f743ab34ceb66992e8b47ae92077d2051f2d8f01ef0eb6aa49161f8ad31'), ('901650362410688513_1', 'data_image/hurricane_harvey/27_8_2017/901650362410688513_1.jpg', 'dev', 'cb5b6661de19544b62ec4cf60f85ca1705f3f38f009924ed1eac3bd0a3448318'), ('908702885202075650_0', 'data_image/hurricane_harvey/15_9_2017/908702885202075650_0.png', 'test', '0dea319834bbef05be69bb1bbf537913f8b14a1bac23de44d203b25f8c7a0f78'), ('906983956553322502_0', 'data_image/hurricane_harvey/10_9_2017/906983956553322502_0.jpg', 'test', 'e25f349c3d7fec4ec89305d9de09998a9835f803f5d3de5537f96725198f5415'), ('905511356568305664_0', 'data_image/hurricane_harvey/6_9_2017/905511356568305664_0.jpg', 'test', '5b7a0625fcfa93645902780676c3b6bc2c4720f4b2d8dabf142fac59182fc8eb'), ('901805448944390145_0', 'data_image/hurricane_harvey/27_8_2017/901805448944390145_0.jpg', 'train', '046afd6b83cd335b1d9753101bb2d7976dfadd4db0222e69c573986cf0601153'), ('904853248384913408_0', 'data_image/hurricane_harvey/4_9_2017/904853248384913408_0.jpg', 'dev', 'f108a0ef1d75724f6b3587d5e665014354777035400f57ceaa8c2e991d1132e7'), ('904335509723467777_0', 'data_image/hurricane_harvey/3_9_2017/904335509723467777_0.jpg', 'test', '3f60e13cdb57e05dd061774fd1adf6480ecf0f73b54a5d1abbd13e82663d9be7'), ('904357631502016512_1', 'data_image/hurricane_harvey/3_9_2017/904357631502016512_1.jpg', 'train', 'e05066943cd80887eeaff695e98740f95852a2132f8f84adc2ea2eccc7885d9a'), ('905105412248260609_0', 'data_image/hurricane_harvey/5_9_2017/905105412248260609_0.jpg', 'train', 'e04e455d730cc20a1ce2f15472631f0a54584a387d5c2a1f0a482231034cb1fc'), ('904331124339609600_0', 'data_image/hurricane_harvey/3_9_2017/904331124339609600_0.jpg', 'dev', '10a5fe0605c02826aacbef20f60b9163f58898473fb3be7b9a706dce3e6c3738'), ('905495596823330816_0', 'data_image/hurricane_harvey/6_9_2017/905495596823330816_0.jpg', 'train', '09e716fc1e73b59e493fa1224abc7807cd76a0ff2ad359c04cfd9735df8dc1c7'), ('904755048302956551_0', 'data_image/hurricane_harvey/4_9_2017/904755048302956551_0.jpg', 'train', '0b1bcfbfd1a0ee21adb902ac33f9d8026ec2ce8900198cf79c1be0f33a5e0708'), ('905570747313217536_1', 'data_image/hurricane_harvey/6_9_2017/905570747313217536_1.jpg', 'train', '8a1f9ca7cd823e4d1ee7da725188e4487c89c993f54519d73414f8f443e95476'), ('904548836126875648_0', 'data_image/hurricane_harvey/4_9_2017/904548836126875648_0.jpg', 'train', '547c17e79aff353aaf92bac73c522ebe2d6e817ba40708f63465c826dfd943ef'), ('908133217009590272_0', 'data_image/hurricane_harvey/14_9_2017/908133217009590272_0.jpg', 'dev', '77ca9de42e28ffbb5ef38a77e18c92e35bb4d78e83cdd2fae1cede5fd8338720'), ('904236893184909312_0', 'data_image/hurricane_harvey/3_9_2017/904236893184909312_0.jpg', 'train', '2ec89cb70ba5119635629a2218460de302e6fac800b9066a9fb7eba2536ae21b'), ('901792190921408513_0', 'data_image/hurricane_harvey/27_8_2017/901792190921408513_0.jpg', 'train', '8ee257c5501b85fc508598eb0b449f93422bb7e7b0a71eb9d8a5ddd4bde0639d'), ('910162402108477440_0', 'data_image/hurricane_irma/19_9_2017/910162402108477440_0.jpg', 'test', 'b35a1071fed582e31bf07a001d57b9fb753cc4f1f94766a0b70f519f362c8454'), ('909890385098440706_0', 'data_image/hurricane_irma/18_9_2017/909890385098440706_0.jpg', 'dev', '56ac89d11a2c42cb973f59d82626bee16da7bc8db65d88e7b8cdd9dde9725b08'), ('909492529854377984_0', 'data_image/hurricane_irma/17_9_2017/909492529854377984_0.jpg', 'train', 'f2044c4058710a4b4813b6b773aca647dbd1802adbbf1285ef72dfc405ea4c82'), ('910222896768983041_0', 'data_image/hurricane_irma/19_9_2017/910222896768983041_0.jpg', 'train', '1577a99e51e96ab7e79fc39b95ecaa4cdd9012c9d54e2afc8e73bdb52c326286'), ('909394026301927424_0', 'data_image/hurricane_irma/17_9_2017/909394026301927424_0.jpg', 'test', '555b6b52ff876d712ef9f2a3cfa337f8635bc0df88552311428fd6583b990c06'), ('909486151760777216_0', 'data_image/hurricane_irma/17_9_2017/909486151760777216_0.jpg', 'train', 'a8627cd6754b8951d4d9124f04b2f9b0d6c370799561cd29db233c2324119656'), ('910084566328766464_0', 'data_image/hurricane_irma/19_9_2017/910084566328766464_0.jpg', 'train', '657ee7d36d419340f2c8f55de61d52bb8751e1e39fb3f45d5edebfa6e86b47ba'), ('910206222204248064_0', 'data_image/hurricane_irma/19_9_2017/910206222204248064_0.jpg', 'test', 'e2f703f930788c4c8ed479717a090359d9d1318cc0105a0688b96b3447a4bfb2'), ('909466733173465088_0', 'data_image/hurricane_irma/17_9_2017/909466733173465088_0.jpg', 'dev', '41d64a62d2e2586eea7c34d4912da5174dacfed20c729ba07cf6fb6a5d281898'), ('910212336522350592_0', 'data_image/hurricane_irma/19_9_2017/910212336522350592_0.jpg', 'dev', '60e21a707c8aee274f761c3c5a348ee375e80058d370108e7e113b0a77976fbb'), ('910193063552745472_0', 'data_image/hurricane_irma/19_9_2017/910193063552745472_0.jpg', 'train', 'b2366a7583317cdc83c1ae0eff28fcedd6b44daf4b6a4dea3b274e0de50f5a8c'), ('909904173621182464_0', 'data_image/hurricane_irma/18_9_2017/909904173621182464_0.jpg', 'test', '8bfb9187c56fbeb3a5af12ff6b10eb01f78d95778aa338bc96c863d6df259b5c'), ('910121933315944448_0', 'data_image/hurricane_irma/19_9_2017/910121933315944448_0.png', 'train', 'd0f90a8814396e25f9499e100ee373a14ced15eebbbf0bceed67ab2b6f2816fa'), ('910099384913862656_0', 'data_image/hurricane_irma/19_9_2017/910099384913862656_0.jpg', 'dev', '0c614bd97a7d12f2dc42ac160b5989c578bb5a9e6e696a1b68c51ee7f1e7cba1'), ('909865332927590402_0', 'data_image/hurricane_irma/18_9_2017/909865332927590402_0.jpg', 'train', 'c068e1d212f4e6b3162e4c6b45732b60e757b40db12a209f6f5fc8f8e708dbd8'), ('909930121645182979_0', 'data_image/hurricane_irma/19_9_2017/909930121645182979_0.jpg', 'test', 'db30be0332a20ed1187af214f574e2ab96505e6eda88f366446bbdedd6c45438'), ('910232317901361153_0', 'data_image/hurricane_irma/19_9_2017/910232317901361153_0.jpg', 'train', '80f77f72b2232db3311e26b7fbc4257f03cf49478dee21dfcb38cdd5b0ff78de'), ('909423458810871809_0', 'data_image/hurricane_irma/17_9_2017/909423458810871809_0.jpg', 'dev', 'cc16caba9002239693a136ffeb7439664fc5f7ed486c918336874e34e415440c'), ('910199425972211713_0', 'data_image/hurricane_irma/19_9_2017/910199425972211713_0.jpg', 'test', 'ea40253125e01f3015b096752a541d9830920ad9feb8acbc0fb057001baf6407'), ('909741634287411203_0', 'data_image/hurricane_irma/18_9_2017/909741634287411203_0.jpg', 'train', 'a4063d2c7e2c2843a616cb83570fd0cbc4c6866f004136932359675b9efc50c1'), ('909401980266860544_0', 'data_image/hurricane_irma/17_9_2017/909401980266860544_0.jpg', 'train', '8371d39c835a55a8ebad30ed4745afcfcf2537dcdff0c9dd120c9bd5829c7c39'), ('909853195064823808_0', 'data_image/hurricane_irma/18_9_2017/909853195064823808_0.jpg', 'dev', '4ca625736be37771ae3ee6222812a4e4d6b26bd84db562059ea26ebf17e9751d'), ('910074634237771776_0', 'data_image/hurricane_irma/19_9_2017/910074634237771776_0.jpg', 'dev', '5db8734906cd49ab261cac8ff283a4d6cba8faafbf8af23989d79764a68063fa'), ('909868133271375873_0', 'data_image/hurricane_irma/18_9_2017/909868133271375873_0.jpg', 'train', 'b8b19a76d88d5fdf1a910b2ec145f0b439783571d82038c0cba3fdf69e371253'), ('910247140441632768_0', 'data_image/hurricane_irma/19_9_2017/910247140441632768_0.jpg', 'test', '0b026b6b84e030054363029c4d532742cc8fb3a7b5ccbdfb38c23ccf0fb6e8ef'), ('910096270609534977_0', 'data_image/hurricane_irma/19_9_2017/910096270609534977_0.jpg', 'train', '45eec95de59c47c7397a9124bd77067ec25907225788528da47f50eb11dda925'), ('909433639540281344_0', 'data_image/hurricane_irma/17_9_2017/909433639540281344_0.jpg', 'train', '76b06bc4544b66686b7e2d5fa6d32dfb0171e37c0f0dac4e984abfe3607e304b'), ('910078085986422785_0', 'data_image/hurricane_irma/19_9_2017/910078085986422785_0.jpg', 'test', '107b3fea18859df0aadeb564aba18df97e64d57516b2af0777b0e5124e982f95'), ('910113251249344512_0', 'data_image/hurricane_irma/19_9_2017/910113251249344512_0.jpg', 'test', '8b9b534463be6f8de41551edd7edb4e1c3372ffe778ac31376e914c13f49731b'), ('909822619276296192_0', 'data_image/hurricane_irma/18_9_2017/909822619276296192_0.jpg', 'test', '499b0eeeb215487e9ca9a885771475b66c2181704fd5ce0724b2cb6ae724802e'), ('909446703237009408_0', 'data_image/hurricane_irma/17_9_2017/909446703237009408_0.jpg', 'train', '27b98fdc7df158d3994f222180463feab1445339c6515553b88a82542c0df91c'), ('910261122187583490_0', 'data_image/hurricane_irma/19_9_2017/910261122187583490_0.jpg', 'test', '3c8fb117657670e978f054a6572273355c23587553f363755cc725450c4ffb9c'), ('909450713377501185_0', 'data_image/hurricane_irma/17_9_2017/909450713377501185_0.jpg', 'dev', '0b82c12d27ec2ba4b98b45b7d1b28a5d53b0aeb373cd8cc20277b1429bca6a25'), ('905745217898283008_0', 'data_image/hurricane_irma/7_9_2017/905745217898283008_0.jpg', 'train', '7adc0bb17ed988d898c554993a900ff5e83ae0aa088e0012f8e2558d0a645575'), ('909398796928303104_0', 'data_image/hurricane_irma/17_9_2017/909398796928303104_0.jpg', 'train', '01d107aa50e993884e737f30460ef607ed6d513d4a46817b220884b7f23b80ae'), ('922560865144012803_1', 'data_image/hurricane_maria/23_10_2017/922560865144012803_1.jpg', 'test', 'dda7d82fc28586c74b48e0c727c4036ab8bf1a29c8241a8dbc6b797c108009fd'), ('914437087591616512_0', 'data_image/hurricane_maria/1_10_2017/914437087591616512_0.jpg', 'dev', 'dd355683dc59618b398fdd8bd81f244dfab1df39e6d053009b739363dea41d61'), ('912991553907458048_0', 'data_image/hurricane_maria/27_9_2017/912991553907458048_0.jpg', 'test', '6e0d01a0f3224a79f9104fda3be5b7fe3a2c82767cd15636271ff11041ae64c3'), ('921507199674998791_1', 'data_image/hurricane_maria/20_10_2017/921507199674998791_1.jpg', 'train', '84b5bc5ff5c42d6f781ae088faa771e9b442aadb5c155fda5d737e2068cdb823'), ('923133826171154432_0', 'data_image/hurricane_maria/25_10_2017/923133826171154432_0.jpg', 'test', 'c562998373672290ccb306b198e820711a05688f9225367259c913e345a9ee6b'), ('922734674832756737_0', 'data_image/hurricane_maria/24_10_2017/922734674832756737_0.jpg', 'dev', 'd0482afa2c601c258bb3b31e3bd5ea533897030aaf6fa04f0570d036a22d9b44'), ('914329492830789634_1', 'data_image/hurricane_maria/1_10_2017/914329492830789634_1.jpg', 'train', 'cc56e5f40ee3f25c3c8ce490f60904ccd926b112dd42bceb4d1bbfcf2efbdfa7'), ('922588041952182272_2', 'data_image/hurricane_maria/23_10_2017/922588041952182272_2.jpg', 'test', '6bb3ea5ed2e20f0f7e1bbceaebdc2b2a223ce28898f6184db7cc6e62b45a1d5e'), ('912411531970793472_0', 'data_image/hurricane_maria/25_9_2017/912411531970793472_0.jpg', 'train', '74e7940e85c9f9a31d7db46467735e5e809821f684a2ae2cc566b51ef5859919'), ('912075698617102340_2', 'data_image/hurricane_maria/24_9_2017/912075698617102340_2.jpg', 'train', 'c4d9bf44f5066e8563240da472e04679ed04fe5a2f67163f0f62152576de8c6f'), ('913152910065897472_0', 'data_image/hurricane_maria/27_9_2017/913152910065897472_0.jpg', 'dev', 'd7f3b01b71124cb78b429e07facd42e88410135b4c33b4568396baebbcc67eeb'), ('922910387653267461_0', 'data_image/hurricane_maria/24_10_2017/922910387653267461_0.jpg', 'test', '284ec93afbc10126a6649573bad5d1d24fd6957173ad8fce0ca87a70ec09e90f'), ('912135083028393984_0', 'data_image/hurricane_maria/25_9_2017/912135083028393984_0.jpg', 'train', '8b88c4932cfff9f8b6946d77dae73b3eee3e2b768c2885864b062c4893c48b95'), ('912078014955114496_0', 'data_image/hurricane_maria/24_9_2017/912078014955114496_0.jpg', 'test', '35b16fbf560dc436b5d60b71ee8e68b3252f6941e7b386f46f19a6542346ee4e'), ('921984809039212544_0', 'data_image/hurricane_maria/22_10_2017/921984809039212544_0.jpg', 'test', 'e67258c13b0a88aab6ffa7f04729d0d40d8085622e570e03778dca6f1d40d855'), ('922093575457951744_0', 'data_image/hurricane_maria/22_10_2017/922093575457951744_0.jpg', 'test', '4df3b397d7989f8083317c34e4130a2e7d79c82efda7c2fe6d8e971df5f5b1ae'), ('913474174580084737_0', 'data_image/hurricane_maria/28_9_2017/913474174580084737_0.jpg', 'train', '1bdd13d9cbfd534ced8386d3bd1f4147df04102a91bb5ef9db261cfaf38e5ae2'), ('923545034691145728_0', 'data_image/hurricane_maria/26_10_2017/923545034691145728_0.jpg', 'dev', '7f0061b8eb66689f3c838b52f54ee21109c86ed37d7f956f70f4aaec30d118bd'), ('914353354503069696_1', 'data_image/hurricane_maria/1_10_2017/914353354503069696_1.jpg', 'dev', '7bc68269d2d0fc882c3fdc442116ec5efc1a90dd2c9f79968072761393ed1125'), ('923562484870582272_0', 'data_image/hurricane_maria/26_10_2017/923562484870582272_0.jpg', 'test', 'cead8e76279a95777cb0cce329d7260a3df1b0778cad08f632a19bb330feccb9'), ('912022246008541191_0', 'data_image/hurricane_maria/24_9_2017/912022246008541191_0.jpg', 'train', '89983886c840b1f3328842bf8021819299457071aad2f7db6983b82734a6b348'), ('911943558399090688_0', 'data_image/hurricane_maria/24_9_2017/911943558399090688_0.jpg', 'test', 'aa33982ad51c1d8dd94553fd6aa3c6a97d7254d0eee9bdd1d4902d5439e8c575'), ('912022246008541191_1', 'data_image/hurricane_maria/24_9_2017/912022246008541191_1.jpg', 'train', '95db2f4da6cf4d0beeadb808b488869db665748456c59933908d77e5ec8e6276'), ('922929746169569280_0', 'data_image/hurricane_maria/24_10_2017/922929746169569280_0.jpg', 'dev', '98d6f193ee756a95d83dfecc17c18456eeb1b66c4be283f1e0e68534300302c7'), ('921379555666989056_0', 'data_image/hurricane_maria/20_10_2017/921379555666989056_0.jpg', 'train', '917ee972a411f95a20bfe45f4be05441505feace276f6180ce3c7b3d74e3ad03'), ('922969963035615233_0', 'data_image/hurricane_maria/24_10_2017/922969963035615233_0.jpg', 'test', '701b6270254f155d7d2a5d192b85045bc02cfa78b414dc262a39b5c76bd541a2'), ('915319640145911809_2', 'data_image/hurricane_maria/3_10_2017/915319640145911809_2.jpg', 'test', 'c614182ca3ee26a7f8a27b12932e535376227aa837fbaf1bf9b3b01e5c9dc54b'), ('914148708559630336_0', 'data_image/hurricane_maria/30_9_2017/914148708559630336_0.jpg', 'dev', '303851d847f361b70d156ce97e1460697d6cb5b7e73ffbd82506b120873af33f'), ('910564018234019840_0', 'data_image/hurricane_maria/20_9_2017/910564018234019840_0.jpg', 'train', '9dffa1fda5dfbbed663740241137c9326055ff333317559d2e08ac93d2a8c6ef'), ('921358561699082240_0', 'data_image/hurricane_maria/20_10_2017/921358561699082240_0.jpg', 'train', '817c7c4ac5c509ffc846c44916057bc410c1e4e7a959450df24f68b8067ce453'), ('923101448279175170_0', 'data_image/hurricane_maria/25_10_2017/923101448279175170_0.jpg', 'dev', '776904a68d4567e7977f2c7bf2c4550af65fa1243ebdf70375a310e016fc34ba'), ('915191406112763904_0', 'data_image/hurricane_maria/3_10_2017/915191406112763904_0.jpg', 'test', '73d604732d0e27b2a330db96ab195d1f3f8206756019ac225c44b89424a04f74'), ('911602843882098688_0', 'data_image/hurricane_maria/23_9_2017/911602843882098688_0.jpg', 'test', '45c77f96917e3bc9d7665003c16cc9b42136ea9b6c50e4af5e48351f10e1e1b5'), ('913014761654284289_0', 'data_image/hurricane_maria/27_9_2017/913014761654284289_0.jpg', 'train', 'f4cd08c78a0a866f4a49b3f33f11f0ba29602b5cfdf0eed99cf1c729407b073f'), ('916029479109251073_0', 'data_image/hurricane_maria/5_10_2017/916029479109251073_0.jpg', 'dev', 'f108f76bf3c9c12f6e2f5972f5c6314dbb1748a93deab9ae49a51f8771e55ec6'), ('912944540226609152_0', 'data_image/hurricane_maria/27_9_2017/912944540226609152_0.jpg', 'dev', '53b785d1c612724d87ab1b7abed4e883f8acd732317f887da69c2e3f4b135028'), ('910701367550992386_1', 'data_image/hurricane_maria/21_9_2017/910701367550992386_1.jpg', 'train', 'eac7eef1c8370e229420686f7d33cf0b2002f2fd8a771a0293b59299d8304d3b'), ('930308604200026112_2', 'data_image/iraq_iran_earthquake/14_11_2017/930308604200026112_2.jpg', 'test', '7a07b380a45855dae952b5ad084b491ba10acd2fe350a66beedc3a4fce5cee34'), ('932027639589507072_0', 'data_image/iraq_iran_earthquake/18_11_2017/932027639589507072_0.jpg', 'train', '1aef9a4e939ada1a2854f142a3183b1a12a1b10014b17bdfd996c41942d066c2'), ('930046898848722944_0', 'data_image/iraq_iran_earthquake/13_11_2017/930046898848722944_0.jpg', 'train', '64c4acd8f19c2a65b9909b7dc890646dcd9a80eb7799709e1ce2f91fdf37102e'), ('930984789724741634_0', 'data_image/iraq_iran_earthquake/16_11_2017/930984789724741634_0.jpg', 'test', '1266719e25812e3403ea88f27c49a7e8c3a4ed30b7be09948e43457db4f4018e'), ('929999388713332737_0', 'data_image/iraq_iran_earthquake/13_11_2017/929999388713332737_0.jpg', 'test', 'b3c5ccedbe8e79483a9687b399acd3c20668d8403f348a3056e8b819f3431be7'), ('930665371153174528_0', 'data_image/iraq_iran_earthquake/15_11_2017/930665371153174528_0.jpg', 'test', '30713d393957669e898a4c8dc5346799f7bae68437018274ccf7d7802ae1ae97'), ('930347490263666688_3', 'data_image/iraq_iran_earthquake/14_11_2017/930347490263666688_3.jpg', 'train', '870bf7400eefde75e8c292c6b72491e4dbee5610cbc922d8903d90aa10d6e330'), ('930139822718468097_0', 'data_image/iraq_iran_earthquake/13_11_2017/930139822718468097_0.jpg', 'test', 'd40842b83d49637d23139530cd07d83dfbe55c9c93ac66f6e6988ae1d4940498'), ('930812113974714368_0', 'data_image/iraq_iran_earthquake/15_11_2017/930812113974714368_0.jpg', 'test', '9ea6f6fd1cffe177fcdcc28b6a3bf1e6ef4e36c3f930d5312f803448be792af5'), ('929989477828911104_0', 'data_image/iraq_iran_earthquake/13_11_2017/929989477828911104_0.jpg', 'dev', 'c20aeadb687dd4a2b3127378076564ac1e6a549e1d6586dcc570636a095a2ea1'), ('930094196064751616_0', 'data_image/iraq_iran_earthquake/13_11_2017/930094196064751616_0.jpg', 'dev', '33ec8ac2b5f2698ecedc728980220e43a7cd27567304e544f0d24b1f753af40f'), ('930133559033053184_0', 'data_image/iraq_iran_earthquake/13_11_2017/930133559033053184_0.jpg', 'train', 'b796fa7f8c5d4048dfc3c4153a364721da7a09ad332bf2ee2bc09c0389dcacfe'), ('930003223800360960_0', 'data_image/iraq_iran_earthquake/13_11_2017/930003223800360960_0.jpg', 'dev', 'c443a65e400d2f1c9d3cedbdd45d0dbb17f15b771faf0f5b11a23918b8292a02'), ('930290541211930624_0', 'data_image/iraq_iran_earthquake/14_11_2017/930290541211930624_0.jpg', 'test', '6a6e8474de0e71a4bea9429ea1da3ed94236266689ad231dbf9a4337f5f694cc'), ('930130379478093835_0', 'data_image/iraq_iran_earthquake/13_11_2017/930130379478093835_0.jpg', 'train', 'da850a45a2ec5d2b692befbce0c2514353efb46111da7c43b285604252f7fc85'), ('930046745974886401_0', 'data_image/iraq_iran_earthquake/13_11_2017/930046745974886401_0.jpg', 'test', '0acee03cad1462d80b80802582c456fbf0506799c677bddd2b7eab998d50b9cb'), ('930049177261887488_0', 'data_image/iraq_iran_earthquake/13_11_2017/930049177261887488_0.jpg', 'dev', '8bf6692924f2b9a2ed002caf353b5a2f57423d29559b55a242e3fb9152afc7be'), ('930523863691288576_2', 'data_image/iraq_iran_earthquake/14_11_2017/930523863691288576_2.jpg', 'train', '0f5aa02f397e790bae80f4fba697777545755a68ac9b6251c9cdebefa70b2108'), ('931608926080159745_0', 'data_image/iraq_iran_earthquake/17_11_2017/931608926080159745_0.jpg', 'train', 'ad6cbe92372a64bd1736691dc4efca6840a6e6f84682873004b6e7c389b5abcb'), ('930501165887959041_0', 'data_image/iraq_iran_earthquake/14_11_2017/930501165887959041_0.jpg', 'train', '178918f8e1ceb246c57c11dd8e612ce8f3b320237da265af53e957568c6676b7'), ('910659107517952000_0', 'data_image/mexico_earthquake/21_9_2017/910659107517952000_0.jpg', 'test', 'fd9d3d5deba8fc123889b7459133b43453986239904d78c10e62d2f365bb5892'), ('910760797487251456_0', 'data_image/mexico_earthquake/21_9_2017/910760797487251456_0.jpg', 'test', '3c9cc845ded7c8017d371acef580c430ff09af474187c57e3eeb263af6ebd1fd'), ('910677538292273152_0', 'data_image/mexico_earthquake/21_9_2017/910677538292273152_0.jpg', 'dev', '2e7737d464c8b88f45aaf87469c9066355bbf2e7c7ad3646170df06bf46b7a8e'), ('910691589034409984_0', 'data_image/mexico_earthquake/21_9_2017/910691589034409984_0.jpg', 'test', '5b08599be94263506b3e53224d1053559fc9eed90ceead9b4e0dcf6e42827d2b'), ('910679991599460352_0', 'data_image/mexico_earthquake/21_9_2017/910679991599460352_0.jpg', 'test', '0c980d3ab668d34d101c652cf047fc07efcd20513f753144fd84e60f40dc4763'), ('910609555821858816_0', 'data_image/mexico_earthquake/20_9_2017/910609555821858816_0.jpg', 'test', '904f1a4965b06990fbaa114f166019c7d2c23554aa8bfd0c41438e2d43f11ada'), ('914159056201424896_0', 'data_image/mexico_earthquake/30_9_2017/914159056201424896_0.jpg', 'train', '26c45819b3857e20569df071cdf46f57ba5fb12537698719736245e423579192'), ('910555424750833664_0', 'data_image/mexico_earthquake/20_9_2017/910555424750833664_0.jpg', 'dev', '08e5c65ccec2d1ab83a9c68af46d52e8ffceddf808ce42363f402cae36b32d09'), ('910563191700185088_0', 'data_image/mexico_earthquake/20_9_2017/910563191700185088_0.jpg', 'train', '1e8c5ccccc088a2f27fc2e13937a2d621697e4da33e06ce94a793c25185096a6'), ('910787443871293442_0', 'data_image/mexico_earthquake/21_9_2017/910787443871293442_0.jpg', 'train', 'e07fc7184bf74518969693cc121025ee2104d7bae21fbb8e220c883a9dd085f7'), ('913772923772719105_0', 'data_image/mexico_earthquake/29_9_2017/913772923772719105_0.jpg', 'dev', '5daeec01e2418e5a91adfffe1f8e07fb40d65f8a42925c1f4b1f187a5087ca8b'), ('913176220254138370_0', 'data_image/mexico_earthquake/27_9_2017/913176220254138370_0.jpg', 'train', 'a7d662cf8e7109c73ec55974c276e60d19331fdca95dec64fda6c9e0983ea800'), ('912186377885716481_0', 'data_image/mexico_earthquake/25_9_2017/912186377885716481_0.jpg', 'train', 'fd9d3d5deba8fc123889b7459133b43453986239904d78c10e62d2f365bb5892'), ('913964666229051393_0', 'data_image/mexico_earthquake/30_9_2017/913964666229051393_0.jpg', 'dev', 'dd610c482e6e8a3ecc3eaf60f4070810bb0b7a9078c33ee9247c7dbe4be52cf8'), ('912184177071620096_3', 'data_image/mexico_earthquake/25_9_2017/912184177071620096_3.jpg', 'train', '04a08e28102e71d6f4c4427c4e0ff3434d9d716860ce1dd9e99af93a3275a484'), ('910523983149252608_0', 'data_image/mexico_earthquake/20_9_2017/910523983149252608_0.jpg', 'dev', '245a19fda7c7e1a203cb4cf441c0e9e232d30b5d5d39128b1ffc8792d96ff9d0'), ('910555154599964673_0', 'data_image/mexico_earthquake/20_9_2017/910555154599964673_0.jpg', 'test', '217ce51081903d41d3a0ffb28d4453bb17c74756f14ad1bc7d748060371c4964'), ('910567699880841217_0', 'data_image/mexico_earthquake/20_9_2017/910567699880841217_0.jpg', 'train', '2c11a52d12479162837d586f445a62771cc0757c75f1adf1ce3e8a984f795911'), ('913818159064416256_0', 'data_image/mexico_earthquake/29_9_2017/913818159064416256_0.jpg', 'train', '3e059a6762bd14a9a4d19c76e21dc2f64904b6049b4d18fe9d6bd6f049bc58bc'), ('910653901539594240_0', 'data_image/mexico_earthquake/20_9_2017/910653901539594240_0.jpg', 'test', 'fd9d3d5deba8fc123889b7459133b43453986239904d78c10e62d2f365bb5892'), ('910555308178722816_0', 'data_image/mexico_earthquake/20_9_2017/910555308178722816_0.jpg', 'train', '032c33e8f18f6b0fe9fbecd7f6b6e605b4a89697b5d412a08c3d218a27ccd72c'), ('911682035269414912_0', 'data_image/mexico_earthquake/23_9_2017/911682035269414912_0.jpg', 'train', '67df91460277f9758bda35cd86267018ae9021d500521a013052f2f7a2eb3a2e'), ('913389115281772545_0', 'data_image/mexico_earthquake/28_9_2017/913389115281772545_0.jpg', 'train', '94597ca51675989f5d5a308187ec2c14cc7bd33f7a7978bb016adbc3a13da470'), ('872816096654508033_0', 'data_image/srilanka_floods/8_6_2017/872816096654508033_0.jpg', 'train', 'd7195a8c5a35a8fb140c661f9199b89dcbcc7d1eb1b2558c7969504c26ed3b95'), ('870294525419192324_0', 'data_image/srilanka_floods/1_6_2017/870294525419192324_0.jpg', 'train', 'bef930fe9e818dee6670d922886c9b5649f94ac15ccd4bfc6df62c84b2536140'), ('870052558928138240_0', 'data_image/srilanka_floods/31_5_2017/870052558928138240_0.jpg', 'train', '9756163c738d7ee88064ae1de8ad71c258c6d299c28fafd86c0f69e2c989ba87'), ('870182877706715139_2', 'data_image/srilanka_floods/1_6_2017/870182877706715139_2.jpg', 'test', '1eb0865d5c2ae7b2fda7ce9b0282012c4b54501f0a5ac478daa6d72abc541510'), ('870274062588805122_0', 'data_image/srilanka_floods/1_6_2017/870274062588805122_0.jpg', 'train', '749170b9eb73aabfc44be3c7308da0593d4c5e23565b24a8fd5e9e8ec33acc40'), ('870033439214510083_0', 'data_image/srilanka_floods/31_5_2017/870033439214510083_0.jpg', 'dev', '1c1e7ab6ec0ab8cfb3977a6cd62fe557362ab8e3f45be57512f639a9e8429dd5'), ('870003422514208768_0', 'data_image/srilanka_floods/31_5_2017/870003422514208768_0.jpg', 'train', 'd1b2b829861e20bfbe2a7b17ba0e0a8f9d7a4e4046ba5a76dd920c5a4fd2d8a4'), ('870559283880112128_1', 'data_image/srilanka_floods/2_6_2017/870559283880112128_1.jpg', 'test', '7f02a95f69f1acf47b151b104b89e07ef906f2868ee24d0c1a022e841aa8cd64'), ('870215735687225344_0', 'data_image/srilanka_floods/1_6_2017/870215735687225344_0.jpg', 'dev', '3cc5a7801f177eec6e6730c1ffd5b0c6183580dadf32d4839f6c42cc5c8c24eb'), ('875116643453632512_0', 'data_image/srilanka_floods/14_6_2017/875116643453632512_0.jpg', 'dev', 'ac4d46f08175b55b2837872556294ee96d96cbb6c508ffcf78d57bc8f97849e4'), ('875116643453632512_1', 'data_image/srilanka_floods/14_6_2017/875116643453632512_1.jpg', 'dev', '32f181e0c78a495488a3740be4465ad24548ccae50ed6850d2bc437b22df49cf'), ('872022117574770689_0', 'data_image/srilanka_floods/6_6_2017/872022117574770689_0.jpg', 'train', 'e93c07e5610d01fcefe628bdcb81ffbfb6eda46709b30550771142b44781b939'), ('875770493336473602_0', 'data_image/srilanka_floods/16_6_2017/875770493336473602_0.jpg', 'train', 'dba8a3d08281a836cc8c0341295ef0eee4e1829a306f959f016a59a63cfeddb4'), ('870349300185706496_0', 'data_image/srilanka_floods/1_6_2017/870349300185706496_0.jpg', 'dev', '5edbbc49c13e9936be77ce755a93f752e0602867bc30f0ed7d164eca57d4c06f'), ('869999741165109248_0', 'data_image/srilanka_floods/31_5_2017/869999741165109248_0.jpg', 'train', '8c4ca721a9d137159f6b58606d187b4648013d20b55bf7db31f79fa5da4d37b5'), ('870855346516111361_0', 'data_image/srilanka_floods/3_6_2017/870855346516111361_0.jpg', 'train', '02f2164f00b302e429390d1dac58aabb6a6b8623fbcb0f6a78bb24a48b7c1b42'), ('870223234855075841_0', 'data_image/srilanka_floods/1_6_2017/870223234855075841_0.jpg', 'dev', '48cbedff4b453cdfd53bcacc55d08d30d583447843135ecf849ebd1660c12289'), ('870347744912711680_0', 'data_image/srilanka_floods/1_6_2017/870347744912711680_0.jpg', 'dev', '9abb06ba9ce52d796d88dae3107b81d123eb87d6835fe8f002ff2134d3cb9a46'), ('870283105151668224_1', 'data_image/srilanka_floods/1_6_2017/870283105151668224_1.jpg', 'dev', 'be638a9cc1eb9d716a8357352648f057666995241c1e2a4902004e98f5fdf0a0'), ('870011540895748096_0', 'data_image/srilanka_floods/31_5_2017/870011540895748096_0.jpg', 'train', '87a477211d2dbedc5990f12b1afb298c094bb1da9553ad5735c44561a22e0a04'), ('871998630718263296_0', 'data_image/srilanka_floods/6_6_2017/871998630718263296_0.jpg', 'test', 'e119ac8099b2207743adf43c9d7a9c907f955773d181c1ebad6d189ade87495a'), ('870288120150765568_0', 'data_image/srilanka_floods/1_6_2017/870288120150765568_0.jpg', 'dev', '358136360fffe142d7239115e39aca7be388ee07fcb00d3d407a5fad6d8792fa'), ('917974072096645122_0', 'data_image/california_wildfires/11_10_2017/917974072096645122_0.jpg', 'test', '6541ca5a5e62f6437479cc68bd2f9592b326b804a1fe8f0cf1e720f02448fefb'), ('919734115364147205_0', 'data_image/california_wildfires/16_10_2017/919734115364147205_0.jpg', 'train', 'ad04c4f63b37365d17bc1e96441af6ad0c27be4fccc76c49c5fa9f0f067d4d95'), ('920040637600337926_0', 'data_image/california_wildfires/16_10_2017/920040637600337926_0.jpg', 'train', 'e10e028b6e284762cb257b66ae2b0aec3e3387c17412f4ca83af9cd856b10987'), ('919987723670233088_0', 'data_image/california_wildfires/16_10_2017/919987723670233088_0.jpg', 'test', 'bcec3c03fea3ee44dcae83d05450992b5a83d81f69e08d025dea13edaea0032b'), ('917815040962695168_2', 'data_image/california_wildfires/10_10_2017/917815040962695168_2.jpg', 'test', '9e5d4a694e70ad6023be0818fd132902568913bcef9a38f2c01f87adefca5ec0'), ('921442451356835845_0', 'data_image/california_wildfires/20_10_2017/921442451356835845_0.jpg', 'train', '5d6ff4b94ce394a42f953a523817a7989941a01f6b63817b43554b26dd7a1420'), ('917871713823502336_0', 'data_image/california_wildfires/10_10_2017/917871713823502336_0.jpg', 'test', '53b4bf76fd152ebd7f3b3bb536da6c3ccdea58cc9f5098a47e5c49d6b0096645'), ('919652631168958464_0', 'data_image/california_wildfires/15_10_2017/919652631168958464_0.jpg', 'train', '86025d49d774a4be086021ac3399bad2c42d87fd79f38f73aa1b807ad747331f'), ('917924741125509121_0', 'data_image/california_wildfires/11_10_2017/917924741125509121_0.jpg', 'test', 'eff7ff72c809085415b3b66c126672199baa10df6858ce868b234af4723b5d79'), ('917820560763183106_0', 'data_image/california_wildfires/10_10_2017/917820560763183106_0.jpg', 'test', 'c812392847c815d3f5acaf6206a60648065e1acd1b796a77aa1ef99e95b143a3'), ('919952400017969152_0', 'data_image/california_wildfires/16_10_2017/919952400017969152_0.jpg', 'test', '10a1d99128731b6fce98a6c683263585b3286654c3d6e748c1a6bedb4877f9db'), ('919776289879453701_0', 'data_image/california_wildfires/16_10_2017/919776289879453701_0.jpg', 'test', '3b249b57037d65bfcac89e032543f9ea2e71bf09bcd3075dcb775b3a8ef99ce8'), ('919653166974750720_0', 'data_image/california_wildfires/15_10_2017/919653166974750720_0.jpg', 'train', 'ea8f7feb36d43de572a8321b209a14dcbfeb840bbb7243547398e23f8368982e'), ('920535960772857856_0', 'data_image/california_wildfires/18_10_2017/920535960772857856_0.jpg', 'test', '94db4026962bf9cfb8130e5fdc4989c977cff0e29dd26163d2d79592a93719a7'), ('921052655375568898_0', 'data_image/california_wildfires/19_10_2017/921052655375568898_0.jpg', 'train', 'c9f1ef5548cc1385fca6194a5040e4d67585eab243e0dd782dcb4f7074360b13'), ('918009429584355328_0', 'data_image/california_wildfires/11_10_2017/918009429584355328_0.jpg', 'dev', '9a2288ffbcc569843627013775a7a7c974c54cf2f70b70627bec68455848fa0c'), ('919580937184260098_0', 'data_image/california_wildfires/15_10_2017/919580937184260098_0.jpg', 'train', '0e7762f6b526ceb366722ec12e7466f530b9c38d876c225980f1404d9f06d520'), ('919684475990130689_0', 'data_image/california_wildfires/15_10_2017/919684475990130689_0.jpg', 'dev', '852050745cabe4885dd4506d295c144711a6c77046cfb20e77b63b407d0f50f8'), ('917844496137601024_0', 'data_image/california_wildfires/10_10_2017/917844496137601024_0.jpg', 'test', '7727b7a91bc5d79b166b141e7d95f50e5e2c86fc846bc4dc07697ef32b75dc40'), ('920765735218978816_0', 'data_image/california_wildfires/18_10_2017/920765735218978816_0.jpg', 'test', '242d254ec04a94dac27a6d791f91a9fdd6d4154071ee7ba0763d281c9ab53e74'), ('919565905872474112_0', 'data_image/california_wildfires/15_10_2017/919565905872474112_0.jpg', 'train', '9400b9b2ad6170ccf80bf5483968e014d82587a196be465bbb84f745ad3ff316'), ('917915442085163008_0', 'data_image/california_wildfires/11_10_2017/917915442085163008_0.jpg', 'dev', '62ac979ddac3cd19532cffdf66009802186d23fbc22a518f655ce8d9865100eb'), ('919888686707175424_0', 'data_image/california_wildfires/16_10_2017/919888686707175424_0.jpg', 'train', '25ecaeaf475a5d0256bd882a2f1c4da8aafeef93e6e4a6c4fccfde8565133b36'), ('917832248048496640_0', 'data_image/california_wildfires/10_10_2017/917832248048496640_0.jpg', 'test', 'b60c54ffc98706d569a064b6bc84277f7afab4f2d7014dec426d5b7d0d251a5a'), ('917834588814155776_0', 'data_image/california_wildfires/10_10_2017/917834588814155776_0.jpg', 'train', '2f34f2a4b61383f2c84b026405a71d9fabc0c372c725fd8c4ef9bb19370f9586'), ('918353302970040321_0', 'data_image/california_wildfires/12_10_2017/918353302970040321_0.jpg', 'test', '38497f7384a596daf09ec5bc9cd24dea80beab138ba122fce75541a533cd71f9'), ('909259117801410560_0', 'data_image/hurricane_harvey/17_9_2017/909259117801410560_0.jpg', 'train', '2d3d84719a6b0578d7ab395dbff926058f53d69d9ce7271b2eac34ae0c088fbb'), ('904271772840841218_0', 'data_image/hurricane_harvey/3_9_2017/904271772840841218_0.jpg', 'dev', '2e216aab768d664bfb9077e740034e0bed44d405acd8c3611fe0c47806ce423f'), ('906023333044424708_0', 'data_image/hurricane_harvey/8_9_2017/906023333044424708_0.jpg', 'test', '081894e9d2f75037c4394dfc3faaa50d9cc571631e623b05ba8b142081e77614'), ('907241824896524288_0', 'data_image/hurricane_harvey/11_9_2017/907241824896524288_0.jpg', 'train', '6123b1940129baca75be91a252b65a1a418deafacdf56f7fb86565b55b266044'), ('904393070363291648_0', 'data_image/hurricane_harvey/3_9_2017/904393070363291648_0.jpg', 'train', 'f09efea2b15854f85876d146510ff14ff71fd88199f2dd044527481c71eeb3bc'), ('901741728520130560_0', 'data_image/hurricane_harvey/27_8_2017/901741728520130560_0.jpg', 'dev', '0ef3f1456cfa0cc5364f661901957d0b97e3d24fbab75b6e277385754984a454'), ('905904205738766336_0', 'data_image/hurricane_harvey/7_9_2017/905904205738766336_0.jpg', 'train', '84dc625a13093b69f718db8f0eb6a0ffec45b59fa8c5a9a53274815ea91a8b6b'), ('907677765293527041_0', 'data_image/hurricane_harvey/12_9_2017/907677765293527041_0.jpg', 'train', '51c1be18bd239be1454cdebc34833683fcbb31cc56a8806f3c65d6483038bb9e'), ('905846674400002048_0', 'data_image/hurricane_harvey/7_9_2017/905846674400002048_0.jpg', 'test', '232289d340128d2f5519b3653b33365860554d45f3de1628ab72d56581d2b6f3'), ('907442878460215296_0', 'data_image/hurricane_harvey/12_9_2017/907442878460215296_0.jpg', 'dev', '9cdc3e47c74682fa27807552e4aeeb2c457c99e345f3a8430f81e6e3b48ac508'), ('904318572708712448_0', 'data_image/hurricane_harvey/3_9_2017/904318572708712448_0.jpg', 'test', '290dbe4140ecf4e5245c1e44107aab4ee3bd58c1d057bbf91b0daa40d9b0fdb5'), ('905912324401266693_0', 'data_image/hurricane_harvey/7_9_2017/905912324401266693_0.jpg', 'test', 'bd6b9e8618f731d2716f53f754a675ffe016e3ffddbe37bd6db8157d9a665782'), ('901675294096060416_0', 'data_image/hurricane_harvey/27_8_2017/901675294096060416_0.jpg', 'test', 'ae0eba7e29f49111abf089a4a9efcfeaa56faa60b5c6c6ea0f09350c9e3051df'), ('901776462189137920_0', 'data_image/hurricane_harvey/27_8_2017/901776462189137920_0.jpg', 'dev', 'd57e37d9fc43f9574ce428233bbbba05f27ad5e90cf747d79630ebb00354a104'), ('901813742933073921_0', 'data_image/hurricane_harvey/27_8_2017/901813742933073921_0.jpg', 'dev', 'a9a10499ecb47922eb44f6935c8d8813cff56b238c1aeb21e0202d08eb320501'), ('906058581048479744_0', 'data_image/hurricane_harvey/8_9_2017/906058581048479744_0.jpg', 'test', '30ac96d2c1b2de48e689ca746d2dcd74b2fcd99e31214fdd3f90471e914e514a'), ('901726788736471040_0', 'data_image/hurricane_harvey/27_8_2017/901726788736471040_0.jpg', 'dev', 'b0711bd41062a6728a0ccef8be1f19fb4338b6a675b49ddcfb3700079f0d9e4f'), ('906174963945799680_0', 'data_image/hurricane_harvey/8_9_2017/906174963945799680_0.jpg', 'test', 'fbabccd7e30fbc21fb69ce0a592cb93225df1c4cd4bfb5e22c67ad9badbf19d6'), ('901796238756114433_0', 'data_image/hurricane_harvey/27_8_2017/901796238756114433_0.jpg', 'test', '15ed767529f72a005ef5a3f92ca4cc21acd0294c0e6753a2f4e60fd31af48135'), ('905894140579381248_0', 'data_image/hurricane_harvey/7_9_2017/905894140579381248_0.jpg', 'dev', '6c3ce8efda76524ec7c6b3d5a7dfef7da62aff9785c49b216a6b6b1350b654ae'), ('906141667853520896_0', 'data_image/hurricane_harvey/8_9_2017/906141667853520896_0.jpg', 'train', '8e42e606b72be1b3b7d7f6108330bfae2182ad967c387457dd73fe5625c83463'), ('901787869521555456_1', 'data_image/hurricane_harvey/27_8_2017/901787869521555456_1.jpg', 'dev', 'd881bd3ea557a79a88d76cac59f11161b01f67a35353e89262d9f47ea0f3f444'), ('907712998642536448_0', 'data_image/hurricane_harvey/12_9_2017/907712998642536448_0.jpg', 'test', '05d22f0ca4d1164b5dd761f17d30fd18815cedd3439ecc3179ee4b27b7052be8'), ('901646377620578304_0', 'data_image/hurricane_harvey/27_8_2017/901646377620578304_0.jpg', 'test', 'a36397b414b0d4ee5324eb0033432fff55abfe011ddf1b339df8971daaa045b4'), ('901701816810962944_0', 'data_image/hurricane_harvey/27_8_2017/901701816810962944_0.jpg', 'train', '8692311733608aa3c0217f85c57d1c116e1f39cb322d8fd4e66518d3b79aec8c'), ('906784627267981313_0', 'data_image/hurricane_harvey/10_9_2017/906784627267981313_0.jpg', 'dev', 'c27368b17da0c3b47aaad584189dcc0279b6f70e8c8c4428e023e79bd7c1568b'), ('904755214879678464_0', 'data_image/hurricane_harvey/4_9_2017/904755214879678464_0.jpg', 'train', '838c87a64e1b15d9dde2ad468bb8a16ed3e810aa943e0ede54f5db2d6e83fa24'), ('909214942137430017_0', 'data_image/hurricane_harvey/17_9_2017/909214942137430017_0.jpg', 'test', 'edef57300158194d66686764281cf5456a1d62c3ad6bcf73bc64d2f8fa5f8473'), ('905277948395184128_0', 'data_image/hurricane_harvey/6_9_2017/905277948395184128_0.jpg', 'train', '9058e0c6ccdd05b7b0d19fc37ab302a64e03a592b941542cd438b8dd210e992f'), ('905161654521913344_1', 'data_image/hurricane_harvey/5_9_2017/905161654521913344_1.jpg', 'train', '21631bd85a8e861976a05bd7136774e38eb3f733246cf9a23326111999293c89'), ('904692810057740289_0', 'data_image/hurricane_harvey/4_9_2017/904692810057740289_0.jpg', 'train', '87fc8bd675cc44b9a7cbdd10ec5e06af1daa72ec54c74e940294928bdcd28fa0'), ('904588625647689729_0', 'data_image/hurricane_harvey/4_9_2017/904588625647689729_0.jpg', 'dev', '3e6907dabced9f888b15a48edd75e5d846d826faa091ec2248c29a36dad8f430'), ('907447444388110337_0', 'data_image/hurricane_harvey/12_9_2017/907447444388110337_0.jpg', 'train', '9309584d9c31c8b562a81edddefe0355c9b62ad5b1c8e74d22ead16a34d812f0'), ('905325481632731138_0', 'data_image/hurricane_harvey/6_9_2017/905325481632731138_0.jpg', 'dev', 'be56268f764cf93d666a12475da1842181a6135a7883c22eb7014ae133ea8b29'), ('909436803853189121_0', 'data_image/hurricane_irma/17_9_2017/909436803853189121_0.jpg', 'dev', '1d2192b53e955a3d1bc69a6f409bbc5bba992c03735292facf0680cee09edf6a'), ('905724879600791554_0', 'data_image/hurricane_irma/7_9_2017/905724879600791554_0.jpg', 'dev', 'd778c4820eba3160a6371b4019ead7aaf935367de405479a26b0be5366b0786b'), ('910243544551628800_2', 'data_image/hurricane_irma/19_9_2017/910243544551628800_2.jpg', 'train', 'ca914a5cc0c8dfc738061294c1d21d5b63c9dda18587aaa717cb3307b74ffcfe'), ('909802252256157703_1', 'data_image/hurricane_irma/18_9_2017/909802252256157703_1.jpg', 'train', '16379a7707fd058ec45b4a1229f9538c0c5dfd4db1db27f0862cfa90ddae5272'), ('909458398466453504_0', 'data_image/hurricane_irma/17_9_2017/909458398466453504_0.jpg', 'dev', 'd4db010db841fe4b28c6dff4956067469a7fcb0d03797b46ec4b8a5c37ac75a8'), ('909833386927251456_0', 'data_image/hurricane_irma/18_9_2017/909833386927251456_0.jpg', 'train', 'cf443156ed187e28f9973456045b567356bb6a8859b8c9a20786f52c1de177f3'), ('909833411598209024_2', 'data_image/hurricane_irma/18_9_2017/909833411598209024_2.jpg', 'train', '78177521b8f6cc84df1008c7046223485e4189ab2caac503a8f048c2dbffdd67'), ('910166284624711681_0', 'data_image/hurricane_irma/19_9_2017/910166284624711681_0.jpg', 'train', '87df97336b8cfd5c45558697be88cd2c02bc484d47fd148f7bc4dbeb9a8e40f1'), ('910085063047606273_0', 'data_image/hurricane_irma/19_9_2017/910085063047606273_0.jpg', 'dev', '213cf253ea4d716710d6bbe27c0f89c207177406feb95857a5438dfa03d1f637'), ('905636901306163200_0', 'data_image/hurricane_irma/7_9_2017/905636901306163200_0.jpg', 'train', 'f06b2220ed79a292c98996001b9d79e9ced5832b9e38a4174991f1335f3401df'), ('910041602340536320_0', 'data_image/hurricane_irma/19_9_2017/910041602340536320_0.jpg', 'test', 'e0ace5a5d8ff2c3823da1e25a96797381d783e24ac4b7346fc569153608eaca5'), ('909752117740654592_0', 'data_image/hurricane_irma/18_9_2017/909752117740654592_0.jpg', 'test', '240a925a662db64f85c4c6e4625bf2847790b2d9a933355141863c4bb0a3a4c9'), ('909902508155392000_0', 'data_image/hurricane_irma/18_9_2017/909902508155392000_0.jpg', 'dev', '08b2a4f40904f5242ad3410eb77ab7ece2f7cef80be6aa3a48f5d4a4df47e9b9'), ('905711934879694848_0', 'data_image/hurricane_irma/7_9_2017/905711934879694848_0.jpg', 'train', '54b1e386c9f06f67c00d841247a9dac37d39f4d2d28f6d83f6908313f3c7ee66'), ('909743170841452544_0', 'data_image/hurricane_irma/18_9_2017/909743170841452544_0.jpg', 'train', 'dc59482599bea11f8ff08291737645066351adf0d3e44b73fd911336492ec4b4'), ('909967052349124608_0', 'data_image/hurricane_irma/19_9_2017/909967052349124608_0.jpg', 'train', 'a3823a1ed13070eca26749b4372329af20397911bb3c3d64f36818d9d0d6ecc0'), ('909386581080969216_0', 'data_image/hurricane_irma/17_9_2017/909386581080969216_0.jpg', 'test', 'b3e3ae7493b6e839fdda92caf12f282aafa6078353a4211067a065b06f695606'), ('910149200410791936_1', 'data_image/hurricane_irma/19_9_2017/910149200410791936_1.jpg', 'test', '87a2d0bd1f6861098071975f00543d12727c5537ab5ac2d96ad727c084bafba5'), ('909447517565325312_0', 'data_image/hurricane_irma/17_9_2017/909447517565325312_0.jpg', 'train', 'd915b0a6737b6ee2fc1dbc2a30b4f37f7353dfe6eea784c0347313b44e8609f3'), ('909779299753971715_0', 'data_image/hurricane_irma/18_9_2017/909779299753971715_0.jpg', 'train', 'fefbb58c1e179275b82f7b2939fa61a99a58733000d7d1a0320ba0a8b2b99866'), ('905669053879951361_1', 'data_image/hurricane_irma/7_9_2017/905669053879951361_1.jpg', 'train', 'cd131ed0cc12546fbacb3b417865f1f6c84f8c8bd96a8f0de274db6149e86e63'), ('909455979460382722_0', 'data_image/hurricane_irma/17_9_2017/909455979460382722_0.jpg', 'train', '3ee838f779e56e6d7605d62fa315dfc79413faba40be8ca22047e218c837c9c1'), ('906927897918820352_0', 'data_image/hurricane_irma/10_9_2017/906927897918820352_0.jpg', 'train', '5dedaa8b70b3ea6ec34b5fea4d1c9f01e65d10996b6d68557e5802ae1863562e'), ('909477370515648513_0', 'data_image/hurricane_irma/17_9_2017/909477370515648513_0.jpg', 'test', '9e484db90b46e1450309b147d44731a03be8085f4fd4b35da9e9a069d4930c33'), ('909924036779630592_1', 'data_image/hurricane_irma/18_9_2017/909924036779630592_1.jpg', 'dev', '068493816f3a9456c5393d0b4caff8f50359aac6087e1cd33fc46ad589abe77c'), ('909418648996999168_0', 'data_image/hurricane_irma/17_9_2017/909418648996999168_0.jpg', 'test', '3af3443a50116a9a3a0a7c3ee3fc1ee6e75c862cd17643487974ed958e9d1647'), ('909768110730809345_2', 'data_image/hurricane_irma/18_9_2017/909768110730809345_2.jpg', 'train', '28285d0b09249e53d66d22e3a58acb71fa2a217694bebd0fc3287b9e87bfe4f1'), ('905709792618041344_0', 'data_image/hurricane_irma/7_9_2017/905709792618041344_0.jpg', 'train', 'b77106ee5479df201d3d5a42578623f440701f8483910542869fdadf97bd1c55'), ('910159121822961664_0', 'data_image/hurricane_irma/19_9_2017/910159121822961664_0.jpg', 'dev', '9440d792d667eaa911296b984765aa134bb715683699607138eaca75e79e27aa'), ('910124040584667136_0', 'data_image/hurricane_irma/19_9_2017/910124040584667136_0.jpg', 'dev', '26a85639ceeedaf52bee5d39193c0eec64e4a7ab1fc10d43b9bee0b6ffe42d6c'), ('909766700853338112_0', 'data_image/hurricane_irma/18_9_2017/909766700853338112_0.jpg', 'dev', '4da33831acc8564424126ea866eb88a73e0d6a51fe7433afdd2c735c5bde8ded'), ('909844516886245376_0', 'data_image/hurricane_irma/18_9_2017/909844516886245376_0.jpg', 'test', 'b847af02e9f568fdfe765253e02aed91e6534bb9e6a0b364bd2fd168120885f3'), ('910156441490780160_2', 'data_image/hurricane_irma/19_9_2017/910156441490780160_2.jpg', 'train', '87508e0b18329f87989159d79eb96c838242e02263dfbe205942557c718b093d'), ('905647759050420225_0', 'data_image/hurricane_irma/7_9_2017/905647759050420225_0.jpg', 'train', 'e8190b4a2b7ae85e6a139fce3a1693056ee762a12435ab5a6e50c019faea6906'), ('905705793869283328_0', 'data_image/hurricane_irma/7_9_2017/905705793869283328_0.jpg', 'test', 'd6287722984106733279a2540f1dfebb820d03633424a2a672b85d334ef89a2a'), ('922560865144012803_2', 'data_image/hurricane_maria/23_10_2017/922560865144012803_2.jpg', 'test', '85c0d3718b838023d1f1c3191db16895122994ded16a31985e0a3af8eb03912a'), ('911486875201937408_0', 'data_image/hurricane_maria/23_9_2017/911486875201937408_0.jpg', 'dev', '4945a35d8da335d6d630b7c335209efa8689d8673cc19ace2728e0a785639460'), ('922579252704358405_0', 'data_image/hurricane_maria/23_10_2017/922579252704358405_0.jpg', 'train', '6b105be4bb0a9bb2264997191bcf4baaa684ba7701bfc497004a763e4a32919c'), ('927252238652887046_0', 'data_image/hurricane_maria/5_11_2017/927252238652887046_0.jpg', 'train', '77b59d1023a1607291350affcac077f9c31683dc52d06b8e38a2138d07250be2'), ('910704791550136320_1', 'data_image/hurricane_maria/21_9_2017/910704791550136320_1.jpg', 'train', '2571bd50e09316fc525b377b61e53b1ea72ef7707bf5711353e12d312e6192f9'), ('921385789635944448_0', 'data_image/hurricane_maria/20_10_2017/921385789635944448_0.jpg', 'train', 'f7081fad85853b905a024e9471e6ec1eaf3d63ba6d855eada1848a06e1dba63c'), ('914570237881745414_0', 'data_image/hurricane_maria/1_10_2017/914570237881745414_0.jpg', 'test', '3018f2875e5aaacef7cc2d0bab07351c7684f104675cf73afe8e3a198591c3b2'), ('913867524705538048_0', 'data_image/hurricane_maria/29_9_2017/913867524705538048_0.jpg', 'train', '9e944282c123e6e1700e5c337e6e095d98849d70cc90c7128c898f0689fe230e'), ('910785253580648448_3', 'data_image/hurricane_maria/21_9_2017/910785253580648448_3.jpg', 'dev', '919383e741143bc1d006edf3836855bb8f84e50d5dd72e7adc7e3f74e8dbf907'), ('910643329511170049_0', 'data_image/hurricane_maria/20_9_2017/910643329511170049_0.jpg', 'test', 'f1d37de027d63f87f99cc91f3d3f8f223764195c3b5e5fc2ed959ab860700f33'), ('914643349214306304_0', 'data_image/hurricane_maria/2_10_2017/914643349214306304_0.jpg', 'dev', '033e5c10d0333f2fd0d3cbe9e2fe0112f624fdd4ee5c3bd5d7a396be31e92f86'), ('910754701234274304_1', 'data_image/hurricane_maria/21_9_2017/910754701234274304_1.jpg', 'train', '9c45b21d67529aeb1ad2f09d17880d3c1232f21253ec462ffc5969ed06819465'), ('910582078470852608_0', 'data_image/hurricane_maria/20_9_2017/910582078470852608_0.jpg', 'dev', 'b78b1f6556d4d1fdaafdc46877c5940ebf06110363499d98f0f1711e0731a80a'), ('913102202423566338_0', 'data_image/hurricane_maria/27_9_2017/913102202423566338_0.jpg', 'train', '61b08efe51d76dc53e19fe7de0385aa09a502f3b6162ce3cea9959d6161714f9'), ('916012313219444737_0', 'data_image/hurricane_maria/5_10_2017/916012313219444737_0.jpg', 'test', 'b72b2d9057ad07f64d50a12538d227bc8560f9cec067b320e55b8a60890dd1de'), ('923466082358263808_0', 'data_image/hurricane_maria/26_10_2017/923466082358263808_0.jpg', 'dev', '395cfdba138faecebfa187b2ce87ea75d394dbabba02f43645f933093f88a52b'), ('922464066622377986_0', 'data_image/hurricane_maria/23_10_2017/922464066622377986_0.jpg', 'train', 'c9d6531d0bff178788de38a8c3df7eb1fe8fcda7f1f560f13d38e439c706d675'), ('923734427859062784_0', 'data_image/hurricane_maria/27_10_2017/923734427859062784_0.jpg', 'dev', '42d914a1bc1ca24bc5c9f94e6faf806845455ab4655fdbe03dbdf954b7d42051'), ('923142857837465600_0', 'data_image/hurricane_maria/25_10_2017/923142857837465600_0.jpg', 'dev', '3c121fca1bb7ba7dee9ffe0836a26ce43c1a1dba54c30d3fb8952149af00a9a1'), ('921865301809123328_0', 'data_image/hurricane_maria/21_10_2017/921865301809123328_0.jpg', 'test', 'a0474736ea6bb6fa43104445b78eca62f78758de64cb4b10b7f86969e336de22'), ('922573383241490432_0', 'data_image/hurricane_maria/23_10_2017/922573383241490432_0.jpg', 'test', '6fa913556af5e40fb42100e26f61f616da91db11704a1aa7b1f89f7916bae7a2'), ('916352092373241857_0', 'data_image/hurricane_maria/6_10_2017/916352092373241857_0.jpg', 'train', '6c337b37319eecb823fac21711542d4b6412a4f717ae7c5d3d6a1969c1206ba0'), ('928331995880374272_0', 'data_image/hurricane_maria/8_11_2017/928331995880374272_0.jpg', 'train', '39870c528b4d6e6ee36f6668008dc5b883de7ec94bd2bae1642a8ff471f7c26c'), ('910763300186505216_0', 'data_image/hurricane_maria/21_9_2017/910763300186505216_0.jpg', 'test', '89c6f9b2a4a84972e1bf0d6c0d63d4126bb249dbfb1b4ee83d4581c0007ebc2b'), ('921317041365245952_0', 'data_image/hurricane_maria/20_10_2017/921317041365245952_0.jpg', 'test', '7967b4dd649545f60797d01020c8225dd49f0d3f1d8c66507812bae7b73b091a'), ('914187089914290176_0', 'data_image/hurricane_maria/30_9_2017/914187089914290176_0.jpg', 'dev', '7e4804ba17ad46eefc3190edffaf427ebb80eec5d7912b94021d6b88a9fac2c0'), ('911701844405170176_0', 'data_image/hurricane_maria/23_9_2017/911701844405170176_0.jpg', 'train', '3b4b164d27966fee361f8a0c6346a1f1d223f8afbacc6afcdb9533a594578a43'), ('910799478491316224_0', 'data_image/hurricane_maria/21_9_2017/910799478491316224_0.jpg', 'train', '6b24279d976ee63862c0d041461a351e2923abbdf763f2959a3645ddfeee37db'), ('922132997326991360_0', 'data_image/hurricane_maria/22_10_2017/922132997326991360_0.jpg', 'test', '5dd9c2791d74d572b83d3cf671abf01dc62ed272ef94ec0716bca01ca316a265'), ('913753360720740352_0', 'data_image/hurricane_maria/29_9_2017/913753360720740352_0.jpg', 'test', '3610ea240b6f0d27a0800a37b25247ad364d7c2cbb7f1addcbbf32234e05b5aa'), ('922977569732988928_0', 'data_image/hurricane_maria/25_10_2017/922977569732988928_0.jpg', 'dev', '1beb2c6cac80cc6315eb37b2bb1dbc5c313e31a2d4249b7ca8e425a75e542109'), ('913092280696868864_0', 'data_image/hurricane_maria/27_9_2017/913092280696868864_0.jpg', 'train', '4723a12cd7f5e32819a538ac3448beed5c13160dc75e39b5e13bda7822942c70'), ('927200670926680064_0', 'data_image/hurricane_maria/5_11_2017/927200670926680064_0.jpg', 'train', '9bfdcfc7a06f5b8dbf2c49111b0c33d3fe729051d3e444a2b6956cd9b8b3c6f9'), ('917782348107796480_0', 'data_image/hurricane_maria/10_10_2017/917782348107796480_0.jpg', 'dev', '0cd88feaabf2a5ae55a3c1b161427ac888aec34a6a9fee3515c4ee4d71495df6'), ('930677315012579328_0', 'data_image/iraq_iran_earthquake/15_11_2017/930677315012579328_0.jpg', 'train', 'c1e84984e95a407c02f29c5d722f70857fb430d637fad92dbfa8668e60cf938d'), ('930527974541193216_0', 'data_image/iraq_iran_earthquake/14_11_2017/930527974541193216_0.jpg', 'train', '5d35a21f6a2e05f3155a26ea31b7533d38059835167cffe7e4cc2cecd38660ae'), ('930401945516462080_0', 'data_image/iraq_iran_earthquake/14_11_2017/930401945516462080_0.jpg', 'train', 'a239e28505a6151da1d6cc3c1849a9a740d3913ebd695b9dc4037820a5df5eb3'), ('930347490263666688_0', 'data_image/iraq_iran_earthquake/14_11_2017/930347490263666688_0.jpg', 'train', 'd54adb38ffad7e1e2d8b90788b9f6af297329cf85ee145f392daab41cc87d3f4'), ('930147282028072960_0', 'data_image/iraq_iran_earthquake/13_11_2017/930147282028072960_0.jpg', 'test', 'cfe427d68d2e0283f17353cd117ad9427a814b38caaeb25a227bcab138cf2635'), ('930556782715555842_0', 'data_image/iraq_iran_earthquake/14_11_2017/930556782715555842_0.jpg', 'train', '59232ab08fd19213bf26f307011ffd60d8312e6ddce65365f19527afbe52b694'), ('930050951020531712_0', 'data_image/iraq_iran_earthquake/13_11_2017/930050951020531712_0.jpg', 'test', '374d00c9b068e7e82067ea5e14c074ae87e01f76ade66a71974e06c37b81148a'), ('929991509499187200_0', 'data_image/iraq_iran_earthquake/13_11_2017/929991509499187200_0.jpg', 'train', '7bc8e2faa1cb55f1d4a7c93dd5807744980b456e2076f3af1b6759321b18cd38'), ('930053449131134976_0', 'data_image/iraq_iran_earthquake/13_11_2017/930053449131134976_0.jpg', 'train', 'f86106afe8e95a0e2e7e69b6657361c85a92fb765f68f9b85a9ce438c38d82ec'), ('930045769163386881_0', 'data_image/iraq_iran_earthquake/13_11_2017/930045769163386881_0.jpg', 'dev', '6e6b5034b4ff13604811d1f143ba8512a10ec5dfebabd409ac1d0a00976cdb32'), ('931533939713306624_0', 'data_image/iraq_iran_earthquake/17_11_2017/931533939713306624_0.jpg', 'dev', '5200b89c3b604e7c7b33d8ede82008235163a8239ef0cd88f2fe760da367c80b'), ('930068973206622213_1', 'data_image/iraq_iran_earthquake/13_11_2017/930068973206622213_1.jpg', 'train', 'b3a12f0432c51c9861636ff2e272447bc7acea7153df905dda3b79d76336c3cd'), ('930477991943131138_0', 'data_image/iraq_iran_earthquake/14_11_2017/930477991943131138_0.jpg', 'dev', 'aa37c5c001dc2e2195b84f19884b29a54b574d8f301055092a63a28435558c40'), ('929989361453621249_0', 'data_image/iraq_iran_earthquake/13_11_2017/929989361453621249_0.jpg', 'train', 'd82fc1d757e58724a84e76479bbe563f8d221d0e4fff63109e4a0344561cb145'), ('931186239931387905_0', 'data_image/iraq_iran_earthquake/16_11_2017/931186239931387905_0.jpg', 'train', 'fae484c3ae4f2b57eb0d9c56d13cd324faac93cfa5b6c99cc3ad90e05419fc02'), ('930438996852985856_0', 'data_image/iraq_iran_earthquake/14_11_2017/930438996852985856_0.jpg', 'test', '720ccb36a20c3da1dfaedbefe571fc701cd9c31bbf303873cec1e6ff6ea2f957'), ('930511669465411584_0', 'data_image/iraq_iran_earthquake/14_11_2017/930511669465411584_0.jpg', 'test', 'e44c282960400b6c0b245d782b22ce3b38481f43220befb493b65dfb360cb913'), ('930213608457306113_0', 'data_image/iraq_iran_earthquake/13_11_2017/930213608457306113_0.jpg', 'test', 'a0edb1d5f082312633f8e0fd1fdff364085cc0d2406e189c5cc0c71b5f986a1c'), ('930115174945587200_0', 'data_image/iraq_iran_earthquake/13_11_2017/930115174945587200_0.jpg', 'dev', '00dbd86f9d895f79a1a079cb1b990d5d124bb5e90576073f9cff12cc720a4e4c'), ('912398804544090112_0', 'data_image/mexico_earthquake/25_9_2017/912398804544090112_0.jpg', 'train', '3746f6141666d8b84432b6fd22da8228451497d50d205b7936682595301b0705'), ('914407560442187776_0', 'data_image/mexico_earthquake/1_10_2017/914407560442187776_0.jpg', 'test', 'e8e2a727f94aa535e9b7accfad1d8f9270d08c25813265283ea321580db3751c'), ('910808994457014273_0', 'data_image/mexico_earthquake/21_9_2017/910808994457014273_0.jpg', 'dev', 'ceddfff9a7f1893ce0b81a992b8f30cb3fb538115c02bf2e3b37abc3e3809aed'), ('914952139209891852_1', 'data_image/mexico_earthquake/2_10_2017/914952139209891852_1.jpg', 'dev', 'c988a4efab5252f27c72fe192b0d0418720508d632e07de4efe08285ceedbf28'), ('911624556292579329_0', 'data_image/mexico_earthquake/23_9_2017/911624556292579329_0.jpg', 'train', '31533270ba2a4c7152f15f44476799c4077ba9d5addb538059463ff7e6b39e91'), ('910524793740513280_0', 'data_image/mexico_earthquake/20_9_2017/910524793740513280_0.jpg', 'train', '25c29d1c0c7136e03002eb8620479ddd7b2924a149a2a3b81f44726f31df6684'), ('910734118165590016_0', 'data_image/mexico_earthquake/21_9_2017/910734118165590016_0.jpg', 'train', '25b01fb014757bc65864d5e722f6c2faf07c6bec85f366dbbc353710b3219a3d'), ('910611807080996864_0', 'data_image/mexico_earthquake/20_9_2017/910611807080996864_0.jpg', 'train', 'd2fd3c623be56e1ee3ff0a1dedb173ee9c24228282af339b71f57054bdcb7ee0'), ('916167674673156096_0', 'data_image/mexico_earthquake/6_10_2017/916167674673156096_0.jpg', 'train', '06b13a96eae49cc3cfca27218aa52d62a48b48845e632f3aa52000079854f457'), ('910784013249343488_0', 'data_image/mexico_earthquake/21_9_2017/910784013249343488_0.jpg', 'train', '3cb40559f241ed45ba631e32e82416d1ed1d5a20806e6e001b512bea2cd61c51'), ('910524793740513280_3', 'data_image/mexico_earthquake/20_9_2017/910524793740513280_3.jpg', 'train', '1800526befdb8d74520e5adb1019294f4361cd412f86f122c99dc76d4350c679'), ('912382620234903554_1', 'data_image/mexico_earthquake/25_9_2017/912382620234903554_1.jpg', 'train', '663033d7ceca7e4c076dd52be83b055dda5463ac7d7fc3345073a8087692c03f'), ('914870161487867909_0', 'data_image/mexico_earthquake/2_10_2017/914870161487867909_0.jpg', 'test', '9b7b8cf83c059b85dce3f81ed2f89c84a75e345a98cb5ae13bd9c83550394817'), ('911833476780953600_0', 'data_image/mexico_earthquake/24_9_2017/911833476780953600_0.jpg', 'train', 'b743c43d870d9e244c8b804607230901178099c08f6a83d4cfc607d20174ae79'), ('911714291992145920_0', 'data_image/mexico_earthquake/23_9_2017/911714291992145920_0.jpg', 'test', 'd32409d97091baac4211a85b2e6f7959bff686263128c265242a2f861d91d138'), ('911664846420234240_0', 'data_image/mexico_earthquake/23_9_2017/911664846420234240_0.jpg', 'dev', 'a862e9dd3829f3b8416121d5ab5e5272a6ab8a333628da670f760df50f7819b9'), ('910708152227602438_0', 'data_image/mexico_earthquake/21_9_2017/910708152227602438_0.jpg', 'test', '5c366f317fd02f561d2b808c55bf56f100843ea3c1526dc728d90b257755ea3b'), ('910658484257030144_3', 'data_image/mexico_earthquake/21_9_2017/910658484257030144_3.jpg', 'train', '059ecdeab9385dab138bd4173546554e56e88f9c2752988de1e00d77dd911254'), ('910681690812465152_0', 'data_image/mexico_earthquake/21_9_2017/910681690812465152_0.jpg', 'test', 'e77578ddbfbc5ffbf26d44679c4c54933e65cb9f3b45670bca1cded3b1a0134a'), ('910523417136316416_0', 'data_image/mexico_earthquake/20_9_2017/910523417136316416_0.jpg', 'dev', '40948a262b5ca1b6ec6711eff3f61136f33fb88aa6e5d836f6ee70b4d3990eac'), ('872359583645356032_0', 'data_image/srilanka_floods/7_6_2017/872359583645356032_0.jpg', 'test', '78f2629ea8c532ef3fac8f6f768b192c3770ad99e8a2c416a23be42d2653bb38'), ('870331956415909890_1', 'data_image/srilanka_floods/1_6_2017/870331956415909890_1.jpg', 'train', 'ff9cc0dbd30e3a6c5d88e267068ebc469e9435c081b1af1723884fb3c82039fb'), ('870917963821535233_0', 'data_image/srilanka_floods/3_6_2017/870917963821535233_0.jpg', 'train', '9d9f55998b54bef130f892da5f4d84c81ce4be74633afbab41abfefea5108c48'), ('873306605122703360_1', 'data_image/srilanka_floods/9_6_2017/873306605122703360_1.jpg', 'train', 'e7892141c36883197367e5776ac39dbdcd729f02963c997de368b51e5ca0d4db'), ('870148339307749377_0', 'data_image/srilanka_floods/1_6_2017/870148339307749377_0.jpg', 'dev', 'c0b43afba07e14eb8fac41512d9e921f132a4557b7fb703fad708fe5b74d0ce0'), ('870300517607591940_0', 'data_image/srilanka_floods/1_6_2017/870300517607591940_0.jpg', 'test', '61a323b7aa717065e1f540191da3d69d2bc5f615fc19af9068c4fbf9a153fdfb'), ('870161539948400641_0', 'data_image/srilanka_floods/1_6_2017/870161539948400641_0.jpg', 'test', 'f4a57526899d810d66476bb74137e064de66d2021234e28586c53faffc4f889e'), ('869968240956420096_0', 'data_image/srilanka_floods/31_5_2017/869968240956420096_0.jpg', 'dev', '23cae63527f3b219a0616e632a0316e67d8deffb89aa1b2040099da04940c0e0'), ('870663249297108993_0', 'data_image/srilanka_floods/2_6_2017/870663249297108993_0.jpg', 'train', '5e881a1ced53cd4ec04a065a7d82524baefb0ce13f252a466dc4afdb41f6e6a5'), ('871142568016453632_0', 'data_image/srilanka_floods/3_6_2017/871142568016453632_0.jpg', 'train', '5f7723b29f303daa871125ef6a69c2bc3d4163cad1790953b07cfdccbbd3ec4a'), ('870061681493155844_0', 'data_image/srilanka_floods/31_5_2017/870061681493155844_0.jpg', 'train', 'f0a656aba775cdda506281325bfa6b2444d0aff8189d168a4dad646f09d882b5'), ('878137481417310209_0', 'data_image/srilanka_floods/23_6_2017/878137481417310209_0.jpg', 'test', 'ad21eeeb447650381fcc7d2772faab1ba189584d8d3c149446829f793c26fc9c'), ('878061646559592448_0', 'data_image/srilanka_floods/23_6_2017/878061646559592448_0.jpg', 'dev', 'b3acc21099b435fa25dcf99180a0136e4c0cb04ef952862cafba9b507b333e41'), ('870661861040988160_0', 'data_image/srilanka_floods/2_6_2017/870661861040988160_0.jpg', 'test', '9d7b48ca709bf4e453b18193afc237512f4d73ecbb9bd66408c5550f3d30618c'), ('872502041255530496_0', 'data_image/srilanka_floods/7_6_2017/872502041255530496_0.jpg', 'test', '5213f57d083ea869ac43004978541ce834c2b388c8b0df7ba87b730f723b15de'), ('871649571058978817_1', 'data_image/srilanka_floods/5_6_2017/871649571058978817_1.jpg', 'train', '96b2909b2d93b50134a75869612e266c703fb6b0fd4d3368cb36cb9f3d61176f'), ('871717741006376960_0', 'data_image/srilanka_floods/5_6_2017/871717741006376960_0.jpg', 'train', '4ce211b7433394a3dd0ea410f15b622c65236b7d956d902286a2725bf8886913'), ('921583344680185856_0', 'data_image/california_wildfires/21_10_2017/921583344680185856_0.jpg', 'test', '1c73b138b59e60f1dd07de90e3118628fd0b8de108b1c42d99c0edfe342e53ae'), ('919942687330942976_0', 'data_image/california_wildfires/16_10_2017/919942687330942976_0.jpg', 'train', 'fe5fac253c27384ee295367aaa29d30453924c8aaef277e091b632142a860870'), ('919638588706263040_0', 'data_image/california_wildfires/15_10_2017/919638588706263040_0.jpg', 'test', '23c7e2a9f74ed41049e814306594ba93fa64edd964324c25fe082f8e66ea1fa5'), ('917972593789063170_0', 'data_image/california_wildfires/11_10_2017/917972593789063170_0.jpg', 'dev', '4724ffa0c291ce163af2cf0590bb59c416468e3c624b31484ee66e041285e073'), ('919562679404003328_2', 'data_image/california_wildfires/15_10_2017/919562679404003328_2.jpg', 'train', '9f88734f260b0aea2858b9c054daff5b25d302843c210b505248eb8019617441'), ('919974245106626561_3', 'data_image/california_wildfires/16_10_2017/919974245106626561_3.jpg', 'test', '77236f2869f409d1a768c9a49e7adca4bbbdfde914f8159b7a187150e6d15fd0'), ('918035671280893953_0', 'data_image/california_wildfires/11_10_2017/918035671280893953_0.jpg', 'test', 'c0e2df313061e2f586c48b53312435eaeb643776e805800496ee491d24c8ebcc'), ('921332920840843265_0', 'data_image/california_wildfires/20_10_2017/921332920840843265_0.jpg', 'train', '717092738122076813f69683296ba5a3910cf88c80cd90e7c1a6c8269aeec7dc'), ('920983411707334657_0', 'data_image/california_wildfires/19_10_2017/920983411707334657_0.jpg', 'dev', 'db75201236993c88e5de17eb36656ea6c797d291b169ac9515b933f2e7de48cf'), ('917864087438049280_0', 'data_image/california_wildfires/10_10_2017/917864087438049280_0.jpg', 'train', '8f25aedf836967b28234790385d08d59f3bcfe6ef2a829a5f483a7df3f858b59'), ('919562679404003328_0', 'data_image/california_wildfires/15_10_2017/919562679404003328_0.jpg', 'train', '65061d850224f6a3cd6804ab616181f39f5a793e8e0d01e1ff9edf736f3891fd'), ('917806204012978176_0', 'data_image/california_wildfires/10_10_2017/917806204012978176_0.jpg', 'train', 'c180d0d8db219b57e1b12e2ee4ff3a8c468d576dc9999d0d4073c821d0559987'), ('918240774462074880_0', 'data_image/california_wildfires/11_10_2017/918240774462074880_0.jpg', 'dev', 'bc2e60568ad45e94d9ac90c39fce2715e8e84326bddb71c82ae3f1e7ecd5cdb5'), ('918050688541822976_0', 'data_image/california_wildfires/11_10_2017/918050688541822976_0.jpg', 'train', 'ca9c28438d8f152633a93972c11a062ee9ffa000bddb54a4ea585bce247c437c'), ('917849898963349510_0', 'data_image/california_wildfires/10_10_2017/917849898963349510_0.jpg', 'dev', '4cfb232fc7c3d68f78ff9fdb3a8cc9d1581a1ccea237cbc865c4d7044b597193'), ('919804068155060225_0', 'data_image/california_wildfires/16_10_2017/919804068155060225_0.jpg', 'test', 'eda9408b5cb5fb7d76a892fb41aa4c537d8bb8b87eb78f9af3c3cb0a093383bc'), ('921438122021412866_0', 'data_image/california_wildfires/20_10_2017/921438122021412866_0.jpg', 'train', '1717ce82b9fa84fcbfa82c69699aca41c9b77922f078aaf419d5922c8f4adc4b'), ('919942286053691392_0', 'data_image/california_wildfires/16_10_2017/919942286053691392_0.jpg', 'train', '6385a8d6362ff74077659937bfe642294fba198de6d968cb7d779860981a7692'), ('920035856957739009_0', 'data_image/california_wildfires/16_10_2017/920035856957739009_0.jpg', 'train', 'd6f4a8a610e2a517b595b02754287a668ab54471bd8e3fa075ce169796b6fc69'), ('919581916973604865_0', 'data_image/california_wildfires/15_10_2017/919581916973604865_0.jpg', 'train', 'f33783903d7b230631fb1c174cccfe21a2bd284c4492d5d51a0ba57102b91c50'), ('918353491453558784_0', 'data_image/california_wildfires/12_10_2017/918353491453558784_0.jpg', 'dev', 'b86b3291d9092720825934211a3e25c8f9d3781201db72e9e3dbcc5860391fe0'), ('919710733058715649_0', 'data_image/california_wildfires/15_10_2017/919710733058715649_0.jpg', 'dev', '4aba03ed1cee593e2e62e13a6aa820c8764c7e9d0cb7d7bd82ab331504e29d97'), ('923591950414041090_0', 'data_image/california_wildfires/26_10_2017/923591950414041090_0.jpg', 'test', 'f960443224ad8dbd23ea110c65a7c85519c25c6d3895a3cbb0d2a2f47f15dbb9'), ('917796940900982786_0', 'data_image/california_wildfires/10_10_2017/917796940900982786_0.jpg', 'dev', '56573e3534171c497a0632cee34467b117d1e31037b566bdaab4f079e0aa4ccd'), ('908517284083306496_0', 'data_image/hurricane_harvey/15_9_2017/908517284083306496_0.jpg', 'test', '450344319cd9cdee813d1b8b6dc7797f0dd0cf5fe7345204d027bb5621f9b2cc'), ('904442140901597184_0', 'data_image/hurricane_harvey/3_9_2017/904442140901597184_0.jpg', 'test', '929ff421e7643e4bdcd1bd78b431a3461d580c101a1dd8a54489b37a23cf030b'), ('904818872817246210_0', 'data_image/hurricane_harvey/4_9_2017/904818872817246210_0.jpg', 'test', '1413615e17d8b7c93a7851a139df18e276bbad0915be883fd99737e04015d487'), ('907638987438874624_0', 'data_image/hurricane_harvey/12_9_2017/907638987438874624_0.jpg', 'train', 'f3946ae4da081633970b379afa05242a240a09ef376818d057bbc92dfadc4cd5'), ('908723134781517824_0', 'data_image/hurricane_harvey/15_9_2017/908723134781517824_0.jpg', 'train', 'd3c9690ad46db30349469ff534caa71fa95c5b0e64c30d1139dfab0705925813'), ('909820533419671555_1', 'data_image/hurricane_harvey/18_9_2017/909820533419671555_1.jpg', 'train', 'f321e84203f2bc9fe0b400b495db9796243ca475b1104922f606f4477c5eed7e'), ('906713843283451904_0', 'data_image/hurricane_harvey/10_9_2017/906713843283451904_0.jpg', 'test', '620e9e35666d146db59207d42e9e58aa20f6d92b9d8c1adf0777830599a52ac4'), ('905600118405124096_0', 'data_image/hurricane_harvey/7_9_2017/905600118405124096_0.jpg', 'train', 'f9af503e472320616a45462db1075620566f119f88e9d25c842f33580242c731'), ('905766527131103234_0', 'data_image/hurricane_harvey/7_9_2017/905766527131103234_0.jpg', 'test', '8e42e606b72be1b3b7d7f6108330bfae2182ad967c387457dd73fe5625c83463'), ('901646496713584640_0', 'data_image/hurricane_harvey/27_8_2017/901646496713584640_0.jpg', 'dev', 'f15d2feab0c8bb9ced508312d9e0af90d5053fa71632b03c3b8bba7b75c42ab2'), ('904991929305370625_0', 'data_image/hurricane_harvey/5_9_2017/904991929305370625_0.jpg', 'train', '742807ed4ec4aa9958ed59c10d488d64d440dfa80d0a8a825099fd5391d28df7'), ('907139822598356992_0', 'data_image/hurricane_harvey/11_9_2017/907139822598356992_0.jpg', 'dev', 'a81a33d2eb1d621c3ccbe5201a674f95d67bff5637cb3c45e57b91c867d01781'), ('905545122150047744_0', 'data_image/hurricane_harvey/6_9_2017/905545122150047744_0.jpg', 'train', 'a3faf80c4a9d7113f015002ac3290f820f19474905d7a9b2a2c97e6eeedf7a78'), ('905274232590004225_0', 'data_image/hurricane_harvey/27_8_2017/901671684478029825_0.jpg', 'test', 'a464a10b2bdd3cc7091f38c43e865592bc84019a3066970cd335812e9529f6a4'), ('907998284383485952_0', 'data_image/hurricane_harvey/13_9_2017/907998284383485952_0.jpg', 'test', 'df6b43b8dc371d1f202623fb33f4f9f7d7f4528a6affed216e791519b945d3e7'), ('906254249314590720_0', 'data_image/hurricane_harvey/8_9_2017/906254249314590720_0.jpg', 'train', 'bd30cc3d008846c38d95fa51357e90847f4a824ed7b3759374831d2b66adb0b1'), ('906934889919991808_0', 'data_image/hurricane_harvey/10_9_2017/906934889919991808_0.jpg', 'dev', '93e96d26c4f65ca75fce24cb1e2c23b2b196374706ac71739eebf75086035d14'), ('905306602416332800_0', 'data_image/hurricane_harvey/6_9_2017/905306602416332800_0.jpg', 'test', '135a497653217612f66d3b385fd52ed7a9b4153cb4019090f7bd1789bdd1c83a'), ('901830988715532288_0', 'data_image/hurricane_harvey/27_8_2017/901830988715532288_0.jpg', 'train', 'd2d6379bd371f76d0d039773a03a350aeaad6019b6d22199323a62302d562950'), ('908660710409953281_0', 'data_image/hurricane_harvey/15_9_2017/908660710409953281_0.jpg', 'train', '900d81bc6a4c5d68b00b5fe34b31e53e0ef1fc0f5fff6b91b7ef7954b567e00b'), ('908519696386818048_2', 'data_image/hurricane_harvey/15_9_2017/908519696386818048_2.jpg', 'train', '5db003d826c136c507d48530b2740f67af5709689b1266755ab7f501865b7949'), ('908130612225167360_0', 'data_image/hurricane_harvey/14_9_2017/908130612225167360_0.jpg', 'dev', 'e93b4b99adb3fbb39eafc1781300fabd75cb2415cb9c9791805ed6148b179382'), ('904796109759021059_1', 'data_image/hurricane_harvey/4_9_2017/904796109759021059_1.jpg', 'test', 'cff0e7dda6715e5c0d417f48665ab2039069a11ae2775b1df7df0453fd51cb2d'), ('908525619037732865_2', 'data_image/hurricane_harvey/15_9_2017/908525619037732865_2.jpg', 'train', 'c0045cdb44096932c3cae6600639215d0a51322232bdaeca27390cfece0dabca'), ('906199568597086208_0', 'data_image/hurricane_harvey/8_9_2017/906199568597086208_0.jpg', 'train', 'ec37b59d9123400defedcccc68b607aa472a7882a1beb73abb9b26e7a93f53c0'), ('905792707053965312_0', 'data_image/hurricane_harvey/7_9_2017/905792707053965312_0.jpg', 'test', '6534405141bcfe9d49408a0aabbd1f7db4cc691531739ca081bf79e5ac3247d9'), ('905065545724252160_0', 'data_image/hurricane_harvey/5_9_2017/905065545724252160_0.jpg', 'test', '68d413365a353efbb07bfbaf973c41630927980dd197f2f4166dba101f2c569e'), ('905137359317897217_0', 'data_image/hurricane_harvey/5_9_2017/905137359317897217_0.jpg', 'dev', '6a5898f92a764f4cb759d78f77cff0134f567846d56be61da2b1471067d87607'), ('909903376875274240_0', 'data_image/hurricane_irma/18_9_2017/909903376875274240_0.jpg', 'test', 'a8db0fa35e70abc9396903e4dbbf50d95b1fdbbd8b943508863dee0c5d67f4ca'), ('909447240959262720_0', 'data_image/hurricane_irma/17_9_2017/909447240959262720_0.jpg', 'train', '61ce2b46b0c178c9fb79d94b01b5a59ed7610c7eca2e1aae576754c3d1ad04e1'), ('909848317835714566_0', 'data_image/hurricane_irma/18_9_2017/909848317835714566_0.jpg', 'dev', '267fe68a948500f1bed1aa2469ca4622990977459770273f5d2c65dbf7089031'), ('910201731052707840_0', 'data_image/hurricane_irma/19_9_2017/910201731052707840_0.jpg', 'train', '08940f3ff45649b9a18abef6500d1dac7849e9c86f62ebd03e88203f8deeb33d'), ('909718820968833030_0', 'data_image/hurricane_irma/18_9_2017/909718820968833030_0.jpg', 'dev', '52ff019b5de95fbf4603b1ddd0be7cc3ff18ab7ab926134c36f4516ac5008e65'), ('909714323156733952_0', 'data_image/hurricane_irma/18_9_2017/909714323156733952_0.jpg', 'train', 'acd9ad649bb06c177b57023b1bd1609dfbb2d3dbec58eb19dc2b2a653f5889d7'), ('910168389846536193_0', 'data_image/hurricane_irma/19_9_2017/910168389846536193_0.jpg', 'train', 'e47b7730b457b31703ae0833f5b3b334e6f340b1cab5e485f8231d94b378d581'), ('910118787080441857_0', 'data_image/hurricane_irma/19_9_2017/910118787080441857_0.jpg', 'test', '012042fcf5aa0e39cf7fb6593aebcea8e7d32ee8c93e54ad6cb3b59214ae3d6c'), ('909957227842232320_0', 'data_image/hurricane_irma/19_9_2017/909957227842232320_0.jpg', 'train', '7055f1b7446b352d608fda8aafebe2b5307afb6782f150d9d98c4f9a7a06ae1b'), ('909474083737341953_0', 'data_image/hurricane_irma/17_9_2017/909474083737341953_0.jpg', 'dev', 'b56ebaf6368488b50e5b71f3b5122db78a9621f092bcae168ddd35f71cbdfc8d'), ('910050984679919616_0', 'data_image/hurricane_irma/19_9_2017/910050984679919616_0.jpg', 'train', '805b7f703bdf18890d2ce9be27f085a7cac9194e129f8fba7c27af4f3c764866'), ('909833582931259393_0', 'data_image/hurricane_irma/18_9_2017/909833582931259393_0.jpg', 'train', 'aa95163cf7b134ceea0151e32045281cb94d177ede4955cacf0993e26931c559'), ('909420528703819776_0', 'data_image/hurricane_irma/17_9_2017/909420528703819776_0.jpg', 'test', '9a15903e673a5cc366719e7f3f23fb3df8ffc25110c4afa6eeddf7927e051c5d'), ('909388509714239488_0', 'data_image/hurricane_irma/17_9_2017/909388509714239488_0.jpg', 'train', 'c629492dd3bdddccd93f3b309d8f7709256c940a5ee436781ff44e001bac4325'), ('909869601923715072_0', 'data_image/hurricane_irma/18_9_2017/909869601923715072_0.jpg', 'dev', 'a022f2b2ce8c924a7c0c7460c5dd4c37f76c9d174dbcaf140840148fab842a6b'), ('910232317901361153_1', 'data_image/hurricane_irma/19_9_2017/910232317901361153_1.jpg', 'train', '00bd79dcc25e0d14dcb4e5ee32200a4472072da363be16bd9dfff461eb396777'), ('909813175671037954_0', 'data_image/hurricane_irma/18_9_2017/909813175671037954_0.jpg', 'train', 'b70aa906d23086a80cca95e9b6d8f805553385a67c817e4d8fab8a4e2551d724'), ('909749533948743682_0', 'data_image/hurricane_irma/18_9_2017/909749533948743682_0.jpg', 'test', 'c4d3dd9ba99b7401efea283e0f7a9f1a1323dcc0e382c8a706219c197bd96c83'), ('905636651447328768_3', 'data_image/hurricane_irma/7_9_2017/905636651447328768_3.jpg', 'test', '579810163366351b5f53c8e21dc72e6ea6dc6f1d282bb38680b5801d595a7005'), ('909494168032071682_0', 'data_image/hurricane_irma/17_9_2017/909494168032071682_0.jpg', 'train', '303910182e8fed37f713d9f45abd4746ac477d22bf748ec153e78ec0e5edd5d6'), ('910223925828739072_0', 'data_image/hurricane_irma/19_9_2017/910223925828739072_0.jpg', 'test', 'c6767c1810f60e069e01179523ed068c3833b6e946f8eb78660f80f6eed6ebee'), ('910209384264777728_0', 'data_image/hurricane_irma/19_9_2017/910209384264777728_0.jpg', 'dev', '2d9de64f5afcfe289a1ca3fb287d999c6efecc8ec8277731ecf81bd16460e60b'), ('910255847346987009_0', 'data_image/hurricane_irma/19_9_2017/910255847346987009_0.jpg', 'train', '363615b6b330860aa479c067f8405639dd4e80f99c15c53489bdd7b311aa80b6'), ('910262796935450624_0', 'data_image/hurricane_irma/19_9_2017/910262796935450624_0.jpg', 'dev', '235aec1237738a613f49ef32f87ceb35be33a7861f1af681e8370afb756448ab'), ('909835094663286785_0', 'data_image/hurricane_irma/18_9_2017/909835094663286785_0.jpg', 'train', '40d1920369880baa5d42b3ee63e54e7c657e59d984ccdbcef03c699da755a155'), ('910170555302780928_0', 'data_image/hurricane_irma/19_9_2017/910170555302780928_0.jpg', 'dev', '0e2f98a55cb78854626535173325acfcc9cb063ea07b5ab4674e07f206c353a0'), ('909748542666887173_0', 'data_image/hurricane_irma/18_9_2017/909748542666887173_0.jpg', 'train', '28521690fb1069bf8eea2f44d7aaa540e42461c32585ea718d2274876401bed9'), ('910115488356864001_0', 'data_image/hurricane_irma/19_9_2017/910115488356864001_0.jpg', 'dev', '17ed2bafe853973f9ea7460f33a75a9f344a66f50c4c6bd880ac27afe07319b2'), ('909385864500989952_0', 'data_image/hurricane_irma/17_9_2017/909385864500989952_0.jpg', 'train', '0edd8e21d1b687987b71c90607b3cc9a161ebc2d57c1b60f66abd35d316bde7a'), ('909475817981440002_0', 'data_image/hurricane_irma/17_9_2017/909475817981440002_0.jpg', 'dev', '470d342768f5f7b454253aace4fb29996486a88c6008ddd19e0804bd3b4ca0e4'), ('909445622796898306_0', 'data_image/hurricane_irma/17_9_2017/909445622796898306_0.jpg', 'train', '4d658ffdff779824545d14f9573f96b907bd6f5a4dcf88db4517defcfc468d6b'), ('910265294370639873_0', 'data_image/hurricane_irma/19_9_2017/910265294370639873_0.jpg', 'train', 'ad285368f3a56e313917bc88d623919c4a8210de3d943e29d4bdb1e97dc69331'), ('913575477498884097_1', 'data_image/hurricane_maria/29_9_2017/913575477498884097_1.jpg', 'test', '87c177e39a8e89965f75a760ccdb6810fe173e528f2458d8f8d37b1a48052a61'), ('914453846470119424_0', 'data_image/hurricane_maria/1_10_2017/914453846470119424_0.jpg', 'train', 'cdb486a7835aecddd9169941824b7c873f69a36018b1a9b98c3d3eab1b910a24'), ('928993274186141701_0', 'data_image/hurricane_maria/10_11_2017/928993274186141701_0.jpg', 'dev', 'd490094a1a71f52025dfe9b1c54f82fbf21ad98b81c3ca0fc012eedf4127ea68'), ('916103663424278528_0', 'data_image/hurricane_maria/6_10_2017/916103663424278528_0.jpg', 'test', 'd84c1789457e28fcd89d813c72639da42d480968d0c275e65e7f2f78e91b6662'), ('912322653578461186_0', 'data_image/hurricane_maria/25_9_2017/912322653578461186_0.jpg', 'dev', '182fc94cefe77f553a8864e4c8c9a141338abc7f12961df85d62cb40642f5b22'), ('911584945470099456_0', 'data_image/hurricane_maria/23_9_2017/911584945470099456_0.jpg', 'test', '68981d6181ee58e3c8edb8d1cb79c1b3bec8ec30abce6e7d4d1945c251b5f15b'), ('912420533689610242_0', 'data_image/hurricane_maria/25_9_2017/912420533689610242_0.jpg', 'train', 'f711e80c1c082ffff8fa724898f688f5f17e05f263c7424422a01946ba7df936'), ('921542604801626112_0', 'data_image/hurricane_maria/21_10_2017/921542604801626112_0.jpg', 'test', 'ee7a21cf70121902c9fbe58305528d10eedec202e7077020839845d8ceacb7bf'), ('913065273888002049_0', 'data_image/hurricane_maria/27_9_2017/913065273888002049_0.jpg', 'test', '631ae8c70fb0acaa9e997366f05cfb4811fbdd3652057a37aa59a1bd36525885'), ('911561478536404992_1', 'data_image/hurricane_maria/23_9_2017/911561478536404992_1.jpg', 'dev', '16faf63da0b45d48c57ded6b0d310a6b1cc41b03f6405bbe9bf406920ae156eb'), ('913558979263025153_0', 'data_image/hurricane_maria/29_9_2017/913558979263025153_0.jpg', 'train', 'ad8219a8671912876867516b97765fea9827bf2280b810b7bbea3fc341a88042'), ('910549105151275008_0', 'data_image/hurricane_maria/20_9_2017/910549105151275008_0.jpg', 'train', '49e208986e8b6b6d15d52dd805dd18e1a4b91a7d8eed89aa80488ecfbc38bc16'), ('928971922628665344_0', 'data_image/hurricane_maria/10_11_2017/928971922628665344_0.jpg', 'test', '978a67a75dac7f46094e53907c9acadc6020311afbc3c466766bd9bdf1ed5d50'), ('913798859809345536_0', 'data_image/hurricane_maria/29_9_2017/913798859809345536_0.jpg', 'dev', '5542488e143e4c238d18925696690ce9e7ed3532d8aa30edb724fdb930b0f397'), ('913638009102270465_0', 'data_image/hurricane_maria/29_9_2017/913638009102270465_0.jpg', 'test', '53e835978ca4afafb96f827105df270a0bd582d7f7dfee4fac8c09f97501f356'), ('921853322570731520_1', 'data_image/hurricane_maria/21_10_2017/921853322570731520_1.jpg', 'test', '33f1c1deb02b2c603fab7021bc754635ed3a7ffb545ba73efdd220205a86d25e'), ('921873063628730368_0', 'data_image/hurricane_maria/21_10_2017/921873063628730368_0.jpg', 'test', 'd85e72f086333fb07aadaae5905cdf438da2e54e81f89555d4c0128dbd64152d'), ('927958507596517378_0', 'data_image/hurricane_maria/7_11_2017/927958507596517378_0.jpg', 'dev', '8fe3b435ce8793a6ad0c5ca50e072af35f336c0ec8520a9e0986e77293d610bf'), ('922049558758248448_0', 'data_image/hurricane_maria/22_10_2017/922049558758248448_0.jpg', 'dev', 'cebd93e80a0096f1f006d3c577691614f903fc0253a5281bd233029db1539700'), ('921392139262660608_0', 'data_image/hurricane_maria/20_10_2017/921392139262660608_0.jpg', 'train', '6cde197ee53c5e91ac78abe9b2843c2fc51ed51c5175a01e817443b65be686a0'), ('916095715813744640_0', 'data_image/hurricane_maria/6_10_2017/916095715813744640_0.jpg', 'dev', '35e5330600de447ccf729c3aa4f065e420f1941436208109d4abe062d2407c81'), ('923042295141806080_0', 'data_image/hurricane_maria/25_10_2017/923042295141806080_0.jpg', 'train', '2a83bbe8480b47110cf9aad3aa39689d94dee47788b746c1500c47966251097f'), ('922009474143019008_0', 'data_image/hurricane_maria/22_10_2017/922009474143019008_0.png', 'test', '2ecafaf9307ea21a28c21e0ab25a356875907244046ff0c95618763e6699ebe9'), ('912376048842665984_3', 'data_image/hurricane_maria/25_9_2017/912376048842665984_3.jpg', 'train', '7c01f97bf9349b23282c5f03c81c48162dcd090857ce6593feb29fdb2ab68446'), ('910806102391918592_0', 'data_image/hurricane_maria/21_9_2017/910806102391918592_0.jpg', 'test', '0bf25fa32aaec31aca402adc509ae2d69077a3f5863a273158e8b0b71d78c6c6'), ('913129892539191296_1', 'data_image/hurricane_maria/27_9_2017/913129892539191296_1.jpg', 'train', 'a4366b8ef272c44a8ff5bc2b8c2adf66422566106470521c760f7dab27c80cf1'), ('911697479057461249_0', 'data_image/hurricane_maria/23_9_2017/911697479057461249_0.jpg', 'dev', '9eda68315d84bcf2189ee044e51bf0b651e9448e55437ac24e446d6c283c95f1'), ('911502259451826176_0', 'data_image/hurricane_maria/23_9_2017/911502259451826176_0.jpg', 'train', 'c14ba25a86e3e2b2ccaf60faa4840052440f1f4342e57badc3364f2bf9c9d96e'), ('923265376963715072_2', 'data_image/hurricane_maria/25_10_2017/923265376963715072_2.jpg', 'train', '57adc6be8de507713299da138859f7aee971da65116d12dc8b82f0b8d90d126b'), ('916179895105056768_0', 'data_image/hurricane_maria/6_10_2017/916179895105056768_0.jpg', 'train', 'a8f20068c6db6677453ace527272f89f345b6ac369bda91d3e6572bea7ef7e23'), ('912417890770579457_0', 'data_image/hurricane_maria/25_9_2017/912417890770579457_0.jpg', 'train', '668aa635d768caa1d4b9fa941ea26d5645d883aff2546dcb1f217ec96ce1767c'), ('927609307377688577_0', 'data_image/hurricane_maria/6_11_2017/927609307377688577_0.jpg', 'train', '601845f0921d7b8aec578352eb7c589e8163ca43ae3bad7d41bdd1125a859cf4'), ('927312549812539393_0', 'data_image/hurricane_maria/5_11_2017/927312549812539393_0.jpg', 'train', '5f189a2ab6a8b0de705e95dd5fe0055dc7fdbfdcca4744870c79357d0e1b94c5'), ('912944540226609152_2', 'data_image/hurricane_maria/27_9_2017/912944540226609152_2.jpg', 'dev', 'fec0bfd0c87ce16b9d82cb5dbeb4c817b9405c84b22160abe978258097cec2b3'), ('931214387318112256_0', 'data_image/iraq_iran_earthquake/16_11_2017/931214387318112256_0.jpg', 'train', '4874a6fee19230a4035940e987e4149a1b9dfff255786dcd6568d341da5ffe8f'), ('931046661274701825_1', 'data_image/iraq_iran_earthquake/16_11_2017/931046661274701825_1.jpg', 'train', '48d199c9f1b3a629d0eef2d3147c01a8191ecdc0e615efc4da8e024955a6182d'), ('930819767874793473_0', 'data_image/iraq_iran_earthquake/15_11_2017/930819767874793473_0.jpg', 'test', '294f2c413f6ef04c243967b9ecd7985eaf068629a53ff1280aaf21f057c9fc4f'), ('930100714625470464_0', 'data_image/iraq_iran_earthquake/13_11_2017/930100714625470464_0.jpg', 'test', '832aca0262792a3374e48d27d7e841ed457602e4b4cde58db940824bfb6ff931'), ('930379264201027584_0', 'data_image/iraq_iran_earthquake/14_11_2017/930379264201027584_0.jpg', 'dev', '7e8524be5b03f161ce4a3dcccca9135bdd58be253e7ad4c4e3e23344f2b0dd86'), ('930066324533006336_0', 'data_image/iraq_iran_earthquake/13_11_2017/930066324533006336_0.jpg', 'dev', 'da0e133b53573a4e5054ada87ec7b9b19e3cb1ba00a71db90bebadb232b997cb'), ('930034325940080640_0', 'data_image/iraq_iran_earthquake/13_11_2017/930034325940080640_0.jpg', 'test', '47101d58b075aadc820be3d59bd276f92b81ba426fe6394f6c96e37d62da797d'), ('929998022867435520_0', 'data_image/iraq_iran_earthquake/13_11_2017/929998022867435520_0.jpg', 'train', '0182716f437d746fa4706e441a6deb7cfc2e1c092ff50134d53f25fd51abe593'), ('930474196257275904_0', 'data_image/iraq_iran_earthquake/14_11_2017/930474196257275904_0.png', 'test', '53564fc19807602dacbb9a5ba3817e985b8eb8438145cf8995eee9f9ff3dfebb'), ('930170726715404295_0', 'data_image/iraq_iran_earthquake/13_11_2017/930170726715404295_0.jpg', 'test', 'ab6ddd397205f57b3a0e8e091e614b01bd558ea2cbe6621607dfedd7303f2772'), ('931164365151178752_0', 'data_image/iraq_iran_earthquake/16_11_2017/931164365151178752_0.jpg', 'test', 'cef6ca7b5d9562fb30edfbdbe0c2074d1033251237ecd32a53284da260a9d14d'), ('930109833474437120_0', 'data_image/iraq_iran_earthquake/13_11_2017/930109833474437120_0.jpg', 'dev', 'cde64cc4363e38e77d7900df0d3140475300421c56045e4eb547d245cf3a80a7'), ('930000308033507328_0', 'data_image/iraq_iran_earthquake/13_11_2017/930000308033507328_0.jpg', 'train', '89c03c2e322e440aa7468a2f19734bef20dbd205bdb7c9ff0126731baa02a706'), ('930523863691288576_3', 'data_image/iraq_iran_earthquake/14_11_2017/930523863691288576_3.jpg', 'train', 'b76908604d99da435f7821a211fdb2962eedf13f24b2a77481b4a5521043f2aa'), ('931321165058060290_0', 'data_image/iraq_iran_earthquake/17_11_2017/931321165058060290_0.jpg', 'train', '40f3e0e976cb956a02d82f29dee9c6b5e514f5964355f4e1c5681133d08d412b'), ('930017315378356224_0', 'data_image/iraq_iran_earthquake/13_11_2017/930017315378356224_0.jpg', 'train', '996d3e95b7929bc29e7734231e20219737b20df32523f747b99e4c14e8c17d2f'), ('913054883779158016_0', 'data_image/mexico_earthquake/27_9_2017/913054883779158016_0.jpg', 'train', '3cfc43dcce700171471e61f4c593eb3e90728b567ac79b8c63922923d7254e1f'), ('914610405707677697_0', 'data_image/mexico_earthquake/1_10_2017/914610405707677697_0.jpg', 'train', '3cbf931f18ea73068d4b9c4097e808bddeb95acb4333cf161e1111fe5d27318e'), ('910544691438944256_0', 'data_image/mexico_earthquake/20_9_2017/910544691438944256_0.jpg', 'train', '5ecba9fe20c4ed7a96834fef8b8e4fdb8e626871bc256475130f058a146d9562'), ('912989234922889216_0', 'data_image/mexico_earthquake/27_9_2017/912989234922889216_0.jpg', 'test', '5048bcd4bc05f51dea45fa453e7feb0ae1e096ef4a2ea28c94b3d3ca2af4eb8c'), ('910525346943873024_0', 'data_image/mexico_earthquake/20_9_2017/910525346943873024_0.jpg', 'train', 'e30a74ebc65063de5adbd06769cb3e4036c6b7c3b8366e61c8ecdecd174def2d'), ('910563191700185088_2', 'data_image/mexico_earthquake/20_9_2017/910563191700185088_2.jpg', 'train', '466b378a38fe50264c5356e2ac763c8bc200bf1c869a07329b3237368b42edd9'), ('910534271764328448_0', 'data_image/mexico_earthquake/20_9_2017/910534271764328448_0.jpg', 'dev', 'd22e5ac68dd00f8464f4b4e8dfe751d45c2d8b7516456bb68097e63aab4fb777'), ('910534454367502341_0', 'data_image/mexico_earthquake/20_9_2017/910534454367502341_0.jpg', 'train', 'acb45a4028dc869a2e44b1529b8b25ee9614359e6f7c40212d836a2082f411bb'), ('911902164251021312_0', 'data_image/mexico_earthquake/24_9_2017/911902164251021312_0.jpg', 'train', '3480a4889004acb8412c192c0ac31de2119e8f61deefd35e36421c526c1d8b92'), ('912408467712901120_0', 'data_image/mexico_earthquake/25_9_2017/912408467712901120_0.jpg', 'dev', '995d9d364f4b032e7b2fc88b1b6fe3da1dff5a496e290c4070b659323cf92728'), ('915167299610521601_0', 'data_image/mexico_earthquake/3_10_2017/915167299610521601_0.jpg', 'train', 'b08a03412c5a3a7de0d68bf663b6b095c84f9edc6f1f77b3ec7e62a0dcae0679'), ('911827596748447744_0', 'data_image/mexico_earthquake/24_9_2017/911827596748447744_0.jpg', 'dev', '32383e7bab8cd866e760df2208b2375955e8f61e1f2e1828735737578f0d0255'), ('912346493809577984_2', 'data_image/mexico_earthquake/25_9_2017/912346493809577984_2.jpg', 'train', 'd4a7b7798a0be735fe41a268cefafeccdf1a03b13ed3408659e682562c691d27'), ('913171165924831232_0', 'data_image/mexico_earthquake/27_9_2017/913171165924831232_0.jpg', 'train', '322adbc4284b2df93ddf667412077d19b1312e76bc7b90e5989051b4a1811ed6'), ('915315863582007296_0', 'data_image/mexico_earthquake/3_10_2017/915315863582007296_0.jpg', 'train', '2273701e091f4b978d9cde5d10fee6da9d2a73310c7b8fa95a0c7920e6458adc'), ('911701484852490240_0', 'data_image/mexico_earthquake/23_9_2017/911701484852490240_0.jpg', 'train', '7a61f897c9cbb1f244a40e3c15c3129358588add79147d81013433de07d944b7'), ('910714915404398593_0', 'data_image/mexico_earthquake/21_9_2017/910714915404398593_0.jpg', 'train', 'dd3ac545d9b0b9f28e8ff790cc5bd85177662b2cd5f17e9d27da03c657bb0235'), ('911717117602316288_3', 'data_image/mexico_earthquake/23_9_2017/911717117602316288_3.jpg', 'train', '9a529a5491f12705ca7878e2022b8f108d3f8372bacd332c791be07d7656c167'), ('911809891882487808_0', 'data_image/mexico_earthquake/24_9_2017/911809891882487808_0.jpg', 'test', '803ab2ef7fd36c7af946eb959064566651656cfd0419af4eb86aa267e6bd17ff'), ('871735126635544578_2', 'data_image/srilanka_floods/5_6_2017/871735126635544578_2.jpg', 'train', 'be99384b5fc2e55bd53b0295de400923b3c91b11add936967134bfdca0d08887'), ('870043559076007936_0', 'data_image/srilanka_floods/31_5_2017/870043559076007936_0.jpg', 'train', '8e03ce72e113f4faeb3745143bcade2a81b7e5b17b39d40bd06a871299a8d952'), ('869962038214131712_0', 'data_image/srilanka_floods/31_5_2017/869962038214131712_0.jpg', 'train', 'ecafc547f2df561d14d99cc2993bfe83ae6bf25f8d3c5338c16da141a72ec7d4'), ('870619854104961025_2', 'data_image/srilanka_floods/2_6_2017/870619854104961025_2.jpg', 'train', '5a0b988d9e5e156c85cdfbe577ac1679f25059684e2d62210d7d094bd13adfca'), ('869954009963970560_0', 'data_image/srilanka_floods/31_5_2017/869954009963970560_0.jpg', 'dev', 'bf7b662bb315201a2ebfcdf9e948d3b7a5494df20d5e6d71294a7bd9bd5324e6'), ('871995437447876608_0', 'data_image/srilanka_floods/6_6_2017/871995437447876608_0.jpg', 'train', '940f5719293fee35d40eb8c40865542793c4e03a8f41125e2b15394cfa507bef'), ('880166436563546114_0', 'data_image/srilanka_floods/28_6_2017/880166436563546114_0.jpg', 'test', '70f071d7a8336f04988dbf4dd12f2408f6790211e041984465cb76bf6e542222'), ('870605057548443648_0', 'data_image/srilanka_floods/2_6_2017/870605057548443648_0.jpg', 'test', 'c15f7a65d366c580b34fe3cc17ab1c21e2333a215daccf25e353a512dbb73aa6'), ('877532804472549377_0', 'data_image/srilanka_floods/21_6_2017/877532804472549377_0.jpg', 'dev', '7cd986a1af55162103e2f01a23ec4dd158d49cddf4812511e12287f1f64298b4'), ('872862344275267588_0', 'data_image/srilanka_floods/8_6_2017/872862344275267588_0.jpg', 'test', '33e3247e8646126049a30f348e0685ed1b37403de0210cceb49110ac306fb1e4'), ('872085722575929345_0', 'data_image/srilanka_floods/6_6_2017/872085722575929345_0.jpg', 'test', 'ff8b769b6e3b6008462937e136c6dc38a1fe149895ea1c171a0e878eaee33d4b'), ('873403196718202881_0', 'data_image/srilanka_floods/10_6_2017/873403196718202881_0.jpg', 'test', 'c0375a574aa533f0894a2cd71f6dd13d848785c09d4ec44b292c7977a9082dc5'), ('870425128731107328_0', 'data_image/srilanka_floods/1_6_2017/870425128731107328_0.jpg', 'dev', 'd8be6dca3731e31d3e0d5995737515c88ff2b332f179a3bb3ac8a5a6a7ce9ef6'), ('921482506922323969_0', 'data_image/california_wildfires/20_10_2017/921482506922323969_0.jpg', 'train', '576a498b1a8519e0c45386321be46d09a3925c541493f672bbdc60f61e1e4a71'), ('920064354002841600_0', 'data_image/california_wildfires/16_10_2017/920064354002841600_0.jpg', 'test', 'e6523074493a58ceae2b263892e936313ec68351cc0c4276db8172c96d6a710f'), ('919639455543070720_0', 'data_image/california_wildfires/15_10_2017/919639455543070720_0.jpg', 'test', '53dfa37b096cd504ad743e19403a41516c34516279f7a0c3d571b184de02d15f'), ('919650008802131970_0', 'data_image/california_wildfires/15_10_2017/919650008802131970_0.jpg', 'train', '37f44edf5558e080409ce92e7b73d3402d623587c505ad9fcb17d12ba24656a1'), ('917846485965959168_0', 'data_image/california_wildfires/10_10_2017/917846485965959168_0.jpg', 'test', '0ffd2e7c9de730f86d4f2f7d8b9e218c82d2e9f30637ba50f4d8f69091592324'), ('918082606545039360_0', 'data_image/california_wildfires/11_10_2017/918082606545039360_0.jpg', 'dev', '5e2aae1b51c314ccc80cd7159cca44db9a0695d754b01be2e5148b3cfeea4541'), ('920169674553278464_0', 'data_image/california_wildfires/17_10_2017/920169674553278464_0.jpg', 'dev', 'b582074b56f8b8e6ea6f50540d1dda9ddb8a22dbbcade92828cd24d393cf6b3c'), ('923272962748440576_0', 'data_image/california_wildfires/25_10_2017/923272962748440576_0.jpg', 'test', '5d2b3bd6e31796ce41062a8d4b70648e1450be6204e5d3de085e7f5488d422fb'), ('923649563746897920_0', 'data_image/california_wildfires/26_10_2017/923649563746897920_0.jpg', 'train', 'bc494a57941b9010369742e66c3b32f98ff3e4a814590e3759b8dd92de633af0'), ('917864819994263553_0', 'data_image/california_wildfires/10_10_2017/917864819994263553_0.jpg', 'dev', '52aeb4c2c653ce7114ef658f2f08da2473ea4f41d96028c6941843dbc977d6de'), ('919828385618186240_0', 'data_image/california_wildfires/16_10_2017/919828385618186240_0.jpg', 'dev', '4ef0d8d24d57a677f0efc8e1a0459580747b098d9737b00104dcf2dd031f2ebf'), ('922569327034769408_2', 'data_image/california_wildfires/23_10_2017/922569327034769408_2.jpg', 'train', '39dc8ce71f23e490bb7433bc41b099c2f05ae1afe6f552235c5bb95e835dc01a'), ('919571896672206848_0', 'data_image/california_wildfires/15_10_2017/919571896672206848_0.jpg', 'test', '51a9d8ca5c0b3046f36a37a24feedc1fc1ac0e90010a9c95f6c6ba52ebad62c9'), ('918318861522313216_0', 'data_image/california_wildfires/12_10_2017/918318861522313216_0.jpg', 'train', '75f61a1dd5d2a85a8846b27d9d305eb21f4ea8e439bb9699165e3eb97128c2b2'), ('920916661519007745_0', 'data_image/california_wildfires/19_10_2017/920916661519007745_0.jpg', 'train', '8834e0e25d73a594ab1796095a56e7dd57c112e9588f6d424e7fb72bead0b7a6'), ('918242805532651520_0', 'data_image/california_wildfires/11_10_2017/918242805532651520_0.jpg', 'train', 'a421fef59e83d05155f430b7f66c58c8a17e93e8170d90c1830e45a261b6215c'), ('919938613466533888_0', 'data_image/california_wildfires/16_10_2017/919938613466533888_0.jpg', 'train', '067351d64979ecd6dc51e69964ffcb3fe6156d5b0d76c56a73145e9316cfa2c3'), ('920307211666116608_1', 'data_image/california_wildfires/17_10_2017/920307211666116608_1.jpg', 'train', '3ad5edbd3768bda38d454cfa59c42a152c774548abe8952d252c52065b77fc20'), ('919933475788066817_0', 'data_image/california_wildfires/16_10_2017/919933475788066817_0.jpg', 'dev', 'cb7eef9f87e96daa96a2d0f154855de2818b08f2104c5ea5076e65e2cd05b0bb'), ('919573757567651841_0', 'data_image/california_wildfires/15_10_2017/919573757567651841_0.jpg', 'train', 'a7d36a1afb7497de90dc93cdbfe39c2da99bc1c0242cd148cfcf40b23753fb68'), ('923725791086526466_0', 'data_image/california_wildfires/27_10_2017/923725791086526466_0.jpg', 'train', 'e1601f70376a9f56152c6bd4e69c2611651de8c9a358893282743b609b837ae5'), ('920770106623496192_0', 'data_image/california_wildfires/18_10_2017/920770106623496192_0.jpg', 'test', '1312d3fb39b2ae349299b20f3022cd7508398139adf1861dc86c2a5888caf312'), ('906020553558917120_0', 'data_image/hurricane_harvey/8_9_2017/906020553558917120_0.jpg', 'train', 'd8e16d1952b22554740a038048639e66cdc258c2e3f36143230b18a41b5e54de'), ('907621145486217216_0', 'data_image/hurricane_harvey/12_9_2017/907621145486217216_0.jpg', 'test', '6af1a9c3e373f885e74727c55e87fcda94faff056c98d08f14eff548905ef65b'), ('905109636356780032_0', 'data_image/hurricane_harvey/5_9_2017/905109636356780032_0.jpg', 'dev', 'ebb73f105da41208c4aa445e0671677ac471ecd0fbff1aa21e72248d8ef2ccf1'), ('901826639310540801_0', 'data_image/hurricane_harvey/27_8_2017/901826639310540801_0.jpg', 'dev', '31de2d5576fa9c2309b58623975d25134297fe528722286ad71b59f696dbcc16'), ('908497182105460738_0', 'data_image/hurricane_harvey/15_9_2017/908497182105460738_0.jpg', 'test', 'f6d2ed9e2b570307e1f58460e2d05900464676a66f28ceb7c38f2c7571e50484'), ('904697308947914753_0', 'data_image/hurricane_harvey/4_9_2017/904697308947914753_0.jpg', 'train', '1242f866a6db7e45723317fa27289eef9017944ac0b51ea48c9da0fdacd06fca'), ('904287900241719296_0', 'data_image/hurricane_harvey/3_9_2017/904287900241719296_0.jpg', 'dev', 'b76d75e64c68e056714944423028b6c3ac150b25413bc4fb3083493172dac388'), ('901743168189812736_0', 'data_image/hurricane_harvey/27_8_2017/901743168189812736_0.jpg', 'train', 'fcbc4d25cd50cd9363e931c18c99f5f2b27fc2878e2be4319f11d956587b1c6d'), ('909873652493160448_0', 'data_image/hurricane_harvey/18_9_2017/909873652493160448_0.jpg', 'dev', '2a11c961242a2320aa3290b3b300cc755795a4c16c2bb5d55f7389b18c1921a7'), ('901646074527535105_1', 'data_image/hurricane_harvey/27_8_2017/901646074527535105_1.jpg', 'test', '3579f6a09a1448a13c03bf8d4e41c5734a603abe22013b58e9ef05cc24acd2d0'), ('904425777072140289_0', 'data_image/hurricane_harvey/3_9_2017/904425777072140289_0.jpg', 'train', 'e26ea2a2b7ad9cac021c316d4b96125f87b2314faf938f3183552c5145318d58'), ('901650362410688513_0', 'data_image/hurricane_harvey/27_8_2017/901650362410688513_0.jpg', 'dev', '4d4961db2d7d73215bb4cdf911b7920eff770f187526977efc81707d18cce918'), ('906563716724019202_0', 'data_image/hurricane_harvey/9_9_2017/906563716724019202_0.jpg', 'train', '46969da47f6b651aa6ad4e2e1af51e2b8f47bee432f73baae8319e2a9f392d8a'), ('907280511709655040_0', 'data_image/hurricane_harvey/11_9_2017/907280511709655040_0.jpg', 'dev', '6d54590d2166e248a64d69bfb979bb7f08fa6f305ddc884e5965722d6172fe8e'), ('901775331530465280_0', 'data_image/hurricane_harvey/27_8_2017/901775331530465280_0.jpg', 'dev', '59bb28b34a62185254ad97e5ec116d13cb4689f9af9a28183948a9e0bdc1970b'), ('907082323614728193_0', 'data_image/hurricane_harvey/11_9_2017/907082323614728193_0.jpg', 'dev', '37f4d8a731f45624c3b57b640527075f0728cbbd758e537f2dee9b22a8f7f817'), ('908854000518234113_0', 'data_image/hurricane_harvey/16_9_2017/908854000518234113_0.jpg', 'train', 'a928006a76ca449a16f7271901c8c8453ca9d5a77a93457a9ae23fb55c51dd3b'), ('907984936598208512_0', 'data_image/hurricane_harvey/13_9_2017/907984936598208512_0.jpg', 'train', 'f9ee65c16e2953a3deb2944ac9fa55aad2b881ac8c22e0963189ed399b9b33e9'), ('906220629812273153_0', 'data_image/hurricane_harvey/8_9_2017/906220629812273153_0.jpg', 'dev', 'a915558f26f1c06d6f2c0bc57fa0f78e9498809ca73e9294573d6a12e2df100e'), ('905732581005590528_0', 'data_image/hurricane_harvey/7_9_2017/905732581005590528_0.jpg', 'train', '035c9d1cda5b339e922203c20960212afa050bca21bd85fd57d2f7bd16619133'), ('906729883396485121_2', 'data_image/hurricane_harvey/10_9_2017/906729883396485121_2.jpg', 'train', 'f19babe8202bf54952201c4645c7be738a8cbabf0ab6957022a3d775aac432f1'), ('909977356546318336_3', 'data_image/hurricane_harvey/19_9_2017/909977356546318336_3.jpg', 'train', '6708747dc2a1b32d386869b374518fd387a3e24cb95269137684241813f321cb'), ('904343743075160069_0', 'data_image/hurricane_harvey/3_9_2017/904343743075160069_0.jpg', 'test', '197fe8589dcdcc0ea5ebfcf993c3d48e8f81aa3deeb480c37e4e06c4074925c8'), ('909176906137948165_0', 'data_image/hurricane_harvey/16_9_2017/909176906137948165_0.jpg', 'dev', '0a603314c3cb1deb0d708a70cd4cebc7198b4b794c96587b36d5ec0751a4a432'), ('905522969723666432_1', 'data_image/hurricane_harvey/6_9_2017/905522969723666432_1.jpg', 'train', '062c765130a2b7727a4f5898bddbe43d8840d31c699b9c6069b50b597458811b'), ('908752026502393856_0', 'data_image/hurricane_harvey/15_9_2017/908752026502393856_0.jpg', 'train', '7dd97562aeb810592bc2947b7537bdcce3c582c59a3b155bec90e09c35d04e32'), ('907952373145980928_0', 'data_image/hurricane_harvey/13_9_2017/907952373145980928_0.jpg', 'train', '16182cac02f065558c302351dbc09e00e6fe4a09d30bf3484404ad7d74840a59'), ('909442846654001152_0', 'data_image/hurricane_irma/17_9_2017/909442846654001152_0.jpg', 'train', 'df1a9c9634782287cd5b31159a4919c09a29d15e174eb7ac17761ca04908dd14'), ('910243544551628800_1', 'data_image/hurricane_irma/19_9_2017/910243544551628800_1.jpg', 'train', '3ce9f8e17ac5d0017a92e7b70fd0cbd6624b58a715af52082d484559f57e4088'), ('910243544551628800_0', 'data_image/hurricane_irma/19_9_2017/910243544551628800_0.jpg', 'train', '2519c970618a586342868d9c556d8b6eb5ddcc85a3050fa353c0bee2a96a6ea3'), ('910152749551144962_0', 'data_image/hurricane_irma/19_9_2017/910152749551144962_0.jpg', 'test', 'becc4de247e89a71622262d813c0e04d912121c2aa9a97d532f8723ab9372685'), ('909454690273730560_0', 'data_image/hurricane_irma/17_9_2017/909454690273730560_0.jpg', 'train', 'c101b9f6c150e3e21b077c713a1151dbc9151217ee4ff7fe2763a0ffc122d91c'), ('909835505747054592_0', 'data_image/hurricane_irma/18_9_2017/909835505747054592_0.jpg', 'test', 'c96006fb0336e93405cf4d20a15cce2a0f047a2210286d447c37f4ef9c3eb705'), ('910192744836059136_0', 'data_image/hurricane_irma/19_9_2017/910192744836059136_0.jpg', 'test', '1b44469b58fac5e87f19ab34864d302e101658f727a5dda792afaffad09916de'), ('909738182031413248_1', 'data_image/hurricane_irma/18_9_2017/909738182031413248_1.jpg', 'dev', '1a45dfd6399604755434bf3fa6b2d6fa3ace3c25eda21ae9379ce95f7e077bb0'), ('905658056465199105_0', 'data_image/hurricane_irma/7_9_2017/905658056465199105_0.jpg', 'train', 'be2022eb8025daf1474bf07f0c479647efabcd17eea040e7aa7e4a07949ba131'), ('909725929714405377_0', 'data_image/hurricane_irma/18_9_2017/909725929714405377_0.jpg', 'test', '0951e55b5d849aaaa7dd30817eb82be8f68572a65a5a9f0c2cf13885a4b6e516'), ('909769034002042881_0', 'data_image/hurricane_irma/18_9_2017/909769034002042881_0.jpg', 'dev', 'f8ab8f24bfa613d886db41c0d6dcf3cd792db2331c2a20b2794f3cc331247935'), ('910123280325124096_0', 'data_image/hurricane_irma/19_9_2017/910123280325124096_0.jpg', 'train', 'e6bbd1497c39327d91aa9129e3777e7163345ca2e72e6b361f543a889b66265c'), ('909935691752144896_0', 'data_image/hurricane_irma/19_9_2017/909935691752144896_0.jpg', 'dev', 'aaa8cf735fc979cecb0785a2cfa270991e608984f63499c06a27161f5eb619ef'), ('905686042526744576_0', 'data_image/hurricane_irma/7_9_2017/905686042526744576_0.jpg', 'dev', '34fdb3d70eaf5549cb50399b1f7a29d8249994acd6ba8d7e7e894db454f5c73c'), ('909863366134566912_0', 'data_image/hurricane_irma/18_9_2017/909863366134566912_0.jpg', 'test', '23dfb976525b3632137ad0601e71e3aee7d5f9ba9dc1c682dfcee1fb0e628bc7'), ('905636651447328768_2', 'data_image/hurricane_irma/7_9_2017/905636651447328768_2.jpg', 'test', '8e81e808c285bb2a8fc5ec7cb08ae592506643214b34022b5d3f4775a140657e'), ('909738182031413248_0', 'data_image/hurricane_irma/18_9_2017/909738182031413248_0.jpg', 'dev', '4c4deca2e2970d70a446a735f6c18ef838bcc4041fa80bec2b960ebe622ca59e'), ('906927991225286657_0', 'data_image/hurricane_irma/10_9_2017/906927991225286657_0.jpg', 'test', 'dc4e771828c0c9178d72d1bed8a756a6c241deff739cc5c895524568cb6722d4'), ('910014707213459456_0', 'data_image/hurricane_irma/19_9_2017/910014707213459456_0.jpg', 'train', '909630e63663766bc0248c998a6c070d24517589ff1cfada0d8226137f8ab31e'), ('909836375716966400_0', 'data_image/hurricane_irma/18_9_2017/909836375716966400_0.jpg', 'train', '794ae2b1e4288b9304132829623ddcda36bb5e0d0bb3cb552cb40e0764b76bef'), ('909778959189016578_0', 'data_image/hurricane_irma/18_9_2017/909778959189016578_0.jpg', 'test', '5e75bf0995095e126e35c70cc6662a6bc4fc050723039930c90ab00c3d965fdc'), ('909864932719685632_0', 'data_image/hurricane_irma/18_9_2017/909864932719685632_0.jpg', 'dev', 'fd098812a340a7c7e1adf113c04f3f1d8dd9c4ab39a291b0fdcc0d3a9023aefb'), ('909384142768492544_0', 'data_image/hurricane_irma/17_9_2017/909384142768492544_0.jpg', 'train', 'b36cb5a8515c34fb06693061e7761cbb4da73f6f2f234cec8c67cb8e4ffb4ae9'), ('909454468135161858_0', 'data_image/hurricane_irma/17_9_2017/909454468135161858_0.jpg', 'train', '70ac79c92131a22460546ab16ac42253bdf35c6ee2d934ba3dab03dfaed73e61'), ('910166121650700288_0', 'data_image/hurricane_irma/19_9_2017/910166121650700288_0.jpg', 'train', '08be206c03e51dde408f144aa0e0f58f2130a9dee32bfdf164be9eca3a9fe483'), ('905670102833446912_0', 'data_image/hurricane_irma/7_9_2017/905670102833446912_0.jpg', 'test', 'f695aa405b85104a2b3ead30e4fdfc72fdefefda1bc6cc7b9007ee7e29aeb792'), ('909394219235672064_0', 'data_image/hurricane_irma/17_9_2017/909394219235672064_0.jpg', 'test', '8f7d6d7b99a2db0b50d80e51eaa77d0e69cb3a7ed67a2cec6c1e171daac589a2'), ('909925028581511168_0', 'data_image/hurricane_irma/18_9_2017/909925028581511168_0.jpg', 'dev', '390925e42650cb9ca99c305f3d98366ed9b8ba459ee33831627b95cfe6ce7eb8'), ('910132205812371456_0', 'data_image/hurricane_irma/19_9_2017/910132205812371456_0.jpg', 'train', 'c4e89403987d36488335b1ef262cc9a32d7937dda27cbee27eb12b9218e6abfe'), ('909997346477494272_0', 'data_image/hurricane_irma/19_9_2017/909997346477494272_0.jpg', 'train', 'b25e8bd443112a84defb3e14ddef27542665eaeb9f818944d9bbe62977f5f7f6'), ('910127478315855872_0', 'data_image/hurricane_irma/19_9_2017/910127478315855872_0.jpg', 'train', '6a70bfc212a13a8f281f22df0b6999865e59946d406aa4e623b14cfe93b66a9f'), ('913833524901187585_0', 'data_image/hurricane_maria/29_9_2017/913833524901187585_0.jpg', 'dev', '4c67018b6ddc8004e788cfd84e954d1876b5c00a40a8c721e95b5a7ae5395d46'), ('927914666180964354_0', 'data_image/hurricane_maria/7_11_2017/927914666180964354_0.jpg', 'dev', '0566fb5520cdfdcdd48809ff15ef553bedbf011ee044acd3cd88f9b193e735b5'), ('913053391244345344_0', 'data_image/hurricane_maria/27_9_2017/913053391244345344_0.jpg', 'train', '98ceb6855ee6377714cb616e07979da01dc8ffb2aba7bab2087622fd04e42b84'), ('928447589287989250_0', 'data_image/hurricane_maria/9_11_2017/928447589287989250_0.jpg', 'train', '673d5b813adc698a8acf410d9d1987e42e7c81b5ae837e730f7609edd1bdc359'), ('912443382525460480_0', 'data_image/hurricane_maria/25_9_2017/912443382525460480_0.jpg', 'test', '648c0e554ec0d880a954fd2892807c4d1c9e55cbaed50cdda947f91d1261dc00'), ('922015756140843008_0', 'data_image/hurricane_maria/22_10_2017/922015756140843008_0.jpg', 'train', '3b1b5da298775304fd442d7fdf9f877681ed1c5c0f8f266ff29baa3b99b54b0f'), ('913897089691774981_0', 'data_image/hurricane_maria/29_9_2017/913897089691774981_0.jpg', 'test', 'c48b39cd207f8d8ff0b3945d0c94720c4ba0a8bb1cacba014a8dbe227f5bce46'), ('923305829629136896_0', 'data_image/hurricane_maria/25_10_2017/923305829629136896_0.jpg', 'test', 'b4939d1afde67b9f46c2d8e58d5c65586e86ac6295c0766a34f63f63e4d975e3'), ('923642736300961799_0', 'data_image/hurricane_maria/26_10_2017/923642736300961799_0.jpg', 'dev', 'cf32b3c8196cc4d889b4280d2c31c638f4022e09c0f5ec49870711a3bfee4520'), ('921437685495029761_0', 'data_image/hurricane_maria/20_10_2017/921437685495029761_0.jpg', 'train', 'bada0d67aa34f707b703e1e82c5304d673092d59b2f62bb7a4ef62f4c92cecd6'), ('916061785404379143_0', 'data_image/hurricane_maria/5_10_2017/916061785404379143_0.jpg', 'dev', '5a7812aa6cc58f338b9dda52383b971309528c799f83a1f11501eb26d562b8d7'), ('910723120297684992_0', 'data_image/hurricane_maria/21_9_2017/910723120297684992_0.jpg', 'dev', 'b8196e89aab67717d4bbccfd2ed22de80bb9be8226460de9b42767fe0a76c99f'), ('911881820559544320_3', 'data_image/hurricane_maria/24_9_2017/911881820559544320_3.jpg', 'train', '506e023929a4a7cbd5fbf52c2ee4ae83586e5210778e4b4caf9ac5d7c6c1f45e'), ('913845125943500801_0', 'data_image/hurricane_maria/29_9_2017/913845125943500801_0.jpg', 'test', '2d4ac0ce3c2235ade4faee21a88635ff506d6c01d3b9d56e54d9755fd5a44443'), ('911881820559544320_0', 'data_image/hurricane_maria/24_9_2017/911881820559544320_0.jpg', 'train', 'd563d36100533b9dba0cc679aeea4bca44f3215a690b2a1a23fd1a8039dc4177'), ('927930744940498944_0', 'data_image/hurricane_maria/7_11_2017/927930744940498944_0.jpg', 'test', '99aad7c46179e38eade31fdcc1518a69d4a0a7fb99fd51a38a45a93a460a8e08'), ('914126824497487872_0', 'data_image/hurricane_maria/30_9_2017/914126824497487872_0.jpg', 'train', '38d0b496d37bd6acdf1310a3b718bc47fa6b9d57314b4172f2a8425184115136'), ('913548443917750272_0', 'data_image/hurricane_maria/28_9_2017/913548443917750272_0.jpg', 'train', '4a7f887e0a8e840fb24a529d4304b402bb265a7a78adb656421e4bd66a5aed1a'), ('915287687552057344_0', 'data_image/hurricane_maria/3_10_2017/915287687552057344_0.jpg', 'train', '0bec90c755c14444ea3bc35a9c276455f2dd7a2a28cbfe4734be4f12276fb164'), ('915986915509145601_0', 'data_image/hurricane_maria/5_10_2017/915986915509145601_0.jpg', 'dev', 'b564947132be7fb3a98cdfccb9d93531fe1855a5fb5ddcd2de2b04a43da04764'), ('910699932704796672_0', 'data_image/hurricane_maria/21_9_2017/910699932704796672_0.jpg', 'train', 'bec786857e4bbf5fad3072f4f7b06366cf951aece04649c04131056c93035c49'), ('927701436699430912_0', 'data_image/hurricane_maria/7_11_2017/927701436699430912_0.jpg', 'test', '5eadfcfbe732aa1b4b76b02e1d1fc3207444d909c2a7cd6838e824e7f71436cc'), ('910524110828171264_0', 'data_image/hurricane_maria/20_9_2017/910524110828171264_0.jpg', 'test', '02e2733bbe90ccc0b881f44e77aaafa4a6311dbe4d342dc40a730a02b4582add'), ('927244371606589441_0', 'data_image/hurricane_maria/5_11_2017/927244371606589441_0.jpg', 'train', '011797fdd4e544fd310cdf2878debc97e26ca2fe4d60fdfef2f2db218b5ff645'), ('911907488756387841_0', 'data_image/hurricane_maria/24_9_2017/911907488756387841_0.jpg', 'train', '1287a7e9d842242bc3c55d3d70ae423f85406b215a5612c63de5cfd3167ab894'), ('922438575991459840_1', 'data_image/hurricane_maria/23_10_2017/922438575991459840_1.jpg', 'train', '23aae306d7f8e46e4190ed5a948212749995452d842072759c354d310ff78388'), ('912075982164643840_0', 'data_image/hurricane_maria/24_9_2017/912075982164643840_0.jpg', 'test', '2334bf937276269c086b5175e0747f5e8939bb89d90d912a1c258029e0649eae'), ('915313085497380869_0', 'data_image/hurricane_maria/3_10_2017/915313085497380869_0.jpg', 'dev', '2b461868ce981b8f4ce1ed4c5005c438cae9be2a992730d08d342806c0fd021b'), ('928716883930550272_1', 'data_image/hurricane_maria/9_11_2017/928716883930550272_1.jpg', 'train', '9c1c7942fd8addcdcd9bf1b59c9ef813afdaed878c4abe66404fb33e4fbf4fbf'), ('910574994127761408_0', 'data_image/hurricane_maria/20_9_2017/910574994127761408_0.jpg', 'train', '492688dba2ad17437e22771987182bd07e974451580af74d7dfb4a8390f2d6ac'), ('913419802525433856_0', 'data_image/hurricane_maria/28_9_2017/913419802525433856_0.jpg', 'train', '6047d6def90946aee7206bb19c2907063643c49aa3e732618fcffe2bdf1895f2'), ('930324267689959425_0', 'data_image/iraq_iran_earthquake/14_11_2017/930324267689959425_0.jpg', 'test', '988e3da5cbb731221d745ecd5698a5513096ba0fd4f8cb89b15542147a3a6792'), ('930564959200858113_0', 'data_image/iraq_iran_earthquake/14_11_2017/930564959200858113_0.jpg', 'test', '851dbc1206a551771698810662875870801732a93e9e921c6ea58163b4f97097'), ('930001999155945472_3', 'data_image/iraq_iran_earthquake/13_11_2017/930001999155945472_3.jpg', 'train', '776936dfe7ba01f69e9f5cc43f3fcc652897f1279dda8756fc0c28d17e942a92'), ('930631448167632896_0', 'data_image/iraq_iran_earthquake/15_11_2017/930631448167632896_0.jpg', 'dev', 'e3f27a6eaa8b4cf0bfba3ad99ad98d70ca275192587bc7d6bb662d6c175dbdee'), ('930383150022193152_0', 'data_image/iraq_iran_earthquake/14_11_2017/930383150022193152_0.jpg', 'test', 'e965758d3580fd314194976aa6f49a5fa6e068960944d4b7da08f912a3658e81'), ('930076476569018368_3', 'data_image/iraq_iran_earthquake/13_11_2017/930076476569018368_3.jpg', 'train', '84187005949f545d8cd27df23f843aab0c6dbe729c567e9eb98b5a6e39cb95c7'), ('930016742730018816_0', 'data_image/iraq_iran_earthquake/13_11_2017/930016742730018816_0.jpg', 'train', 'c6aae9a8af0110c16167f3cdf1a757e89d1c3bc6b908d0a75922b32ea54b8ea9'), ('930020107698556929_0', 'data_image/iraq_iran_earthquake/13_11_2017/930020107698556929_0.jpg', 'train', '9d6e36760076efce7164a80d004c592e83c4cb524538b4cfd6c29b9403cba691'), ('930693871239073795_1', 'data_image/iraq_iran_earthquake/15_11_2017/930693871239073795_1.jpg', 'train', '5c6b3defd3f374403d0b4f88afbbbdf3b8f054723a841938b5d7874b1bf046fb'), ('930068973206622213_0', 'data_image/iraq_iran_earthquake/13_11_2017/930068973206622213_0.jpg', 'train', '58ffcd653d5b23d20c1a18bdcb01dc1a6f35ab432e1fc495faf4a165949a85d9'), ('931880697752567808_0', 'data_image/iraq_iran_earthquake/18_11_2017/931880697752567808_0.jpg', 'test', '05e2974afef9db745966f5f1a339033754df6def36d97c10c8f680fd0a7ca3ec'), ('930435764336209920_0', 'data_image/iraq_iran_earthquake/14_11_2017/930435764336209920_0.jpg', 'dev', '53ec24e11046ed246410da6d62b7b909755811f23e07923c3a102728d8a6b62a'), ('930090178990895106_0', 'data_image/iraq_iran_earthquake/13_11_2017/930090178990895106_0.jpg', 'dev', '81d60b21f26bee10237b752a13ba3048ac316788ae43b32cfc59e04ab8673810'), ('930767353658437634_0', 'data_image/iraq_iran_earthquake/15_11_2017/930767353658437634_0.jpg', 'train', '455627a49114fe6005f88e4991ecfb935207876f98122f0cae2019262df91091'), ('930151625032740865_2', 'data_image/iraq_iran_earthquake/13_11_2017/930151625032740865_2.jpg', 'train', '417da3f247843b77f6ea83c3ea180824b27743d3235c69fd21d0d29b71f670d4'), ('930339539733831680_0', 'data_image/iraq_iran_earthquake/14_11_2017/930339539733831680_0.jpg', 'dev', '15077be681e3a1987599ee46e20e63a114c372d565ab45f8fe85d611fa2a475b'), ('911621942536474624_0', 'data_image/mexico_earthquake/23_9_2017/911621942536474624_0.jpg', 'test', '4495b07f1a1318a58d983fa2bb6efd9f70fccf358b9498b8bc932290f21dbc1e'), ('914104822131040256_0', 'data_image/mexico_earthquake/30_9_2017/914104822131040256_0.jpg', 'train', '5a69e51b846bdcdcec74be01a96e7ae3918deff5888702c1ffd9cb589bc88e3d'), ('913914328348114944_0', 'data_image/mexico_earthquake/29_9_2017/913914328348114944_0.jpg', 'dev', '258a344db53bc9bf35e40af3eea611ed00cf7f63dd7ccf8ebb03f9236cdc1094'), ('910523388598280192_0', 'data_image/mexico_earthquake/20_9_2017/910523388598280192_0.jpg', 'train', '9bfbf99f085e78726615ac096550c862dbaebdb4662181830d899d6ef6806a18'), ('912111866595471360_1', 'data_image/mexico_earthquake/25_9_2017/912111866595471360_1.jpg', 'dev', '38870e4d4799f7e00933d47759b0b7f43c4e1106a471c715a7776e9ce64ab5d5'), ('910723569499308032_0', 'data_image/mexico_earthquake/21_9_2017/910723569499308032_0.jpg', 'test', '6ef6b640bbd99af0ba1fef0778ef9b587edf217c56f2c84c69b69ed4789cee05'), ('910744812722237441_0', 'data_image/mexico_earthquake/21_9_2017/910744812722237441_0.jpg', 'train', '0f0a451828d3e8e00f54b64f4e7cb8b7cbf8178c9ceeb2714949a02791e23981'), ('911673077100146688_0', 'data_image/mexico_earthquake/23_9_2017/911673077100146688_0.jpg', 'train', '2a8df15a265acd7f71b723b9a3cc0ed2d04166952c37d1cf41ceae06f92cc18e'), ('913886101688979456_0', 'data_image/mexico_earthquake/29_9_2017/913886101688979456_0.jpg', 'test', '20994c58b2af17ad5b13dab89d8b7980a77c43f6e98df8e0789aaca5c7e8ffc2'), ('912205877121298432_0', 'data_image/mexico_earthquake/25_9_2017/912205877121298432_0.jpg', 'test', '66d3474b478af1d37a9b6755be34d9132a03144a03b510ecdd95373dad47e003'), ('912466253150851073_0', 'data_image/mexico_earthquake/25_9_2017/912466253150851073_0.jpg', 'train', 'babf31e709a455ae971100317d4f899c12e0387daa828d42c01fe0ca051edd24'), ('910627398789541889_0', 'data_image/mexico_earthquake/20_9_2017/910627398789541889_0.jpg', 'train', 'bef3608c21ceb8ba3ef6465681300bad9be41f620bea69f88e0a890ed8471a95'), ('910705424764211200_0', 'data_image/mexico_earthquake/21_9_2017/910705424764211200_0.jpg', 'test', '9d4a1bd3ec842fa145e479eef242872ee53fb2d93ecec6317a3324a5d1748bcf'), ('910577546978263042_0', 'data_image/mexico_earthquake/20_9_2017/910577546978263042_0.jpg', 'test', 'bca86776e1f77783f048530853f6ffade1d89a7fe04bf60da6823b755b581ba6'), ('912340425566023680_0', 'data_image/mexico_earthquake/25_9_2017/912340425566023680_0.jpg', 'test', '43adcd43d326ea604f4b8e25474d6473e62bd8b97f6dfb3e3ec4a199b1de3735'), ('912122876546953216_0', 'data_image/mexico_earthquake/25_9_2017/912122876546953216_0.jpg', 'train', 'ba65b53b400f1320e5f82d7815a683f8481d4b6fc0af36dd0c06b220e9f346f8'), ('910625461402607617_0', 'data_image/mexico_earthquake/20_9_2017/910625461402607617_0.jpg', 'train', 'd7abbbcad7d8fb3a9b03b6a7eecce440dc3af4f161a8de99e96d5166a38537a8'), ('872400492705361921_0', 'data_image/srilanka_floods/7_6_2017/872400492705361921_0.jpg', 'train', 'b278319eb0eaf6a6d678219c9c79e9e2cb8673be7fdeb8987f2fe653ed3db569'), ('870013706935119872_0', 'data_image/srilanka_floods/31_5_2017/870013706935119872_0.jpg', 'dev', 'f08e635360e432d8e27fd35503ab9239f71d2ef0d7e03c782b09eb92fb5b999c'), ('870574893280092160_1', 'data_image/srilanka_floods/2_6_2017/870574893280092160_1.jpg', 'train', 'ee4e5b0837a785fa5d798b973b509ea5800f75b78f8647d7ba9f4f54da263a86'), ('870668890564567040_0', 'data_image/srilanka_floods/2_6_2017/870668890564567040_0.jpg', 'train', '20e0f7a1ed8c60695907e6538fc18804ae155a45e9c7d3054ed8b4b2f83f2c85'), ('872235889820291072_0', 'data_image/srilanka_floods/6_6_2017/872235889820291072_0.jpg', 'train', 'f45df34f93da3f9b4c756742ae680319a4e371a6bd1547d1c361dfc09332478d'), ('870161183927660544_0', 'data_image/srilanka_floods/1_6_2017/870161183927660544_0.jpg', 'train', '55156dc1d05ac602eec5b9d1a233655dbebb5a60f5220862195f95b8fa41901a'), ('879128419987988480_0', 'data_image/srilanka_floods/26_6_2017/879128419987988480_0.jpg', 'train', 'bd51740b5f7a925ccf323df1f7ff131369e7a36b85a8599a8beeee378480139d'), ('880766663418945537_0', 'data_image/srilanka_floods/30_6_2017/880766663418945537_0.jpg', 'test', '96025bf1912ddb33346c176fec7bb79b0a1ac706f3b9618bce53b6edb7f5f280'), ('870237610362380289_0', 'data_image/srilanka_floods/1_6_2017/870237610362380289_0.jpg', 'test', '8eaa9b91e0adbb7804a8eef3fa4b47d307d43b5e4573660dd5a43adf92ba5780'), ('875461354584289281_2', 'data_image/srilanka_floods/15_6_2017/875461354584289281_2.jpg', 'train', '71e85a553541b9fa199b7c88ef8edac8f51dc0f118b5547aaa42aa543c2b54ec'), ('870885758055333888_0', 'data_image/srilanka_floods/3_6_2017/870885758055333888_0.jpg', 'train', '0d6db0e4aa5b99d057f451cb1a0859a0022fbc4109a4f987c7c39c0326860125'), ('917945356696072192_0', 'data_image/california_wildfires/11_10_2017/917945356696072192_0.jpg', 'test', '1c66dc4b0c31f8c7d3ff3cd95615247e62f8e43aaafd9590342fa6fdd19d1f08'), ('919641467206774784_0', 'data_image/california_wildfires/15_10_2017/919641467206774784_0.jpg', 'train', 'fa13f524b4e8fa5a7ea174e0c15c59b21851a35f8a60494645a29428fe031a4d'), ('920048678252855296_0', 'data_image/california_wildfires/16_10_2017/920048678252855296_0.jpg', 'dev', 'ea19449464f498bae04957807565894dc9d531b734fb2db976e38cc76f4e3ea4'), ('920034434845360133_0', 'data_image/california_wildfires/16_10_2017/920034434845360133_0.jpg', 'test', '0f6763c3b887587edc965e3c56d999625e8d14dba887ef7a9f9a730bac38fad2'), ('922161026912669698_0', 'data_image/california_wildfires/22_10_2017/922161026912669698_0.jpg', 'dev', 'fc20d16f07b1a4d805984034a4388e73aaac881de6034b8e32e54d11366e0862'), ('919566772528865281_0', 'data_image/california_wildfires/15_10_2017/919566772528865281_0.jpg', 'train', '7fc9eeb9d67a297b83a5b534e4aee4c23334362de45db0590dc549030bd8dd32'), ('917920562248269824_0', 'data_image/california_wildfires/11_10_2017/917920562248269824_0.jpg', 'train', 'a282a4fa4e00e2599ad8cc324098c02c3223aad1c5a0f5c1124fb1d6e34664da'), ('920154343738880001_0', 'data_image/california_wildfires/17_10_2017/920154343738880001_0.jpg', 'dev', '5a329a386eea5c654c5336437c2a820db8f78ecaa4f181f508150f93266273bc'), ('921030096051408896_0', 'data_image/california_wildfires/19_10_2017/921030096051408896_0.jpg', 'test', 'c9833d872db8daed319f3568f46845dcf881c16770ba55c2b69e3e06c8d1b772'), ('917794360581869569_0', 'data_image/california_wildfires/10_10_2017/917794360581869569_0.jpg', 'dev', 'e9646a486e9d1cf28ee12be5f99978d4bf9e76ac18ff817af787e06d0130f399'), ('920167628303491073_0', 'data_image/california_wildfires/17_10_2017/920167628303491073_0.jpg', 'train', 'e54011df27335fed917833d39723a298a69d900e3b32cdf4a46983284f963982'), ('919928977673695232_0', 'data_image/california_wildfires/16_10_2017/919928977673695232_0.jpg', 'train', '68558c22e1325196674bc25cb04fe6f17b00510da0d0b1820f1369bed4503b34'), ('922555362267590658_0', 'data_image/california_wildfires/23_10_2017/922555362267590658_0.jpg', 'dev', '55520925a2050b754d22c69a880b91a4639ad2ebdff0c02628745ee0e6b02ebc'), ('919698307655577602_0', 'data_image/california_wildfires/15_10_2017/919698307655577602_0.jpg', 'dev', '3e0e99ea60d2195abd3554b6567d7d9241d407927c373838d6a83b62e79425bd'), ('922018534863011840_0', 'data_image/california_wildfires/22_10_2017/922018534863011840_0.jpg', 'train', '009ccb62d16a82a74ee4b28f917978aa30631dd1d07d840d9e7ea3389a3f9b02'), ('918312725813628928_0', 'data_image/california_wildfires/12_10_2017/918312725813628928_0.jpg', 'dev', 'cbaedeeef974beaaf2480e2a9f96ffd01174b155911628bed09360e8b6355bad'), ('917844519890153472_0', 'data_image/california_wildfires/10_10_2017/917844519890153472_0.jpg', 'test', 'b7ae42e9c80885468e1cf4d10549f3cee55e39a39ea124ba18d3603ab022f441'), ('918041990008328192_0', 'data_image/california_wildfires/11_10_2017/918041990008328192_0.jpg', 'train', 'ea28b527d1604a7ecec0631b8975fc0a717e7206af45482c5eeb2e4fb99abfdd'), ('917926173975474176_0', 'data_image/california_wildfires/11_10_2017/917926173975474176_0.jpg', 'train', '505a81a6aa965043b9f50af7bad5eb730a7d9d60fc8b57450df64dbc75386adb'), ('919575878241280000_0', 'data_image/california_wildfires/15_10_2017/919575878241280000_0.jpg', 'train', 'ef526d80e2efb50d980fb2cf507d2580856440d953e45a0af302bdb0a4e50ff6'), ('919902803761430528_0', 'data_image/california_wildfires/16_10_2017/919902803761430528_0.jpg', 'test', '0221a6494cc99fa3eddb3f57b30f01b9c9fec9ca30cf0871192a335954a46ef2'), ('922144749632479232_3', 'data_image/california_wildfires/22_10_2017/922144749632479232_3.jpg', 'train', 'f0fdff0a9690c432c807ba25d3ad5e5aedda031deabed770d5a574964f07eab7'), ('908270102289805312_0', 'data_image/hurricane_harvey/14_9_2017/908270102289805312_0.jpg', 'dev', 'a34de5639568da4abf4e96e70ffbf2ced7125cefce641f6dc734513d47ceefcc'), ('905578522730799105_0', 'data_image/hurricane_harvey/6_9_2017/905578522730799105_0.jpg', 'dev', '4983fd305b6968058583ec089b94628a3672afec1897496b0e1b25c844c5435f'), ('904796009854754819_0', 'data_image/hurricane_harvey/4_9_2017/904796009854754819_0.jpg', 'test', '47737ab2ce9e5cf136a00e3bd514df646d5f4c440d8e40b389553c45fa457c04'), ('908701985809068034_0', 'data_image/hurricane_harvey/15_9_2017/908701985809068034_0.jpg', 'test', '16357d2f7e90d4de5acad670270d9bb6e869ad29c23f6862e67165933913e7a5'), ('908372808564056064_0', 'data_image/hurricane_harvey/14_9_2017/908372808564056064_0.jpg', 'train', 'b1c19d5bac182edfe54bd2923f188fddeae758fc06e730c543b85f6dbc885ff7'), ('904612540847718400_0', 'data_image/hurricane_harvey/4_9_2017/904612540847718400_0.jpg', 'test', '8c73f89b227458a90f493296069a6e89fc4e16676819cb8f7b5a45713d8beb32'), ('907277725165146112_0', 'data_image/hurricane_harvey/11_9_2017/907277725165146112_0.jpg', 'test', '8a2f58b98b679ea6407bc4ee8d813676ce53e187e70708f949102c37e14cdea1'), ('905082136612540416_0', 'data_image/hurricane_harvey/5_9_2017/905082136612540416_0.jpg', 'train', '4c8b140b660c27e0d5ab0dc97aac68da31ee55f56fbc65f1f586f7c43524b1a9'), ('906635791778160640_0', 'data_image/hurricane_harvey/9_9_2017/906635791778160640_0.jpg', 'train', 'a92d962327c6bd7d43afcdbc802892c036f67c9901d6cd853596bcf38af0d0c4'), ('904780011621670912_0', 'data_image/hurricane_harvey/4_9_2017/904780011621670912_0.jpg', 'train', 'd6aadd46d87d0804a80ae9659d48b7af0451293477c63f5c2eb2360a4b95e83a'), ('904830112671178757_0', 'data_image/hurricane_harvey/4_9_2017/904830112671178757_0.jpg', 'train', '718b03c4e86a6c52547dbc7fbe76880ab107a4dee1bbdd4fd38e85bb59d7d494'), ('901743093493489664_1', 'data_image/hurricane_harvey/27_8_2017/901743093493489664_1.jpg', 'dev', '45c251b5cfa07e547f4107bdb1981f5ebeddb9399d1f1e97580efa9d6c815d41'), ('905777977249681409_3', 'data_image/hurricane_harvey/7_9_2017/905777977249681409_3.jpg', 'dev', '06edccebc5d2ab18b5bdc080295da6091988bc8f271056f4ef902fda2e7eacb1'), ('908712278467465216_2', 'data_image/hurricane_harvey/15_9_2017/908712278467465216_2.jpg', 'train', '1d563bae0fa301941a01497d73051606a218a2e65ea875feaac5aadb8cd7de62'), ('905316510935199745_0', 'data_image/hurricane_harvey/6_9_2017/905316510935199745_0.jpg', 'test', 'd258a1456b5cc90698f9c81016475e6565af2cadb5c327db446f02e46472bfcd'), ('906306158994100225_0', 'data_image/hurricane_harvey/9_9_2017/906306158994100225_0.jpg', 'dev', 'a9369e62b93db6f930448c0e8dc4ceda493053548ba0be1914088a66fba0d3d9'), ('910070012693561344_0', 'data_image/hurricane_harvey/19_9_2017/910070012693561344_0.jpg', 'train', 'f228aa4aa0b9bb7ff97a38c16f7154e4490acd41b24ddb3c572dc03643d2e981'), ('904237573811339264_0', 'data_image/hurricane_harvey/3_9_2017/904237573811339264_0.jpg', 'test', '024cb63a863b843289fcdd1ec5813b5ebae1bfee0e097a65b281b648742a190b'), ('901717268199067648_0', 'data_image/hurricane_harvey/27_8_2017/901717268199067648_0.jpg', 'test', 'fee3e6d348ddfa3ecf2f5e41305cc9f23b6cc20e3f73326aaa0213d33d62c9a0'), ('908308407899377664_0', 'data_image/hurricane_harvey/14_9_2017/908308407899377664_0.jpg', 'train', '1662369b5a2f900578e832cd95773d66d43eb3fa096831b1a79b1867225f837d'), ('905897170007711744_0', 'data_image/hurricane_harvey/7_9_2017/905897170007711744_0.jpg', 'test', 'cf6a2a2c571ecd47583a0c67c8d5fe5003759f55eb5645261f739ca943a16ba9'), ('905963555282984960_0', 'data_image/hurricane_harvey/8_9_2017/905963555282984960_0.jpg', 'dev', '644e1aa6687387c7e2d1af2666709743b9bbe981256bdcea6bcb20ded85ae7b1'), ('904221977925349376_0', 'data_image/hurricane_harvey/3_9_2017/904221977925349376_0.jpg', 'train', '19406e565dcec7151b57edfacc26cd417c3bb0ae1897d8c62695b715d77b961d'), ('904746274834046976_0', 'data_image/hurricane_harvey/4_9_2017/904746274834046976_0.png', 'dev', '91b472c66590e072188c63383859658e7145bf9a4c0cec8d169ba7740649d2c8'), ('905047654543093760_0', 'data_image/hurricane_harvey/5_9_2017/905047654543093760_0.jpg', 'train', '7662300a4015ad08f4ec69e8e83737f46bb0c0c373ebbe6da4eb814dca7c9a67'), ('904918235337785344_0', 'data_image/hurricane_harvey/5_9_2017/904918235337785344_0.jpg', 'test', 'bbcfa8cea64a99997dbb37970338bd04b85fc4822cafbe9167fd7bc5ceb716b7'), ('910178011756027904_0', 'data_image/hurricane_irma/19_9_2017/910178011756027904_0.jpg', 'train', 'e92c98bdb0d595dfb3495b64dedb71f9d5e156ec18fab50f6f88fc799e8d3b5c'), ('909447617230381058_0', 'data_image/hurricane_irma/17_9_2017/909447617230381058_0.jpg', 'test', '883a16292bb221562495663b76937bf8cff645a9853526fd1ffeae7b8ae4784a'), ('909796226169819140_0', 'data_image/hurricane_irma/18_9_2017/909796226169819140_0.jpg', 'test', '5f5638abf51eb0b10cee2338f3218410469525ed410c4124c9ec6d451af831eb'), ('909975544036458497_0', 'data_image/hurricane_irma/19_9_2017/909975544036458497_0.jpg', 'test', 'c3ab22e271d79c24d436aad50bda797d62827cfd6e0c76af0a295dbecb14386b'), ('909398796928303104_1', 'data_image/hurricane_irma/17_9_2017/909398796928303104_1.jpg', 'train', '436d67eb7f8f0d812d8b871f3f9e2add00f3b0e0d7164f5db0b9666f68513248'), ('910066988596305922_0', 'data_image/hurricane_irma/19_9_2017/910066988596305922_0.jpg', 'train', 'f8677e7761c30e2ccb03a800308d917bce54ba3ecd183ecb9e4e7fbb74412ffc'), ('905625009430949888_0', 'data_image/hurricane_irma/7_9_2017/905625009430949888_0.jpg', 'dev', '01eb3a16194f582393f6e6148f3cab5ce25cdd34693da7454f236d00251e3923'), ('909741340887412737_0', 'data_image/hurricane_irma/18_9_2017/909741340887412737_0.jpg', 'train', '597678fc42658f2a667329a7640b6a16afe509abe0e8938d1128b924a5f7a216'), ('909840252239409152_0', 'data_image/hurricane_irma/18_9_2017/909840252239409152_0.jpg', 'train', '0c6c10b1fb43e3649758e4f2369ba94caf7b87cbe88f832b1326d81746ebbbfb'), ('910213624056164352_0', 'data_image/hurricane_irma/19_9_2017/910213624056164352_0.jpg', 'test', 'b8b19ac34dd487b46bcbfc4f4ee888ee21ffdfa570a5e528d02dcd8dfa5add64'), ('909429266697248768_0', 'data_image/hurricane_irma/17_9_2017/909429266697248768_0.jpg', 'test', '49acf9bcc7fac53df341c8b9f8953e2c7a526764d682e3f06f8d4c489122d6ea'), ('905685853892169728_0', 'data_image/hurricane_irma/7_9_2017/905685853892169728_0.jpg', 'train', 'ac58da4cca9241e741a904a627d94ad00988f4190fe261092040a89d8c936b26'), ('909482048653537282_0', 'data_image/hurricane_irma/17_9_2017/909482048653537282_0.jpg', 'dev', 'd8f7184e29fd25b59d207da13458c20275e5d7276063d0198fc603b0e09da19b'), ('909833411598209024_1', 'data_image/hurricane_irma/18_9_2017/909833411598209024_1.jpg', 'train', 'e7d3fbab913532962081ee84d061cb4833aec388999ba414d7119b4fceb9c28d'), ('910166366015119361_0', 'data_image/hurricane_irma/19_9_2017/910166366015119361_0.jpg', 'dev', '09ce2c5b68acd047849b8a2015b0b77e57e01d97e7cbfdab5f2e6496b99781c2'), ('909731790931337217_0', 'data_image/hurricane_irma/18_9_2017/909731790931337217_0.jpg', 'test', 'f51fd7ca2723062a3840674e038fe23f8ae707f39dff990f315c22d1c6b061a7'), ('909450753420513280_0', 'data_image/hurricane_irma/17_9_2017/909450753420513280_0.jpg', 'train', '4ed574faa67d7bca4e71a48f09e1e3ed277fefaee284a5bf09c14552b682a9b6'), ('910163300662562816_0', 'data_image/hurricane_irma/19_9_2017/910163300662562816_0.jpg', 'dev', 'c372a603ba59eb1f0ca6c87d61d3e5abe977e1bebc15c29d18ec7d3ef909350a'), ('910119192594145280_0', 'data_image/hurricane_irma/19_9_2017/910119192594145280_0.jpg', 'train', '20f23c83d6574c3e11d4aee048095da716cc71d081a12364b4489bec521d25b3'), ('909440826933223425_0', 'data_image/hurricane_irma/17_9_2017/909440826933223425_0.jpg', 'test', '1fb503f9403cec757dbc8ba55a341407b1e39a239104b9f913f5499303fd8c5e'), ('905729999235403779_0', 'data_image/hurricane_irma/7_9_2017/905729999235403779_0.jpg', 'train', '2f2088c6fe1e1c8dadeecd61f70e2861d05e173f0651796777036df849c787d9'), ('909407154234384384_0', 'data_image/hurricane_irma/17_9_2017/909407154234384384_0.jpg', 'test', '65ba5367e177fa2c4a27b8bf800c714722e57600cc7bbc802e1f4cca0a7ae631'), ('910189039852097537_0', 'data_image/hurricane_irma/19_9_2017/910189039852097537_0.jpg', 'test', '379043a692fb338a076b5c150a21fa7c87307cafec78852b357f344f360ca0a0'), ('910198685685944321_0', 'data_image/hurricane_irma/19_9_2017/910198685685944321_0.jpg', 'test', '3e5b30f28f7c27f7d57018e9dcfcab597203754c45af0db70f673d8d9c5fd68b'), ('909385864500989952_1', 'data_image/hurricane_irma/17_9_2017/909385864500989952_1.jpg', 'train', '28e30dedec12ae866a1dcbda576bc52a70a3acfd7c157da1f97de1cec889a999'), ('909683788225097728_0', 'data_image/hurricane_irma/18_9_2017/909683788225097728_0.jpg', 'test', 'd773a4833c34f8ecfb95c3eea32ba0f6d504e2fa2e81764f80c4878bfcca5285'), ('909424491427778560_1', 'data_image/hurricane_irma/17_9_2017/909424491427778560_1.jpg', 'train', '120909b86cace6d8291357c4b906abc10b8d98a856398fd2984d7a1b1d5e8583'), ('905685988428656640_0', 'data_image/hurricane_irma/7_9_2017/905685988428656640_0.jpg', 'train', '20e119af4c077f42c1b85ce800cf8367d82f11ccb6db59ecfa294041a6fde298'), ('923338620307730437_0', 'data_image/hurricane_maria/26_10_2017/923338620307730437_0.jpg', 'train', 'f6574b96698ef6d946e0374658827f9251c05d87f36c434345f2a2298e0e3f2c'), ('913428258926063616_0', 'data_image/hurricane_maria/28_9_2017/913428258926063616_0.jpg', 'test', '6804383be399dd93c570d21e19852ce0209e4473af2c95e1ca1f13a6f0d2d734'), ('913450346525184002_0', 'data_image/hurricane_maria/28_9_2017/913450346525184002_0.jpg', 'test', 'aba2ad5fb0baf8a5202bd14af17768e31861192e5d092f049d61859c27c35f8d'), ('921532750552883200_0', 'data_image/hurricane_maria/21_10_2017/921532750552883200_0.jpg', 'dev', 'aa2bf1fe0c2b589fd53c60e3061bf60376ea2ae1e6d83adcf7d8ca057587ab0d'), ('913594815408984065_2', 'data_image/hurricane_maria/29_9_2017/913594815408984065_2.jpg', 'test', '198b15ef919d8fbec0f8348fd77252af30927dfcb847c6c60ea7a0c4f7b8417b'), ('916150091966177280_0', 'data_image/hurricane_maria/6_10_2017/916150091966177280_0.jpg', 'dev', '65b19108bd1044c939cf0b7187cb81b4659e783bf913da8b95490f33d146ee19'), ('911580776759742464_0', 'data_image/hurricane_maria/23_9_2017/911580776759742464_0.jpg', 'train', '1e1dcd3aa9ea110fe7b4ee401620b98021e2f00c28bc0ac7f16ce05a05400a40'), ('910540043126808577_1', 'data_image/hurricane_maria/20_9_2017/910540043126808577_1.jpg', 'dev', 'e22a08968858eb3b23efa886a03784b79ad6afdc1bdff9a4d9df4066e83394ac'), ('923703821431316485_0', 'data_image/hurricane_maria/27_10_2017/923703821431316485_0.jpg', 'train', 'e5b8cbb8527cd50ae9eded313c35acbdfda0e9532639812df53749340620ce9b'), ('910563826424274945_0', 'data_image/hurricane_maria/20_9_2017/910563826424274945_0.jpg', 'train', '96395920188ef55ae7cd273ce14fdc56bb5672ca21f47d9c0e3b060e24fa95b9'), ('922966952166985733_0', 'data_image/hurricane_maria/24_10_2017/922966952166985733_0.jpg', 'dev', '213a14935165d4b31491bfd44b4740c9129a9bab0e2520c7116909a0a6b7f082'), ('913795571193978880_0', 'data_image/hurricane_maria/29_9_2017/913795571193978880_0.jpg', 'train', '123d231f67874afd0bcf300ad17236045b692c62b37f0de412e36450d96f3d5a'), ('923038602132631552_0', 'data_image/hurricane_maria/25_10_2017/923038602132631552_0.jpg', 'test', '88696f9c027af52998859b769d74cfc4d860808ebb4aa99b33d0ecb539fcb609'), ('927949811759824896_0', 'data_image/hurricane_maria/7_11_2017/927949811759824896_0.jpg', 'test', '30b636256124a226b299a7274e774d9051e5652152453c4604b26719ef531f49'), ('922104773821718529_0', 'data_image/hurricane_maria/22_10_2017/922104773821718529_0.jpg', 'train', '6757e33621cefd91dbdb35c1279683b05fab4a6572df121a5836f6ba41defb70'), ('915973610723999745_0', 'data_image/hurricane_maria/5_10_2017/915973610723999745_0.jpg', 'dev', 'c12c40393a515fcae488c0e5d8aba863d35c15769ef92cc109e25e31c161d203'), ('928007930682556416_0', 'data_image/hurricane_maria/7_11_2017/928007930682556416_0.jpg', 'test', '506e3306ecf3c21950dca5dee0f1cdda5d9b7a17b4bd0ad16f5cce87bc09812e'), ('921969679869997057_0', 'data_image/hurricane_maria/22_10_2017/921969679869997057_0.jpg', 'train', 'fac6657b8c491f021ef878365f6062a3cd1cc89747cc9728f4e7219d2aa3b44b'), ('914937344981159936_0', 'data_image/hurricane_maria/2_10_2017/914937344981159936_0.jpg', 'dev', '3412d7adee0e219b6f41c07174e6940cbd86f763b67f63347cba63f2ea0b3892'), ('913104613636808711_0', 'data_image/hurricane_maria/27_9_2017/913104613636808711_0.jpg', 'test', '5b600ff374cf8295898e70bb81bf3b6537ff0d2bc76c61843ded59b80469fbc7'), ('927568993438887940_0', 'data_image/hurricane_maria/6_11_2017/927568993438887940_0.jpg', 'train', '09174f536ea213ada09dc2c6089646d26c53f31caab4966a6da96d36999809a8'), ('910524854612393984_0', 'data_image/hurricane_maria/20_9_2017/910524854612393984_0.jpg', 'train', '521d68c9bc9a970706a3ca78f4a61dba01396169a61ac3d02ad9c0bb9ff4f235'), ('922039995292160000_0', 'data_image/hurricane_maria/22_10_2017/922039995292160000_0.jpg', 'dev', '66d07c7fa147eb3f0ed0889f600594d9732625bbfb57f8e2828920011eb42ab0'), ('914113249934811136_1', 'data_image/hurricane_maria/30_9_2017/914113249934811136_1.jpg', 'train', '30b7a4f301431e17150ad7fd941bafa0cece37df1057c53237f706bcd0f45b9e'), ('914188148623671296_0', 'data_image/hurricane_maria/30_9_2017/914188148623671296_0.jpg', 'test', '6032e50eb3bb9980ab6f6ccd937f3566aa2df1f1d1bfdfd3e460e261cc7f7cb6'), ('922856336915877888_0', 'data_image/hurricane_maria/24_10_2017/922856336915877888_0.jpg', 'dev', 'bcb06b7183e46a9eac3124bdec759a2ce19f6bcf70bc4daa1bfccbed5ba56c26'), ('913843711909429248_0', 'data_image/hurricane_maria/29_9_2017/913843711909429248_0.jpg', 'train', '2eae8e5481b441d1a95311c36c4ae999e710c5c2932116aa64217a0de7e41552'), ('910619430073323521_0', 'data_image/hurricane_maria/20_9_2017/910619430073323521_0.jpg', 'dev', '86183b4380af24ca84c65f8d3c8683bed99e8a11ee9ebcd5e6a3e5d685ce21b5'), ('910781377158692864_0', 'data_image/hurricane_maria/21_9_2017/910781377158692864_0.jpg', 'test', '9741057f7b7e7ccee38e033b6882577ad7bba94c84ff1ea9a273504aba11493e'), ('930102201825091585_0', 'data_image/iraq_iran_earthquake/13_11_2017/930102201825091585_0.jpg', 'train', 'bb41722928a4727ea1e47d62252d751f082a005efd2cfc65927e47bb263fbb42'), ('930415579294584834_0', 'data_image/iraq_iran_earthquake/14_11_2017/930415579294584834_0.jpg', 'test', 'b68ec2f92df2b0014d6360cd970e264dfb40022b02fcf58f18cff51ac7347ae9'), ('930000573411110913_0', 'data_image/iraq_iran_earthquake/13_11_2017/930000573411110913_0.jpg', 'test', 'd3bcc0bf8440644f6ea1328873fdc0c23fa2296ba6c4dbf8e4da8c5afda36da1'), ('929994458153803776_0', 'data_image/iraq_iran_earthquake/13_11_2017/929994458153803776_0.jpg', 'test', '5f4f2cffbf0ebc76a1f788a0e620de70eb75270af04e746b775ef9f1d9035d79'), ('930193943408586752_0', 'data_image/iraq_iran_earthquake/13_11_2017/930193943408586752_0.jpg', 'train', 'c8ab815a69a615c26b19b885eca6b0ad3ced8c2b161a969bb0f8d2dfc84cf419'), ('929989752400564226_0', 'data_image/iraq_iran_earthquake/13_11_2017/929989752400564226_0.jpg', 'test', 'bd25a9aed651d34f333e40910ef512adc279fd6d914d51d1432cb6b62b5a4cc1'), ('930107946285101058_0', 'data_image/iraq_iran_earthquake/13_11_2017/930107946285101058_0.jpg', 'test', '46aa2c152efccaf94c421c14718f022b6dd5912063f84ba4852f643d9ef411e8'), ('930693871239073795_2', 'data_image/iraq_iran_earthquake/15_11_2017/930693871239073795_2.jpg', 'train', '5469f6aa59a989ea886ac73aea0d032b9110a346a90572b4b4b9335a56c0171b'), ('930054026133213186_0', 'data_image/iraq_iran_earthquake/13_11_2017/930054026133213186_0.jpg', 'test', '73996f90f49423114d7543c48ce69972b599cd64686b3f550aef715f4273b151'), ('930014391596208128_0', 'data_image/iraq_iran_earthquake/13_11_2017/930014391596208128_0.jpg', 'test', '0316257f3afc68f738196b342e400b71a59d5124093640dc4a480f611cf5a27f'), ('930014288714092544_0', 'data_image/iraq_iran_earthquake/13_11_2017/930014288714092544_0.jpg', 'train', '31293e8edb82651d2bccc528bbac161be97fee17fd9a41f86525520011d62e6b'), ('930337898217594880_0', 'data_image/iraq_iran_earthquake/14_11_2017/930337898217594880_0.jpg', 'train', '87fcc4c2b562847f982b5961c0c82478fff6a85de10d37af4e712ccd25195a6b'), ('930523863691288576_1', 'data_image/iraq_iran_earthquake/14_11_2017/930523863691288576_1.jpg', 'train', '32466eba61d16cb7d3b9fdfa15c3cc23154457bc30f098d782c065d269633959'), ('931192541654941696_0', 'data_image/iraq_iran_earthquake/16_11_2017/931192541654941696_0.jpg', 'test', '27c63396d94265930d1603bb99223afe6a81b183e4fca78f131944fe8942664e'), ('912430414781808640_0', 'data_image/mexico_earthquake/25_9_2017/912430414781808640_0.jpg', 'train', '5cb8c01cb0689bd5389f3a52fea3ac94317f5efdfc86d6c66242be6551312896'), ('910655620566548480_0', 'data_image/mexico_earthquake/21_9_2017/910655620566548480_0.jpg', 'test', 'aa3b1fff9b881a9ce62442f3063ae5dbd718b618dc58a7f9e84678ef0568b362'), ('910623424309088256_0', 'data_image/mexico_earthquake/20_9_2017/910623424309088256_0.jpg', 'train', 'bdf4313efdc703c46ba3fecc10e43b02394a2b0f965852c53742954c5a208899'), ('910542340967718912_0', 'data_image/mexico_earthquake/20_9_2017/910542340967718912_0.jpg', 'train', '8b8ec66e52d4f875bdbd0f0193cee4daabd22fa98758b2120f57be19204608b0'), ('913215455334199301_0', 'data_image/mexico_earthquake/28_9_2017/913215455334199301_0.jpg', 'train', 'c686e61e9c62e002a60af16a07d14099a0468feb3e119b8bf0813067062e7b11'), ('910579578934059008_0', 'data_image/mexico_earthquake/20_9_2017/910579578934059008_0.jpg', 'train', '1656d25c9c0946fe26226cc4c3a957f28f89a70dc6fa05d7206efbf9ff283492'), ('913089638583152641_0', 'data_image/mexico_earthquake/27_9_2017/913089638583152641_0.jpg', 'train', '6eee7b1cc3ff61717202a4216c38f38d24f81abe255e0d4bc627c9170804eab4'), ('911889087782629376_0', 'data_image/mexico_earthquake/24_9_2017/911889087782629376_0.jpg', 'dev', '38e5c67c0bb653ffd15c69222c661e6156bf954d29dd24424ccc1919218e7bb3'), ('910538402759618561_1', 'data_image/mexico_earthquake/20_9_2017/910538402759618561_1.jpg', 'train', '62450152af7eb42d75fea8f01c05dde33813435de9a8fb97afb26f1fc679bff1'), ('910699551673208837_0', 'data_image/mexico_earthquake/21_9_2017/910699551673208837_0.jpg', 'train', '7f0a457af210d0583f5a0b74b7be9a70df092b08cb97560db17060de78ec99af'), ('911674637410594816_0', 'data_image/mexico_earthquake/23_9_2017/911674637410594816_0.jpg', 'train', '067ff0077bb08d993dfd2daa6c679fc05b09e2f5acc2c0585f9f4094bae3311c'), ('912002101324206081_0', 'data_image/mexico_earthquake/24_9_2017/912002101324206081_0.jpg', 'dev', '0d35dff940989f0b5b8433d6babf751f85c57f6ade9f769dfd65fc5ce63ea271'), ('911934448098258945_0', 'data_image/mexico_earthquake/24_9_2017/911934448098258945_0.jpg', 'dev', '8b858443c36abdca5b7c2184dceb5336ab43bd26f4254cd232b5b58ddb4a255b'), ('911683426595590144_0', 'data_image/mexico_earthquake/23_9_2017/911683426595590144_0.jpg', 'train', '38de343b47a192fd162aefe6d0d20a723d93ff3dcefbcdcb4eb5b8a80228e497'), ('910538402759618561_0', 'data_image/mexico_earthquake/20_9_2017/910538402759618561_0.jpg', 'train', '44f780c5ce2653bef08861ee421c9395fd459802be55905c4ffb1cc55e9a5ba8'), ('913052415926161409_0', 'data_image/mexico_earthquake/27_9_2017/913052415926161409_0.jpg', 'test', 'bb7f3fcab7c903c52ebf787df08507bbed456aa1cff1bba402d6de8a3ad44a17'), ('910533045400231936_0', 'data_image/mexico_earthquake/20_9_2017/910533045400231936_0.jpg', 'test', '4c4b7b433f4b9ebb6dcb2252013d3ae8cd9756683fd66df53c9f2c9de1817c14'), ('872517849616613384_0', 'data_image/srilanka_floods/7_6_2017/872517849616613384_0.jpg', 'dev', '2a3ddbc52cde1ba9609b99f1c8c8918a45017b37e04581dea50831065bf0029d'), ('870176000109182977_2', 'data_image/srilanka_floods/1_6_2017/870176000109182977_2.jpg', 'train', 'a41c4800909a1a14ffb553df0df239788e87186f0095c0d2c1e6dae6c07fb84b'), ('871260385252343808_0', 'data_image/srilanka_floods/4_6_2017/871260385252343808_0.jpg', 'train', 'f6d18881183e8cb158dc3964e1a9be316f26620dfb20ee01dc320b74984af476'), ('871245851359948802_0', 'data_image/srilanka_floods/4_6_2017/871245851359948802_0.jpg', 'test', '5160ca39a7d500262837e06f21dba7a2126ffe69a085b30539f427ce464490db'), ('870207474493685761_0', 'data_image/srilanka_floods/1_6_2017/870207474493685761_0.jpg', 'test', '5a23584cb4f56b140ceea890fd1694e2c5280db4d7e724c6205e211b618df400'), ('875443828966273025_0', 'data_image/srilanka_floods/15_6_2017/875443828966273025_0.jpg', 'train', 'cc328f5e68a0bc22561cdc710abd333cbe6798981b6748fc5e43875b61912407'), ('877818186967666689_0', 'data_image/srilanka_floods/22_6_2017/877818186967666689_0.jpg', 'train', 'a29e2766e4bbec46311d929c822eacb1b110ddfddb86ecf7ab4cb9287a470dd7'), ('871271893474492417_0', 'data_image/srilanka_floods/4_6_2017/871271893474492417_0.jpg', 'train', 'e4ed44423763131d877651206889a608ffaf108f54f5417ef6b0973ad9d63487'), ('870845845133971456_2', 'data_image/srilanka_floods/3_6_2017/870845845133971456_2.jpg', 'train', 'ba356d6e1a61c4921a43342d306f797cd23dfb749b1411d1e2bde7f9029f9b79'), ('870272043538886657_3', 'data_image/srilanka_floods/1_6_2017/870272043538886657_3.jpg', 'train', '06bc90e06e8b1755e1049eb0d236ed366b6e5864b475e131742371f196b8eecc'), ('874137651384918016_0', 'data_image/srilanka_floods/12_6_2017/874137651384918016_0.jpg', 'train', '9b02f220579f622259f61b64864ce91bbe3825d00757504323237eb512a9d5be'), ('919563091406286848_0', 'data_image/california_wildfires/15_10_2017/919563091406286848_0.jpg', 'dev', 'f5e46ef5cea17eb3c09e992a1a6d1b24d83bbd67fddb0b026b221ad382db2734'), ('919627708476350465_0', 'data_image/california_wildfires/15_10_2017/919627708476350465_0.jpg', 'train', '4ac8355e2814f3c5d2bd68c6c46bb1bd512fa2a0957a2ffb6c8d1337ce7df605'), ('919784701656625152_0', 'data_image/california_wildfires/16_10_2017/919784701656625152_0.jpg', 'dev', 'c1241a768203d4eb39dfa083963af7d45834063daa532f8435b5c312c5f5e479'), ('920349606717792256_0', 'data_image/california_wildfires/17_10_2017/920349606717792256_0.jpg', 'train', 'bc0e27c131e6c2b7f08ecb7e6fcf373cfedb114ef5d65fc29f24038489bb5d92'), ('920013231862833152_0', 'data_image/california_wildfires/16_10_2017/920013231862833152_0.jpg', 'test', '4b7d0c5171483305ad4c5ea04d53a185788a359a8ca9c6252b29bc85974d0847'), ('917842240646664192_0', 'data_image/california_wildfires/10_10_2017/917842240646664192_0.jpg', 'test', 'e2f2ef41f1e229222976b560331db671733ae863b5589f0e9f9a53a106c49493'), ('917832022017445888_0', 'data_image/california_wildfires/10_10_2017/917832022017445888_0.jpg', 'train', 'c6481d8e31ac65f0a49f0c492ce7fe3f6700142bf89882f07b72836df8b40017'), ('919562980370604032_0', 'data_image/california_wildfires/15_10_2017/919562980370604032_0.jpg', 'test', 'c94dfbf9c6a21ed7a70e4899e8fbf1d480307516bf4957a1e68a67c2f14a70eb'), ('921135437279911936_0', 'data_image/california_wildfires/19_10_2017/921135437279911936_0.png', 'train', '3e2b29d160758ae9b3d8aadc95c58043ef2e4e23bdb8329decada67a4fa18121'), ('919633543474577408_0', 'data_image/california_wildfires/15_10_2017/919633543474577408_0.jpg', 'test', '1612aa7de3090ef2c6b04fef084d61574a0e1836f52944b87fde16f2315b815f'), ('919562679404003328_1', 'data_image/california_wildfires/15_10_2017/919562679404003328_1.jpg', 'train', '43724ab8dc425a35e22dc11510ba65aea54674776f2250d99dcf6d1b280c9ac3'), ('919630733542592512_0', 'data_image/california_wildfires/15_10_2017/919630733542592512_0.jpg', 'train', 'f6bd75921b0859b1d7e14eebd1dc36d0d5c3d5c188210ce22a99e28515a4ad44'), ('921118973743325184_0', 'data_image/california_wildfires/19_10_2017/921118973743325184_0.jpg', 'train', '413f3014fb6c8e51f8deb91e03db54f9988e9958b451468f8d851afdfaf605c4'), ('922601333399719936_0', 'data_image/california_wildfires/23_10_2017/922601333399719936_0.jpg', 'test', '0cc34e5cf562203a51cdb48343b05cdcc0a0289ea294d604be6e58929a0f0bb3'), ('921975972085280769_0', 'data_image/california_wildfires/22_10_2017/921975972085280769_0.jpg', 'dev', '7c7e116b166b2db022b955a3248a14a10ac2f2ec0e058ca733fb4c9cf1bcccfd'), ('920486171985166336_0', 'data_image/california_wildfires/18_10_2017/920486171985166336_0.jpg', 'train', 'a1c0941b882dc24786364485cb2dc0338bd9f2a335278bbdec89bbb8b1ec599c'), ('917932244643311617_0', 'data_image/california_wildfires/11_10_2017/917932244643311617_0.jpg', 'dev', '720ef6df14297b0ec78919008bff73a1feb9a1529a9cfc870bc4a77f35449062'), ('919612428501110784_0', 'data_image/california_wildfires/15_10_2017/919612428501110784_0.jpg', 'train', 'a9e6474d677fd0b359611fd7a39489fea67122c76c47ea0b3168bf94e4f3f2d0'), ('918095867218817026_0', 'data_image/california_wildfires/11_10_2017/918095867218817026_0.jpg', 'dev', 'c4925faa64f4c3407391315818da99d4acbc18a8b6b9ccee0d7409adc3440a87'), ('919979254447820800_0', 'data_image/california_wildfires/16_10_2017/919979254447820800_0.jpg', 'train', 'a5a6466b96823977df173f2ba8c49d3fcea309fb6e81c492c3a67ff6b2cf4285'), ('923224900344545280_0', 'data_image/california_wildfires/25_10_2017/923224900344545280_0.jpg', 'train', '8d09c2b0c6aa49cfcf7b0d71c15540feb40bcc2c005b8bb682ba811479e7df3c'), ('921371052063821824_0', 'data_image/california_wildfires/20_10_2017/921371052063821824_0.jpg', 'dev', '1b103a79873d05c52de51d024357fc5ca78b8678a38b4a653ce772e09a7ebf18'), ('905248425746288641_0', 'data_image/hurricane_harvey/6_9_2017/905248425746288641_0.jpg', 'test', 'dce0b4b742d7ae5561381a8307a6210f1fa0137bcdeba11104effccef397e943'), ('901804808105066496_0', 'data_image/hurricane_harvey/27_8_2017/901804808105066496_0.jpg', 'train', 'c17bba9fd9f28c5ecea99da62427b5ea6e149c74e2718e3d2ccffdc188c9c38e'), ('907491611222745088_1', 'data_image/hurricane_harvey/12_9_2017/907491611222745088_1.jpg', 'train', '694ab01df9aba162a78d8e64bc4edbfdb080c142589562f62d0805985e591e92'), ('909938076423675905_2', 'data_image/hurricane_harvey/19_9_2017/909938076423675905_2.jpg', 'train', '02e4478a5fd5662dc23eb2304da87624f2b57745456c31f82da2aa6fd154d4d4'), ('906713368563556353_0', 'data_image/hurricane_harvey/10_9_2017/906713368563556353_0.jpg', 'train', '6b0cec8f299b48276b45adc3590e44fa58126a50127b54e40f725ed65cf3d4cd'), ('907877049779539968_0', 'data_image/hurricane_harvey/13_9_2017/907877049779539968_0.jpg', 'train', '23ef0bcd20e2364fce93440baab48eeabf51e054d0174c6108a46fe8bbcaad3c'), ('905587869963735041_0', 'data_image/hurricane_harvey/7_9_2017/905587869963735041_0.jpg', 'dev', '25c00009c2ed00678fedb4bc79be2cd748ced1913527bb12abf4265d6b43f223'), ('901646610916106240_0', 'data_image/hurricane_harvey/27_8_2017/901646610916106240_0.jpg', 'train', '4e732a7195ff022c1b02e0d4c7d0eb50bc01e2feabcbc0b00849fe8d9f76480a'), ('904469110028951552_0', 'data_image/hurricane_harvey/3_9_2017/904469110028951552_0.jpg', 'train', '7fbf55c0edcf01dc862591c65c501fdb6a9e5c2608c3c60929bfe7a54f1457a7'), ('901646074527535105_2', 'data_image/hurricane_harvey/27_8_2017/901646074527535105_2.jpg', 'test', '8923be686eece16e85901ad1b043a6c4762076ee830d304c1624e7d2ddfbde09'), ('904759493468160001_0', 'data_image/hurricane_harvey/4_9_2017/904759493468160001_0.jpg', 'test', '5fe44fe762029b961e6a72c56f3a5c46f51964317a8df390c0b22ea3999ec9bd'), ('905664802214858752_0', 'data_image/hurricane_harvey/7_9_2017/905664802214858752_0.jpg', 'test', '2674fd3579f73f26b3b61f6ba6ee4fe0480210057db82748c33fb347aa88f5b1'), ('905552797357170688_0', 'data_image/hurricane_harvey/6_9_2017/905552797357170688_0.jpg', 'test', '2ef33e1ae332194da33074532307059393910f488ecd332b5dd74c70ee546e72'), ('905793985670721536_0', 'data_image/hurricane_harvey/7_9_2017/905793985670721536_0.jpg', 'dev', '1d4570a9609c4048a2e28a334f8e91d773deb667d3239658fadc26c4356acfcf'), ('906262165618753536_0', 'data_image/hurricane_harvey/8_9_2017/906262165618753536_0.jpg', 'train', 'f8b207cab1e0732cea88c7a8b37f4eef46832c681fc822c251bc5d17fd283cd2'), ('901787699648004097_1', 'data_image/hurricane_harvey/27_8_2017/901787699648004097_1.jpg', 'dev', '5783015d94df363849c10eb174000ba4327bbe12719d31b5c17b126e919197a9'), ('908871077660692480_0', 'data_image/hurricane_harvey/16_9_2017/908871077660692480_0.jpg', 'dev', 'f76d9cca4363f98ea6f768f35831f8ab3618feea0c1735fc938d98d41a4d353e'), ('904878370088816641_0', 'data_image/hurricane_harvey/5_9_2017/904878370088816641_0.jpg', 'test', 'b21a2d557725d96c2b8624e93bb91b3cbb02184e7022fc6ab1f593b2d9d2c18d'), ('901646131628830721_0', 'data_image/hurricane_harvey/27_8_2017/901646131628830721_0.jpg', 'train', '1b7ae9c9af14a036e4c499f4077dadd12bcf8bdd69597186977bee6d93fe6071'), ('909147681917886464_2', 'data_image/hurricane_harvey/16_9_2017/909147681917886464_2.jpg', 'train', 'a8553f4f6fb23a9a357c3ccb227badb9ca806e93ee51376b548804737922d1fb'), ('904994908016058368_0', 'data_image/hurricane_harvey/5_9_2017/904994908016058368_0.jpg', 'test', '1b46cebe80b46d60358ecffcd23b9338503baa5a7cde1cd8abe58302744a52ff'), ('907307110307962885_0', 'data_image/hurricane_harvey/11_9_2017/907307110307962885_0.jpg', 'test', 'e6d295286e2a0091b42209eaa07ab5f0766b7751ce9e0c84e1428eb6d5bac8a3'), ('901814164406009856_0', 'data_image/hurricane_harvey/27_8_2017/901814164406009856_0.jpg', 'dev', 'f23bdc3eea25c8415e3d155f8a20316e5f0da30b544aa02e9d8829d8355754d0'), ('905375783232950273_0', 'data_image/hurricane_harvey/6_9_2017/905375783232950273_0.jpg', 'test', '11e2b1359c2b41372ca08d57c84d8f62c49b2fd503a02f8fc64c80ce034ffac5'), ('908113078620958720_0', 'data_image/hurricane_harvey/13_9_2017/908113078620958720_0.jpg', 'dev', '106ab4aa40d78e872abe5fb07a4a0d35ab65738c676471a24af951dab072ffbf'), ('904642434591752192_0', 'data_image/hurricane_harvey/4_9_2017/904642434591752192_0.jpg', 'dev', 'eb8f2415fb1140b3bf566ba6a04ff7f468aacfd42571c3dcb415d83abe73695e'), ('909768766094897152_0', 'data_image/hurricane_irma/18_9_2017/909768766094897152_0.jpg', 'dev', '9e09ba5614240cff4156e40e8214c51e662cd9bba4c9f33bf6b140ea52e8ce79'), ('909399529119715329_0', 'data_image/hurricane_irma/17_9_2017/909399529119715329_0.jpg', 'test', 'fd883c80265e04f1df50321c8dba43baecf4edaabd42880fd62b48365e281e33'), ('909397777792118784_0', 'data_image/hurricane_irma/17_9_2017/909397777792118784_0.jpg', 'train', '7f1322323223e4aa61d2ef8ed7a9a465735f938e7ff02caec693802657589039'), ('910266304476667906_0', 'data_image/hurricane_irma/19_9_2017/910266304476667906_0.jpg', 'train', 'e9ac783f0cbea7dd48f2188a84bf87fac869333fa68f0511927cd58200cec338'), ('909888420905930754_3', 'data_image/hurricane_irma/18_9_2017/909888420905930754_3.jpg', 'train', '71df5edabdcff69e4cc762d9efa634ba4f0d12a0767132a05d84c4f4a84b5b7f'), ('909439327779852288_0', 'data_image/hurricane_irma/17_9_2017/909439327779852288_0.jpg', 'test', '9539e2cce88d774aa2808a9c5962bb99f5a22d1bd8999bfadbfcafde0e06f036'), ('909480824826888192_0', 'data_image/hurricane_irma/17_9_2017/909480824826888192_0.jpg', 'dev', '4bfec1e1b49f7e81fb4340ec30221142320dcc9524ea62e74bb4f130406e1e39'), ('905630785084039168_0', 'data_image/hurricane_irma/7_9_2017/905630785084039168_0.jpg', 'train', 'b92ff4cb939531df2d17cf9f026cb982332f2e9a62b3e9e9b1b9e038ab9d2c79'), ('909988214240305152_0', 'data_image/hurricane_irma/19_9_2017/909988214240305152_0.jpg', 'train', '1b13c2587dbb8fc47ca0e2964e3225a093bbfbb9693f2a99b46573bfb554b8f5'), ('905721072250736640_0', 'data_image/hurricane_irma/7_9_2017/905721072250736640_0.jpg', 'train', 'b03e5d05e0612300244fada3062f21a3039b8c05e17526b374795a1f811fcedc'), ('909986865041874944_0', 'data_image/hurricane_irma/19_9_2017/909986865041874944_0.jpg', 'train', '5fc210e7048d82d9f6ed63aaddcf77fd5da00e7ebb1dc850daa5510d1e6ea9ee'), ('909475867587436544_0', 'data_image/hurricane_irma/17_9_2017/909475867587436544_0.jpg', 'train', 'a94f3a601405e4db006e8d33e0e074c754144e53597e0fc26630c30401a813a7'), ('910080834153123841_0', 'data_image/hurricane_irma/19_9_2017/910080834153123841_0.jpg', 'test', '367890f2a23116b7a2a05309286ec6addbf1713593e69d57f28bbd5c0ff70b1a'), ('909837615830065153_0', 'data_image/hurricane_irma/18_9_2017/909837615830065153_0.jpg', 'train', '65bfbf6a93868092f06ca62ea2147e9582879a8e4ec9cf9363d729bf637af6d3'), ('909885019136708613_0', 'data_image/hurricane_irma/18_9_2017/909885019136708613_0.jpg', 'dev', '0f9ef1479b64e4351664e28de5f9d4780eea85c5453e6706eed26150e8233d43'), ('910211201182789632_0', 'data_image/hurricane_irma/19_9_2017/910211201182789632_0.jpg', 'train', '0e76f88ff29fe10ba6d46288ac81cd52a8547fbb524bb325f87a0ceb0f88cb00'), ('909475803628412928_0', 'data_image/hurricane_irma/17_9_2017/909475803628412928_0.jpg', 'train', 'e5f65bb0c8708b7ca600aac638a4863a3331c1b5d974209aa40333be633001cf'), ('909383753830674432_0', 'data_image/hurricane_irma/17_9_2017/909383753830674432_0.jpg', 'train', 'ccdbfc07e082980eeb3d0a72476b403bd0f7c21d1e0e10c729a50dded843418b'), ('909927596439293952_0', 'data_image/hurricane_irma/18_9_2017/909927596439293952_0.jpg', 'train', '136e898cc2d90b248495de550324886ad6e00abc7e298d47cc1a9370a8d18567'), ('906927949118672896_0', 'data_image/hurricane_irma/10_9_2017/906927949118672896_0.jpg', 'train', '088abef120792ebe5ea7e9ab3ae2baa7ca9855f2b51fa7dd634e210ecb5ddfec'), ('909404123266764801_0', 'data_image/hurricane_irma/17_9_2017/909404123266764801_0.jpg', 'dev', '3e290da6b04b6bae1a6daeb02f70e4f641b5914f3959a7b714ceb577cd91962c'), ('910133170493956097_0', 'data_image/hurricane_irma/19_9_2017/910133170493956097_0.jpg', 'test', '1d089134bf4b5dca7adaa78635efb156ee640fb0ffd84f085c9275da48980d1f'), ('909458165640552448_0', 'data_image/hurricane_irma/17_9_2017/909458165640552448_0.jpg', 'train', '5746e6c24bb6263f27ee448c948c9b0464a10ae82803a8ee50354ac4052e172c'), ('910133170372198400_0', 'data_image/hurricane_irma/19_9_2017/910133170372198400_0.jpg', 'test', '3548abc2b6bc41477915a8572a9b2dc46b1f0a49a49f7e871a7fce2c6def57c3'), ('909431953434251264_1', 'data_image/hurricane_irma/17_9_2017/909431953434251264_1.jpg', 'train', '82b7b23dcdef19c35b2a378573baf2b0c905ced7d2130ea0d370769f99853ba1'), ('909425041003212800_0', 'data_image/hurricane_irma/17_9_2017/909425041003212800_0.jpg', 'train', '1dea4bae65ca93836217ed87aef99c03bb34d826f7e05a0fc95460ecbcace5b1'), ('913474359813181440_0', 'data_image/hurricane_maria/28_9_2017/913474359813181440_0.jpg', 'test', '97292b7ddfe706efed9a8734f8eb98f7b85418025a910742d37de770971541a4'), ('913776925071441922_0', 'data_image/hurricane_maria/29_9_2017/913776925071441922_0.jpg', 'test', 'ca9c930df550f7e69c2fe2b21767a85f4defc1e11e5956dbee6e048fdaae3040'), ('922610290281402368_1', 'data_image/hurricane_maria/23_10_2017/922610290281402368_1.jpg', 'train', '33cdc1b554b7ae059552b9078042326b62d472a7b7f0e6734eab21b1f7686818'), ('916085953084710914_0', 'data_image/hurricane_maria/5_10_2017/916085953084710914_0.jpg', 'train', 'da0dcb7bd41a86c6c9b341a1f977d37ea6196d42272b5b6fd1b3367ec3e1f992'), ('913214732122849280_0', 'data_image/hurricane_maria/28_9_2017/913214732122849280_0.jpg', 'test', 'bd7893144ea5aaad2182d21bfd3f35664acd5b74b2454fe0aee8e28bf8d71e71'), ('927998178636587008_0', 'data_image/hurricane_maria/7_11_2017/927998178636587008_0.jpg', 'test', '59338badece22977942df5c73ed67871db0b47102e3b83b5ca6feaeffdc6376b'), ('910540043126808577_2', 'data_image/hurricane_maria/20_9_2017/910540043126808577_2.jpg', 'dev', 'd1e68cadae5c247dcc8ed87e6c988516f25c0aa7aef6e31abb9ebabee2bffe21'), ('922565155874996225_0', 'data_image/hurricane_maria/23_10_2017/922565155874996225_0.jpg', 'test', 'b5b5cbfb3e014c9701481598d8ba8b0bfd2cccd3956fc4718f24a78163d35863'), ('916262078893248513_0', 'data_image/hurricane_maria/6_10_2017/916262078893248513_0.jpg', 'train', 'cb2c0a4555ca96b9d83ba219c2558ac973fa7ab3aedd16872015a4a57fb31fc6'), ('911636787239559168_0', 'data_image/hurricane_maria/23_9_2017/911636787239559168_0.jpg', 'train', '77a6371187de3d4b32e97f6f5fbcbac021b3688b377f3af680b631fae20a8301'), ('923357285497401344_0', 'data_image/hurricane_maria/26_10_2017/923357285497401344_0.jpg', 'test', 'e9d34e46daa42377043035052580344c37654c844edb0a281da7171f99cc8965'), ('922015756140843008_2', 'data_image/hurricane_maria/22_10_2017/922015756140843008_2.jpg', 'train', 'a6a96e512776caf58321e0c159befcc41d5a539153d5ba7af3725e8995796b2c'), ('928291159943692288_0', 'data_image/hurricane_maria/8_11_2017/928291159943692288_0.jpg', 'test', 'af112ab3ddcb64535b270f2b59ef85b24d3f740fdd2f1419548cdf4f5e0f06cb'), ('913048103728635905_0', 'data_image/hurricane_maria/27_9_2017/913048103728635905_0.jpg', 'test', '8b06444774d7a7ca8a9f1f98a2bc86c9fff3f6c1bba903a02e27e343fc2e8521'), ('913473823701266432_0', 'data_image/hurricane_maria/28_9_2017/913473823701266432_0.jpg', 'train', '3861295d6f47eda768853bfb41ceda858b29aa1c1585520c133dbd6c807145e2'), ('911914695803666432_0', 'data_image/hurricane_maria/24_9_2017/911914695803666432_0.jpg', 'dev', '50d5eb5a39285d3abbc6eaf891f9b1c094b6c60d38f37fc4d9caf985d4f073f9'), ('914372204787355648_0', 'data_image/hurricane_maria/1_10_2017/914372204787355648_0.jpg', 'test', 'f2da38815f1c9c383791130dca0bb395b8e56a7f7748882076516ed1e8a73363'), ('915995595923435520_0', 'data_image/hurricane_maria/5_10_2017/915995595923435520_0.jpg', 'test', '37ec702aa03e9a437fc812bbc999d97bd109256e408554ac87b95964d7ec1aaf'), ('923578493656600576_0', 'data_image/hurricane_maria/26_10_2017/923578493656600576_0.jpg', 'test', '185a2979586a25fc67330cac14dca631c09b1660d135e0ed4a05c70fb05007fa'), ('910626406513676288_1', 'data_image/hurricane_maria/20_9_2017/910626406513676288_1.jpg', 'train', '6e0158844c9404b5a2a6bcb4bea218404829ee5c4eccd592f52e5cfdade8019e'), ('912368411770056704_0', 'data_image/hurricane_maria/25_9_2017/912368411770056704_0.jpg', 'test', 'f74682ed3c28addbef8ea684839a2972c4aa3d35f25808486a6c6c432888ba40'), ('927192810146619393_0', 'data_image/hurricane_maria/5_11_2017/927192810146619393_0.jpg', 'train', '4a4cbe0ff12ec1895e9c217d0546fa45c6fa665ddfdbede13c6ed8f592542ecc'), ('911742118787469312_0', 'data_image/hurricane_maria/24_9_2017/911742118787469312_0.jpg', 'train', '884aab11af549a37ee18e1a58c8e2607d832ef6a2c9f7ec362259c76daae7b46'), ('915094814080798720_0', 'data_image/hurricane_maria/3_10_2017/915094814080798720_0.jpg', 'test', '2f0706b2793bfc9e1778e947f991a3030f833b73db255384e81f4a4c5b7a99ee'), ('923654546764611584_0', 'data_image/hurricane_maria/26_10_2017/923654546764611584_0.jpg', 'test', '4004314f39d0b0350ccd5c23f3ec2c15b7e6630c985837a25d5ce3c60da52d80'), ('927967121786617856_0', 'data_image/hurricane_maria/7_11_2017/927967121786617856_0.jpg', 'train', 'cb3869d5aa685b11fd77f99f30142e052115e7c5c293f1a18f8ae1e464033497'), ('910566721735475200_0', 'data_image/hurricane_maria/20_9_2017/910566721735475200_0.jpg', 'train', 'c4779570fd45de200fcfcd2b391a8667a4b8842fe1ba4d0dfe4d79430378b336'), ('929989443913691136_2', 'data_image/iraq_iran_earthquake/13_11_2017/929989443913691136_2.jpg', 'train', 'd1310f612f4ab904b51408e2843e3f9e2c4cc526f7f4064427e2c24c6fed3368'), ('929996311478272000_0', 'data_image/iraq_iran_earthquake/13_11_2017/929996311478272000_0.jpg', 'train', '46d83eab4b9fb4b02c6662ee2c54789a4a773f7d99f773f252751642702a8a85'), ('930023493160366080_0', 'data_image/iraq_iran_earthquake/13_11_2017/930023493160366080_0.jpg', 'test', '0855e2a1424963512222e09c56a6898a1d8332acd8aab18048fbab9d742d7661'), ('930293724990410752_0', 'data_image/iraq_iran_earthquake/14_11_2017/930293724990410752_0.jpg', 'test', '71c91f756cbf156801eb6e60faa70427ada93d5a16804baeb86a85c16b433294'), ('929991089150406656_0', 'data_image/iraq_iran_earthquake/13_11_2017/929991089150406656_0.jpg', 'test', '8e42e606b72be1b3b7d7f6108330bfae2182ad967c387457dd73fe5625c83463'), ('930475123634769920_0', 'data_image/iraq_iran_earthquake/14_11_2017/930475123634769920_0.jpg', 'train', '1229ef09ad981f1969ce0bdd84e5a1944cff8616fff209ff0e68378452db6e23'), ('930059422033956869_2', 'data_image/iraq_iran_earthquake/13_11_2017/930059422033956869_2.jpg', 'train', '04e8907873b10786a6d6d28ea176081c87ee0de75290fe4057e5f07a4e2c08e2'), ('930539345043341312_0', 'data_image/iraq_iran_earthquake/14_11_2017/930539345043341312_0.jpg', 'test', 'b243cc95b97db8eaee75c1816b2c75edf0ff51874f38f080fc1cb6acfe6a9dae'), ('930273853783162880_0', 'data_image/iraq_iran_earthquake/14_11_2017/930273853783162880_0.jpg', 'test', '4b8e16658f3b1eacd51635589997384a0a63630faff313e3a0b0f29c6e701517'), ('930169117893685248_0', 'data_image/iraq_iran_earthquake/13_11_2017/930169117893685248_0.jpg', 'dev', '412020100b4036d01fb8e02f228f36a9468a1b14ed2ff8cd9f696c3f451711fb'), ('931100946721136642_0', 'data_image/iraq_iran_earthquake/16_11_2017/931100946721136642_0.jpg', 'test', '1be5bf45616ad94ce80b459810ec69cb64165ce4210c6a5c9300e8503bdc4cb7'), ('930468660392710144_0', 'data_image/iraq_iran_earthquake/14_11_2017/930468660392710144_0.jpg', 'train', 'af6dfbc1736ffbb33f51cf941bb23d15080cc194413381905d73fdc358fdd630'), ('930775648121090048_0', 'data_image/iraq_iran_earthquake/15_11_2017/930775648121090048_0.png', 'test', '2d19238ed29d770ce83ddad0d2519a7898fd5b7cb1fbbe1e0d7eb68b50e6bf15'), ('910715482742849537_0', 'data_image/mexico_earthquake/21_9_2017/910715482742849537_0.jpg', 'train', 'acf9a6499e6eede4e23ab057e04c2b47cf0dafdd865a0078c05b375baf8819e7'), ('910542244830212096_0', 'data_image/mexico_earthquake/20_9_2017/910542244830212096_0.jpg', 'train', 'b64b69ad6053c5708d00be34a007005fc60763a50052401fd6028403a612dab5'), ('910523537860866048_0', 'data_image/mexico_earthquake/20_9_2017/910523537860866048_0.jpg', 'train', '9ca263f6855c6ef6995049c8b7101493dbc433dece0c4eedc901b827fa2c9c59'), ('910750236053708800_0', 'data_image/mexico_earthquake/21_9_2017/910750236053708800_0.jpg', 'train', 'dd3e00b9668e77b6ce2fc6c2f4d455054cc1986236d99eb7329bccbee61bfabb'), ('910586144462585857_0', 'data_image/mexico_earthquake/20_9_2017/910586144462585857_0.jpg', 'train', '99060a4d6692c504522773d50e51d9aa79a6324f03caae63890f8096a3990870'), ('910798273367171072_0', 'data_image/mexico_earthquake/21_9_2017/910798273367171072_0.jpg', 'dev', 'a11e93d2b6d0295738550240ab627d6a68f5d6091c6df5b049bf02aaf199bf65'), ('910580950605733889_0', 'data_image/mexico_earthquake/20_9_2017/910580950605733889_0.jpg', 'train', '10ded482e3269fe4181050e75c3d66f32e693ae872708bd25f98674e29059e91'), ('913230949294465024_0', 'data_image/mexico_earthquake/28_9_2017/913230949294465024_0.jpg', 'test', '906c3986da1a51262a0756d8be77fbb5e538d6800462d7e51f40cdf56145dccb'), ('911664304725913600_0', 'data_image/mexico_earthquake/23_9_2017/911664304725913600_0.jpg', 'test', '500f4ba8e16888203d3c92d6dabd5cf502e5e8f1cb6de03b50ef062591aae9ec'), ('910524679777001472_0', 'data_image/mexico_earthquake/20_9_2017/910524679777001472_0.jpg', 'train', '8e25742c5aa18ba2fa05aba81231001be3e7dfbaa416f5d4ffce56d75f505b0b'), ('914849104320180225_0', 'data_image/mexico_earthquake/2_10_2017/914849104320180225_0.jpg', 'train', 'a505361d482249431ea699f0075fa3579dc3427825aaeaeb667059002c2595d5'), ('912341595667779584_0', 'data_image/mexico_earthquake/25_9_2017/912341595667779584_0.jpg', 'dev', '2d4ed8dd1b3d58924eb70cb94759868b6a92dca6862ef08dfed11378b2826071'), ('910658421338275845_0', 'data_image/mexico_earthquake/21_9_2017/910658421338275845_0.jpg', 'train', '801eb2a16bda08f78d747495505012ee719e1c94990c8762aadac639f7696e84'), ('910538030347423745_1', 'data_image/mexico_earthquake/20_9_2017/910538030347423745_1.jpg', 'train', '79ea0a24e76b2dfb3f8c8d9f98d4a3f4c3b604d8a28e2a79c6027e33a1a1cca0'), ('911625458420240384_0', 'data_image/mexico_earthquake/23_9_2017/911625458420240384_0.jpg', 'dev', '7db1eff72da579faed606b5c554cfdd0abe2bf95325b681ef7479d105cf5929b'), ('910549663459110913_0', 'data_image/mexico_earthquake/20_9_2017/910549663459110913_0.jpg', 'train', '8e0f00a55d55b5a71d6907a6a71ff88d1b36af27cee3723a55226d18f0ecda28'), ('870295706384318464_0', 'data_image/srilanka_floods/1_6_2017/870295706384318464_0.jpg', 'dev', 'b6157a49e47611e407f00c7f6a41b91a5e423aa06e4b21fc9bcdda038d8e95dd'), ('870004426542505984_0', 'data_image/srilanka_floods/31_5_2017/870004426542505984_0.jpg', 'train', '271395d375bb35e77da250cd62692717d40e9a602a56349615231f42a76f4a77'), ('872044691885305858_0', 'data_image/srilanka_floods/6_6_2017/872044691885305858_0.jpg', 'dev', '0b016247b4bf4a939752356f336e145f8f400c22789ad72c82e587de974f653e'), ('870147991163875328_0', 'data_image/srilanka_floods/1_6_2017/870147991163875328_0.jpg', 'dev', '9a362529e22a2012b9441452a3ec4d1e087c57a16fc8f03bc7e67f08a0ed6b03'), ('873818174466121728_0', 'data_image/srilanka_floods/11_6_2017/873818174466121728_0.jpg', 'train', '4b5e7504938871b58f57138d20300e25dc0701d5bdc1094f1688d4e8ea5fe572'), ('871935702132445184_1', 'data_image/srilanka_floods/6_6_2017/871935702132445184_1.jpg', 'train', '0c6e966fc2b74256510cbca7fe0c99bff27441fef11b8fc4bc756454eea870d5'), ('871735126635544578_1', 'data_image/srilanka_floods/5_6_2017/871735126635544578_1.jpg', 'train', '710ac1ebd50ccfb22bc9b0c3c5acb86ef917f0bd2d62fd133e7a93f2b685f623'), ('870855346516111361_1', 'data_image/srilanka_floods/3_6_2017/870855346516111361_1.jpg', 'train', '37fca0e6f7d6d5446c71a751e3e87a594af01180a2762cdc67af0ccab1f15105'), ('871757822056202240_0', 'data_image/srilanka_floods/5_6_2017/871757822056202240_0.jpg', 'dev', '77973f5bd12bc039b592571229bf6f0c4d9f82425023b4e6ead47c8057d535d8'), ('873491045442441216_3', 'data_image/srilanka_floods/10_6_2017/873491045442441216_3.jpg', 'train', '0a5ec245b6c01a185f349d377ecc02b2db1249346497e8ef7c15682a66a9709e'), ('871269851108708357_0', 'data_image/srilanka_floods/4_6_2017/871269851108708357_0.jpg', 'train', '8bb677b262afd6d6a5426c8ebf0d463993570c0457ac78ab66e46d6a730b3fd7'), ('921617294945525760_0', 'data_image/california_wildfires/21_10_2017/921617294945525760_0.jpg', 'train', '7eebe769638fd8163e72424d12684bb7f8c9ae60c03b91274a9bcf3bda86e12d'), ('919734115364147205_1', 'data_image/california_wildfires/16_10_2017/919734115364147205_1.jpg', 'train', '13ec6f6139d1b571d61b84bc40e1e873d1701a573df65c64ee9461ce10504ea7'), ('919631918077005826_0', 'data_image/california_wildfires/15_10_2017/919631918077005826_0.jpg', 'train', '82e2a9e95dd1cf47bdc8f92ee5c7290ba4e0b80e43667b13d59d47255ffc7e9b'), ('921571773501640709_0', 'data_image/california_wildfires/21_10_2017/921571773501640709_0.jpg', 'dev', '3f21dae3ce41b66f496599f74d202dd3495f15db66d4cc0883644d7af4d7fdbb'), ('919943278220341249_0', 'data_image/california_wildfires/16_10_2017/919943278220341249_0.jpg', 'dev', 'f40194e885865282735a23511b7fd24b6f750e7f3b6f9023ad9ffbd9a0c1b72c'), ('917842724551667712_0', 'data_image/california_wildfires/10_10_2017/917842724551667712_0.jpg', 'test', '092cbc9f8f0e8e7bb92f3cfd87456f7e89a28c80f2237683569288a782ecaa46'), ('922915671347945472_0', 'data_image/california_wildfires/24_10_2017/922915671347945472_0.jpg', 'test', 'f9d007be4b8212b1f28d7386e92855ff2550142ed8591deeb2d295aa2a5c7e3c'), ('917793158251077632_0', 'data_image/california_wildfires/10_10_2017/917793158251077632_0.jpg', 'test', '98ac440da1fc73ddb808aa155096fa8f607961e78b1a8c9e5f322c47f796b17b'), ('918013344006602752_0', 'data_image/california_wildfires/11_10_2017/918013344006602752_0.jpg', 'test', '1ae3d69603f83ee80ee741efaf9cf096611a80e4eec651bba8bf0821151cc7f4'), ('921009874372366336_0', 'data_image/california_wildfires/19_10_2017/921009874372366336_0.jpg', 'dev', '66fa75f34735db91681fd0f165e99b26b30a3fa34cfd2c48f35a3c12f720e8a7'), ('919639737215963138_0', 'data_image/california_wildfires/15_10_2017/919639737215963138_0.jpg', 'test', '3335b2804bca6165ca2ce06fb75d0cc2cd2090db97a975419b1d69d59762c712'), ('922948121939927040_1', 'data_image/california_wildfires/24_10_2017/922948121939927040_1.jpg', 'train', '0a9f1dae43dd7a5762fbc73de5f2b85e841ce58fc34c4d54a4888a794929b641'), ('923043964063711237_0', 'data_image/california_wildfires/25_10_2017/923043964063711237_0.jpg', 'test', 'ce0cb98b780db7d20f8d329dc129e374a01b502cac4f2ceb05fa14cb2b1b8889'), ('920859460725280770_0', 'data_image/california_wildfires/19_10_2017/920859460725280770_0.jpg', 'test', '3dd6831678c3f0de82b1dd1298ee77e8a721c3eb41afceca0fbd83b060908da8'), ('918356075782463488_0', 'data_image/california_wildfires/12_10_2017/918356075782463488_0.jpg', 'dev', '9351bc6b00da5843221a853224be27d9c3ec41e4e24a6b27dad16118912f9c30'), ('918058954718035968_0', 'data_image/california_wildfires/11_10_2017/918058954718035968_0.jpg', 'test', '0678cda6fcb811982d51c2ad73f19d9d83b678311fb6a31aa18a0576457750d1'), ('922159926620934144_0', 'data_image/california_wildfires/22_10_2017/922159926620934144_0.jpg', 'test', 'b3048aeffe6db8be994e50fef050799c1487629df5e7fddd7db96e2dc7928373'), ('920774765064810496_0', 'data_image/california_wildfires/18_10_2017/920774765064810496_0.jpg', 'train', '7f3f8d75bdc01065f693c6c9465221e5bf8272cd951efe2ae20090bb84b0e2ff'), ('921414148747091969_0', 'data_image/california_wildfires/20_10_2017/921414148747091969_0.jpg', 'test', 'c6fbbc7243772f22ae6b358e9c103d37bdaa624953b98d8e6cb9d23b6883cce4'), ('922656568063217665_0', 'data_image/california_wildfires/24_10_2017/922656568063217665_0.jpg', 'train', '1f19ee338610cf84ff409f2e2d1f6d32f758ed929052e35e7a7dbcea9ab3505f'), ('919595039658885126_0', 'data_image/california_wildfires/15_10_2017/919595039658885126_0.jpg', 'test', '73c78d8863af4f5263d2c34d2a36e235967f2d909ae1d3a5c61816324a885d0b'), ('907361650600140800_0', 'data_image/hurricane_harvey/11_9_2017/907361650600140800_0.jpg', 'train', '26d74666816239d916c53fb1eb599b27058e38e3908839e6994bad227cbb8226'), ('904606499707555840_0', 'data_image/hurricane_harvey/4_9_2017/904606499707555840_0.jpg', 'train', '339b6b8451e03f52876f4d8902954d14f3409685f83bb9956ab9e653c0e0cb2b'), ('905830176931569664_0', 'data_image/hurricane_harvey/7_9_2017/905830176931569664_0.jpg', 'test', 'd35d84692f2f6e3451be535b2581f5fba75c4f471255fee3a36b4ab9b9baa670'), ('908111834418466816_0', 'data_image/hurricane_harvey/13_9_2017/908111834418466816_0.jpg', 'train', '35062bc7016df616f1a2de74104a457b09cd2b2cb25f6740c7f52f6ba0c76944'), ('904240721917616128_0', 'data_image/hurricane_harvey/3_9_2017/904240721917616128_0.jpg', 'train', 'c02946e6946cb27e1fec7f66cb003a0f421afd5287ecf2e2162ad034f57c2d8f'), ('904804014419824640_0', 'data_image/hurricane_harvey/4_9_2017/904804014419824640_0.jpg', 'dev', '8d398f95c34deda0a321dc2496aa1fee1193494f341b9e6c1a10ef6cfd3ca912'), ('906032635318640641_0', 'data_image/hurricane_harvey/8_9_2017/906032635318640641_0.jpg', 'train', '3dc6dc29431db53ea554d6123f5a8a84e85784a61b9c98a637e0fb47d70000d3'), ('904206894264352768_0', 'data_image/hurricane_harvey/3_9_2017/904206894264352768_0.jpg', 'dev', 'af4f5e70392ba81fb40e2514c1a14d82ce8510523e65a905f6243f4ea5b24d8b'), ('905686377114755072_0', 'data_image/hurricane_harvey/7_9_2017/905686377114755072_0.jpg', 'train', 'f0e9f6f6e3a6a1bdc18fe44cfcd5bdbbb5d14c29d91956393381d9c1f08d0dc3'), ('901688227647684609_0', 'data_image/hurricane_harvey/27_8_2017/901688227647684609_0.jpg', 'train', 'ba65f414f72e8e077e18b320cc3207c4a7c6e2bdf38220756e981dadfbfac4bf'), ('901709640282591232_0', 'data_image/hurricane_harvey/27_8_2017/901709640282591232_0.jpg', 'test', 'c6a5984ba87f3e429795a29d4e39ce299b1fe0d07b5441fa1d1158835dbee00d'), ('908406994461224960_0', 'data_image/hurricane_harvey/14_9_2017/908406994461224960_0.jpg', 'train', '40bb9dc3093cce3bcbb4990142796b7e7425646f094e23a10ac86a08c04e1dbb'), ('905777977249681409_2', 'data_image/hurricane_harvey/7_9_2017/905777977249681409_2.jpg', 'dev', '9c8eac3f885cc517c23a35a7a5397eab59a38c248cc29bad35438b581f8f4139'), ('905355683599491072_0', 'data_image/hurricane_harvey/6_9_2017/905355683599491072_0.jpg', 'train', '322adbc4284b2df93ddf667412077d19b1312e76bc7b90e5989051b4a1811ed6'), ('906243309424840705_0', 'data_image/hurricane_harvey/8_9_2017/906243309424840705_0.jpg', 'train', '0f45af08bde760afb020bbfeec155a5b1ccae8f5d719921204a90e461cf0de66'), ('906706479440384000_0', 'data_image/hurricane_harvey/10_9_2017/906706479440384000_0.jpg', 'train', 'd587080ba432ec7f9c784ddcf27f766f00373ab17e1c57c9eefe442e6af7a986'), ('907735778914058241_0', 'data_image/hurricane_harvey/12_9_2017/907735778914058241_0.jpg', 'train', '86e88c7f987c59228cd878cdec8c9d3fec791a3b79c474b9ba460319a5971e33'), ('907251173551493120_0', 'data_image/hurricane_harvey/11_9_2017/907251173551493120_0.jpg', 'train', 'eba964fc9b97f1b9654b7d9d9c7cf7e76b53c3c9cd1f2cd47d093f2cb652f6c9'), ('910254709289684992_0', 'data_image/hurricane_harvey/19_9_2017/910254709289684992_0.jpg', 'train', 'cca4adc0a0353a18dacd3705be9f77ecde9a110c8c3a994993f425487c366ab4'), ('908468618022785024_0', 'data_image/hurricane_harvey/14_9_2017/908468618022785024_0.jpg', 'train', 'd94a169241cf3282c8d389fd55c90f5b7208dfb76e4b5090db3cc554a65a0b8f'), ('905498554332000256_3', 'data_image/hurricane_harvey/6_9_2017/905498554332000256_3.jpg', 'train', 'e1e9df295ec73907bdd51310652b5596e560c348498a08c6aa398601dc184093'), ('907591129239101440_0', 'data_image/hurricane_harvey/12_9_2017/907591129239101440_0.jpg', 'train', '97603892428048c6b126d96b5f58b8b849fb8aa101793af83605aa89d465f099'), ('905947586967855104_0', 'data_image/hurricane_harvey/8_9_2017/905947586967855104_0.jpg', 'train', 'a8b3ef62ce3f16923b68d15efa2733a629f1435469d5d07875db9b544f0fac1e'), ('908793499096633345_0', 'data_image/hurricane_harvey/15_9_2017/908793499096633345_0.jpg', 'test', '790108ad0af45c0dffbfce1e4d70af97f5286c375fe06ac3c5470b5b2375625a'), ('904516442485338114_0', 'data_image/hurricane_harvey/4_9_2017/904516442485338114_0.jpg', 'dev', '0ace3be0ba8055b66c1453d9a947c2aa8b3d5f9b23f2c52c18e7585dc976cca6'), ('908726003555586054_0', 'data_image/hurricane_harvey/15_9_2017/908726003555586054_0.jpg', 'test', '4b60713e55cc084d495ed4935797a7a201e131f1dab95a9cf6d108a77cc9a297'), ('910065687082680320_0', 'data_image/hurricane_irma/19_9_2017/910065687082680320_0.jpg', 'train', '1a56c467ec718d5fa182300b55b4d51f258d32179e472b59f899eab98b704d54'), ('909816306182103042_0', 'data_image/hurricane_irma/18_9_2017/909816306182103042_0.jpg', 'train', '091e26939e18358aba1c445f199bc91c793843cc36893832d07e244c93d237a0'), ('905669512434917376_0', 'data_image/hurricane_irma/7_9_2017/905669512434917376_0.jpg', 'dev', '67838d56816004e07c9901dc29c21e92e6f3ccc7b59cc4a6d4f063cce498bde9'), ('910198598054367237_0', 'data_image/hurricane_irma/19_9_2017/910198598054367237_0.jpg', 'test', '04b0c48cea56ae1f1a26d616ba84334c3b5b95d7d9c370b54dcddef221788181'), ('910005101019799553_0', 'data_image/hurricane_irma/19_9_2017/910005101019799553_0.jpg', 'train', '48a28f5ce1b8a3ca8eeee1cce9dd9f088c5bd8c9806d8ce1626526598543d134'), ('909768110730809345_0', 'data_image/hurricane_irma/18_9_2017/909768110730809345_0.jpg', 'train', 'bd8705204b8fd254f29e67970c5747e7c5935a145b4b340818dde60ee0a945e1'), ('909928094026387456_0', 'data_image/hurricane_irma/18_9_2017/909928094026387456_0.jpg', 'test', 'b9b0dea43cc2af9a15d3fc41428e01053b027b2cc7023a6ada70188b7acb0221'), ('905691457591410688_0', 'data_image/hurricane_irma/7_9_2017/905691457591410688_0.jpg', 'test', '1c9fe3789393a942b5094131224c988d982bd6f0bef89da2b21c5e595cf6a01b'), ('909823209641521152_0', 'data_image/hurricane_irma/18_9_2017/909823209641521152_0.jpg', 'train', '875bf345598b9bb493f6509b4ece0e35a5dc99ea5961ceb0e51b88b66be564ec'), ('909960548737044480_0', 'data_image/hurricane_irma/19_9_2017/909960548737044480_0.jpg', 'train', 'ce7837604b6ae44af14d052cf76fe8c8b038aba3a3847fa05bc60b1fd01792d0'), ('909445607550459904_0', 'data_image/hurricane_irma/17_9_2017/909445607550459904_0.jpg', 'dev', '8146862e602a0c17e2b0ef4d1293ed86589298f3c0e00ed08a1ccc4b814b72a9'), ('909863172156284928_0', 'data_image/hurricane_irma/18_9_2017/909863172156284928_0.jpg', 'dev', 'c098365ec4ee1676db7292862ee1d84e3bf59760ce08c6c8f5e26a01ace5c261'), ('909383948563869696_0', 'data_image/hurricane_irma/17_9_2017/909383948563869696_0.jpg', 'train', '54c3d59c315535730fd64b80d9a5345c6de6d9895592f397b198548f44c1898d'), ('909473412187291648_0', 'data_image/hurricane_irma/17_9_2017/909473412187291648_0.jpg', 'dev', '5a94740d91f43056fb45c875df8331941a833b5bce48e7bbcbefe8949c22eb4e'), ('909490925268692992_0', 'data_image/hurricane_irma/17_9_2017/909490925268692992_0.jpg', 'train', 'ec3dc6310ff60258f7dccc8edb23fec38bdc9b4674f846bc6df9b3f06444e71c'), ('909465020932739077_0', 'data_image/hurricane_irma/17_9_2017/909465020932739077_0.jpg', 'train', '4c63674e2e8b10d66c19234b858902166e24bea87f1370406280cefa7ff75c0a'), ('909907796958830592_2', 'data_image/hurricane_irma/18_9_2017/909907796958830592_2.jpg', 'test', '86e218affe042c0a1ca48f072b42cee31d04118ba42b948ecae3497810f06ee0'), ('910122575380000768_0', 'data_image/hurricane_irma/19_9_2017/910122575380000768_0.jpg', 'train', '544dcaa834454c8ad643dc9f0b481d241807ea2ec3a608457c41d9c0e39d54cd'), ('909866386356412423_0', 'data_image/hurricane_irma/18_9_2017/909866386356412423_0.jpg', 'test', 'a93b93c1f1ce1751aa44ba355b9f8301491466f51d24ab2cd92d99fbc55fef73'), ('910206013617332224_0', 'data_image/hurricane_irma/19_9_2017/910206013617332224_0.jpg', 'dev', '0e0d82f093eea02eec18c8a16e208e7bdd06f421812ab3dceb191accb5756c4b'), ('910188712415387649_0', 'data_image/hurricane_irma/19_9_2017/910188712415387649_0.jpg', 'dev', 'a3bf01aee88abeab81bbcd72b2b803351206cd35ff0677f389b7630e4c77b47e'), ('905679676428963840_0', 'data_image/hurricane_irma/7_9_2017/905679676428963840_0.jpg', 'dev', '05e1634b07d4690eca0f0f9c74628a1b23b22a803048b9410261739e7d6527d2'), ('916086403523596288_0', 'data_image/hurricane_maria/5_10_2017/916086403523596288_0.jpg', 'dev', 'fe6cabe842215292619f2354e9639e865b691b51c47562950266ce44144848a2'), ('921460085196328960_0', 'data_image/hurricane_maria/20_10_2017/921460085196328960_0.jpg', 'train', '546b21965b354fdbf2cc22c1b89fd3916f242603f51f77a19a980b957b03437c'), ('911711789246693377_0', 'data_image/hurricane_maria/23_9_2017/911711789246693377_0.jpg', 'train', 'a55fdd44d5bd8e301a89662d09eea8ed6406de03c18fb1a80696a720569458a0'), ('921383665917218816_0', 'data_image/hurricane_maria/20_10_2017/921383665917218816_0.jpg', 'test', 'cee8e4c43ba635d19964f69ab147b72c7ba66d53d20c8b50f3f3d793b74676cb'), ('912114770710581248_0', 'data_image/hurricane_maria/25_9_2017/912114770710581248_0.jpg', 'dev', '72c0052945db2958c056db623f243ed8ec5e8191a85a2a7f3d87d20558c7f00b'), ('921629571128143872_0', 'data_image/hurricane_maria/21_10_2017/921629571128143872_0.jpg', 'train', 'd45aa29fb96236d5cd9307b1043e63fbb16dbfa8fcbafeddbe527fb4255632d1'), ('911677412714393600_0', 'data_image/hurricane_maria/23_9_2017/911677412714393600_0.jpg', 'test', 'f02a3f660cdd63d58665175113552d266858e21990eebf60f7d118162818c453'), ('927096986310008832_0', 'data_image/hurricane_maria/5_11_2017/927096986310008832_0.jpg', 'train', 'bc1d947a6c1515662292e6cf8c494cd847e61a28e7d4489412e7d860cfe8f713'), ('914608656452341760_0', 'data_image/hurricane_maria/1_10_2017/914608656452341760_0.jpg', 'train', 'e7b95c67dbe362231d359eef59b4acf77267a5bcea728fd43441abefb6cd4127'), ('928597439321395201_0', 'data_image/hurricane_maria/9_11_2017/928597439321395201_0.jpg', 'dev', '80a7f28467b2a35006b8fb1a903598a5ef22918588beee668f7a763a73d38f86'), ('913380968769228802_0', 'data_image/hurricane_maria/28_9_2017/913380968769228802_0.jpg', 'dev', 'f64418b2b7ebf81fd68ee11b9907efcd677535e77530d5e95fab4b03ce246675'), ('921529194730983424_1', 'data_image/hurricane_maria/21_10_2017/921529194730983424_1.jpg', 'train', '7b8289326c85e394339440161632b6516fe6a6f23a0ed5db90ffc9220f7c5e69'), ('921419527489032192_0', 'data_image/hurricane_maria/20_10_2017/921419527489032192_0.jpg', 'train', '156d21447316d70ed1dd551e46bd7dbfa4af4fe88449b987bd6c958d52729ed5'), ('921422259897733122_0', 'data_image/hurricane_maria/20_10_2017/921422259897733122_0.jpg', 'dev', 'fbc3fe3f35b270fd88edfcb00ec83c4f9cf15e90399dcbec877eac0d7058a25a'), ('927974818275872774_0', 'data_image/hurricane_maria/7_11_2017/927974818275872774_0.jpg', 'dev', 'd6ea3a6c98c5e90d81cfb07f35b88802ee0ca0dae36a5e31d7d6d11c71f2433c'), ('923171800187588609_0', 'data_image/hurricane_maria/25_10_2017/923171800187588609_0.jpg', 'test', '4b43041588ee8dc8ffb1ea4cd754cdcf5ad91559edd3b618ff23ec34356fff83'), ('910628198307819522_0', 'data_image/hurricane_maria/20_9_2017/910628198307819522_0.jpg', 'dev', '0d7a24194d6493369e9048fcc6eeab3a19690bc41374d33b55df9fa18c0a5df5'), ('910542863150202886_0', 'data_image/hurricane_maria/20_9_2017/910542863150202886_0.jpg', 'test', '1bef9f6262ddce7eb5eea6e804819c8e8cfb100026c0cad420724a70798dfe01'), ('923163712835538944_0', 'data_image/hurricane_maria/25_10_2017/923163712835538944_0.jpg', 'dev', 'a2e7bba5c0231b036b8a464efae139703126cbd20e7d5073f9f00a604298bd1a'), ('914184867306491904_0', 'data_image/hurricane_maria/30_9_2017/914184867306491904_0.jpg', 'test', 'fc0dfa8adae23ce23ade706edf1c3066f7b8b1213baba92d06c5dbf2810c80af'), ('910658239880204288_0', 'data_image/hurricane_maria/21_9_2017/910658239880204288_0.jpg', 'test', 'a65d6b74722aa1373687280b7bb91d0a246c8dcd36b946a2aafc77f22fea7373'), ('916168959971303424_0', 'data_image/hurricane_maria/6_10_2017/916168959971303424_0.jpg', 'train', '8d9afc903624be3a532c49c6ad92f84eb9b8b3defce75e7d2930a1c324044f55'), ('921619592342097921_0', 'data_image/hurricane_maria/21_10_2017/921619592342097921_0.jpg', 'test', '044df7f22ea26ed8812b4d3f3ad603586ae8a5ec4814782a019572a61552fe66'), ('928201809994268672_0', 'data_image/hurricane_maria/8_11_2017/928201809994268672_0.jpg', 'train', '884df2fd708804b4f187ddd81a0294a8eba3594b22797c3a11dffa95e5681565'), ('929382361338400768_0', 'data_image/hurricane_maria/11_11_2017/929382361338400768_0.jpg', 'train', '44ec646dfa9fdfed4121f439ae125955ba632d1ad6287948231e84f91f057529'), ('927576648807043072_0', 'data_image/hurricane_maria/6_11_2017/927576648807043072_0.jpg', 'train', '3a9dd3b48516b23493d95d3903704cc347c8b338cdc6048c001881f1c8b2b098'), ('931084029599928322_0', 'data_image/iraq_iran_earthquake/16_11_2017/931084029599928322_0.jpg', 'test', '08327bb745a77a7120129b317c3bf0180883412c67ed39cc34f50a08d93924e4'), ('931505905736306690_0', 'data_image/iraq_iran_earthquake/17_11_2017/931505905736306690_0.jpg', 'dev', '5b40f9cd9f1cc656380d8be71bd92662ad2a65dca10247523ab4831e5e601d4e'), ('930037422422941696_0', 'data_image/iraq_iran_earthquake/13_11_2017/930037422422941696_0.jpg', 'test', '8106d58fa5ab3da61e67860758e7910be287762ec8bf21d0a22d4e51906f67cd'), ('929989443913691136_3', 'data_image/iraq_iran_earthquake/13_11_2017/929989443913691136_3.jpg', 'train', 'c6b8984419cab944f7b211dad9bd2c8bf201b5768204c9952db4d6bbbae5bb49'), ('930800150498377729_0', 'data_image/iraq_iran_earthquake/15_11_2017/930800150498377729_0.jpg', 'train', '2c8acf836fb8265e7db8ffeced279c8d697a5e61eea34c682c045fa260322311'), ('930205140287741953_0', 'data_image/iraq_iran_earthquake/13_11_2017/930205140287741953_0.jpg', 'train', '8f41d1f1481022c885fcc53d51560a0ebc703de5cb2d177c947d2b1f2b7d4095'), ('930401239002701825_3', 'data_image/iraq_iran_earthquake/14_11_2017/930401239002701825_3.jpg', 'train', '36a1a9ba46d54c4f434c644bd8caea6fc665f8d8b9e00a758ddce2c420af72c3'), ('930436669223845888_0', 'data_image/iraq_iran_earthquake/14_11_2017/930436669223845888_0.jpg', 'dev', 'c0bd7420731bd2cf1cc023357cf98fe6f4ac808eb54ccb17d35c9933dce48fba'), ('931232098874281984_0', 'data_image/iraq_iran_earthquake/16_11_2017/931232098874281984_0.jpg', 'test', '16c5813a9a7cad30047417850e187e6ffbaf8fa26cb9c8fabd81e739f0332e60'), ('930136678488043522_0', 'data_image/iraq_iran_earthquake/13_11_2017/930136678488043522_0.jpg', 'train', 'e5361aa5e65fd3d7cd7caabbd96ad938d5f5cf997cf083ff6723b31dfb7af994'), ('930007000632713217_1', 'data_image/iraq_iran_earthquake/13_11_2017/930007000632713217_1.jpg', 'train', 'a025686b0692b45c4b9cbe976c9a966e679863f2b11951d7739069b77edf4e1b'), ('930658203708416000_0', 'data_image/iraq_iran_earthquake/15_11_2017/930658203708416000_0.jpg', 'test', '0562fc44848a85118a4ffa0f752b18def57fd7ef40130192f32619e8a51c460c'), ('910630400527147014_0', 'data_image/mexico_earthquake/20_9_2017/910630400527147014_0.jpg', 'train', '0de806f2812ea2f36b6d39c0be6847c5a9aa0485d88eb1c373cf4b9d948b90bc'), ('914952139209891852_2', 'data_image/mexico_earthquake/2_10_2017/914952139209891852_2.jpg', 'dev', '031488e7fcb60002b2a1791f723219538cb14d490c7290ffbf05b904876f3605'), ('911846454242578433_0', 'data_image/mexico_earthquake/24_9_2017/911846454242578433_0.jpg', 'test', '5f35be9f18b72ab18b49f3fdba54ea67283d81ae6ecf9c0cbe31c7ac43464cec'), ('910558750913658881_0', 'data_image/mexico_earthquake/20_9_2017/910558750913658881_0.jpg', 'test', '118fba44d21d1e79e67a60ce35a83218d67711ed79b05d031fc335ab5ee847ca'), ('911905889904234497_0', 'data_image/mexico_earthquake/24_9_2017/911905889904234497_0.jpg', 'test', 'c8552584be5197d817a351cde65229a2f4ae0fae36a1e3c1e3e5250e8c815b67'), ('913460305732218881_0', 'data_image/mexico_earthquake/28_9_2017/913460305732218881_0.jpg', 'test', 'f711d7de5d9ae5f71a7aefa7cee01dbaa62554f5cf0087acb2a71e9ebdebc620'), ('911663933790064640_0', 'data_image/mexico_earthquake/23_9_2017/911663933790064640_0.jpg', 'train', '674bf2aac9070a581d05c5d6995accca62bf9091ab9e5357d1eb4414ee57cdd4'), ('910747357700067329_0', 'data_image/mexico_earthquake/21_9_2017/910747357700067329_0.jpg', 'dev', 'b561272dd13646a18f96e760c42b77d624f7808f533221860e1205fc9a9ef36a'), ('910657941295947776_0', 'data_image/mexico_earthquake/21_9_2017/910657941295947776_0.jpg', 'train', 'd60169f3ba4095976443449c4b5325eea6f113299c761c61a71a3049125254a0'), ('910812992908005377_0', 'data_image/mexico_earthquake/21_9_2017/910812992908005377_0.jpg', 'train', 'ada058aa1e308fbc0e855aed67bf4a1cc8b9f02fb868c9ab84cf32a854cce996'), ('911775074318995456_0', 'data_image/mexico_earthquake/24_9_2017/911775074318995456_0.jpg', 'train', '81aa0928cd939b372f0a99f221fff1831430dddc5b1c4b8b9a11423e8d28233b'), ('914411563972534272_0', 'data_image/mexico_earthquake/1_10_2017/914411563972534272_0.jpg', 'dev', 'c5edf096c0b3c0827eb4eca1a2da44710389fd8c06a1818e81ff73a8da4aa9af'), ('912359993680236554_0', 'data_image/mexico_earthquake/25_9_2017/912359993680236554_0.jpg', 'train', '76a45d2312852b13dc2a21a7fe9fa25e68bbb978b89be4ca5b9ff5743370e3aa'), ('910614165076750337_0', 'data_image/mexico_earthquake/20_9_2017/910614165076750337_0.jpg', 'train', '9d318836ec2f83fc81c31dd74b3b430749bf102ec8787306216c5a5d27165592'), ('912128815509364736_0', 'data_image/mexico_earthquake/25_9_2017/912128815509364736_0.jpg', 'dev', 'ed238bb9a933df43eeae0bf253ff7c239b3c6fcd00fa37d70872a9c4465f4708'), ('910579326894071810_0', 'data_image/mexico_earthquake/20_9_2017/910579326894071810_0.jpg', 'train', '77567aaec556ce13dc97951c4e7f716b4a0813a8273d66e55efb4be320520173'), ('870176000109182977_3', 'data_image/srilanka_floods/1_6_2017/870176000109182977_3.jpg', 'train', 'ee6eb231cf1f94c806b5f3c9d1536e7dc27fa07ba4eed1a28353933c98ba3de6'), ('870283105151668224_0', 'data_image/srilanka_floods/1_6_2017/870283105151668224_0.jpg', 'dev', '7ea4ac07dbb644a00ff28462967a323816e76f168bce4923f747431e3fa0f07d'), ('870199337996787712_0', 'data_image/srilanka_floods/1_6_2017/870199337996787712_0.jpg', 'train', 'dfc6352643e8d61d31158cf84d66b89237e76dcd10ce438c39ffb09c933b1ded'), ('869993768467873792_0', 'data_image/srilanka_floods/31_5_2017/869993768467873792_0.jpg', 'train', '822b6aa523ee943a0aa5ebfb9a60376985eb1570f221c29b98c2d494719fb194'), ('870841147119325186_0', 'data_image/srilanka_floods/3_6_2017/870841147119325186_0.jpg', 'dev', '394a7c608d0cb24ee60749efdfe54ff8a9132e1ff86582c7173ad2c6915cfd8a'), ('879374567487221761_2', 'data_image/srilanka_floods/26_6_2017/879374567487221761_2.jpg', 'train', 'fc58f614896e978cde9cc8848b16aa50f70428515f8a1edbe9c6b17406118e0b'), ('871060191520980997_0', 'data_image/srilanka_floods/3_6_2017/871060191520980997_0.jpg', 'train', '9ee03d5fcd1226913e20c3357169246151b404c02cc462d62a18e0559049c9c8'), ('872672448394780672_0', 'data_image/srilanka_floods/8_6_2017/872672448394780672_0.jpg', 'train', 'd5ccef8f04e2784de54b9b8b95b795017c908cddf4571aef241250da292895fa'), ('871494277951508482_0', 'data_image/srilanka_floods/4_6_2017/871494277951508482_0.jpg', 'train', '9abc3781c657edb70550e20544cc2d1209cc45ba21b4bc40e1265afb8144d852'), ('869918891664977921_0', 'data_image/srilanka_floods/31_5_2017/869918891664977921_0.jpg', 'train', 'a9e9826f7bc5f75405107f917bd9fd2662afa66059e4cf4b13675423019019a6'), ('873491045442441216_0', 'data_image/srilanka_floods/10_6_2017/873491045442441216_0.jpg', 'train', '99c23f309b9931d695f327b8ab8127386d74dcbcbd0f0cd64f7f476e6721fc02'), ('917834588814155776_1', 'data_image/california_wildfires/10_10_2017/917834588814155776_1.jpg', 'train', '2f34f2a4b61383f2c84b026405a71d9fabc0c372c725fd8c4ef9bb19370f9586'), ('917857531656933376_0', 'data_image/california_wildfires/10_10_2017/917857531656933376_0.jpg', 'train', 'b29bcb9ec8cc7aced2c93f10cb3e485a1b8ad8f20a524ca8b609f95fc95e4c24'), ('917818978629808129_0', 'data_image/california_wildfires/10_10_2017/917818978629808129_0.jpg', 'dev', 'fb69e0a93cd668e723fca7f17c432ddc6ea202b496b692105b8669197b4348d3'), ('919784813535612929_1', 'data_image/california_wildfires/16_10_2017/919784813535612929_1.jpg', 'train', 'fbfda50b3355d42ad53e82ab7b52a358cdfc905b5affce7c667aee3e78faa78e'), ('921455344425029632_0', 'data_image/california_wildfires/20_10_2017/921455344425029632_0.jpg', 'test', 'd050ffcb6e3679a1ca81db851ea09177967cb43107e4202a481e390b5391c573'), ('922274902299824128_0', 'data_image/california_wildfires/23_10_2017/922274902299824128_0.jpg', 'train', '0e3eadbfdb5cf09bd63e8953d0a74289390092ef62cd865b711d76c4777f195c'), ('918086493134098432_0', 'data_image/california_wildfires/11_10_2017/918086493134098432_0.jpg', 'dev', '7f9acfd49c5edc7ba485a5cbc7d6bd9288c0eb50f1f47b55121dec6597cb5504'), ('921106179807481858_0', 'data_image/california_wildfires/19_10_2017/921106179807481858_0.jpg', 'test', 'f58febbfff7750e3fedc1ad1dc224a420abbb6531e44d2d1210033e7641e2229'), ('920528819626434561_0', 'data_image/california_wildfires/18_10_2017/920528819626434561_0.jpg', 'test', '1df53794780837dce1825f7a4d4baa043a3109b498929ea3c78543cb0cd96c44'), ('918354135350616064_2', 'data_image/california_wildfires/12_10_2017/918354135350616064_2.jpg', 'train', '72ebfa57c7ca6549e6d8051ee0d3fc4919a1774a8024ad8816e4359211e1da27'), ('923292980068237313_0', 'data_image/california_wildfires/25_10_2017/923292980068237313_0.jpg', 'train', 'fb49484209ec7dc9b974a2cce8adb8401651b159ddd8618c9b7ead36eab5d8dd'), ('919937237676040192_0', 'data_image/california_wildfires/16_10_2017/919937237676040192_0.jpg', 'test', '2d72fe97b74a8e052bd8be255f7f58cf484d909533d5cdfd7c45447d5095bf2e'), ('920643131837739008_0', 'data_image/california_wildfires/18_10_2017/920643131837739008_0.jpg', 'train', '249af4a795cefb57f328b19a70c04d62494254142204ee1ff2e1a1d4c53cfd0a'), ('917983113015365634_0', 'data_image/california_wildfires/11_10_2017/917983113015365634_0.jpg', 'test', '200c80e98bdb41d6b8571eefbf27079f8b5be1747ccabda7393b5972a34b678c'), ('917855444475023361_0', 'data_image/california_wildfires/10_10_2017/917855444475023361_0.jpg', 'test', 'c273971be72002846877bfc5198f329620c7fa72b463e9a8abe2d79512b32459'), ('919624193762717696_0', 'data_image/california_wildfires/15_10_2017/919624193762717696_0.jpg', 'train', '2424b7905292c105a73e0aa9dfe44a81d59f8910f6c177c7de7aa392541c0240'), ('918391582197510144_0', 'data_image/california_wildfires/12_10_2017/918391582197510144_0.jpg', 'dev', '7935e55390c9360715ea7495f647068abd843b050007df9d995b3b8d66b73a99'), ('920300145253613569_0', 'data_image/california_wildfires/17_10_2017/920300145253613569_0.jpg', 'train', '52fa106406fb65dd541757fc7d255301074a8bc084012ba90c4330be0dc91295'), ('921623155835224064_2', 'data_image/california_wildfires/21_10_2017/921623155835224064_2.jpg', 'train', 'e251108429e68f96c0b7cbaa88678e280bb26bc8300a448c804eeef95c7058b1'), ('905499135754866689_0', 'data_image/hurricane_harvey/6_9_2017/905499135754866689_0.jpg', 'test', 'be9cb66986aa1496b6d7c234bb5cb0aab672595d0503c59aacd4f427e2a85fe3'), ('906000931170295809_0', 'data_image/hurricane_harvey/8_9_2017/906000931170295809_0.jpg', 'train', '3ccad179395fb75e0c6e963a275d885fdc110481b5a454599fc55294f2e36fbd'), ('906250037629505536_0', 'data_image/hurricane_harvey/8_9_2017/906250037629505536_0.jpg', 'train', '25d10fa2ff7791aa83947acf621d09a562a7074c4f794980b9f49a3d0c8680f5'), ('909220520461180928_0', 'data_image/hurricane_harvey/17_9_2017/909220520461180928_0.jpg', 'dev', 'e846f0bd927398bab7ef251aba9f68b9509a3d8043f93419dcb2d80919ea2120'), ('906090345137659905_0', 'data_image/hurricane_harvey/8_9_2017/906090345137659905_0.jpg', 'test', '59b7e841ad54eaef4f333d73ffbfedd3c9570aefa7b3ae91c3e05366912bdcbe'), ('905926924467363841_0', 'data_image/hurricane_harvey/7_9_2017/905926924467363841_0.jpg', 'dev', '0cff4491714992b950147d245dbfad3f47e7d3d9e9b1292c07e21eeb357b9c17'), ('907301000351907840_0', 'data_image/hurricane_harvey/11_9_2017/907301000351907840_0.jpg', 'dev', 'b1871f1a1d3768151a964b87d0b39b423c08480ab3fb37870693d1f71a3c90eb'), ('904324002176532481_0', 'data_image/hurricane_harvey/3_9_2017/904324002176532481_0.jpg', 'train', '9d628c08da10c9dec3d14cd2376f6dfbb10af64083d1be60ced2e5871a0e2245'), ('901667596151517184_0', 'data_image/hurricane_harvey/27_8_2017/901667596151517184_0.jpg', 'dev', '70eeebd284c5a0231accf972b1df4a3a247caf640f8b49b14f0b03e3365d76d5'), ('901792186026516482_0', 'data_image/hurricane_harvey/27_8_2017/901792186026516482_0.jpg', 'test', '31a57be69a7b7da5ea830736f4043e1c889852c3129e70af005584f2f4eaff90'), ('904464153036894208_0', 'data_image/hurricane_harvey/3_9_2017/904464153036894208_0.jpg', 'train', '3b2f7645c5cfe2cafb6aaa8cb53f62eb287ecd40c0a4f0135ca6e38611fa9745'), ('905555972642541570_0', 'data_image/hurricane_harvey/6_9_2017/905555972642541570_0.jpg', 'test', 'f8ba3bdabe97b7a1adc2b8293a8dfb79ea506fd6f09f9559e84209cec2a2336c'), ('907355599083655173_0', 'data_image/hurricane_harvey/11_9_2017/907355599083655173_0.jpg', 'train', 'df8139b4739b860ade2502579d4c627c234a8fe73834020f3b4eee866d3b5009'), ('905487148652036096_0', 'data_image/hurricane_harvey/6_9_2017/905487148652036096_0.jpg', 'test', '4787cdd19048be9b9664e3235cdc5bb3140c0f685d5ad769b5ce6308a7334774'), ('901835409440768000_1', 'data_image/hurricane_harvey/27_8_2017/901835409440768000_1.jpg', 'test', 'becedf7ef6b4f9657e88f01501e58a324a07f955f3cd0d6e4714cecf3c9e1b09'), ('904322912206950401_0', 'data_image/hurricane_harvey/3_9_2017/904322912206950401_0.jpg', 'dev', '73d4627888fdea9658f64f8f565b2bf4a1d2d70eefb88772919d75e2f79bc4fa'), ('907215000233635840_0', 'data_image/hurricane_harvey/11_9_2017/907215000233635840_0.jpg', 'test', '337804c432e381089868da9d2d977b829cf31724ad0c51de05e96f9c3e663729'), ('905119957821915138_2', 'data_image/hurricane_harvey/5_9_2017/905119957821915138_2.jpg', 'train', 'abdf3f2b481042ea7fdfa77aec65f5f88e9df06340efb19d796161d8192dacf4'), ('905890951117381634_0', 'data_image/hurricane_harvey/7_9_2017/905890951117381634_0.jpg', 'train', '2198f7ec313b0b4e0970d00e3987a30eb4da0a1c975ea9e47e4cf29df0482f3c'), ('907257101029974016_3', 'data_image/hurricane_harvey/11_9_2017/907257101029974016_3.jpg', 'train', 'e4ba52643f56f109823ed8f99a19412c1147009e13ee0c4c5dbe3f3b6e8058df'), ('907272819100327936_0', 'data_image/hurricane_harvey/11_9_2017/907272819100327936_0.jpg', 'train', '0ae94a5309a835ea38ca8226d191f8e337873218e4354805c4e29c5aa4527d39'), ('904825961165795328_0', 'data_image/hurricane_harvey/4_9_2017/904825961165795328_0.jpg', 'test', '346fbf20272e94cd313637fe65a97242bb8498944d7b34ea5450b5c3d55698b6'), ('905052168721506304_0', 'data_image/hurricane_harvey/5_9_2017/905052168721506304_0.jpg', 'train', '6e6edae210e6e100a641b9b149f1f5796e4207747b08c67cc6a4f7d50fd5d9a1'), ('905454433122697217_0', 'data_image/hurricane_harvey/6_9_2017/905454433122697217_0.jpg', 'test', 'a4798462ee3a422fd8d0089dc38c37370fe924e00640c2bc321400667d8dd9a3'), ('905486837162147841_0', 'data_image/hurricane_harvey/6_9_2017/905486837162147841_0.jpg', 'train', '35460cf2b500335664d0980aac08be5d1195662f150e24fe434b0a8676e3d08b'), ('909772570911805445_0', 'data_image/hurricane_irma/18_9_2017/909772570911805445_0.jpg', 'test', '497f4f64df46cd54424bbd82e429feb0151bba91a39a416e3bdb8dda01b90cd4'), ('909915987360727041_0', 'data_image/hurricane_irma/18_9_2017/909915987360727041_0.jpg', 'dev', '354cf0d38651afc2721f13b36e123d4644aa8f342866b81d39f8cecfc37d87ab'), ('910174179605057543_0', 'data_image/hurricane_irma/19_9_2017/910174179605057543_0.jpg', 'dev', '5464b0062d9327a2f05fe9aafca69ee721269863c47bac925231998b256d651f'), ('909434397077753861_0', 'data_image/hurricane_irma/17_9_2017/909434397077753861_0.jpg', 'train', '4d2508492a413bfa2fe6c88013a1ba95d822bdd460510f19a361a971d1c7362b'), ('909792210861694977_0', 'data_image/hurricane_irma/18_9_2017/909792210861694977_0.jpg', 'test', '5275a83843659e0a728adb61055be23d80473a41a2fe72396d138fa1287d109d'), ('910194680045895680_0', 'data_image/hurricane_irma/19_9_2017/910194680045895680_0.jpg', 'test', '85ffc6cf29060d62dc9657252aa4a5bb2fd578ffe740996677403de3dcfe48b2'), ('910091660062789632_0', 'data_image/hurricane_irma/19_9_2017/910091660062789632_0.jpg', 'test', 'fb85d16328d391d0605f46436ab78b431a052decb5d396045e97aaac6ba4028b'), ('905625395042574336_0', 'data_image/hurricane_irma/7_9_2017/905625395042574336_0.jpg', 'train', '56d5d71703bde8c8709152a3468c1cd57fa590f668341273bab354fc8edf4a24'), ('909396171235487745_0', 'data_image/hurricane_irma/17_9_2017/909396171235487745_0.jpg', 'train', '37e0f7c06d57b67672d31d5f3f72c68355d434e5dff350c8213ce0b5e0491a82'), ('910119925565526018_0', 'data_image/hurricane_irma/19_9_2017/910119925565526018_0.jpg', 'dev', 'a71f862011a46d2cae8de2aa52f3a7aa3d7672ac734aaf979b46f3d621106e96'), ('909793306716049408_0', 'data_image/hurricane_irma/18_9_2017/909793306716049408_0.jpg', 'test', '3dd70c65f1900f6ecd2231f47f8a92e335524e7e992f230b9bc772e56a7298f3'), ('910155666899292160_0', 'data_image/hurricane_irma/19_9_2017/910155666899292160_0.jpg', 'test', '74231ac04b920b3b8134d7d44358da8333fbed59e15a8cb98c58875123f871e8'), ('909468250953658369_0', 'data_image/hurricane_irma/17_9_2017/909468250953658369_0.jpg', 'test', 'acd8a2970676b56e646d32520347f1c700c71b895426200f06cf3c1c0c939529'), ('910000060225728512_0', 'data_image/hurricane_irma/19_9_2017/910000060225728512_0.jpg', 'test', 'fb21409447d2a23bd3a9f48b8b5d268f0500449429427b2a6a026f200c729c93'), ('910177380408446976_0', 'data_image/hurricane_irma/19_9_2017/910177380408446976_0.jpg', 'test', 'cff11a8b3bf5043102d5892654fea2197e268badfb16f867025fb471df9d5837'), ('909443048249155584_0', 'data_image/hurricane_irma/17_9_2017/909443048249155584_0.jpg', 'train', 'a9bb359cfdc1bba4dad54048b9e5397a1b01be8a241660f6e893e8ab696c311e'), ('909880224631857152_0', 'data_image/hurricane_irma/18_9_2017/909880224631857152_0.jpg', 'test', '8e9c3e5e9ec9f12dc9973b1e34f9267a5390a384faab352758fa6c704fb96266'), ('909839895077834752_0', 'data_image/hurricane_irma/18_9_2017/909839895077834752_0.jpg', 'train', '60e16dca4989b98889fba4c52fb3e0e3cbde9bd0c2330bfa29e1283908df23f3'), ('910060789159137281_1', 'data_image/hurricane_irma/19_9_2017/910060789159137281_1.jpg', 'dev', '241c58c8cd4585f02d7036d51050f233c982246bb55b573039396db6ff48f95f'), ('910183711034966016_0', 'data_image/hurricane_irma/19_9_2017/910183711034966016_0.jpg', 'dev', '95f9733e25cbe962ebe93eaabac94e5abb26320e13d86c2c4002d76b5c274f04'), ('909914945566810115_0', 'data_image/hurricane_irma/18_9_2017/909914945566810115_0.jpg', 'test', 'f8fd931ac5ab64449c11ba236eff2aac573806b8cb72b33ade0efb68cf069fd2'), ('909728487577083904_0', 'data_image/hurricane_irma/18_9_2017/909728487577083904_0.jpg', 'test', '92d7a53473dc843cbb49ffd5f66a8afcf2a7d8462fab3a0ed9360b02d5fa8f62'), ('913123015726125057_0', 'data_image/hurricane_maria/27_9_2017/913123015726125057_0.jpg', 'dev', '5edc76c4d51b92241bbfa5ae02c3cec8b859d2e3ece0dfceb06a7d812e6416ca'), ('913820679056785413_0', 'data_image/hurricane_maria/29_9_2017/913820679056785413_0.jpg', 'train', 'd7a4d160207a77db237bd7e9bcf9b94714a3e2d007cf8421d25177f3893c9293'), ('912165621198188544_0', 'data_image/hurricane_maria/25_9_2017/912165621198188544_0.jpg', 'train', '6dd367dab04ce8c5708d716f353121b512c451a6a7c3d82f7199624b6fb0ae30'), ('913549726083141632_0', 'data_image/hurricane_maria/28_9_2017/913549726083141632_0.jpg', 'test', 'c1b267e78b9138027815d355f4a7fb7d7e15083ffc21f7d4a3c682df7d738f7e'), ('911818109123747846_3', 'data_image/hurricane_maria/24_9_2017/911818109123747846_3.jpg', 'dev', '62233f808405a42653768971d059ebc102e268c29f7853772ca40be853a3d09c'), ('928972399139344384_0', 'data_image/hurricane_maria/10_11_2017/928972399139344384_0.jpg', 'dev', '6f5ffb47cdeee96c718a7cf4626d04c0a303425aae1b901fd0b6f8e3e571eb0c'), ('911818109123747846_1', 'data_image/hurricane_maria/24_9_2017/911818109123747846_1.jpg', 'dev', 'd9e9f7d73cd2bb8a5c63f5269261321498788ab3a7e214ef846bcab084d92d2d'), ('928473762688794624_0', 'data_image/hurricane_maria/9_11_2017/928473762688794624_0.jpg', 'train', 'bf0c50cd0ffb102b6c684f285280a95b370d3a5bc0dd9a1b9331cac098540c3c'), ('912103009819250688_0', 'data_image/hurricane_maria/24_9_2017/912103009819250688_0.jpg', 'test', 'fc93ce02df411a26b0a3b10b8a75ecab8ccc2a135efb57e9f175f420488e0228'), ('928651012298104833_0', 'data_image/hurricane_maria/9_11_2017/928651012298104833_0.jpg', 'train', '044e107e2e6528171c92b47046c2b104f515d626ac1dbf0c449022f30d3cc5b7'), ('912021400734617600_0', 'data_image/hurricane_maria/24_9_2017/912021400734617600_0.jpg', 'dev', '6530e492dbdfe3595bc66ef5cba998034bc8e433fbb551d5b32d848eaad6107f'), ('916104794288394240_0', 'data_image/hurricane_maria/6_10_2017/916104794288394240_0.jpg', 'train', 'f22d741ebc3f7e271861879ab0fb77f0bf4be85ca476f4283acf7bc8889650f6'), ('921842963843551233_0', 'data_image/hurricane_maria/21_10_2017/921842963843551233_0.jpg', 'test', 'c214f223f9e5a2896e47238a92c4dc282d371e0ea68ce230b7ff156e304648ce'), ('921746970527653888_0', 'data_image/hurricane_maria/21_10_2017/921746970527653888_0.jpg', 'train', '33f034a5a2a4de085c877323fdac4e5db621fdf46d18b300bfe60ecc59b7431a'), ('927302076086005760_0', 'data_image/hurricane_maria/5_11_2017/927302076086005760_0.jpg', 'train', '830b14e9c6c2e46fceebaf8552a3fb40cb64112d140bbe67cc15b653ad288d9b'), ('921528967533805568_0', 'data_image/hurricane_maria/21_10_2017/921528967533805568_0.jpg', 'dev', '36b4f22e3f6da77fae80c98f652469fde2c9091cb5214e56e3ad4c0c4dec5e80'), ('929456086737854464_3', 'data_image/hurricane_maria/11_11_2017/929456086737854464_3.jpg', 'test', '193189fcb4fd4d2d00fa8500d9f302a49b4a186e7e28d94f34b2a330932ab8cb'), ('923578730114514945_0', 'data_image/hurricane_maria/26_10_2017/923578730114514945_0.jpg', 'test', 'cac2f9995e65ec11b7b3a83da6d82f8142fd68a5214be7ae27dd19fbe26bd0aa'), ('913501170173337600_0', 'data_image/hurricane_maria/28_9_2017/913501170173337600_0.jpg', 'dev', '1bb1c599bab8647aa88f3b1fcbe9b901e185338bae6111fccbfb1a47fc3bb220'), ('916187309103607808_0', 'data_image/hurricane_maria/6_10_2017/916187309103607808_0.jpg', 'train', 'ae78b68f5f2375fe5a4bb5f72f80968440118795e485073d36ea9476a50989d1'), ('911492333702656000_0', 'data_image/hurricane_maria/23_9_2017/911492333702656000_0.jpg', 'test', '207621bb8cd4f21c788039d157c50ebb07491e3cabf2034ddda8159e8f4a3c73'), ('923085525677936640_0', 'data_image/hurricane_maria/25_10_2017/923085525677936640_0.jpg', 'test', '5f0d263ffc15c0152090937c74caafbeca170e7f97e39a3b75e1e799bd1148ab'), ('923205262613405696_0', 'data_image/hurricane_maria/25_10_2017/923205262613405696_0.jpg', 'dev', '4941422fa0b71415e62f10227829109fc483f60f1c6071ced29abab5db7854cd'), ('921784458218315776_0', 'data_image/hurricane_maria/21_10_2017/921784458218315776_0.jpg', 'test', '79f1aa46a60a5a9c6d7efa4b68ab66f27e3311d21bb1158ebc3fabd00ee0db94'), ('927563668103073792_0', 'data_image/hurricane_maria/6_11_2017/927563668103073792_0.jpg', 'train', '819af81df514d22e18e4cd82753060447fb62b97674b28793608807937ca934b'), ('913408134638477318_0', 'data_image/hurricane_maria/28_9_2017/913408134638477318_0.jpg', 'test', '6a01da20ccf28cf5a7ae4794d2e3c1fa72404101a50eb34fb19fc7884e751eda'), ('929997322351673344_0', 'data_image/iraq_iran_earthquake/13_11_2017/929997322351673344_0.jpg', 'train', '7a70469db0cbf74df91bf332479e1c77066735926032463e961fc0b5ccd1ae6f'), ('930197285417443329_0', 'data_image/iraq_iran_earthquake/13_11_2017/930197285417443329_0.jpg', 'dev', '0f4342117e7b0ac322de5179c3129ed9ede2d47d674cfb53d2f6c118a88ac913'), ('930071199157293056_0', 'data_image/iraq_iran_earthquake/13_11_2017/930071199157293056_0.jpg', 'test', 'a19a20a0da358969e2d9112c411bff4607640d43c4ff578ac0d51c1dc8710e19'), ('930262966460485632_0', 'data_image/iraq_iran_earthquake/14_11_2017/930262966460485632_0.jpg', 'train', 'e27055c33ee8c63594bff526ade300d6534f1d208891230657dd3a54a7697c9d'), ('931824949509140480_0', 'data_image/iraq_iran_earthquake/18_11_2017/931824949509140480_0.jpg', 'test', 'de941f924af386b4dec4674bd4b41081482ba774558fdb3c88346827f9b832bc'), ('930624137143508992_0', 'data_image/iraq_iran_earthquake/15_11_2017/930624137143508992_0.jpg', 'train', '63f435a908b62805775ba431e202e090707a35fb7e848b251f910fa75c565609'), ('931062729779138560_0', 'data_image/iraq_iran_earthquake/16_11_2017/931062729779138560_0.jpg', 'train', 'ee5c675b7618d05b2e125c77d1d7b4b93e6b2f8b5d6fdd872ef2e59b0a591543'), ('930074444151869440_0', 'data_image/iraq_iran_earthquake/13_11_2017/930074444151869440_0.jpg', 'train', '5abb118cfa16c8e1edd56a77eb55f8aaa37f58743a8c2597f57d5c2c7c0ed431'), ('931202226952986624_0', 'data_image/iraq_iran_earthquake/16_11_2017/931202226952986624_0.jpg', 'test', '17cf77fc3300b3d42273ff989346a873643ccb0ec2a1e565491727a74ee33919'), ('930992584347938818_0', 'data_image/iraq_iran_earthquake/16_11_2017/930992584347938818_0.jpg', 'test', '20674df585090b9a18130ceed56f87e6eb85f935ba61f5a2e819d8379fa310db'), ('930380460726865921_2', 'data_image/iraq_iran_earthquake/14_11_2017/930380460726865921_2.jpg', 'train', 'ec2463372fdaf5c1ce9447cb4f687f8b0752edb8a821bc24640fe2a54952d0a6'), ('930151625032740865_1', 'data_image/iraq_iran_earthquake/13_11_2017/930151625032740865_1.jpg', 'train', '1613aaf1502cc34ed8df53a8273f9dcf79038f7f1151597776e734e3b57dda1e'), ('912987423327637504_0', 'data_image/mexico_earthquake/27_9_2017/912987423327637504_0.jpg', 'test', '7db89e8d87cb21a7e937ec04e39f9462161c25b05c1a8778e033f67bfaf4fe6b'), ('910524793740513280_2', 'data_image/mexico_earthquake/20_9_2017/910524793740513280_2.jpg', 'train', 'ceb47951bca87e9ca3ace16d5f7653ea2ff516401137bf0be50d6897041c4c08'), ('910525286009102336_0', 'data_image/mexico_earthquake/20_9_2017/910525286009102336_0.jpg', 'train', '7bb99f3b33deeca00454c7b4f58c684a04e244d0d0a4307ad3e3cce2dfb1b492'), ('910718933019320320_0', 'data_image/mexico_earthquake/21_9_2017/910718933019320320_0.jpg', 'test', '63760704fd58d3d4910f0d1a86154b6e64cc11bde62f2f2684276f1be4cf6a37'), ('910784474337574912_0', 'data_image/mexico_earthquake/21_9_2017/910784474337574912_0.jpg', 'train', '6c366e7690b31f484591cb36ff4be7ace4d8a2b8df3424a6cc6c6a2bfa4bef83'), ('910613883613847553_0', 'data_image/mexico_earthquake/20_9_2017/910613883613847553_0.jpg', 'test', '93ef1e473ac074aa6645263f964dd3ee3131353dc01ab3e19599424b95e87191'), ('910713422358204416_0', 'data_image/mexico_earthquake/21_9_2017/910713422358204416_0.jpg', 'train', '53eddbd9100d283ad8877073dd8b2b76b9c4db8597c4c9a72e3663c660c2a694'), ('910790915840774144_0', 'data_image/mexico_earthquake/21_9_2017/910790915840774144_0.jpg', 'dev', '2d2e326835c7f694c66e52b1126db0448937732e7ed644eb9c604a3a10b8bd9f'), ('910712437266989056_0', 'data_image/mexico_earthquake/21_9_2017/910712437266989056_0.jpg', 'test', '6e1dc86664df0f89cd13f46a055e61239801031d15500ae5e42861ca23257de8'), ('910732598434971648_0', 'data_image/mexico_earthquake/21_9_2017/910732598434971648_0.jpg', 'dev', '2a8797c27ca54b5ed0ad78d3f51e93f2863792862113a545dd480fa4e34ff8cf'), ('912950452223975425_0', 'data_image/mexico_earthquake/27_9_2017/912950452223975425_0.jpg', 'train', 'f9210d84880b85f49c43968bc3f35da51d5ef217a704b5087eb8cd7d713b07aa'), ('910708503681032192_0', 'data_image/mexico_earthquake/21_9_2017/910708503681032192_0.jpg', 'train', '76b975be8d298b828ea60b1020d3c633d671aaebee87bcc942b7491f9daa85c2'), ('910534117099372544_0', 'data_image/mexico_earthquake/20_9_2017/910534117099372544_0.jpg', 'train', 'fb0773930cf2e97777f5998092b1a80850fceed0c3f4e67c3f7d446e790cd41b'), ('913581869588811776_0', 'data_image/mexico_earthquake/29_9_2017/913581869588811776_0.jpg', 'dev', '5e89b74c22aaa18d7f57f1ebb09436beabe98bb56aab9b7151a05912fdad18f0'), ('911655195217719296_0', 'data_image/mexico_earthquake/23_9_2017/911655195217719296_0.jpg', 'train', 'fc1d24b7620540d8b0c5fc647123fe67377d7b9993e5ed0223604afb3d42874c'), ('872297618147340289_0', 'data_image/srilanka_floods/7_6_2017/872297618147340289_0.jpg', 'train', 'fa829b7ea74e322e75b7c2fbd0ad20360e86f154e73661e2a2907d0034237393'), ('870015709555834880_0', 'data_image/srilanka_floods/31_5_2017/870015709555834880_0.jpg', 'train', 'beca749bfae509f0dff425a293184e70973330aa1d67ee741948b36cdf0477cc'), ('870023473166401536_0', 'data_image/srilanka_floods/31_5_2017/870023473166401536_0.jpg', 'test', 'd11883512c721941b02dddad965a3c1b7ba6a6bdbf0d5b218fa71c5a7bded26c'), ('870054333122883584_0', 'data_image/srilanka_floods/31_5_2017/870054333122883584_0.jpg', 'train', '68d315c6f7e6cacd35cac90179905aed8062afb9d1558ade9ed1a45d307d1f58'), ('873258646397833216_1', 'data_image/srilanka_floods/9_6_2017/873258646397833216_1.jpg', 'train', 'b3801cb709816eafef3c5274919ac269420e9e8f7839d968e28482bb5a3efb2d'), ('871012375096422400_0', 'data_image/srilanka_floods/3_6_2017/871012375096422400_0.jpg', 'dev', 'bbf916c89f3bcf617d7ae6ed3856c48c38cd93e71d42ecd2972b1aea1633719e'), ('878326464734609408_1', 'data_image/srilanka_floods/23_6_2017/878326464734609408_1.jpg', 'train', '1a44895868e0672ebe1ee73e46fd30a892d3f09ed7fffb7407693cbf502b4a3e'), ('870419183632228353_0', 'data_image/srilanka_floods/1_6_2017/870419183632228353_0.jpg', 'dev', '747eef1df94098a886d8b13d7537e9110addb96d7486758ab77edbb5d3adf26f'), ('871272409667559424_0', 'data_image/srilanka_floods/4_6_2017/871272409667559424_0.jpg', 'train', 'f8e35de4319643c0a718a671990934663c662635b548f2cee93ec61d75c937f9'), ('871389377590644736_0', 'data_image/srilanka_floods/4_6_2017/871389377590644736_0.jpg', 'test', '8ec027e98893ac20cd530f8b94bd38ad02f97611c3ed35d9ad3611275860a066'), ('919909006508019712_0', 'data_image/california_wildfires/16_10_2017/919909006508019712_0.jpg', 'train', 'ea45dae140e17f8caddb4bbed75f1abe43925ba89c6286577328e0493bd0664b'), ('919912231952928768_0', 'data_image/california_wildfires/16_10_2017/919912231952928768_0.png', 'train', '3e2b29d160758ae9b3d8aadc95c58043ef2e4e23bdb8329decada67a4fa18121'), ('917828328475680768_0', 'data_image/california_wildfires/10_10_2017/917828328475680768_0.jpg', 'test', '6bf4741d9f30c51935709ffdc5886f2b2f523fd0d8802ed72f285d2b8900f7c7'), ('917794232160661505_0', 'data_image/california_wildfires/10_10_2017/917794232160661505_0.jpg', 'train', '0c8b881694c043b87439c263633721196197ecbe7f46df5ba9360e40691065f8'), ('918053549912838144_0', 'data_image/california_wildfires/11_10_2017/918053549912838144_0.jpg', 'train', 'dc856fe8c58b01b3d46a5cb239ac68778cb4945170dcdc5a03a786b1dd4616c6'), ('917923292148912129_0', 'data_image/california_wildfires/11_10_2017/917923292148912129_0.jpg', 'train', 'a0277f96ee2255a6f9979fe26edafb58ca008a340774fd185796914386ccf952'), ('919562679404003328_3', 'data_image/california_wildfires/15_10_2017/919562679404003328_3.jpg', 'train', '67b52605a78f50f25ffb00b032c5c0d0d13cd7284194f41324c5551eb950ac5f'), ('917847260200886275_0', 'data_image/california_wildfires/10_10_2017/917847260200886275_0.jpg', 'test', 'd5996f72fcf865f9beafc3e9cc6af5816902aa49ca68c581d6af086e5d864949'), ('920552749485428736_0', 'data_image/california_wildfires/18_10_2017/920552749485428736_0.jpg', 'test', 'd59c4e4929c98917268c247feac27c9953d7b3624b5603921864efcbfce7a70f'), ('921705168885944320_0', 'data_image/california_wildfires/21_10_2017/921705168885944320_0.jpg', 'train', 'c26303d3f190d2766710c065a3182d9c6be9dc4b544bbbf7ee7d23cbf6c416ae'), ('919799276271800320_0', 'data_image/california_wildfires/16_10_2017/919799276271800320_0.jpg', 'test', '0345fd32eb6a57ef63fb8aed614ad65525dbae6f8d5adacf6901233ed34e5e03'), ('917926207492120576_0', 'data_image/california_wildfires/11_10_2017/917926207492120576_0.jpg', 'test', '3a993144b306d67e417b71f4f20c5873f0dae95a9712194337d9e7297d5dd5fd'), ('920700423497048065_0', 'data_image/california_wildfires/18_10_2017/920700423497048065_0.jpg', 'dev', 'c64fc06428e676ced90ccb44a439cd80834dcbb0001b7949a8273f175c39c8ad'), ('920628640005738497_0', 'data_image/california_wildfires/18_10_2017/920628640005738497_0.jpg', 'dev', 'b65e3c83bf9cc483f9932b62d675691875780c9ec53ed9d0615d4b76de45343b'), ('920009093376491521_0', 'data_image/california_wildfires/16_10_2017/920009093376491521_0.jpg', 'dev', 'd7a94a4b6f8ddad587b4427652b2b3bfa76aa9f8c4d25402240d2903dc6da0cd'), ('919647928360296450_0', 'data_image/california_wildfires/15_10_2017/919647928360296450_0.jpg', 'train', 'dda6ddfb47807b480ed94cf60436be459c70ed964daff9a70160934cc41fa483'), ('920300150869737472_0', 'data_image/california_wildfires/17_10_2017/920300150869737472_0.jpg', 'dev', '748ff3467b7fcfb83a86edd97183d077d1a8291208740c5a12144b1ec5338dfb'), ('918100629922504704_0', 'data_image/california_wildfires/11_10_2017/918100629922504704_0.jpg', 'dev', '8cb77f092f8ccace632db29296dd32c28e32bb9d11339835093ba6e829f41403'), ('906464461946531840_0', 'data_image/hurricane_harvey/9_9_2017/906464461946531840_0.jpg', 'train', 'f4b50d5fa9079e697f404f8c99ffc84eb506c735742f8b5bcf4f5a711b8fc412'), ('904295228256989184_0', 'data_image/hurricane_harvey/3_9_2017/904295228256989184_0.jpg', 'train', '732166fa8870aabfdffe58315956f0dbde2057675637843ed707c495459f3a20'), ('901779677429342208_0', 'data_image/hurricane_harvey/27_8_2017/901779677429342208_0.jpg', 'dev', '2fd199c5628084815ec28a92708019f86894b17f45964fa13446b0e5b4ffcb7f'), ('909157252518563840_0', 'data_image/hurricane_harvey/16_9_2017/909157252518563840_0.jpg', 'test', 'ee557b302cde90eef51189e47279c9a1593ce264cf6d2521a18838ede95d74c8'), ('907772978116653056_1', 'data_image/hurricane_harvey/13_9_2017/907772978116653056_1.jpg', 'test', 'bea239e0aa0ad5ec35e3b4678c860bafa13ff94b3d6bbde4d401bee4d9bf77df'), ('901809537828696064_0', 'data_image/hurricane_harvey/27_8_2017/901809537828696064_0.jpg', 'train', 'cb21c579f48f9adb9b273f422e02445173d5be70642ed091389e9bf6256ee4aa'), ('909154298579374083_1', 'data_image/hurricane_harvey/16_9_2017/909154298579374083_1.jpg', 'train', '79bc44bc1d0849a8c6ce143dc4ea661a550368085fb2112b222beff38ed62012'), ('905499549430636545_0', 'data_image/hurricane_harvey/6_9_2017/905499549430636545_0.jpg', 'test', 'c8324fa48c0a32f64d2c1ae4b490ef10825ef4b3038b45cce23e684f4212b11a'), ('905540825261772800_0', 'data_image/hurricane_harvey/6_9_2017/905540825261772800_0.jpg', 'test', '450f0a49e70ccfd8df917108c1e7808789d87dd304e36f695e619f27d3c05c47'), ('906907801192222720_0', 'data_image/hurricane_harvey/10_9_2017/906907801192222720_0.jpg', 'train', '0ee1425517575a6f2731444ab28a708299e6bc8d6e377cc76f338f1cdf3ce680'), ('904630236712665088_0', 'data_image/hurricane_harvey/4_9_2017/904630236712665088_0.jpg', 'test', '4188ffb29817c50b0b7241a8936d9bb7c375dee191006322f364fcbc165d62ee'), ('901650362410688513_2', 'data_image/hurricane_harvey/27_8_2017/901650362410688513_2.jpg', 'dev', 'c221e2646472d0afaba1bf3e57ac298cbe9d5cc6b5ac300f217e2677d80a8a2d'), ('905432778019307520_0', 'data_image/hurricane_harvey/6_9_2017/905432778019307520_0.jpg', 'train', '74976ea3dc6140417272954b706deb53d50dda397f9d1c995b339b474e819b87'), ('907584474547933185_0', 'data_image/hurricane_harvey/12_9_2017/907584474547933185_0.jpg', 'test', '07dfd158424f71825b6035223da9a1a0098b66e7201aa546fc42368622af91d6'), ('904281502539628544_0', 'data_image/hurricane_harvey/3_9_2017/904281502539628544_0.jpg', 'dev', '3b04a64ad19eb363c3fdb0492318082e5c3d0bf1b668c718ddf3e5f3fece99e1'), ('904556347164950529_0', 'data_image/hurricane_harvey/4_9_2017/904556347164950529_0.jpg', 'test', '98008ff2e64539eb95b4653c1b01132ad62c8f2569fbd57953b75dc825b17800'), ('906048371714084864_0', 'data_image/hurricane_harvey/8_9_2017/906048371714084864_0.jpg', 'test', '090168910c2d66aa07e827c9e73ff784e3ee034969c9ebb2146ca0ba82edb659'), ('904410882624679941_0', 'data_image/hurricane_harvey/3_9_2017/904410882624679941_0.jpg', 'dev', 'bb1b537d10f9995b92fc90a8be2f1f52293b753be259949e7f35fb3bd7653582'), ('908299694849347584_0', 'data_image/hurricane_harvey/14_9_2017/908299694849347584_0.jpg', 'train', '248c0395856e8992a52ff76318ef35c617d8e332123a0d8ba2dde4c430d9a30e'), ('908478161070972928_0', 'data_image/hurricane_harvey/14_9_2017/908478161070972928_0.jpg', 'dev', 'fb0d6ba9faf5fb9e99c4546c189eefdd096861f34faee428af2435a6f8021bc2'), ('908525619037732865_0', 'data_image/hurricane_harvey/15_9_2017/908525619037732865_0.jpg', 'train', '4e34a947715b5a6c0dbcbfc7c53e5df8c5caf62315e543709ab675eb545380bf'), ('905400513201446914_0', 'data_image/hurricane_harvey/6_9_2017/905400513201446914_0.jpg', 'dev', '0a86666d16f75c8270a8441d7afb51d52efa2c4d86bb47154dbe7273ace74ce0'), ('910145294595223553_3', 'data_image/hurricane_harvey/19_9_2017/910145294595223553_3.jpg', 'train', 'bc8d7324c5b07ee2fe1a0986485b1f42eef7933d5db2dc03f3b10c756c9d02cd'), ('904952256415531009_0', 'data_image/hurricane_harvey/5_9_2017/904952256415531009_0.jpg', 'test', '105a9ff323de9c3a3e039fe20e625228dd10d4f0796d7c2dd23a4a5b4776a16a'), ('909855087144718336_0', 'data_image/hurricane_irma/18_9_2017/909855087144718336_0.jpg', 'test', '53d1d6faf0beceaa9903c8adb71c4ecdcf7fd114ae7394df3d8e3c4a0ad2d52a'), ('910070466265731072_0', 'data_image/hurricane_irma/19_9_2017/910070466265731072_0.jpg', 'train', 'e0cba9f406358d5937e1830ab981d7b54b0da727789390270de57711754d2a22'), ('910270036861886465_0', 'data_image/hurricane_irma/19_9_2017/910270036861886465_0.jpg', 'test', '23baf4afc731c6c894a1637cd45cf40419cf045ede84cfbe4eb9fd381df7224c'), ('909890256299745280_1', 'data_image/hurricane_irma/18_9_2017/909890256299745280_1.jpg', 'test', '812010ed036a673a523048bd7ae1b4c9713ba3ed73b6beff72613fe40e614f81'), ('905705134361120768_3', 'data_image/hurricane_irma/7_9_2017/905705134361120768_3.jpg', 'dev', 'e132a59c865fc3a9bf1f150bc9850bc3d22008c8ef2586e3dad4803e9e1ab80c'), ('910203290645823488_0', 'data_image/hurricane_irma/19_9_2017/910203290645823488_0.jpg', 'train', '4cf19840edb05a79f4453f4b460084a7f1f6dd7e7e4a257211ca293c8358ea38'), ('909879689610743810_1', 'data_image/hurricane_irma/18_9_2017/909879689610743810_1.jpg', 'train', 'ed79f8965be5efb777adc921c5294d70380c115e0ab160451fb032424f8803ca'), ('910240814097248256_0', 'data_image/hurricane_irma/19_9_2017/910240814097248256_0.jpg', 'train', '301b578aaa8a21e6be2878df676ae1369ce7f0f3f949b55a96102caa1c43b159'), ('905714507657469953_0', 'data_image/hurricane_irma/7_9_2017/905714507657469953_0.jpg', 'dev', 'a1095838ddb4a3dc1ad9142e7eee9add02851d588685e03c17dad6645b57eab8'), ('910156250368888832_3', 'data_image/hurricane_irma/19_9_2017/910156250368888832_3.jpg', 'train', '9c16aff2bf2f2ec5b5602c80348c240b07188c27409bd1b98deb181230f46a3e'), ('910182954441330688_0', 'data_image/hurricane_irma/19_9_2017/910182954441330688_0.jpg', 'dev', '8a3b03e7b1e4ea3f8c5d559009f6ff7753eef63efa7d11fd27e910ae3d9d8018'), ('909940694625726464_0', 'data_image/hurricane_irma/19_9_2017/909940694625726464_0.jpg', 'train', '90f43543ae2de4d8ca605d4304ede7f691d9a77f79278da3bfa585ba69175452'), ('909723445289799680_0', 'data_image/hurricane_irma/18_9_2017/909723445289799680_0.jpg', 'train', 'ecdce7719db939ca17a33100016dd626a098f8bc2adaae232e333ba8896e84c4'), ('909484490237636608_0', 'data_image/hurricane_irma/17_9_2017/909484490237636608_0.jpg', 'test', '5f865767481079fdf8f65f6dd60297ff276d57ec24bb38fc8a38a92059c60e6e'), ('909907257852170240_0', 'data_image/hurricane_irma/18_9_2017/909907257852170240_0.jpg', 'dev', '84a30b4b751519649f65915a28d35c0fbeea620a146c4e2dad810dbe5d90aa12'), ('910156539561988096_0', 'data_image/hurricane_irma/19_9_2017/910156539561988096_0.jpg', 'test', 'b35bf1b855b86204bfe7e85246e957f989401f7acc30649a7f49c15972bba82e'), ('909854830725910528_0', 'data_image/hurricane_irma/18_9_2017/909854830725910528_0.jpg', 'test', 'aeb9fe29487764e2ed4902fa417f4faa9822af3dc0a1784ded20d15c4827c7ca'), ('910124040584667136_1', 'data_image/hurricane_irma/19_9_2017/910124040584667136_1.jpg', 'dev', 'fe59a4b8d7a824757dc3d7f15f7e19cc139e8df028c2fd932064615b7d89cfec'), ('909804272677675008_0', 'data_image/hurricane_irma/18_9_2017/909804272677675008_0.jpg', 'dev', '4223322f903efff0dc3cf718f56a48e78417377bf54f933ef21cd3115d803840'), ('909890390504882176_0', 'data_image/hurricane_irma/18_9_2017/909890390504882176_0.jpg', 'dev', '0d9e176e9b1483dd841d87d0a7f5eecd6cec15c07f332265fcd74c71fe5793be'), ('910177030741864448_0', 'data_image/hurricane_irma/19_9_2017/910177030741864448_0.jpg', 'test', 'ab96b647a70ddf193af18004c8d752eb9ae3f18ae14fe7ee384fc77925ca5af5'), ('909870984223690753_0', 'data_image/hurricane_irma/18_9_2017/909870984223690753_0.jpg', 'train', '18779e74ee595b6c05d7db840994be635f9fcf9880ba4f02d60a7345cf19e744'), ('913627390043197440_0', 'data_image/hurricane_maria/29_9_2017/913627390043197440_0.jpg', 'dev', 'a4da92996e451a5c6e4d3a97ed8880a23e11db953976262b4f870384555e73d7'), ('921299994556149762_0', 'data_image/hurricane_maria/20_10_2017/921299994556149762_0.jpg', 'test', 'ee83d269f9e17f707f388306cf8bc8c2f648dc6595cb3943e1ea8101d0829f85'), ('922496274129129473_0', 'data_image/hurricane_maria/23_10_2017/922496274129129473_0.jpg', 'dev', 'fdbbf072b9ff66f0d17970ba9654c61356c8464047fd5faa7e3257eb63ebdf5a'), ('910538720117391360_3', 'data_image/hurricane_maria/20_9_2017/910538720117391360_3.jpg', 'dev', '96bb654d58bc2c668d5778f0c846ae439a8cf33d81c1ca22bfe0a788d0081b69'), ('922606066034270208_0', 'data_image/hurricane_maria/23_10_2017/922606066034270208_0.jpg', 'train', '3b5c1b0f3c6789a5431f66cb3125fdd2f6ae5a75641cce2a9bd58c7cb8ed63fa'), ('921806731625357313_0', 'data_image/hurricane_maria/21_10_2017/921806731625357313_0.jpg', 'train', '01e4ef36febca8cdf7ad1738e479df13d77b845d8ef971bcfcce30e861a69f54'), ('911646354685997056_0', 'data_image/hurricane_maria/23_9_2017/911646354685997056_0.jpg', 'train', 'bb46bd0906276f76da1aa708ecfce4e00f75454d91502860a2d82dd52bafc690'), ('914272099329507328_0', 'data_image/hurricane_maria/30_9_2017/914272099329507328_0.jpg', 'dev', '7c504b94a4161aa182ce623723a143805591a7ef91ed7b546c65ef5c63125495'), ('912286890845511681_0', 'data_image/hurricane_maria/25_9_2017/912286890845511681_0.jpg', 'test', '1fc855602d7cf99b2e4b9d76228232b15f00f31605bdc8a95b43b7e2d96d6669'), ('921306645833175040_0', 'data_image/hurricane_maria/20_10_2017/921306645833175040_0.jpg', 'dev', 'f791f5b9c70193de793105331d957d354c4d1c51e9a21369a7f5a60af04e831c'), ('927699646159380480_0', 'data_image/hurricane_maria/7_11_2017/927699646159380480_0.jpg', 'dev', 'd6540d975e8aa763fdc27f39ad785b0082ee097a1457d05374c435fcebfc5354'), ('929379143506153473_0', 'data_image/hurricane_maria/11_11_2017/929379143506153473_0.jpg', 'test', 'b9c88bb31d0bbd5f1a699eebfe654bf177110214e74b52a7f4ce6d6a0067c5d1'), ('910540043126808577_0', 'data_image/hurricane_maria/20_9_2017/910540043126808577_0.jpg', 'dev', 'f34f93ee25d80d4976d621102787402c04cdf38f7a73c7243642f84034e457a2'), ('927913661485715456_0', 'data_image/hurricane_maria/7_11_2017/927913661485715456_0.jpg', 'train', 'cb8e34fed0a8dcd9ef73ffd0c3c18d8039d487dacf3699fde52a15ce8a06e17e'), ('928009110779514885_0', 'data_image/hurricane_maria/7_11_2017/928009110779514885_0.jpg', 'test', '4c694d24366e890169de807b87d38aad138793ca36e50308c0cf01d997d3d7d7'), ('922881031463755776_0', 'data_image/hurricane_maria/24_10_2017/922881031463755776_0.jpg', 'train', '56584d46b91863e33f2e243a15be413be926e7cf42de6155323d33aed82364b5'), ('916148763105943552_0', 'data_image/hurricane_maria/6_10_2017/916148763105943552_0.jpg', 'dev', '09b9dfd311c29fde12c2a603036d074ada22dd12d7809aebbe8c593315203ab8'), ('922474988472492037_0', 'data_image/hurricane_maria/23_10_2017/922474988472492037_0.jpg', 'test', '5ab3539b43220624eb613c9c38b290c3c631c105638b8b1c65da4f021c86b203'), ('916062103856918529_0', 'data_image/hurricane_maria/5_10_2017/916062103856918529_0.jpg', 'train', '326154c08f1450667bb632144c9e611cad37bf21ee32eb60e000126fefdba735'), ('916341078013435904_2', 'data_image/hurricane_maria/6_10_2017/916341078013435904_2.jpg', 'train', '7ad7a417363b5beb2908df20e07f44153110c40b10918ad946c6f056bca0796f'), ('928442150899081216_0', 'data_image/hurricane_maria/9_11_2017/928442150899081216_0.jpg', 'test', 'c73f062d0148e909bce06382c0dd776f8a995b964af4c520e0b019bb77e660be'), ('913260146121469953_0', 'data_image/hurricane_maria/28_9_2017/913260146121469953_0.jpg', 'train', 'c56568cf4e403b75b8652bb8376d71de04042cfc52265be16908855920f065c2'), ('916331694596214786_0', 'data_image/hurricane_maria/6_10_2017/916331694596214786_0.jpg', 'train', '5e79f08efe1ef21f474e2008c194dd05c880a09625f3f798a3d5820637e0949d'), ('914085315698192384_0', 'data_image/hurricane_maria/30_9_2017/914085315698192384_0.jpg', 'test', 'c08b03858ea37ed326b81093bb2fc59acf734132d7cd59cdf244577cf14d9f22'), ('911954851298496512_0', 'data_image/hurricane_maria/24_9_2017/911954851298496512_0.jpg', 'test', '15a7d57659e42c9393e4a0039a61e71c0a31f1f36594062dc5befa49f22ca2fe'), ('929996293245718528_2', 'data_image/iraq_iran_earthquake/13_11_2017/929996293245718528_2.jpg', 'train', '55cfabae52daa11e3147b76fc446407cad139b375b877a0a2580c4804d9fba15'), ('930163638819786754_0', 'data_image/iraq_iran_earthquake/13_11_2017/930163638819786754_0.jpg', 'test', 'a8c9e1ed862f38c11b6c5a1f447c919edc07cadbad208542276e958a354cc339'), ('930444320095789057_1', 'data_image/iraq_iran_earthquake/14_11_2017/930444320095789057_1.jpg', 'dev', 'd59e563eb16e064e90c682b057a8d38b73e487ecfd647311c36f644e43dfdcf9'), ('930725990371905536_0', 'data_image/iraq_iran_earthquake/15_11_2017/930725990371905536_0.jpg', 'train', '420d7d94bf6f72b88d5d5e0a2f1a0c9b7ab89de4f8a1b15688f136ab07465e01'), ('930076620018167809_0', 'data_image/iraq_iran_earthquake/13_11_2017/930076620018167809_0.jpg', 'train', '6d4011e8100e143c98edcfef1a3a5d151a9d31ac173db5c94eddf11f5c28dda7'), ('930952369461964800_0', 'data_image/iraq_iran_earthquake/16_11_2017/930952369461964800_0.jpg', 'train', '34a3696223fffba58f357415b572bc00828eb2eb997a5a35b589f33bbdafcc3c'), ('931531506564157440_1', 'data_image/iraq_iran_earthquake/17_11_2017/931531506564157440_1.jpg', 'train', '0c03b8ec36f1c67151d7752760bfab85a6a0a6ea5c25a51b8fc972ffc3d755fd'), ('930022991311917058_0', 'data_image/iraq_iran_earthquake/13_11_2017/930022991311917058_0.jpg', 'train', '5ae7c7e0aacc76e23e0c3b8b412cd7e41912ca10414ab319686b1e0dbf4513bd'), ('931493766816559104_3', 'data_image/iraq_iran_earthquake/17_11_2017/931493766816559104_3.jpg', 'train', '6fe18baab216c95064ae1c4d927da716704e5284226e0a9d58ae7aa1cf916777'), ('930271699727200262_0', 'data_image/iraq_iran_earthquake/14_11_2017/930271699727200262_0.jpg', 'dev', '3285905aa4eff94eaa83737e2f645c138ef9794bb737cd5eef013125f323ecc1'), ('910749963423948805_0', 'data_image/mexico_earthquake/21_9_2017/910749963423948805_0.jpg', 'train', 'ee202d9108f9f65ae088c15b7c144ac3f1ca5c1cf1e2c60e5c8c468ba1bd21e2'), ('912346493809577984_0', 'data_image/mexico_earthquake/25_9_2017/912346493809577984_0.jpg', 'train', '450041c4d6c3a34f1aaf148da7cfcb92844632307c4a4498109278d120e6d8b6'), ('910524778896691200_0', 'data_image/mexico_earthquake/20_9_2017/910524778896691200_0.jpg', 'test', '466f6fbce34b7109f1e27714484e7d7f88d6a5dbc7324a79a927a2d4ad6108f3'), ('913699317747273728_0', 'data_image/mexico_earthquake/29_9_2017/913699317747273728_0.jpg', 'train', '7f3f5243372d3e874af2f2ec067bba3ccb34da4d8f1519a272378bca2b91b220'), ('910727196393771009_0', 'data_image/mexico_earthquake/21_9_2017/910727196393771009_0.jpg', 'test', 'bb8fde4093d4f3877fbc660625440a06bea7578207f5eb46ce018e5266135e61'), ('910576213818671104_0', 'data_image/mexico_earthquake/20_9_2017/910576213818671104_0.jpg', 'dev', '459d5b8d9b0ffe34f5b3bb01d775dc68a5527a13290c3cfd3b3eb114c5238cf4'), ('911916099238793216_0', 'data_image/mexico_earthquake/24_9_2017/911916099238793216_0.jpg', 'train', '1763e1afa9ef731c8650b2191ca298ab9935396ffc9bbb75c6142316eae04144'), ('910696303679512576_0', 'data_image/mexico_earthquake/21_9_2017/910696303679512576_0.jpg', 'train', 'a5dfd69ad93b4be32d956fc4388c5cc2a1aa5deb0fbeb27d4828ce66ac493a6c'), ('913965634437992448_2', 'data_image/mexico_earthquake/30_9_2017/913965634437992448_2.jpg', 'train', '760b5994f7baeb56b7b95674fe7c10a3eddbcb4aee3041f0149d70b743beebe8'), ('910550433550057473_0', 'data_image/mexico_earthquake/20_9_2017/910550433550057473_0.jpg', 'train', 'f029fe25e2612d0a9cb1e9df4c3922d257cd132d26dd83308bb4431f3a3ead30'), ('910798374940667904_0', 'data_image/mexico_earthquake/21_9_2017/910798374940667904_0.jpg', 'train', '58e478e47f6ea28cd2a372fc5d6a34512d45ecb538abb585a9b24b2252911eb0'), ('910690620389310465_0', 'data_image/mexico_earthquake/21_9_2017/910690620389310465_0.jpg', 'train', '76b975be8d298b828ea60b1020d3c633d671aaebee87bcc942b7491f9daa85c2'), ('913323978554519552_0', 'data_image/mexico_earthquake/28_9_2017/913323978554519552_0.jpg', 'dev', '7b15a7746054f67bd93d1c421a8d0172aa39270b04924f55621e04473a622e84'), ('911717117602316288_0', 'data_image/mexico_earthquake/23_9_2017/911717117602316288_0.jpg', 'train', 'd584e394179279243326c2d6c85a3038599fdf34696e9c6767869842568d158c'), ('911665527415750656_0', 'data_image/mexico_earthquake/23_9_2017/911665527415750656_0.jpg', 'train', 'e174bbae3908d1676cd7b62ca949fd15454fc1162102a43a2f78b27419abb037'), ('869991393124466688_0', 'data_image/srilanka_floods/31_5_2017/869991393124466688_0.jpg', 'dev', '0fd0040de9c4666a01fe6afd14f35e4d72138dd52a053cc850f615c46821a4e4'), ('870140525889822724_0', 'data_image/srilanka_floods/1_6_2017/870140525889822724_0.jpg', 'train', '7ee58803d160265fa466b077ae98a43386f3322bcf9432cbc7680be49c2ef245'), ('870528490184908800_0', 'data_image/srilanka_floods/2_6_2017/870528490184908800_0.jpg', 'train', '9337f937ebefd2f8cfe0e1a7d25389004b20799ac51e85bed43d3ce17f74dc88'), ('870176000109182977_0', 'data_image/srilanka_floods/1_6_2017/870176000109182977_0.jpg', 'train', 'a0eb9a075045f1621f91a47ed4a590fc04b11b585fed39678ebfd85dacc648b9'), ('873210513588719621_0', 'data_image/srilanka_floods/9_6_2017/873210513588719621_0.jpg', 'test', '61a8d79d31cde0d638d0e400b9af872abed1ec327a4a72f3748287a173fa6182'), ('870143714852655105_0', 'data_image/srilanka_floods/1_6_2017/870143714852655105_0.jpg', 'test', '90e5c62d8c7e29ad55fc04c651d5010bfc8649f18ff659117badb31d5487bd4c'), ('878803403157061632_0', 'data_image/srilanka_floods/25_6_2017/878803403157061632_0.jpg', 'test', '008c721374c3de32ad316fef80c6525d0538d4e1e6075fa9f46e08805345ffc0'), ('870272043538886657_1', 'data_image/srilanka_floods/1_6_2017/870272043538886657_1.jpg', 'train', '5af3e23815af0b6144fbeb69700c56decf9c3d9423fb462f0ee97b3c3bd371b1'), ('871769027223461888_0', 'data_image/srilanka_floods/5_6_2017/871769027223461888_0.jpg', 'train', '304a1a3b03c3fc38c7d11f8fe71453a17f5ea5eb3c9e91389a4e5caff639cc29'), ('918315399397216256_0', 'data_image/california_wildfires/12_10_2017/918315399397216256_0.jpg', 'dev', 'f073a3e78a240b4495d1731d5a4a08bc805bd6a097191b9638d14a65d1140bb0'), ('922558661469331456_0', 'data_image/california_wildfires/23_10_2017/922558661469331456_0.jpg', 'train', 'fe5300a31c68929504ead4fd83d2b53e8374a8a56796eeb5b65cc862d0fc182d'), ('919677043192741888_0', 'data_image/california_wildfires/15_10_2017/919677043192741888_0.jpg', 'train', '95f70f005a9d923ee8e9a07bf22a5e3bffc79a522b4aa822f575f9ff2949e73d'), ('918357597735895040_0', 'data_image/california_wildfires/12_10_2017/918357597735895040_0.jpg', 'dev', '776e23d62e8f1b4677f2ea4e027ac239afa61e734d460dd9e3474f2dd381dc01'), ('918106736019591169_0', 'data_image/california_wildfires/11_10_2017/918106736019591169_0.jpg', 'train', '32cf1000472e1e302b040be128b707399a723bb5dd910de84be45b0ccd45c1cd'), ('919679294544195584_0', 'data_image/california_wildfires/15_10_2017/919679294544195584_0.jpg', 'train', 'c1faaf90fa52a524367bb05f07668fa8398d67e63626076ef740753852c8c1fb'), ('922976152234360832_0', 'data_image/california_wildfires/25_10_2017/922976152234360832_0.jpg', 'dev', '26628cb4b81bdfead92737b8de60a017ddde2ab4ca0fbd003f96494fd8e8f166'), ('922946867893190656_0', 'data_image/california_wildfires/24_10_2017/922946867893190656_0.jpg', 'dev', '6ebb5066aa6e0c30d2fffbee78ea9c96e3b9418afca0b10d0796f3d3a309113f'), ('919592996521132032_0', 'data_image/california_wildfires/15_10_2017/919592996521132032_0.jpg', 'train', 'ce2248236bf7ae01a08946b83604e78a14fa8413bfa9b5cd92e3d4a3bf5bfe9c'), ('919607858106265601_0', 'data_image/california_wildfires/15_10_2017/919607858106265601_0.jpg', 'test', '1ba74cdad8348b675cf709ea68e17378ef3bbf54d9e872b48c42c098fd36ec0f'), ('917929562029416450_0', 'data_image/california_wildfires/11_10_2017/917929562029416450_0.jpg', 'test', 'b70039a69afac0af7fd1292f3cd6f5c16fbce4c38e9ab1c1ae709c97b9704e81'), ('920303079714471936_0', 'data_image/california_wildfires/17_10_2017/920303079714471936_0.jpg', 'train', 'e4e7fe61c7cc7270cd964c169766d76c20c00963acf1acf7b5d82d3b48f15a36'), ('920281575748788224_2', 'data_image/california_wildfires/17_10_2017/920281575748788224_2.jpg', 'dev', 'cc93496aab9c5844699e7bbf1656855d300c02fd8f58e075747879b9fa6f2225'), ('920337866521284608_0', 'data_image/california_wildfires/17_10_2017/920337866521284608_0.jpg', 'train', '300fc29fc03ba14515b96ddee3310a80ad8e9700bdea6deb825f16dcac87d864'), ('922734446276640768_0', 'data_image/california_wildfires/24_10_2017/922734446276640768_0.jpg', 'train', 'cf651f820c54435db0a48c2c3a9655e39c925fd133cdb01c0dbd1cc24935f41c'), ('918112951592280064_0', 'data_image/california_wildfires/11_10_2017/918112951592280064_0.jpg', 'train', 'f89c711a49c1dbf4a4494f5189622ef5f4960a9da9bb55b231ab63d2488cf812'), ('905230326359891968_1', 'data_image/hurricane_harvey/6_9_2017/905230326359891968_1.jpg', 'train', 'fa1a46fd8b0f771b6e4884e8e43849cf51dec5d05569be02515436d0d81e6ee3'), ('904269818542653440_0', 'data_image/hurricane_harvey/3_9_2017/904269818542653440_0.jpg', 'test', '99ecd020d4fae94f25d4ff2de4d58cbc8724c27f897aa28f41877d33037244e3'), ('907281334615396353_0', 'data_image/hurricane_harvey/11_9_2017/907281334615396353_0.jpg', 'dev', '7843f3f2a7c89e0bb68083c5dab8c4cd586068dea75a42727385dcfd4254d8c7'), ('905826747395870720_0', 'data_image/hurricane_harvey/7_9_2017/905826747395870720_0.jpg', 'test', '2d0ddf2174667a5ae61d5598dbad52a7c5006f089ab43cbccab953b6b7c84750'), ('906563342189375488_0', 'data_image/hurricane_harvey/9_9_2017/906563342189375488_0.jpg', 'test', '7cd520f47c439443edce13b143bed65ad5c28a5730b2ff734744e2cbee63563a'), ('906162712937451520_0', 'data_image/hurricane_harvey/8_9_2017/906162712937451520_0.jpg', 'test', 'cf7d786e92e5aa8b3391af28c23ca7dd5edc03cdd80f0a457fa695dca2aacb45'), ('904433649545658368_0', 'data_image/hurricane_harvey/3_9_2017/904433649545658368_0.jpg', 'train', '9533d187409d8197bc5556bd996b2800cea888aeffcbaafb0a432ac7aa4e1c64'), ('904906277511364608_0', 'data_image/hurricane_harvey/5_9_2017/904906277511364608_0.jpg', 'train', 'ed18d7f2106f37cedecf860d63c983451c5298e753dd0dd070f69d11e869de76'), ('905268948828119044_0', 'data_image/hurricane_harvey/6_9_2017/905268948828119044_0.jpg', 'train', '86a49fb68c7eade6478e84dda65ce4080e2662511058a7d005cbab5da60d47be'), ('904827282551963648_0', 'data_image/hurricane_harvey/4_9_2017/904827282551963648_0.jpg', 'train', '6885d40cf03303e08a3ace6b5623cb0cdaf43d71ab485ce58535324a61d045e1'), ('905037826286387200_0', 'data_image/hurricane_harvey/5_9_2017/905037826286387200_0.jpg', 'dev', 'dd18cd27c953a6d9beee281688565882b9b0767f690379c77944887c5d271f02'), ('901787831433125889_0', 'data_image/hurricane_harvey/27_8_2017/901787831433125889_0.jpg', 'test', '2d9cd88e49f671e3d8dc24ec1965357056704b9e758c1e132d44a4f5ea0fa9d0'), ('908424137953091584_0', 'data_image/hurricane_harvey/14_9_2017/908424137953091584_0.jpg', 'dev', '7e217cb62f3d88981b4cc528fec58d7a7b82c3f3df114b0be4f37ff02c76571c'), ('904261999558881280_0', 'data_image/hurricane_harvey/3_9_2017/904261999558881280_0.jpg', 'train', 'e70eb9528511636ebd0866112bcd9427bc38f9c404f9055e6517b19f0a81f6a2'), ('906200820961693697_0', 'data_image/hurricane_harvey/8_9_2017/906200820961693697_0.jpg', 'train', '1a27972ec1eccb1c7f995fea8d3bee335b02a68ee013c2ed238c16f4c5351bce'), ('904256691939762176_0', 'data_image/hurricane_harvey/3_9_2017/904256691939762176_0.jpg', 'dev', '26a9d77cbc3647beaf515b4f669c7f07f72f604221232761321f65a5f9c5ac90'), ('905213262039539717_0', 'data_image/hurricane_harvey/5_9_2017/905213262039539717_0.jpg', 'train', 'd7d2bd90f3b4d79a0661c31e4e821466c2ff279aa2b0c92b7584d448bdd912aa'), ('901775187796004865_0', 'data_image/hurricane_harvey/27_8_2017/901775187796004865_0.jpg', 'dev', 'c931758c89943c276cf95ea2f470b41f416461097314deb1e88c13b3fbc9f2c2'), ('906847194371821568_0', 'data_image/hurricane_harvey/10_9_2017/906847194371821568_0.jpg', 'dev', '3b071afb3aea642041081e9f4b1161167ea6f16e880ca75bb8294e7b67f5ac58'), ('906242339861037056_0', 'data_image/hurricane_harvey/8_9_2017/906242339861037056_0.jpg', 'train', '138c8a6e5eb97b511d781f8f5eacbac502c82f210e555132b9fcca36f2c96802'), ('906478412281401344_0', 'data_image/hurricane_harvey/9_9_2017/906478412281401344_0.jpg', 'test', '134cb27e7d57c419673ca1b70ca8b79943261220435eaf65649b7fe95341389e'), ('905768224809263105_0', 'data_image/hurricane_harvey/7_9_2017/905768224809263105_0.jpg', 'train', '74d83065065dc43d765bfbe71ab1a6bad430b9703243f58234064ba239956c5d'), ('908200014668804097_0', 'data_image/hurricane_harvey/14_9_2017/908200014668804097_0.jpg', 'train', 'c181e01fdda8b56883b0b32545267f1545098715c955f18f106bfb1313237da3'), ('908092076226760706_0', 'data_image/hurricane_harvey/13_9_2017/908092076226760706_0.jpg', 'train', 'a91fbff7d71ac770d9cade2768943bba897b5277a14fd7f5d0a77f6f18a86f34'), ('909783378110623744_0', 'data_image/hurricane_irma/18_9_2017/909783378110623744_0.jpg', 'dev', 'd698d1533a7fecda5fdd028a1e11c429c66fa0c803a0f6b2ccdea2a4da0d2863'), ('909862878747951104_0', 'data_image/hurricane_irma/18_9_2017/909862878747951104_0.jpg', 'test', 'f7db858c38c5069b1a2bcbe88a154d36e9896323ca1e9046dd9fb45f4dd4cfe8'), ('909448877790695424_0', 'data_image/hurricane_irma/17_9_2017/909448877790695424_0.jpg', 'train', '1eb9d27a14c2c6adf65d519f7d4e702ad59928bb64527d0c2b4148ef9a176569'), ('909421586893070338_0', 'data_image/hurricane_irma/17_9_2017/909421586893070338_0.jpg', 'dev', '191a145d7108b3077ccca3d3c423ac37b57f7f8276b6048cae904117b616a10c'), ('909395278519775232_0', 'data_image/hurricane_irma/17_9_2017/909395278519775232_0.jpg', 'train', 'f5afee487b5d0102b664fe43286244fb06e40c5c53483574d8c552f9213806eb'), ('909912459640700930_3', 'data_image/hurricane_irma/18_9_2017/909912459640700930_3.jpg', 'test', '267c6cb16e12e9965a2cca2d798c79849cbae6ef913b0fbc28ea9df28cbf3f3c'), ('909720610078695426_0', 'data_image/hurricane_irma/18_9_2017/909720610078695426_0.jpg', 'test', '146f43c249eec8548ad2edcd8bc02eea988bcabcd096f3b99e8690c07a612221'), ('905692122434519040_0', 'data_image/hurricane_irma/7_9_2017/905692122434519040_0.jpg', 'dev', 'ba1e716ee944e1c275f8ff28bd0512fadece8af9be4678b0651f1e2b413ea26d'), ('909387455924666368_0', 'data_image/hurricane_irma/17_9_2017/909387455924666368_0.jpg', 'test', '2979369fa292a2cc420f37268f6cf334cf0d90343764f5c29fcde169d66982f3'), ('909403344095076352_0', 'data_image/hurricane_irma/17_9_2017/909403344095076352_0.jpg', 'test', 'b12b01a4dff841c0dfd96a6a11a7a875a9188d5ddf95415aa101e3a367aa3f0b'), ('909874091779379201_0', 'data_image/hurricane_irma/18_9_2017/909874091779379201_0.jpg', 'train', '25aeffbcb9a1a70266aca014eb68bdab3bbb287ef36b46a0ed240b881c825700'), ('905723534730641408_0', 'data_image/hurricane_irma/7_9_2017/905723534730641408_0.jpg', 'train', 'df1d7c5f584dc842e95504edf01367908ca7bf03056a1d706c166ca829282567'), ('909474115114774528_0', 'data_image/hurricane_irma/17_9_2017/909474115114774528_0.jpg', 'test', 'a864082330954aa752a9e3311157ff7c7167fad899d94d0276e42d0b5b7c175e'), ('905761884808335361_0', 'data_image/hurricane_irma/7_9_2017/905761884808335361_0.jpg', 'train', '0225613e2d3f8e1e5c1ed8e5135dc5dd13198bba0fa5b7a8472dd1f81e2464fb'), ('909394672698740737_0', 'data_image/hurricane_irma/17_9_2017/909394672698740737_0.jpg', 'train', '47ec4fa2ea5a48bf3724e446755f010f0b4b43a2db37e3eb0c600a2b9bfeab23'), ('910130721406189568_0', 'data_image/hurricane_irma/19_9_2017/910130721406189568_0.jpg', 'train', '0537abcd3e7879d5bdbec47d03ae0ed4a67fa41325f890e5503663901e22851c'), ('910126603946090497_0', 'data_image/hurricane_irma/19_9_2017/910126603946090497_0.jpg', 'test', '747aab8f62826ad1dda7901fe76b61738996622fab10b4827fab49932212817d'), ('909932318965604352_0', 'data_image/hurricane_irma/19_9_2017/909932318965604352_0.jpg', 'train', '5ccaa7c1cdebd00541ed57ca905bf4818b128ab3194db300b1ffcf5b886a5b43'), ('909774149698179072_3', 'data_image/hurricane_irma/18_9_2017/909774149698179072_3.jpg', 'train', '889cb6039c85c0dca288855f87747d7c72bf3e87c775284b0d76a486224f79e2'), ('909800633942888448_0', 'data_image/hurricane_irma/18_9_2017/909800633942888448_0.jpg', 'train', '68cd54bb9e8029d9c87a7b1ae5afe984dd59fded341b27f75945d05ce33a99b0'), ('909824106954149888_0', 'data_image/hurricane_irma/18_9_2017/909824106954149888_0.jpg', 'train', '4c02ef563c8c22cdf470cc949551ce2fe8efac86ac26c56e02bb9d2bb606daed'), ('905744911907074048_0', 'data_image/hurricane_irma/7_9_2017/905744911907074048_0.jpg', 'test', 'c13270707ff48d63811a1b3f8e93545babe99179d208ec13ea2aa8934d3cc278'), ('929049019124535302_0', 'data_image/hurricane_maria/10_11_2017/929049019124535302_0.jpg', 'dev', 'b1fd679154ddf642ea825721670e1349c241d99045ad2e02149374a3fa20263e'), ('913451020373696513_0', 'data_image/hurricane_maria/28_9_2017/913451020373696513_0.jpg', 'test', 'c4e6ae24b805eebb6a42d6071a99bf49e60a4076cab0d61a07e4f13ae0aafe43'), ('913261865266028544_0', 'data_image/hurricane_maria/28_9_2017/913261865266028544_0.jpg', 'dev', 'c87fc8e4718557d21ac3ea452fb22f5c1b50c0eac21fb7b122553fdcaaf64af4'), ('913884389305659392_0', 'data_image/hurricane_maria/29_9_2017/913884389305659392_0.jpg', 'train', '5af35101cfd6b60a0a95320b2e80e634974d6a332f4f887421bb30f66a8c9acb'), ('911558544780152832_1', 'data_image/hurricane_maria/23_9_2017/911558544780152832_1.jpg', 'dev', '55fc13561a386a9ed2432c95d598d3a70e26888d38a9869d5d0c10d2cfe7229a'), ('910614549744758784_0', 'data_image/hurricane_maria/20_9_2017/910614549744758784_0.jpg', 'train', '83ac8b144eafd8200634b8f6a373bb737d77b525244bcb57f0e80d79f08cdd35'), ('923876037540372480_0', 'data_image/hurricane_maria/27_10_2017/923876037540372480_0.jpg', 'train', '3155b603f3fc4159fc7f22c70fe03cd5572ff8d11ad2ae11145be09379a79ea3'), ('910571811020062720_0', 'data_image/hurricane_maria/20_9_2017/910571811020062720_0.jpg', 'test', '370c645f79a40012960e296e8afd405cbb30e92165072318f0d37988f73736ec'), ('928473090971054082_0', 'data_image/hurricane_maria/9_11_2017/928473090971054082_0.jpg', 'dev', '89812e45b24c06cbee1d72f889fc1bed8867431e59a2afc95c76242359e5cd03'), ('911818109123747846_0', 'data_image/hurricane_maria/24_9_2017/911818109123747846_0.jpg', 'dev', '252b09255277c51874caaaede436bd630a70bf4ab9f7af8dc6bdb93cc454d2a3'), ('910532852911153152_0', 'data_image/hurricane_maria/20_9_2017/910532852911153152_0.jpg', 'train', '759f843b386033a0d1d3bcb76dac3e6abe4c4a123977d94b26fd4a84c1acbda1'), ('921428147861708800_0', 'data_image/hurricane_maria/20_10_2017/921428147861708800_0.jpg', 'train', '7ec80dc99132af10af606ed2e36f8ea7ffe6ac138cb885be08c558846b576e0b'), ('921433090177863680_0', 'data_image/hurricane_maria/20_10_2017/921433090177863680_0.jpg', 'train', '22f4784b0439c686434b7a1e30cb89dfdc8dc2614f7387f3ca34e750d772b596'), ('914516960804839424_0', 'data_image/hurricane_maria/1_10_2017/914516960804839424_0.jpg', 'dev', '2cd4d6be4a8e80731b953f39f986e55277a65604e57f5c95d58cf98ffcc7a135'), ('912128706293960711_0', 'data_image/hurricane_maria/25_9_2017/912128706293960711_0.jpg', 'test', '55eff0715683b9b89944d408f44d8c634d68d65daed4ae6161a7c8675eecc57f'), ('912460997603999744_0', 'data_image/hurricane_maria/25_9_2017/912460997603999744_0.jpg', 'test', '86cd3ad62018fc44eac60632032c564bf95305a0a54ae44fba26ed3bb67b570b'), ('910555135759278082_0', 'data_image/hurricane_maria/20_9_2017/910555135759278082_0.jpg', 'test', '7b8c717c51b1d4f52d2d6a4851459f9fb57193eb8656ba8673091c5bb6f2b799'), ('913059096944893952_0', 'data_image/hurricane_maria/27_9_2017/913059096944893952_0.jpg', 'test', 'a05e338701c589922613db489b7eaccb0b2e934c4140cdc037867f534b2081b6'), ('910534801999806464_0', 'data_image/hurricane_maria/20_9_2017/910534801999806464_0.jpg', 'train', 'ffd5467d0f52d5f8679695a842bef7b43bb1b77b611902440a7dfc5e892e28fd'), ('916282920482099200_0', 'data_image/hurricane_maria/6_10_2017/916282920482099200_0.jpg', 'dev', 'c928bd4bda7aae2949597f38a79e9070a7d96655508b7d49f46a8084b89bc80e'), ('923265376963715072_1', 'data_image/hurricane_maria/25_10_2017/923265376963715072_1.jpg', 'train', 'd4c74ff423d1bf5c8d4b6386c3387a97a9704395dd88ee7a6a799cbb4bffa34b'), ('913131967167778816_0', 'data_image/hurricane_maria/27_9_2017/913131967167778816_0.jpg', 'dev', '3cd6d4a2f8c7fd2bb0c3921eb50049d6f939e3283f1feb806aa8054189352a31'), ('913804552289255424_0', 'data_image/hurricane_maria/29_9_2017/913804552289255424_0.jpg', 'train', '334a5cbe806be37259006afd7e65ea396b3429c5d4fe1b173e5ef75ea1dc0ae1'), ('913191254623649794_0', 'data_image/hurricane_maria/27_9_2017/913191254623649794_0.jpg', 'train', 'ed5d720ca79a6d6823c1481f594ff592abdc7aa345c1c3ac97fa184c52c36cbd'), ('912409535918288896_0', 'data_image/hurricane_maria/25_9_2017/912409535918288896_0.jpg', 'test', 'a8df8ff0b4050f71ed40cd67eb7a5780f7a8e75d68452082fb8080d159bd47df'), ('930114382268334080_1', 'data_image/iraq_iran_earthquake/13_11_2017/930114382268334080_1.jpg', 'train', '28f9af7f76e269f62154d2b59d021748ce53d9916139f712f8b77476733a4e50'), ('930015958340718593_0', 'data_image/iraq_iran_earthquake/13_11_2017/930015958340718593_0.jpg', 'dev', 'aae237f709e9d5e4ea0879b809247e9ed4d4483cf5d27fabf57879a716990aa1'), ('930064582290403329_0', 'data_image/iraq_iran_earthquake/13_11_2017/930064582290403329_0.jpg', 'test', '0e3c1a606dd388fdd2adf0bab9b6f5e971be0c30d2f61e86cff768d59bc931d2'), ('930723148818993153_0', 'data_image/iraq_iran_earthquake/15_11_2017/930723148818993153_0.jpg', 'dev', 'cc612fb3ae56a5168e4e0730066b524f41bdb4249e34fc14837126f5e20228c9'), ('930944762529615872_0', 'data_image/iraq_iran_earthquake/15_11_2017/930944762529615872_0.jpg', 'test', 'b8f2a05ccd7e00a15d2d386f25b14ed23646c130e824663ed520cfa5f3be5c3f'), ('930116035671416833_0', 'data_image/iraq_iran_earthquake/13_11_2017/930116035671416833_0.jpg', 'train', 'b3c4a29e14bb4b222385a52134855d61c20f5b2ddcd02f3062c0b4d6390fb14a'), ('930810047835398144_0', 'data_image/iraq_iran_earthquake/15_11_2017/930810047835398144_0.jpg', 'test', '5af9289628035d5c4cdd55113403d241b93ccde00765097e5c0980bfc9386af2'), ('929989443913691136_1', 'data_image/iraq_iran_earthquake/13_11_2017/929989443913691136_1.jpg', 'train', 'a27afc9196c9b880647778f9533be15fb21c7ce42e2c379b83baa4cf8dfce79d'), ('931240676951433216_0', 'data_image/iraq_iran_earthquake/16_11_2017/931240676951433216_0.jpg', 'train', 'a1943df178fd3ade7aa2ac587a9c172282c7c295be943f88f0fda2c899b991e6'), ('930178474102358016_0', 'data_image/iraq_iran_earthquake/13_11_2017/930178474102358016_0.jpg', 'dev', 'f2f7839fe90571633acf1b8f04fb1cb782fff96b85af3245e46e63efbfea18a8'), ('910736159285882880_0', 'data_image/mexico_earthquake/21_9_2017/910736159285882880_0.jpg', 'dev', '76fadb988f61a04a91d0c34bd53c79ef5aa03d83820bcc76d9765760ddfb12f9'), ('910586697754308609_0', 'data_image/mexico_earthquake/20_9_2017/910586697754308609_0.jpg', 'train', 'b0a9b8482e8cc0aeb353f7e9c9bea7195fa0283ed5ebc09528bd10971394523d'), ('913479005122236421_0', 'data_image/mexico_earthquake/28_9_2017/913479005122236421_0.jpg', 'test', 'a569281af93b68daaca9076f6708a68f333a8464b6e2849fc7defbb37ea4022b'), ('910632195068354560_0', 'data_image/mexico_earthquake/20_9_2017/910632195068354560_0.jpg', 'train', '6be777ed2d68d6eed0a9ed59c2384aeddb7b0a1fbfce8fe81cf8d49eb67b6c1c'), ('911720094266310656_0', 'data_image/mexico_earthquake/23_9_2017/911720094266310656_0.jpg', 'train', '0f0b67f0ea03262ede9f437f90e176a4ec18358d7bb3b979f73cf69b128ebf96'), ('910670466356072448_1', 'data_image/mexico_earthquake/21_9_2017/910670466356072448_1.jpg', 'train', 'd112a2834ad8c6fa3c6c7c20c9872a4ea66703efcc9158c3b00e94f3776155a3'), ('913012851736023040_0', 'data_image/mexico_earthquake/27_9_2017/913012851736023040_0.jpg', 'test', '4f04dc41fb294a8498dfc63bca8caae70c7c892f1b8efde5aa85f17122832268'), ('913965634437992448_0', 'data_image/mexico_earthquake/30_9_2017/913965634437992448_0.jpg', 'train', 'c993e611697cbd4eb5e108357e9dd785a51315cd5b4cf6a0a8c7a8f3b26b05e2'), ('910688930663714816_0', 'data_image/mexico_earthquake/21_9_2017/910688930663714816_0.jpg', 'train', '1e69e11cc0621a537a75ccffab0327f0154c8a82ee448229e7024b67be7b4179'), ('911928865135751168_0', 'data_image/mexico_earthquake/24_9_2017/911928865135751168_0.jpg', 'train', '8f72787f028628215d529f362932d8cf4fc281813e7e82bff7f7f7b4c7bc22bb'), ('910760553303420928_0', 'data_image/mexico_earthquake/21_9_2017/910760553303420928_0.jpg', 'dev', 'bf87d05775b46f96b08306071842dce4b814aea29048c2d0c3172ddc5d5c3d44'), ('912356666343022592_0', 'data_image/mexico_earthquake/25_9_2017/912356666343022592_0.jpg', 'test', '06e812d64ced1dc7efe9312e0a4cceab6d51358a77ff0f810965bc220481bcbd'), ('913389114229063680_0', 'data_image/mexico_earthquake/28_9_2017/913389114229063680_0.jpg', 'test', '94597ca51675989f5d5a308187ec2c14cc7bd33f7a7978bb016adbc3a13da470'), ('871346616740962305_1', 'data_image/srilanka_floods/4_6_2017/871346616740962305_1.jpg', 'train', 'd1656c8e1fe73809668eb4cb7e68173ae605d7255cd6286af5cc146e24151213'), ('870942570704019456_1', 'data_image/srilanka_floods/3_6_2017/870942570704019456_1.jpg', 'test', 'c78bda2450166e2498ea838a82d272effee6773ba1f5e133e4d5480faf6cc1d4'), ('869967714948849664_0', 'data_image/srilanka_floods/31_5_2017/869967714948849664_0.jpg', 'dev', '669ec999b27b223290bf3024b9db7a941b76a119596f88a02a6433e09dcc8636'), ('870318701760974848_0', 'data_image/srilanka_floods/1_6_2017/870318701760974848_0.jpg', 'test', 'e9783036e94d169aa5263164dd524daa47e5dd82476abc38cc8c90c7c3ccf447'), ('870636021314191362_0', 'data_image/srilanka_floods/2_6_2017/870636021314191362_0.jpg', 'train', '14f2dad2820a33ed974f9d7a0a48ad029ca47179715a10e6000abc04d8fa4adc'), ('871242972385296384_0', 'data_image/srilanka_floods/4_6_2017/871242972385296384_0.jpg', 'test', 'c3a172d0b6cd2336b6201c98efa7d55d8aaae6bd481dc0f71f08965aa40bcb10'), ('870636058656026624_0', 'data_image/srilanka_floods/2_6_2017/870636058656026624_0.jpg', 'dev', '9ed81ed646cf17146f330d561e1f114d314328fd84be2d2c6b12e91fce690d92'), ('870882793118683136_1', 'data_image/srilanka_floods/3_6_2017/870882793118683136_1.jpg', 'train', '70bb76493e8890148d6f2e58ced0116a8dc7cd1645bad85170aa846701aadb7c'), ('870279064996777985_0', 'data_image/srilanka_floods/1_6_2017/870279064996777985_0.jpg', 'dev', 'bd420b43a5e9fba71f802fa4ebcbd82c7a6737c4671f53fbb83a65363f2fe366'), ('921794491949944833_0', 'data_image/california_wildfires/21_10_2017/921794491949944833_0.jpg', 'train', '0a7238f72181dff8e3dc185247b94ff260c36bc46b0ff33a00fe75a3d11d085c'), ('919970733375217664_0', 'data_image/california_wildfires/16_10_2017/919970733375217664_0.jpg', 'dev', '97bfef948e268247c2627512c0c3af4d83547a1d30e71f21ac6b15c960d808fc'), ('920702272996245507_0', 'data_image/california_wildfires/18_10_2017/920702272996245507_0.jpg', 'test', '445123e9c4425485116b6b93c5b5015725b8bbf433b498883d02548f703ac922'), ('919562294924840961_0', 'data_image/california_wildfires/15_10_2017/919562294924840961_0.jpg', 'test', 'd11d89a65e117e9799c034d69ea4b2bed577932f9dde48d2eb9d4233c5ab4054'), ('917794892113498113_0', 'data_image/california_wildfires/10_10_2017/917794892113498113_0.jpg', 'test', 'b40ffb47b544192a8e3d5f451fb89599d46298b3c696ab14814e27fa5f9a14c5'), ('917803284936589313_0', 'data_image/california_wildfires/10_10_2017/917803284936589313_0.jpg', 'train', 'bc8c9b1616b37206e6cf0b9cbe3414429c9237bcb85610ea1bd6271b2c46ae83'), ('917833850746822656_0', 'data_image/california_wildfires/10_10_2017/917833850746822656_0.jpg', 'test', '1ccd97c49cd798490ee1b24dde756fd8264a43bf2db1875a417049ef4aa14323'), ('918021529614434306_0', 'data_image/california_wildfires/11_10_2017/918021529614434306_0.jpg', 'dev', '8e32a463429cca587f9f97e37db567c4cbc84db6c7016fa7ffdbeeeb7b6fe3c7'), ('918111395555749893_0', 'data_image/california_wildfires/11_10_2017/918111395555749893_0.jpg', 'test', '0495c01749f26f2e36cc06873d4cef28a637dc74c8e69b439a5a5ff7e89660fb'), ('919609553603280896_0', 'data_image/california_wildfires/15_10_2017/919609553603280896_0.jpg', 'train', '4bc5b7bb4846e48c5af3ce30655a89c6b4eb56e5aa0fa134bdaf49c42739d492'), ('922583410635186176_0', 'data_image/california_wildfires/23_10_2017/922583410635186176_0.jpg', 'dev', '72324c5744b569cf09c4fa5ef1c26c0649818d6f6041a2057fb69371316f0b33'), ('919974245106626561_0', 'data_image/california_wildfires/16_10_2017/919974245106626561_0.jpg', 'test', 'c7e780c3846b801058f475c47b8aecd0e445b61fb99e429d5131a2f85b5efadb'), ('919827031562510336_0', 'data_image/california_wildfires/16_10_2017/919827031562510336_0.jpg', 'train', '327a340d7f06977db75925ba4a6a0e757e8d94e17687bf7cb53c6948f46f91d4'), ('922924297533865984_0', 'data_image/california_wildfires/24_10_2017/922924297533865984_0.jpg', 'test', '276b3febc36ff96da4245271f0c820f75a0ca16274149866806daf8da8cdcbed'), ('920613959643705344_0', 'data_image/california_wildfires/18_10_2017/920613959643705344_0.jpg', 'dev', '67e903810db47bfa5c45c67c99fdf323ac35694df44fbd7e454fa1839a07a45c'), ('917818543831261184_0', 'data_image/california_wildfires/10_10_2017/917818543831261184_0.jpg', 'test', '39d777cdda09468f5963017489b92b8cd7c789a6bf5b9d169299bbdf5f158151'), ('904389009538998272_0', 'data_image/hurricane_harvey/3_9_2017/904389009538998272_0.jpg', 'train', '99f9e332c9a2243656fadfbc17f91cecf3b323cd256efe0642062200220dd016'), ('905197126426157056_0', 'data_image/hurricane_harvey/5_9_2017/905197126426157056_0.jpg', 'test', '76f55eb48e87ad86f2d2156f10b9d401dea23c746fcf07b6ff79e898a785232f'), ('905064429544501248_0', 'data_image/hurricane_harvey/5_9_2017/905064429544501248_0.jpg', 'test', 'e52939a7712e9e59c6d264b6d404ed376176c3fec3d9e1b464385ebdef8c3cfd'), ('905819071849140224_0', 'data_image/hurricane_harvey/7_9_2017/905819071849140224_0.jpg', 'test', '1bcf50680381a82427932ab05634ed9aaec0e4d961328b417cfe5a6fd138b5c8'), ('901750080943902720_0', 'data_image/hurricane_harvey/27_8_2017/901750080943902720_0.jpg', 'test', '114bcf70fbb7d784c7da2fe24d8ccc712f807f570132f9353a435f1aaf0d41ad'), ('907661128213217280_0', 'data_image/hurricane_harvey/12_9_2017/907661128213217280_0.jpg', 'test', 'a6956e9bc22c23d0dffb0a8c28987a1487eb4e9d177ed832e11659aa969b900d'), ('907654982144544769_0', 'data_image/hurricane_harvey/12_9_2017/907654982144544769_0.jpg', 'dev', '30b85026c37bf7e19ba67263061953a1be7f2b6a4ea7449c22b6bf6504ea8cc5'), ('907200075687964672_0', 'data_image/hurricane_harvey/11_9_2017/907200075687964672_0.jpg', 'train', '8a71b98a764760af7d2610f7fa6504a884a9f4ed820e67a5f8a8f6d5ef5c886d'), ('905164133934485504_0', 'data_image/hurricane_harvey/5_9_2017/905164133934485504_0.jpg', 'dev', 'f7aa6701300732e345da59ff580a13f5da3757a3a6807c7478e8f6737eaf3725'), ('906732992797696000_0', 'data_image/hurricane_harvey/10_9_2017/906732992797696000_0.jpg', 'test', '6bfb5b83d8a3a47622d8746b3f0d1208e8f03dcef94052f5ebbcd506f74255ac'), ('908118102587539456_0', 'data_image/hurricane_harvey/14_9_2017/908118102587539456_0.jpg', 'train', '92233d9d9370c4d6ce196c13d860296a0c49252bb38c3afe17b34dbb6a8d1589'), ('901650362410688513_3', 'data_image/hurricane_harvey/27_8_2017/901650362410688513_3.jpg', 'dev', '94ca96a61ddebdf2e3ccdf67e2c321a2b08d5b9a9660c07a1e8f251d4620ede5'), ('905010450978484224_0', 'data_image/hurricane_harvey/5_9_2017/905010450978484224_0.jpg', 'test', 'a55848ae14a4b85f3673a932642f4c33dce927ac2a881c5e1de3b81b1242a9ef'), ('907257101029974016_2', 'data_image/hurricane_harvey/11_9_2017/907257101029974016_2.jpg', 'train', 'fade3db3b4130bd25b27dd8a9a9fd38fb832af9c285f410a05b455ba23337f68'), ('901701789279547392_1', 'data_image/hurricane_harvey/27_8_2017/901701789279547392_1.jpg', 'train', '350623fb89445521af89d231a778b2f740a1aeaa71d6786b4d1d3a86ec36ea2b'), ('907638041661108224_0', 'data_image/hurricane_harvey/12_9_2017/907638041661108224_0.jpg', 'test', '5ff59ca2d8653548cdbd5a2d8f2b201076a12cf0a002fda7013b30a3ce35b55a'), ('901655268215341056_0', 'data_image/hurricane_harvey/27_8_2017/901655268215341056_0.jpg', 'train', '61f87d0baf7c53a8d6d1600ff8d4dad8b03a1efd8ebb86770a90726556cf8925'), ('906373710684098560_0', 'data_image/hurricane_harvey/9_9_2017/906373710684098560_0.jpg', 'dev', 'b0ca2b978b59080f3ec7558eca522a4fff3be69cd40b273675668e87edfcddbe'), ('907963801621770241_0', 'data_image/hurricane_harvey/13_9_2017/907963801621770241_0.jpg', 'test', '46aaabebc39b3a855f9f52dd4b93650d6a71c350a49b05f3ae9dd5358d58aebf'), ('906598440360083456_0', 'data_image/hurricane_harvey/9_9_2017/906598440360083456_0.jpg', 'test', 'fa9d17eae2c13af194d24f31af02a108114a526fec774e71f0592e1df3645d82'), ('907315378187055105_0', 'data_image/hurricane_harvey/11_9_2017/907315378187055105_0.jpg', 'dev', 'adc7d69e80651adffc99d9eabbf56bf73d1e0bb2f1072dc72fc32a2a78b004d0'), ('905064623199719425_0', 'data_image/hurricane_harvey/5_9_2017/905064623199719425_0.jpg', 'dev', 'bbef22c457dc34114fc8846d7f6b7a31e94c17589e986490018334fa8a1cfcfa'), ('906258170472890368_0', 'data_image/hurricane_harvey/8_9_2017/906258170472890368_0.jpg', 'train', 'a421854b7c59a88770fc96abf6803f42d93c68d92942f76f9550f04b515ce9c1'), ('904504140021293056_0', 'data_image/hurricane_harvey/4_9_2017/904504140021293056_0.jpg', 'train', 'bb8e34d74838c5cf2412880b9089dd1837b63d266fb25b74f5dbe8b38ed9dace'), ('906927964083847168_0', 'data_image/hurricane_irma/10_9_2017/906927964083847168_0.jpg', 'train', 'eb4f21f09b5b46cd5544bb2dad4b6abd4802fd913ceff865bee9cf9f9d4882cc'), ('909842335424503813_0', 'data_image/hurricane_irma/18_9_2017/909842335424503813_0.jpg', 'train', '6e015d032688caeb22f32c15accf1a28ea0fc19d00dabebf5e55938caf292d44'), ('910131060033196034_0', 'data_image/hurricane_irma/19_9_2017/910131060033196034_0.jpg', 'dev', '7d17818efb317fff044bb1b501f40a7d686fb33367ea04c39c8e8aa69092297b'), ('910160326737281025_0', 'data_image/hurricane_irma/19_9_2017/910160326737281025_0.jpg', 'train', 'ed72395adec3606e8dd87c4c608c8b731f9648a7382b03d62e52a89260b18215'), ('909438423068495872_2', 'data_image/hurricane_irma/17_9_2017/909438423068495872_2.jpg', 'train', '8f09f8013f08d4987b84a1e268dc9a8a159f54ec3aea6626b859eebd721f65e6'), ('909962040063623168_0', 'data_image/hurricane_irma/19_9_2017/909962040063623168_0.jpg', 'dev', '28f70257a70dabbf3399975029f46e881436e49be280238a6d74628b79f63333'), ('909719616133517312_2', 'data_image/hurricane_irma/18_9_2017/909719616133517312_2.jpg', 'train', '049e0de4494a821b6a6d541e756710415c169dcf12adce1f5c621bee8dcc3d06'), ('905734818058657792_0', 'data_image/hurricane_irma/7_9_2017/905734818058657792_0.jpg', 'train', '7ff8f5c198ed067e2278a1a1647ec64ea31f83bdfe1c801b189aec340c63b769'), ('910103777901981696_0', 'data_image/hurricane_irma/19_9_2017/910103777901981696_0.jpg', 'test', 'f7301782ecb7f483042f0853502f6b2c936bd97f3c5895901387e1eaf340baf3'), ('909472350298689536_0', 'data_image/hurricane_irma/17_9_2017/909472350298689536_0.jpg', 'dev', '6b225fdfb5e0a8deb86e84b92ab7712f4323a3906952b0a9406406b04db7d51f'), ('905642249706340352_0', 'data_image/hurricane_irma/7_9_2017/905642249706340352_0.jpg', 'train', 'c87d9c770a4cc8aff02e9d9a9a2ccf9ed9e71cf91059eeb9ea137d1cc66f27d2'), ('909771963958267904_0', 'data_image/hurricane_irma/18_9_2017/909771963958267904_0.jpg', 'dev', '0492534073a9b6fa77d13e2c4986ed9e5b95134bbce903c93b2b40350f629a3a'), ('909779524107325440_0', 'data_image/hurricane_irma/18_9_2017/909779524107325440_0.jpg', 'test', 'e0deefc9aaa91f8477c2f2a8418fdaec8caf8d6d78616d575117db99ee439084'), ('909485900278534144_0', 'data_image/hurricane_irma/17_9_2017/909485900278534144_0.jpg', 'train', '9498b75d3f83aa67084122dd14cc69a7a7e08bb6794e1d12ec74f9f1c877e8eb'), ('909799240184078338_2', 'data_image/hurricane_irma/18_9_2017/909799240184078338_2.jpg', 'train', '1815e92a3144117bc3d7c552eeed222a76ebcc8a3262cbc7f43700a994ea3bbe'), ('905630591437197313_0', 'data_image/hurricane_irma/7_9_2017/905630591437197313_0.jpg', 'test', 'c220d24532cfd5ad8369d0e60822682b98487522f7fcb74477c4283a43567445'), ('909384935785582592_2', 'data_image/hurricane_irma/17_9_2017/909384935785582592_2.jpg', 'train', 'cc676acab5745df6e7f951586fff6c9372ae9b9734e428df86bef567ab225ad5'), ('909494895693398016_0', 'data_image/hurricane_irma/17_9_2017/909494895693398016_0.jpg', 'test', '21e05ea172564134d986a7a276198bd2c0a9d4301dee9ade05c1553f5af99245'), ('909899971767529472_0', 'data_image/hurricane_irma/18_9_2017/909899971767529472_0.jpg', 'train', '30c84207b36ec808680d29985a7c200be76554caa6321ec0ed905f7a2701e333'), ('909429809989681152_0', 'data_image/hurricane_irma/17_9_2017/909429809989681152_0.jpg', 'train', '104539c52851638e42b6a4db25bf10008fdc134d2caef598b3f9a36e6bf6ae0a'), ('909395209204764677_0', 'data_image/hurricane_irma/17_9_2017/909395209204764677_0.jpg', 'dev', '2b7023af8f95b20388117b7cd045e516f30f5ce76c809112818cb0db48c16c8b'), ('914188789681152001_0', 'data_image/hurricane_maria/30_9_2017/914188789681152001_0.jpg', 'dev', 'df74b05da84df52d92ce62a407076e7b661f708ee8f5eb6bbf8e05ae90c8fc1a'), ('922954964913278978_0', 'data_image/hurricane_maria/24_10_2017/922954964913278978_0.jpg', 'test', 'c61a106e5c23079b0394261d440d16527c519dcc869321ec3a0af0eecbe0aff0'), ('923368626035576832_0', 'data_image/hurricane_maria/26_10_2017/923368626035576832_0.jpg', 'train', '7b6c242e0f99ae330c04a9bbd30dfb4243d40f7fe6cad4754184a432494db093'), ('914992466918551552_0', 'data_image/hurricane_maria/2_10_2017/914992466918551552_0.jpg', 'train', '0ccb95c27db980bc7eef8886468c8dd74a0f0a7f1677d6ec7fb912950a9b00aa'), ('913879651398930432_0', 'data_image/hurricane_maria/29_9_2017/913879651398930432_0.jpg', 'test', 'd1dc8d87395e94c7a2b6e2dd6f24c38f8e8180f640f22fda3dfe13ec20ee081c'), ('911558544780152832_0', 'data_image/hurricane_maria/23_9_2017/911558544780152832_0.jpg', 'dev', 'db53bee7583e9b7d1240c02886ed6b97cf422f77ccc57924382857094f3c4a4e'), ('922588121186705412_0', 'data_image/hurricane_maria/23_10_2017/922588121186705412_0.jpg', 'train', '59dd02fa50e1023949575eeb8ce24b3fac81deda3eefcfba45739260ed097809'), ('916078054811799553_0', 'data_image/hurricane_maria/5_10_2017/916078054811799553_0.jpg', 'test', '1b6ff6cc517d778fff2c5d754be16b547381bb7f8714caa7fe1e60dab1bdc9fb'), ('915306985960148992_0', 'data_image/hurricane_maria/3_10_2017/915306985960148992_0.jpg', 'train', '6c5000a9d4835b998af9ed97d3cb9805bd9d7fa51c8d0ab036cf4befecce814d'), ('915282034628730884_2', 'data_image/hurricane_maria/3_10_2017/915282034628730884_2.jpg', 'dev', 'e506109147ab6f0fc9aaa782f9db749c66f1e4de80df41b92cf6aed426f5b585'), ('915313106473103360_0', 'data_image/hurricane_maria/3_10_2017/915313106473103360_0.jpg', 'test', '5d134cfd6992bb98ee1b52b04b3a13dd76fa1809abd1ae5aed1bd87328e0a3ed'), ('911784798162735104_1', 'data_image/hurricane_maria/24_9_2017/911784798162735104_1.jpg', 'test', '462ca3cff43652db1d169e3b501406660a53d16787b826f07834f6e07a32baec'), ('922610290281402368_2', 'data_image/hurricane_maria/23_10_2017/922610290281402368_2.jpg', 'train', '8e9963563aec34428050b188548c360710a5ae48dbc36bbfb089a9a0236fecb9'), ('914428444166955008_0', 'data_image/hurricane_maria/1_10_2017/914428444166955008_0.jpg', 'train', '531c705a1dc067603b2141d4ade6f4806f785cbd3f8fadbfa36a89e538f943db'), ('921462391426240512_1', 'data_image/hurricane_maria/20_10_2017/921462391426240512_1.jpg', 'train', '0464d939c806f2eb82e98a663328d43889b7321134944abb0fac834d41ab353f'), ('910585595629527041_0', 'data_image/hurricane_maria/20_9_2017/910585595629527041_0.jpg', 'test', 'f4143df5a92aec94d94cc24a897b1132e822028b5d2dc56d2af56bd74f492e07'), ('912163976850690050_0', 'data_image/hurricane_maria/25_9_2017/912163976850690050_0.jpg', 'test', 'a418f010faefdab9ad0fe9d63635c0c5e66442d5a766baaa0e2a9911e10b4bd5'), ('913463677285740544_0', 'data_image/hurricane_maria/28_9_2017/913463677285740544_0.jpg', 'test', '61b3948b81838d0cdc2dfcee5274fe356f5e617cfed3bccfeb9598ce75d92a44'), ('914540213875200000_2', 'data_image/hurricane_maria/1_10_2017/914540213875200000_2.jpg', 'train', 'f7f6601b6446081d1672dc007c69d126b55c58507918b56ab60faea9d097f3a2'), ('916292237700497408_3', 'data_image/hurricane_maria/6_10_2017/916292237700497408_3.jpg', 'train', '46098171110516ca4d34020429707b067b5268f47af347c05aaf4d186fc8d2a9'), ('922424804132524034_0', 'data_image/hurricane_maria/23_10_2017/922424804132524034_0.jpg', 'test', 'e867fc9c8d753a2dd04da0c477dc2e059027a987888619cd983d300bd9e29be6'), ('913048338253205504_0', 'data_image/hurricane_maria/27_9_2017/913048338253205504_0.jpg', 'dev', 'a174b9c983fbf1b2d0ad83b76baf3311c74d8ae3c98a9c1c172d37f84ea594c7'), ('923388823702265856_0', 'data_image/hurricane_maria/26_10_2017/923388823702265856_0.jpg', 'train', '665f672c8d63e352c0db6fb69fe0daab17c781b517474ce0344e1d7895ef1967'), ('910757545261768709_0', 'data_image/hurricane_maria/21_9_2017/910757545261768709_0.jpg', 'train', 'f88580d43fcfe70e3deb811e75e634c2a6a64674c33b675760a562a95b9ec264'), ('930314802752258048_0', 'data_image/iraq_iran_earthquake/14_11_2017/930314802752258048_0.jpg', 'train', 'b98f4adbfe29049ef1a5b8a6adb78d220c13cd95941fc535957132d3a2c69170'), ('930002970640388097_3', 'data_image/iraq_iran_earthquake/13_11_2017/930002970640388097_3.jpg', 'test', '6c0d80148234bfbf68908fb76db468870f0e4ef8a33d4690ee29326782327314'), ('929994458153803776_2', 'data_image/iraq_iran_earthquake/13_11_2017/929994458153803776_2.jpg', 'test', '02c834df685105091320bdabda14fb9a320ee41e1d27abe72c4247a55328ad0e'), ('930137569106583553_0', 'data_image/iraq_iran_earthquake/13_11_2017/930137569106583553_0.jpg', 'train', 'd3469a1639778b83f9e2955d2b8a687d689fa064353bad20b0bfba892e732f39'), ('930007000632713217_0', 'data_image/iraq_iran_earthquake/13_11_2017/930007000632713217_0.jpg', 'train', '13fc7f6d8d02a6a45cb992c4d6c80f893af43d00f249dbceaee6a6aa24099a47'), ('929995053619138560_0', 'data_image/iraq_iran_earthquake/13_11_2017/929995053619138560_0.jpg', 'train', '9dbc43d84ae85d939ffc3ec70ae662904f1b818598c45428f4d0371acbba6602'), ('930001999155945472_1', 'data_image/iraq_iran_earthquake/13_11_2017/930001999155945472_1.jpg', 'train', '96edc52e38991d07b5fdb4accc45a01153c42c38d63074735d91ccb651cb7648'), ('930008792410001408_0', 'data_image/iraq_iran_earthquake/13_11_2017/930008792410001408_0.jpg', 'test', '631bb8f4d5f183d44d9e32e947373111116abc81d74c44b16785de9bd6a2bd1b'), ('930066495648026626_0', 'data_image/iraq_iran_earthquake/13_11_2017/930066495648026626_0.jpg', 'dev', '1a592bc407a8b49e7930cfbf4ed049bd7777b6ac890fe901c1511a2672b8c93e'), ('930380460726865921_3', 'data_image/iraq_iran_earthquake/14_11_2017/930380460726865921_3.jpg', 'train', '63af8dcfc4bc5b3de94948fd9c45c24252a14486f3acc78ec56fc04c3d43653b'), ('911622015815122945_0', 'data_image/mexico_earthquake/23_9_2017/911622015815122945_0.jpg', 'dev', 'd49d36863ea26db853d7f2d1879cfdd08fb249bd6f9012f70dacde1c1f55ae3c'), ('912048104651788289_3', 'data_image/mexico_earthquake/24_9_2017/912048104651788289_3.jpg', 'train', 'a1a2984d34d8ac1093153ba8eb24594bd6084fde9e0c42e40e52e7121ed10ac6'), ('910803599680327681_0', 'data_image/mexico_earthquake/21_9_2017/910803599680327681_0.jpg', 'test', '33e5f063678284e641ff8b4cadd610863c9e28c230a07897c6b417cf35c70c8a'), ('910639835953680390_0', 'data_image/mexico_earthquake/20_9_2017/910639835953680390_0.jpg', 'dev', 'd9e7271c3e6cbb415190fadf5317c748bcc989c42ba0c1cea7cbe3c1f2d1f981'), ('910637916635959296_1', 'data_image/mexico_earthquake/20_9_2017/910637916635959296_1.jpg', 'train', 'dc4d7702471bc77a8e098d6eb8517240867dc7cc75a13d1fa07d2c8ce71f2922'), ('911825724205658112_0', 'data_image/mexico_earthquake/24_9_2017/911825724205658112_0.jpg', 'dev', 'd70c5478494c5f135b671572614b67e269339f2c57694c944006d0942f346517'), ('911705733955297281_0', 'data_image/mexico_earthquake/23_9_2017/911705733955297281_0.jpg', 'test', 'a467bbf49943fcbb83e51d68746df298bf8be5f3ae1b2ddf39ea1cae8108e2e2'), ('914823897601605632_0', 'data_image/mexico_earthquake/2_10_2017/914823897601605632_0.jpg', 'train', '1e34b5e2012d72dc5d69b38c39f3f09e359983c8d26de83342c5a2b34760c8fe'), ('913302721725378561_0', 'data_image/mexico_earthquake/28_9_2017/913302721725378561_0.jpg', 'test', 'b642acfd382c3ff4b1a526f8078c51ca9d5791ef9cdd6c933e25f6613799847e'), ('910553306619027456_0', 'data_image/mexico_earthquake/20_9_2017/910553306619027456_0.jpg', 'train', 'dd693bc2d981248e232c6dcdf797f3c9775ea46c7eded287a127c801d00ff1ac'), ('910533452520394757_0', 'data_image/mexico_earthquake/20_9_2017/910533452520394757_0.jpg', 'test', 'a5131e006df5d659e13a88dec0c13ab153a45163f0beecbbace4f2e8bd437d56'), ('911715501684002817_0', 'data_image/mexico_earthquake/23_9_2017/911715501684002817_0.jpg', 'train', '115a2a8021f58e361db7d47a605451a87a46d3ec4edc46f8b5a64f6ed2308469'), ('910674385685680128_0', 'data_image/mexico_earthquake/21_9_2017/910674385685680128_0.jpg', 'test', 'a12d66a9622f8edc93be691a35e53df1f68b57cfec7aa2154084feca9ec29cd4'), ('871932273108627456_0', 'data_image/srilanka_floods/6_6_2017/871932273108627456_0.jpg', 'train', 'fa829b7ea74e322e75b7c2fbd0ad20360e86f154e73661e2a2907d0034237393'), ('870942570704019456_2', 'data_image/srilanka_floods/3_6_2017/870942570704019456_2.jpg', 'test', '870a0902452a4ebb63e5b257217fff7ad44649e13fa30e4818a01a464a99a7ef'), ('870134495650906112_0', 'data_image/srilanka_floods/1_6_2017/870134495650906112_0.jpg', 'dev', '2a3b9a5a32b8f84c37ed216f8367fce668de31b4135fb06dd5a5fd9082af22e8'), ('869987261160542208_0', 'data_image/srilanka_floods/31_5_2017/869987261160542208_0.jpg', 'test', '792c27794fef47cecd541f3029f11e021ac267b75544eb90c1d75f265462bc61'), ('870985716456505344_0', 'data_image/srilanka_floods/3_6_2017/870985716456505344_0.jpg', 'train', '80014d784f50bf2e8a353360c7e1557f1151c50173d430d4eabc30878a821556'), ('870023530624233472_0', 'data_image/srilanka_floods/31_5_2017/870023530624233472_0.jpg', 'train', '00a93a897fb8a8ff4429c36ca4b61f978913bf3960811ce9563f7d55ee0b68da'), ('870035816990425089_0', 'data_image/srilanka_floods/31_5_2017/870035816990425089_0.jpg', 'train', '99680134abddd26cb0ee636cd34090257800e88aaedb2a37589a1b71a7ddab27'), ('879374567487221761_1', 'data_image/srilanka_floods/26_6_2017/879374567487221761_1.jpg', 'train', 'acb855f078c425c05bd9d9b0187601b91be2c44ecb663a93fb20bdf15a760079'), ('918059058074210304_0', 'data_image/california_wildfires/11_10_2017/918059058074210304_0.jpg', 'train', '9596c0537a51d21453f9900d5aafd08a3fb27955a9c28bbb90202b8c78302382'), ('923717489883693058_0', 'data_image/california_wildfires/27_10_2017/923717489883693058_0.jpg', 'train', '3f7dcab85810b6030225ef9793c80a5d0f35324908991d400b024d4f4a23c51d'), ('917793137925459968_0', 'data_image/california_wildfires/10_10_2017/917793137925459968_0.jpg', 'dev', 'ac3505495bb3e23dc201b9e45fbec79d81ad50db2933400e57a36ba2db3797cc'), ('921143211468324864_0', 'data_image/california_wildfires/19_10_2017/921143211468324864_0.jpg', 'train', 'a5d5f724dc3b53f09b71f6a7e0fdf558aa7135f0622c374e53f6c6579e0c40a1'), ('917795236595863552_0', 'data_image/california_wildfires/10_10_2017/917795236595863552_0.jpg', 'test', '9bc34a6ca15aa4c168f3dc5b673acce8cbedc4e5681189d9df1b7287b3f6a14f'), ('917836981291413507_0', 'data_image/california_wildfires/10_10_2017/917836981291413507_0.jpg', 'train', '2a55300bf3937b80c3ab07dd3db8451b6da43b669fe15e4fa05543435b5677b4'), ('922226378363363328_0', 'data_image/california_wildfires/22_10_2017/922226378363363328_0.jpg', 'dev', '81bca5723a61f002bb713fce766b5a3985d08b3621ee29ee5f5f3a2d7ab499d4'), ('917868703705735168_0', 'data_image/california_wildfires/10_10_2017/917868703705735168_0.jpg', 'test', 'e36313e4947c47378cd7407b6779980513abc48ffeaa4cdc07ca30b4cda2fd22'), ('918313986621362178_0', 'data_image/california_wildfires/12_10_2017/918313986621362178_0.jpg', 'dev', '565662e7337709e7806e7d1f0abf9c32b67d3ef5af2fa5d485199fda3a2b063f'), ('920354122058125312_0', 'data_image/california_wildfires/17_10_2017/920354122058125312_0.jpg', 'test', '85733369a80a9fa4240fcc2231b8378106cca02bafd3be35dea82ed5fd1bde77'), ('917988836541632512_0', 'data_image/california_wildfires/11_10_2017/917988836541632512_0.jpg', 'dev', '4ddb7b52156f5a46d5d3eff63245b8fc11ddf52469ab1d8125b7d8805a56cc13'), ('921392455328473088_0', 'data_image/california_wildfires/20_10_2017/921392455328473088_0.jpg', 'test', '5e3eb099913a3b47a28278754db73ce9078fbf69d05da62a765fbbee15590d52'), ('917963263396270080_0', 'data_image/california_wildfires/11_10_2017/917963263396270080_0.jpg', 'train', '550d9c1e6a3598de9567750334b2ffee5cef078ddc2f6def1c51d69794f97dc2'), ('919626960577314816_0', 'data_image/california_wildfires/15_10_2017/919626960577314816_0.jpg', 'test', '15aa025e933b77b847a53bbe62970d46f04d3471b418d7b969a67dea1cd936f7'), ('923725791086526466_1', 'data_image/california_wildfires/27_10_2017/923725791086526466_1.jpg', 'train', '9877a6fe6c241bfd7b447f37cd324040bfa7ab6290a9164fbfc3532b9ee462e1'), ('918104972268535810_1', 'data_image/california_wildfires/11_10_2017/918104972268535810_1.jpg', 'train', 'b86283f2f40c26d76f311ff3936ee4cecca184fb438a963fc1de00682bf92265'), ('904393848356569088_0', 'data_image/hurricane_harvey/3_9_2017/904393848356569088_0.jpg', 'test', '47d0d4c2baae251564bb75d40982f0d13e7a2773db680f0ff49f71535326c739'), ('906043792255569923_0', 'data_image/hurricane_harvey/8_9_2017/906043792255569923_0.jpg', 'train', '24b6a013f59098488e7c8b3018cfe6e1f4ecfd05fd7404ae36caab46b7faf64a'), ('905073390612090880_0', 'data_image/hurricane_harvey/5_9_2017/905073390612090880_0.jpg', 'train', '4b31d2e19e040ee5dd54de950dbf10432d15470fd8b5ffc0870981d3f982dc81'), ('901646234880024576_0', 'data_image/hurricane_harvey/27_8_2017/901646234880024576_0.jpg', 'train', 'c804511f90d1d89823e775529970876ea26861c75a22b58e52cbeb31af4ee505'), ('904511606612037632_0', 'data_image/hurricane_harvey/4_9_2017/904511606612037632_0.jpg', 'test', '15f511ece63fd054182fb4d21dae02cfd79d5d7fe4764d83399d6f51814c37e7'), ('901805267989475328_0', 'data_image/hurricane_harvey/27_8_2017/901805267989475328_0.jpg', 'test', 'fd3c50a25a60cfcdbea30e2852cf1c466296de7a936d4634331f27a3d8bf9dae'), ('909128086985338880_2', 'data_image/hurricane_harvey/16_9_2017/909128086985338880_2.jpg', 'train', 'c8aaab10e89fed147ebe6df3b4756314204a79a2c18481617e74e03bb14bfcdc'), ('908100686273024000_0', 'data_image/hurricane_harvey/13_9_2017/908100686273024000_0.jpg', 'train', 'f548634f0bc100fb2fe22ef0f73271f3db482f88c1524cd8f0cc1b6474d5c508'), ('904787823181520896_0', 'data_image/hurricane_harvey/4_9_2017/904787823181520896_0.jpg', 'test', '3d81fb9013679a8337d1c96b2285b1f0600c44bf92025bf24540ed0fe796eed3'), ('904759353856557056_0', 'data_image/hurricane_harvey/4_9_2017/904759353856557056_0.jpg', 'train', '17007e5581fc5cbba6d850a7c27634478065deb8d1d09988904c2b540863e6b6'), ('904849577450160128_0', 'data_image/hurricane_harvey/4_9_2017/904849577450160128_0.jpg', 'test', 'ea5fc85f7c3bead9770e9d1d7aef23fa130ada96892b890e80a382c7266bf870'), ('906187513101860864_0', 'data_image/hurricane_harvey/8_9_2017/906187513101860864_0.jpg', 'test', 'e916fb918a95e41f68a899792ae6fec06c96932e68b74025170f83efae068e5e'), ('905756920392409088_0', 'data_image/hurricane_harvey/7_9_2017/905756920392409088_0.png', 'train', '3e2b29d160758ae9b3d8aadc95c58043ef2e4e23bdb8329decada67a4fa18121'), ('908320017221132288_0', 'data_image/hurricane_harvey/14_9_2017/908320017221132288_0.jpg', 'train', '8e42e606b72be1b3b7d7f6108330bfae2182ad967c387457dd73fe5625c83463'), ('909142245554532352_0', 'data_image/hurricane_harvey/16_9_2017/909142245554532352_0.jpg', 'train', '3e21958103e25c8bb51de168459414a84de9b1f83852165d039db84af9486f3d'), ('907967554789847041_0', 'data_image/hurricane_harvey/13_9_2017/907967554789847041_0.jpg', 'dev', '966dc90121bee88cb347b3d7e1ea9d7a64d7d1c4fbbf75aeff849143066148ae'), ('901813825711853568_0', 'data_image/hurricane_harvey/27_8_2017/901813825711853568_0.jpg', 'test', '50d9dd600da0305dcd8b0f6e5565e9fe356e78eb2e3c96535ed01422e936b56a'), ('908271399545925633_0', 'data_image/hurricane_harvey/14_9_2017/908271399545925633_0.jpg', 'test', '94d0c02b6dbfc32846437a52fb4dc2b2f8e03b2153a482c7884f4d73450848cb'), ('909694844901183488_0', 'data_image/hurricane_harvey/18_9_2017/909694844901183488_0.jpg', 'train', 'fc7072fbde1eff0f9ed696bac1a77f2124231e39d7f045e3a1ea5087fbf2c198'), ('907257101029974016_1', 'data_image/hurricane_harvey/11_9_2017/907257101029974016_1.jpg', 'train', '20b654c395839c1f4b1ee34b3327db0fbaec9dde1ad37c842b7b71867083f076'), ('904799982200303616_0', 'data_image/hurricane_harvey/4_9_2017/904799982200303616_0.jpg', 'test', '301f2276cbc5adc96fd89ac16e9416602b8136438801a67ede076e6d71f3c75d'), ('907271519289692161_0', 'data_image/hurricane_harvey/11_9_2017/907271519289692161_0.jpg', 'test', '8eb51418d051d9e404a6203df3322e7be21ab87d2ea7ac0224d7ad470dcf014a'), ('904944352060989440_0', 'data_image/hurricane_harvey/5_9_2017/904944352060989440_0.jpg', 'test', 'c9ef38763620566ded950478edfa9363f24bd50852b215c0c930968ef99c4d23'), ('901646591035138050_1', 'data_image/hurricane_harvey/27_8_2017/901646591035138050_1.jpg', 'train', '9bd4efeff554dc472bdcb1dffcbf5fe4aa862c5b859ae9f7812dc44fe25810fd'), ('909986190086025216_0', 'data_image/hurricane_irma/19_9_2017/909986190086025216_0.jpg', 'test', 'b99067bbb0b07d1660bcd84ff73acb9fa25c9be574c6d99540db2ab1e7f0339a'), ('909765980519370753_0', 'data_image/hurricane_irma/18_9_2017/909765980519370753_0.jpg', 'train', '31a7ff74ebfdaa92c8c02f4c273963f42bd0cdcded41a1c72425cd2e86d19104'), ('909715026872762368_3', 'data_image/hurricane_irma/18_9_2017/909715026872762368_3.jpg', 'test', '2dc20902ddaddad1f57827200f4c62c2fe1771b1083cd65fc6cacd32794ea9b9'), ('910070947725692928_0', 'data_image/hurricane_irma/19_9_2017/910070947725692928_0.jpg', 'dev', '219df5778749e7a38dc13ba83d2979c582e64a18295e12f562b3c5269acef4b9'), ('909719418887909376_0', 'data_image/hurricane_irma/18_9_2017/909719418887909376_0.jpg', 'dev', 'd70a7a1a2bc07b22137aff69c7a24deb8a2d5af531822516a83df1b24aed4840'), ('909900730735190016_2', 'data_image/hurricane_irma/18_9_2017/909900730735190016_2.jpg', 'train', '18a22cbb903f5841f4a00bddbf15df317ea55ea5877cd6c476724ede0cec8f43'), ('909873915178123270_0', 'data_image/hurricane_irma/18_9_2017/909873915178123270_0.png', 'train', 'f0f7bb2442075ca01bb92a6e934746f0dc162b3623344b69554c385f664b522e'), ('905738954103824384_0', 'data_image/hurricane_irma/7_9_2017/905738954103824384_0.jpg', 'train', '2b53a64423a0323b3c4215cbcc252a973a951f618395c18b07bbea5e63803546'), ('910189070072057856_0', 'data_image/hurricane_irma/19_9_2017/910189070072057856_0.jpg', 'dev', 'e26ba6834d2c53d93c4255625c37a389b940de4097caa1ca0a0a6b943c8f70e5'), ('910182750346465281_2', 'data_image/hurricane_irma/19_9_2017/910182750346465281_2.jpg', 'train', '7032415a05f308dd9675759cf0d76928c93cc6731a249789c8d59738ce8f4aeb'), ('909710844811993088_0', 'data_image/hurricane_irma/18_9_2017/909710844811993088_0.jpg', 'dev', 'c2cc5f071be180e37ba3f6dfc63b4cd8f1d3c1e1730d5786f0aa2c5caa961498'), ('909876949614444544_0', 'data_image/hurricane_irma/18_9_2017/909876949614444544_0.jpg', 'test', '20b7b2e709b0871b712731fc42dc1cbdad4a2bb841bb94dc3544ba3f24b89c33'), ('909805021767901184_0', 'data_image/hurricane_irma/18_9_2017/909805021767901184_0.jpg', 'dev', 'dcf8d53e7d54c6cbf96609f439d27e6f704ed3bd378521406a76e846c1e54154'), ('909789893479497728_0', 'data_image/hurricane_irma/18_9_2017/909789893479497728_0.jpg', 'train', '58fd43d08624583bf157049c51b1ca71bcaaa61ada13247f5fec720548110c53'), ('909796523713609728_0', 'data_image/hurricane_irma/18_9_2017/909796523713609728_0.jpg', 'train', '7d9d65f8f8c3397eb4b871ef4f04dbf304f13ef368f8e361cfd0db1e2181f9d4'), ('910225493877813249_0', 'data_image/hurricane_irma/19_9_2017/910225493877813249_0.jpg', 'dev', 'ec8a4432aa9572be4dcbbc6572caf434d0cefe3f8bd0378c598736c8c7b043c2'), ('909771657140670465_0', 'data_image/hurricane_irma/18_9_2017/909771657140670465_0.jpg', 'train', 'dd9882278c54c487f4018458d0072eee3a30ec65bd42edda61fa9dc3e85c1b26'), ('910253973516435457_2', 'data_image/hurricane_irma/19_9_2017/910253973516435457_2.jpg', 'train', 'c5c55b80b784cbd918b123298abe632bbfc2a8f487d01e3ef11002578b252128'), ('909381091131236352_1', 'data_image/hurricane_irma/17_9_2017/909381091131236352_1.jpg', 'train', '7af33f48c685ce7b38a6d697b4e9800c20229495612c3a076b42a58b70216af3'), ('910149254097850368_2', 'data_image/hurricane_irma/19_9_2017/910149254097850368_2.jpg', 'train', '06d07e37a71f8cc3aceae84ca19a9233f7b2f9ad0b71e34afd31154d6051394e'), ('909408865992798208_0', 'data_image/hurricane_irma/17_9_2017/909408865992798208_0.jpg', 'train', 'f7c4772237083d2cf4fe6ebc1bd7f1c35b32673dbffc78fa0f689a7afe673f9d'), ('910523362950053888_0', 'data_image/hurricane_maria/20_9_2017/910523362950053888_0.jpg', 'train', '74ec7c60832f448d9c07d1e3d28c5bdec150f691a6c3c644d6f0a76ce7869cb7'), ('921507199674998791_0', 'data_image/hurricane_maria/20_10_2017/921507199674998791_0.jpg', 'train', '98222c9760fb52cbae9a2b9956c9d707adcbad2d925af37c508510518abad87e'), ('914928771865694209_0', 'data_image/hurricane_maria/2_10_2017/914928771865694209_0.jpg', 'train', '353a7f59fc2fe6566fa3e0686db382f0b2a4b2ac47fc758f7a72215573450f7e'), ('910655381247950848_0', 'data_image/hurricane_maria/21_9_2017/910655381247950848_0.jpg', 'train', 'a90bcff3f74dc2f2a909159751526dc574cbaa67655eb8b3a039dd2417faedd8'), ('913119569023299584_0', 'data_image/hurricane_maria/27_9_2017/913119569023299584_0.jpg', 'test', 'dfe25ac850d1de5f8dbb6aa5ba734c72daba20faa48673bb3de0c3710e48f60f'), ('910552318709567488_0', 'data_image/hurricane_maria/20_9_2017/910552318709567488_0.jpg', 'dev', '1912b8abefb067f45b6a4bf8196f6c0a4fe5300f8288feb5316cb6cea48adf90'), ('912973226493530112_0', 'data_image/hurricane_maria/27_9_2017/912973226493530112_0.jpg', 'train', 'f4d9de068b2ac893ffc0e72e5876681d17fd9277ddcbbdc1b02fa0da2ecade96'), ('911564230507339776_0', 'data_image/hurricane_maria/23_9_2017/911564230507339776_0.jpg', 'test', 'cb4e001a59bb09e519fcf29d38fec76492de902b504eadf116df31d79def92d9'), ('916119415665684480_0', 'data_image/hurricane_maria/6_10_2017/916119415665684480_0.jpg', 'train', '58c4e4a6906a6672795c7b4de6d760008831e577ab683125c3db904b94632a9c'), ('911944421507297280_0', 'data_image/hurricane_maria/24_9_2017/911944421507297280_0.jpg', 'dev', '8bbf83efc2094f3a5ef0aa6cae24a3cb7d884dca5185c4c1841ac34f263d01e2'), ('915287851478085632_0', 'data_image/hurricane_maria/3_10_2017/915287851478085632_0.jpg', 'test', '99ee85a407e2366820746ed8a1149d23780d99fb9930d465cdfeec6767381242'), ('916086878088163329_0', 'data_image/hurricane_maria/5_10_2017/916086878088163329_0.jpg', 'dev', '9097914a9b27f0e45547c32d8607cf25a67e34f1b7e7b13ae08ac7a88fd6cd3f'), ('916028011023126529_0', 'data_image/hurricane_maria/5_10_2017/916028011023126529_0.jpg', 'dev', '2d70c916970f1f4d86d912484b5fef435c25e8db5b2bd50cd2d6364b2aa5e394'), ('922557506894909440_0', 'data_image/hurricane_maria/23_10_2017/922557506894909440_0.jpg', 'train', '0c84b702e07ddeb20de064406b817d15f4ad51927408d42bb9b1f2623a7094d0'), ('929464595407917056_0', 'data_image/hurricane_maria/11_11_2017/929464595407917056_0.jpg', 'dev', 'd5f64e2a15eafaccc911c5a85bea81e43f76340cada74899d9d7fbcb7a54106f'), ('923255652025294848_0', 'data_image/hurricane_maria/25_10_2017/923255652025294848_0.jpg', 'dev', '3751f03a1c7404fbe94a66037815cfb88ab03d988b8e82467412813578ffa6c4'), ('911644108523032577_1', 'data_image/hurricane_maria/23_9_2017/911644108523032577_1.jpg', 'train', '76d0661a16a423148272780642e4be42f711052a036f69b9c6289cb8f6d08269'), ('910723784474349569_0', 'data_image/hurricane_maria/21_9_2017/910723784474349569_0.jpg', 'train', 'a572966f2e7a1a0af90467bc469021996722101e2587e8e6da1eb19949f4d512'), ('916060300717297666_0', 'data_image/hurricane_maria/5_10_2017/916060300717297666_0.jpg', 'test', 'e217c5994ce42f7f2497b9c287e9341bad44a0e4005f508337ccce1ba9cabff0'), ('914098838897651712_1', 'data_image/hurricane_maria/30_9_2017/914098838897651712_1.jpg', 'train', '7d27071d2a544bf10cbfba9a021320d78df117ffc214e10c61b144738121eeaa'), ('911874137542578176_0', 'data_image/hurricane_maria/24_9_2017/911874137542578176_0.jpg', 'test', '625fa3d5b096bf12bd727a127aab1cb3b393e9574f235322f2379096f3d9314c'), ('922975296894242816_0', 'data_image/hurricane_maria/24_10_2017/922975296894242816_0.jpg', 'train', '9b8bfdef17b5d5f674d846fbd3dcee1fb036b0f843cff5b8a28b5ee90355da79'), ('923652116383756290_1', 'data_image/hurricane_maria/26_10_2017/923652116383756290_1.jpg', 'train', '6aa7d4b10a78ae64ee3b63965f19dec95a4faabf7ba75b88827dc118d7717916'), ('913008996335878144_0', 'data_image/hurricane_maria/27_9_2017/913008996335878144_0.jpg', 'test', '4ccf8cdd82fc1fb1d2d1d71f12b231bb0966b1886fef0895151e5e4d73044720'), ('930427544372776960_0', 'data_image/iraq_iran_earthquake/14_11_2017/930427544372776960_0.jpg', 'dev', '4d7346f2b81fcf8700ac97f76461ef4d22741f9c23d67adf6ffe751ca1b6892d'), ('930570718168928256_0', 'data_image/iraq_iran_earthquake/14_11_2017/930570718168928256_0.jpg', 'train', 'f078c4c2c6f2a4555ecc806828ce7c44e3b58b8a5aafbba55c2095a25f1efeaa'), ('930042585770536960_0', 'data_image/iraq_iran_earthquake/13_11_2017/930042585770536960_0.jpg', 'test', '886c78c89fbc3945a185060a16932225151bfb189ff9a750e0334fee2aba8cda'), ('930875630832545792_0', 'data_image/iraq_iran_earthquake/15_11_2017/930875630832545792_0.jpg', 'test', '1a89ff63275343c72f4a5d7384ae98d699e24ebe211f70176862b6b4d14f2121'), ('930009524945276929_0', 'data_image/iraq_iran_earthquake/13_11_2017/930009524945276929_0.jpg', 'train', '29658615dfb830af997819f2b7ccbe0a0400c95748687ba8242ec64711fc3771'), ('931465546129989632_0', 'data_image/iraq_iran_earthquake/17_11_2017/931465546129989632_0.jpg', 'dev', 'ce574bf8b0d9fb876b9b5c36a56e409d1c5a132d6d6f7df97cec15f3b42e4ca2'), ('930080763273412608_0', 'data_image/iraq_iran_earthquake/13_11_2017/930080763273412608_0.jpg', 'dev', '2d79356e391480e4d4899b78dc8e012aefc6e92f594eec43961b014380b61e6a'), ('929997660345397248_0', 'data_image/iraq_iran_earthquake/13_11_2017/929997660345397248_0.jpg', 'dev', 'b398e360c961120d601ae1ef7c1e6aef79a1f7cdd43d22adf2167f2392a29d5c'), ('930106897100951552_0', 'data_image/iraq_iran_earthquake/13_11_2017/930106897100951552_0.jpg', 'test', '7b3cb58d0c71b1c24b54f7d4251c6bb72b3deb2ca41a72eafcac9e9ac5f4f33d'), ('930026404036726785_0', 'data_image/iraq_iran_earthquake/13_11_2017/930026404036726785_0.jpg', 'test', '7b133bb64a1b20e20175b47bdc0aaed5551c59470e4ae98960829df388cd9e7a'), ('912950452223975425_2', 'data_image/mexico_earthquake/27_9_2017/912950452223975425_2.jpg', 'train', 'ccad47eac8d0cc9459cc36cef2cacd43ee23e06a1479f59114f867198b088b78'), ('915253376912379905_0', 'data_image/mexico_earthquake/3_10_2017/915253376912379905_0.jpg', 'dev', '1c2b00ad64c8d7c89c753aebca0332b5eae80599841d00813578c51781298d7b'), ('915201277432418305_0', 'data_image/mexico_earthquake/3_10_2017/915201277432418305_0.jpg', 'train', 'f1545fc2ce25d17fe5c83246557d91e1b1bb4861bf5e4b69b4f4f585d3f1d5e0'), ('910523485889470465_0', 'data_image/mexico_earthquake/20_9_2017/910523485889470465_0.jpg', 'train', '10be0a84f56d971bd91f95264ae3bbaa7d4d14ca789c2ad0a32674b37cb4116e'), ('910706225549062144_0', 'data_image/mexico_earthquake/21_9_2017/910706225549062144_0.jpg', 'train', '17a56f8a63c612bf9f8aaf9eee951383f0855c47a5d6c6023a61e66def9c08f7'), ('913811073005842433_0', 'data_image/mexico_earthquake/29_9_2017/913811073005842433_0.jpg', 'test', '2ea268fd787f58e86f814fdba005cd1df3b49683c707481f534752d1a806b27e'), ('910681818843582465_0', 'data_image/mexico_earthquake/21_9_2017/910681818843582465_0.jpg', 'train', '10403e0dfe8d5dc4713c2d113e392d8d55eb7165223e8766074110e6c2917211'), ('912050610396188672_0', 'data_image/mexico_earthquake/24_9_2017/912050610396188672_0.jpg', 'test', 'e6e93d4578b5489854f01ec2a76765e5012699072290bac65c1ef513255ab8b5'), ('910536199470768128_0', 'data_image/mexico_earthquake/20_9_2017/910536199470768128_0.jpg', 'dev', 'ff1af2351df8548433b61138cc8dfc78e4e4e8cfab0fcd0a4a9aea842c7e5213'), ('911623309724233728_0', 'data_image/mexico_earthquake/23_9_2017/911623309724233728_0.jpg', 'train', '944007b239f7f2756441743736b0c65da65d00aae031da6b724f29fd82fb7027'), ('910621512209539073_3', 'data_image/mexico_earthquake/20_9_2017/910621512209539073_3.jpg', 'train', '3125f689e5c236d97b3b07b7b4d6ab829957d792881aba28d2d4ed1cf545bc47'), ('915246492096565248_0', 'data_image/mexico_earthquake/3_10_2017/915246492096565248_0.jpg', 'dev', '7f08ce3bfcc89e1e10fd7d1c47f9ea61345480e77e0b627bb482b6ef980c4c3f'), ('910553534470516737_1', 'data_image/mexico_earthquake/20_9_2017/910553534470516737_1.jpg', 'train', '5866eb8cbd4e8b52cb9ee9cae26826e22788c1a1e97acebe3cc200a61af9dc97'), ('871703976097583105_0', 'data_image/srilanka_floods/5_6_2017/871703976097583105_0.jpg', 'dev', '14dcf9f28870ca79a6680eb56ea72dbb6c40a658c5d0d1d879a53d71ae1f9455'), ('870278054697881600_0', 'data_image/srilanka_floods/1_6_2017/870278054697881600_0.jpg', 'train', '38003016736919cb7c6a5428ee5a24f5e9566ddbad7224f47f300e68a4dcc319'), ('873113700944584706_0', 'data_image/srilanka_floods/9_6_2017/873113700944584706_0.jpg', 'dev', '5684f1af129edc43dd78337ccd950953e7c7da1a2143f5803a5275473e826339'), ('870110651624022016_0', 'data_image/srilanka_floods/1_6_2017/870110651624022016_0.jpg', 'train', 'd7c60852807cc17e5dd804cca17e741c0cd20da0cf9f742caa417d15959aa09b'), ('870383429929578498_0', 'data_image/srilanka_floods/1_6_2017/870383429929578498_0.jpg', 'dev', 'bbf16e83aa7d801d4f85e4e8d4e6fa6cfbb5272656a0f36e5a9f25eac1395632'), ('870190791452438528_0', 'data_image/srilanka_floods/1_6_2017/870190791452438528_0.jpg', 'dev', 'c512768fa9888620f86d080e5e97f2b1e785a2d9b0abb3347a6946d7dd1d01db'), ('875722518899961856_0', 'data_image/srilanka_floods/16_6_2017/875722518899961856_0.png', 'dev', '965acadac4bc6d9a4ea0b91f20f8e905a1f2b1b484c9c82631065528d241c0f1'), ('870491727886405632_0', 'data_image/srilanka_floods/2_6_2017/870491727886405632_0.jpg', 'dev', 'aba506ff07798d674a0ed3b797e53afc0c526f643c83979a16a5a1d32afb6dc4'), ('919663276740329472_0', 'data_image/california_wildfires/15_10_2017/919663276740329472_0.jpg', 'train', 'd497ab8504253986dea5a077fa9e061971880a5a042af0fbad87ad1799f31754'), ('920684183105794048_0', 'data_image/california_wildfires/18_10_2017/920684183105794048_0.jpg', 'train', '9b0ac5d43f537d46b67c9752572393b53933662f4f5bcc71ae56b34e9e11ad70'), ('917869210499399680_0', 'data_image/california_wildfires/10_10_2017/917869210499399680_0.jpg', 'test', '4f774a817afbf1d03e116e81ac275695ab8dd4603b3a7ba4dde1e4343937bf1f'), ('923238299245752320_0', 'data_image/california_wildfires/25_10_2017/923238299245752320_0.jpg', 'train', 'ae914bb08296da4e87dcdebc6a9047138899ca2b250a2899666b37ccf2aad161'), ('917934616878530560_0', 'data_image/california_wildfires/11_10_2017/917934616878530560_0.jpg', 'test', '9fb66005eb179a44b471a6322e8f8790808741333113771a5a4e0c0bd6e75b45'), ('918071206984781825_0', 'data_image/california_wildfires/11_10_2017/918071206984781825_0.jpg', 'dev', '5bc3cd953479a35084295b9980e233dd8ff8b785ef895d48844d9db0218091f8'), ('919618650134343680_0', 'data_image/california_wildfires/15_10_2017/919618650134343680_0.jpg', 'train', '44c21fb9ce997e03354fb5575a6fbc33aa4d88617d74aa85a73a4daa4f95bf56'), ('919996197057843201_0', 'data_image/california_wildfires/16_10_2017/919996197057843201_0.jpg', 'train', '42a47ce98adb5096fc98f1a5aebb3926324b19afc0c7220ebc003f068c6aab46'), ('921178221084897281_0', 'data_image/california_wildfires/20_10_2017/921178221084897281_0.jpg', 'dev', '4fae73aad80ba66687ddf1c364355a37f01b736e27b24b4b6a2239a9e9c175c8'), ('922553813202034688_0', 'data_image/california_wildfires/23_10_2017/922553813202034688_0.jpg', 'dev', '7379d14b1512af3481cd19ece59f5ae9322b8a3c3d4cf4b1d27e476052b7e1bf'), ('917866615936618497_0', 'data_image/california_wildfires/10_10_2017/917866615936618497_0.jpg', 'test', 'de5668e1796d95ed650d1d9bb3316260f3b8d96ecdd07efff826791eb0400b4e'), ('920278093222129665_0', 'data_image/california_wildfires/17_10_2017/920278093222129665_0.jpg', 'train', '52df929b2dcceda941c2323491c615999d80d7dbf91650b080a4c11e6b08f47f'), ('921044916158869505_0', 'data_image/california_wildfires/19_10_2017/921044916158869505_0.jpg', 'test', '2efc38173eb0870cba8896603903d1fd54b0313317a9eacd48b89506559a0520'), ('919587932901642242_0', 'data_image/california_wildfires/15_10_2017/919587932901642242_0.jpg', 'dev', 'ad7aa279d24699468b6d980a95465556dee6b57331790a6ca428f209eef2dd87'), ('920210041075990528_0', 'data_image/california_wildfires/17_10_2017/920210041075990528_0.jpg', 'dev', 'a25344cd0fafd8a5cdff61b54960c7f22c6fb64dd39142622bfcb50b0a8105ec'), ('901683922676969472_0', 'data_image/hurricane_harvey/27_8_2017/901683922676969472_0.jpg', 'train', '3a278889a0b2efb27b45d9c6e4f957015a16aa36c4e4c6ef11742e7127c184c1'), ('904869860546674688_0', 'data_image/hurricane_harvey/5_9_2017/904869860546674688_0.jpg', 'dev', '05d19b0fdaeaf780ad684fe5fb69253c481e7c57bb2b31bacda4e25685a9e71b'), ('904795388862881795_0', 'data_image/hurricane_harvey/4_9_2017/904795388862881795_0.jpg', 'test', 'c089e15d7778472c6561717ba68dcefe61742ba3864b105c7ce741475a5c4527'), ('906964197053014016_0', 'data_image/hurricane_harvey/10_9_2017/906964197053014016_0.jpg', 'train', 'f3c7626685835c6f1b3286ded81cb586f98808d24def235c2f62cc9af4e41062'), ('904914448099495936_0', 'data_image/hurricane_harvey/5_9_2017/904914448099495936_0.jpg', 'train', 'bd2b1ee1ff90426e833f351630020b5b33e098a03aaa33f561475ea5cef70526'), ('901713432759566336_0', 'data_image/hurricane_harvey/27_8_2017/901713432759566336_0.jpg', 'dev', '9ffd64217270fb3e9bfdf6459c040a24629592e3b6e20bc8e1d902d11b1a95c7'), ('904299444249690112_0', 'data_image/hurricane_harvey/3_9_2017/904299444249690112_0.jpg', 'train', 'b809622aea34a287f5e2bfd425dbcdbb1dd84d427af80ea6a4d14ae681a0bf8e'), ('906625964112826368_0', 'data_image/hurricane_harvey/9_9_2017/906625964112826368_0.jpg', 'train', 'af3982843ae83ed20c62266b68b13fc1c0b79aa2ccf981b9fd944cfc334072b7'), ('906234333555433473_0', 'data_image/hurricane_harvey/8_9_2017/906234333555433473_0.jpg', 'train', 'fa68eb8bcb7183ea9dd12749aa74c8b73a08633f63ae56bd1ced9df3ae73c257'), ('905693387092787200_0', 'data_image/hurricane_harvey/7_9_2017/905693387092787200_0.jpg', 'train', 'ed92286495a393958f46a581d96a8957a2fa542da3b2e0edb56ff8d2dc37d58c'), ('907928859064061953_0', 'data_image/hurricane_harvey/13_9_2017/907928859064061953_0.jpg', 'train', '2944d40f88f455e00ea7fef592c6cac0285066b24648aa73b07aa79a988b372b'), ('904414219604459521_0', 'data_image/hurricane_harvey/3_9_2017/904414219604459521_0.jpg', 'train', '62239aa9dc7f33ad5eafe85b9acd282156d7dd510e5fc492b6413ca36324a25c'), ('905036991644958720_0', 'data_image/hurricane_harvey/5_9_2017/905036991644958720_0.jpg', 'test', 'ba130a03144a482cbb3c914acc83cfa6fdf99f9d98fce134efb5e98b97d4f5b4'), ('905637549737213955_0', 'data_image/hurricane_harvey/7_9_2017/905637549737213955_0.jpg', 'dev', 'f8894d8fc496ebd54e199ad447f282293c526c087521ba64f07975ef89e486d9'), ('908454080749727746_0', 'data_image/hurricane_harvey/14_9_2017/908454080749727746_0.jpg', 'dev', '87cec299bf2a122751e2a7940f0adea881ce165826c898a2686b7196918cb65a'), ('905988543142653956_0', 'data_image/hurricane_harvey/8_9_2017/905988543142653956_0.jpg', 'dev', '5a820268369a54640e0f7bf90bff639d80e37ddfc18d4b04412246afd0af025e'), ('905392976574205952_0', 'data_image/hurricane_harvey/6_9_2017/905392976574205952_0.jpg', 'test', '2564b2fecda7009eda2bab3c772ba5df7f83089eb19064a74a2f5df66a3888ba'), ('905184065061576704_1', 'data_image/hurricane_harvey/5_9_2017/905184065061576704_1.jpg', 'train', '8765e2a93ed495bdc43f9a7437196d4a8fdc895c792d414926b0ce5026d48958'), ('905092083047964672_3', 'data_image/hurricane_harvey/5_9_2017/905092083047964672_3.jpg', 'train', '191feb3b433eb55f43351fe292c0aba81a27f1f9636e5edac48f3f3730f86af9'), ('907612395278934016_0', 'data_image/hurricane_harvey/12_9_2017/907612395278934016_0.jpg', 'train', '30f2e1af3d5b6dab1e3f9542386b95a8e410e40f51305857182670da2ab4690e'), ('904357429806120961_0', 'data_image/hurricane_harvey/3_9_2017/904357429806120961_0.jpg', 'train', '6c3bb79849906afb6f2eee89b023180126d968c25dcdec6a62f93f2a4a9c924e'), ('904881597501108224_0', 'data_image/hurricane_harvey/5_9_2017/904881597501108224_0.jpg', 'test', '683a5fcb29d8ac307c197243531627d09264ad4121128a1f9805187513e912e5'), ('910262990540242944_0', 'data_image/hurricane_harvey/19_9_2017/910262990540242944_0.jpg', 'dev', '2ae45ca46b37469a16722f4f798d1cd35d113c0ebdae0a2c339509e6f02d2b85'), ('909956798937001984_0', 'data_image/hurricane_irma/19_9_2017/909956798937001984_0.jpg', 'train', '603ad0614b580632d9849f7e474376fb84d8fd737879ddff160d3fd7e786cc80'), ('909793989691355136_0', 'data_image/hurricane_irma/18_9_2017/909793989691355136_0.jpg', 'test', '005c29d42857d2dce139037f1b3a18693f04a2915dc0f050b443bdb0fbe9277f'), ('906928042865504261_0', 'data_image/hurricane_irma/10_9_2017/906928042865504261_0.jpg', 'test', '3d968efcde911c9b09082d8b39ce2fe47c151fe467c019f2a1f2c8b96ab01d4f'), ('909814108610072577_0', 'data_image/hurricane_irma/18_9_2017/909814108610072577_0.jpg', 'dev', '688bfdf2ac5c77e43e0cb4da5e239ae03159eff77e1cf437b3696e4dcdc2a5e5'), ('909809255498129408_0', 'data_image/hurricane_irma/18_9_2017/909809255498129408_0.jpg', 'train', '775a8ad82f99e4ea9bfcc46f801f089dcbdf8528f02f55d4b52f5eff9376fc2d'), ('909421707038986240_0', 'data_image/hurricane_irma/17_9_2017/909421707038986240_0.jpg', 'test', '3adaad8e89a6ade6ba14fc3233c3ade112f284a679cb79b52a3e3f70aeb8d41c'), ('910202225905029120_0', 'data_image/hurricane_irma/19_9_2017/910202225905029120_0.jpg', 'train', '6ea1a0201af6a619da6d2cdaa3f58907385b1fca9cbaf793327c40f0d5eea3e7'), ('910240396688658433_0', 'data_image/hurricane_irma/19_9_2017/910240396688658433_0.jpg', 'train', '6c10f848bc24cd22a806e5292786df0c48e2c88459d9cb8b3a1b4ec337a70b3e'), ('910074974328651776_0', 'data_image/hurricane_irma/19_9_2017/910074974328651776_0.jpg', 'train', 'ae3d6688519c3d4233cfe60e034305c9b8fb7083f54b1b68c7142e71b031030c'), ('905691597005889536_0', 'data_image/hurricane_irma/7_9_2017/905691597005889536_0.jpg', 'train', 'aff49388b41e874dd27d913ef13c6e11043b8eb377c9ee8d37d30704b9fc0464'), ('909890037436653568_0', 'data_image/hurricane_irma/18_9_2017/909890037436653568_0.jpg', 'train', '1a78574347c1ec4a709f193010905461329710c0c95c61300fc80f041e3d7c94'), ('910233174663458818_0', 'data_image/hurricane_irma/19_9_2017/910233174663458818_0.jpg', 'test', '3bfd9e938753c9c85d9b12fb5e08e0b813ba1694bafc1b6b30f46923e315cc2d'), ('910218229397602304_0', 'data_image/hurricane_irma/19_9_2017/910218229397602304_0.jpg', 'test', 'b37f951d3a8ea5373294dbefd5ee393604304f31cc5efd5e6d8ffcd7c0ba8f57'), ('909492601702813697_1', 'data_image/hurricane_irma/17_9_2017/909492601702813697_1.jpg', 'train', '3b143765f1fe18e42b057e1fd69a7d98e7647525641a3e7e987d3a4a4f982259'), ('910082333268676610_0', 'data_image/hurricane_irma/19_9_2017/910082333268676610_0.jpg', 'test', '0430721ed48b142a50200a160a9c3b8ae31036ad6854f08943c3b0fc23e9eb2b'), ('909838084786786304_3', 'data_image/hurricane_irma/18_9_2017/909838084786786304_3.jpg', 'dev', '31d1ede67e4627ffca58f62b38be97eae682818bd98417e2b906dffbb833c841'), ('909885632582787072_0', 'data_image/hurricane_irma/18_9_2017/909885632582787072_0.jpg', 'test', '15a09c30c979cb169c1d5b89bea2505cc3c973b72f00dfe4fc296ba10480560c'), ('910161511108300800_0', 'data_image/hurricane_irma/19_9_2017/910161511108300800_0.jpg', 'dev', '0b0a258e3498ce822ff52982a95f90a7dc4047cb6d08ab0449d2020647b34af7'), ('909813091470397441_0', 'data_image/hurricane_irma/18_9_2017/909813091470397441_0.jpg', 'test', '0fe5302578380ce4b9da358266c683e6970244a70d009ca42cf1ca08789bfc29'), ('905721215180042241_0', 'data_image/hurricane_irma/7_9_2017/905721215180042241_0.jpg', 'train', '370d4c836909bc86e598bcad7bc1a81ec97e8218943688858c57fb0ef2e36bd8'), ('911933848962953216_0', 'data_image/hurricane_maria/24_9_2017/911933848962953216_0.jpg', 'test', '0ec81b49e584e7b12a71a71b5fa3c24976b310f38499864a553ed86790c6ba5a'), ('923287403481223168_0', 'data_image/hurricane_maria/25_10_2017/923287403481223168_0.jpg', 'dev', '917df32eec0115794a3bd8b90957f78ef9db86dbc5a923503da212302c2cb055'), ('910639429664026624_1', 'data_image/hurricane_maria/20_9_2017/910639429664026624_1.jpg', 'dev', '74c47738f125250d20e56eb83a6b1b38c5fcd57e6567d6cf921f95b764f75dc2'), ('913416972129509377_1', 'data_image/hurricane_maria/28_9_2017/913416972129509377_1.jpg', 'train', '18b73069c2c44807db75f9de50fea022a31d80a86db8e003764eb74de6834d4f'), ('923717142587105280_0', 'data_image/hurricane_maria/27_10_2017/923717142587105280_0.jpg', 'train', 'e2ceaa8eb2737a15c8f12ef05bd36bc2f7014130a9d80b0b7cfe68294d563031'), ('911820532022046723_0', 'data_image/hurricane_maria/24_9_2017/911820532022046723_0.jpg', 'test', '576e6d2c2a90a2e4fe3e0f1fde8a347972c6923b12db2e2c8c2d5c00f1d97681'), ('927875856516689921_0', 'data_image/hurricane_maria/7_11_2017/927875856516689921_0.jpg', 'train', 'c7dcaf8d582b533904c12a2d1d4d467bf4ca78e91834483c92a3e9188e371053'), ('921802691587735552_0', 'data_image/hurricane_maria/21_10_2017/921802691587735552_0.jpg', 'test', 'a8af845cf8fe4379b06761d1740da6e612070c39c64a675046373eb81c016475'), ('928862568629669889_0', 'data_image/hurricane_maria/10_11_2017/928862568629669889_0.jpg', 'train', 'a4f6f367741ba0844958ca8968e6e81e13d2355e54cf07c1e8b6becb6151af98'), ('921914898665394176_0', 'data_image/hurricane_maria/22_10_2017/921914898665394176_0.jpg', 'train', 'ac87e444381fd5891b851a5dad2aee01509c2c71709b9be6d96e06263814c790'), ('922979668352958467_0', 'data_image/hurricane_maria/25_10_2017/922979668352958467_0.jpg', 'test', 'a523e66b870b2a30efe2877475a937388809e2620acb62bb05fd43681b7bb0fc'), ('914207898250735616_0', 'data_image/hurricane_maria/30_9_2017/914207898250735616_0.jpg', 'dev', 'c3a459531e779109b081106247271e0a24cf681675de4f3fe2d3123f057b3d49'), ('914867683023364096_0', 'data_image/hurricane_maria/2_10_2017/914867683023364096_0.jpg', 'dev', '1586a8531d59c761fbf905ce9a364eb540925e3774727e445f208b2004ad951d'), ('929160865672245248_1', 'data_image/hurricane_maria/11_11_2017/929160865672245248_1.jpg', 'train', 'b9373b5ac58cff0135023e1d4ccf8247613ca574bd422d7ec14c2e313babf5c1'), ('914854749245145090_0', 'data_image/hurricane_maria/2_10_2017/914854749245145090_0.jpg', 'dev', '8fa077e23875e1333616f0b5b934eaea28c18ddb33d46d42a568629e9f175516'), ('911926937148211200_0', 'data_image/hurricane_maria/24_9_2017/911926937148211200_0.jpg', 'dev', 'f9a9353f45543931738de0a788f07a29e6163fd8aadd68ecd90c25fd376ad98b'), ('911658201552101376_0', 'data_image/hurricane_maria/23_9_2017/911658201552101376_0.jpg', 'train', 'f704acf70ad515829a30b91c5b6f45e7b8b995f9e6b87a131476e72dc4a84ffc'), ('923261630313402368_0', 'data_image/hurricane_maria/25_10_2017/923261630313402368_0.jpg', 'test', '8b0f035b010848724a06dfab076bd3e6ceb04bdccd59182ac3c6572b29d8bbb0'), ('915973750276857856_0', 'data_image/hurricane_maria/5_10_2017/915973750276857856_0.jpg', 'dev', '7a13ec58a33e860190c25b0bb7dc63d7a5e1f768e79d270619263dea504f7ab4'), ('913727986368749568_0', 'data_image/hurricane_maria/29_9_2017/913727986368749568_0.jpg', 'test', '0ad4f19811abfd605b70c28b2bd1bc6df79ddb6bf598b6f4f8fc21d671fdf623'), ('916011739812188161_0', 'data_image/hurricane_maria/5_10_2017/916011739812188161_0.jpg', 'test', '1c20ee7ac121d623d197c4c10505f6e949fd7b33d698d6121efd253b36a29af9'), ('923635710564843520_0', 'data_image/hurricane_maria/26_10_2017/923635710564843520_0.jpg', 'train', '847023bce4df3d66a698705aa669876b04d0dd2c1b50a85bfb56f7d8ff1c4b78'), ('910809445692841985_0', 'data_image/hurricane_maria/21_9_2017/910809445692841985_0.jpg', 'test', '4e06b223009da7357062069893a8c7b4418a1c415a05222554fa1ffdaefda842'), ('930108299776856064_0', 'data_image/iraq_iran_earthquake/13_11_2017/930108299776856064_0.jpg', 'train', '15bf2a9fd3f721a8e7515b11e2be2ef54afc5abab7e1f8f10b2dafb72b0aff7f'), ('931103180741595136_0', 'data_image/iraq_iran_earthquake/16_11_2017/931103180741595136_0.jpg', 'test', '6635b001e1ca572b133f68792d99401f13ddb52f36aab64dd0f3007eb3d8b040'), ('929994458153803776_3', 'data_image/iraq_iran_earthquake/13_11_2017/929994458153803776_3.jpg', 'test', 'dd38c0b872db8eb0701ae5d68d4e47d10dd5e19a44014d05d2d69593abc7e409'), ('930086947674116097_0', 'data_image/iraq_iran_earthquake/13_11_2017/930086947674116097_0.jpg', 'test', '3238ffcdacd2ed4102651eb314505d1a12b703bdbd180a6c5cfc5726ad3c4592'), ('930440061644148737_0', 'data_image/iraq_iran_earthquake/14_11_2017/930440061644148737_0.jpg', 'train', 'aa0338fd8992be713acec345ffcf3865bff38b2d4004c6aa2e834061d3d3a15f'), ('930101728342675456_2', 'data_image/iraq_iran_earthquake/13_11_2017/930101728342675456_2.jpg', 'train', 'f79f0dac252366a70d1c944ba65e4c532239de5781fe486390dbab93fea73fa4'), ('930002970640388097_0', 'data_image/iraq_iran_earthquake/13_11_2017/930002970640388097_0.jpg', 'test', 'b40af0ae885e73bd2a9044cdf6636459273edeff494851424fbc63867fd4d4a6'), ('931499665635397632_0', 'data_image/iraq_iran_earthquake/17_11_2017/931499665635397632_0.jpg', 'train', '475e6c649c6b5eb013937adec3aac63d61af93ec78300ac1cfee2eb260adb4aa'), ('930026420746645505_0', 'data_image/iraq_iran_earthquake/13_11_2017/930026420746645505_0.jpg', 'train', '15bf1b2a0eb48d437beaccb09726f1b8c0081d0cc676a6940dfef850c09a79b9'), ('930832085375504384_2', 'data_image/iraq_iran_earthquake/15_11_2017/930832085375504384_2.jpg', 'train', '42d8a62dc51bd565d9dad6a91f425451c65f0bdee1e68c200b600efa4b4ffd44'), ('915214074945982465_0', 'data_image/mexico_earthquake/3_10_2017/915214074945982465_0.jpg', 'train', '3be2aebc1394312e922c8c7c77050aa7ad0b6971b92fcffcf2402497ba794dd2'), ('913533790437257217_0', 'data_image/mexico_earthquake/28_9_2017/913533790437257217_0.jpg', 'dev', 'bc7ba6848de39024c1e31ff1e10bc17efe3dcd2bba306eda23668f1ab08bdb71'), ('913735879931314177_0', 'data_image/mexico_earthquake/29_9_2017/913735879931314177_0.jpg', 'test', 'fcd7264fc199d6e601804e2dd54ce9c70d4148526d8af58f021d7923b5ecf988'), ('910658484257030144_1', 'data_image/mexico_earthquake/21_9_2017/910658484257030144_1.jpg', 'train', '39b8a80d4281d70b2eabe1853ad693ea6b0223a7d82a38f6ba6519810961cba1'), ('910525106329329664_0', 'data_image/mexico_earthquake/20_9_2017/910525106329329664_0.jpg', 'dev', '1444b6b15ab39ed3cc1d4da81102e479ea35621dea9f06af3b43bba89d3240bb'), ('912219901615472640_0', 'data_image/mexico_earthquake/25_9_2017/912219901615472640_0.jpg', 'train', '01f9e6831b4b3f1fb776b1222e2c26399b05ebe92782cc586481df177511276c'), ('913264857360867329_0', 'data_image/mexico_earthquake/28_9_2017/913264857360867329_0.jpg', 'train', '0b814370c9c0ea3ddf2b4aa862c91bbccdb444a44bf03db1e83a782eaae77be9'), ('910566762839543809_0', 'data_image/mexico_earthquake/20_9_2017/910566762839543809_0.jpg', 'test', 'afd6f0cfcc60de1a53c0135bf53ceae5758df4841da2d5e4b54e0d5bdbb401a8'), ('910631675113197569_0', 'data_image/mexico_earthquake/20_9_2017/910631675113197569_0.jpg', 'test', '6f99a96b2a5c29e457631bfde784e3235c87ea57888d0ee074ecebe4fa7ec1a2'), ('910722746551865345_0', 'data_image/mexico_earthquake/21_9_2017/910722746551865345_0.jpg', 'dev', '4e423c1aa962b6eb1db755b67ae3f4f8a26bbe32c850c015919b49ed0123c964'), ('912191868967907328_0', 'data_image/mexico_earthquake/25_9_2017/912191868967907328_0.jpg', 'train', 'fdae33315b9c2469582b0505f7c426672906d4ce81f35acedece2947117931c4'), ('910817025072812032_0', 'data_image/mexico_earthquake/21_9_2017/910817025072812032_0.jpg', 'dev', 'a15fbdd7e51c0c61348b82dfadca3b6639b19c95703c8b61d67324e130870475'), ('877877059162210304_0', 'data_image/srilanka_floods/22_6_2017/877877059162210304_0.jpg', 'test', 'b42b7a36772483f5b8e4a256ede361e5248ddc8b571671512e29622ef62d3925'), ('870258521635381248_0', 'data_image/srilanka_floods/1_6_2017/870258521635381248_0.jpg', 'train', '7dd7aa03d0ec9573be970039b0522a7c31aec856ecf9c402b6a23a8b5f4a1d02'), ('872486971796598785_0', 'data_image/srilanka_floods/7_6_2017/872486971796598785_0.jpg', 'test', '5ca49b6ac77dfb3bd60fb2fdd3d5ecd78b805dfe2d06b1f7b4d8602105e71146'), ('870145228077969408_0', 'data_image/srilanka_floods/1_6_2017/870145228077969408_0.jpg', 'train', 'bfa55c2a3f74fc9805ba3c81c5ec83de19541bbfc4717d45f2d6b7577f4f6f47'), ('877579878660534272_0', 'data_image/srilanka_floods/21_6_2017/877579878660534272_0.jpg', 'train', '99d4c58f90daa2ebe8e51edef3f84dcd90673e7c69869fee11e6a6009ec668ab'), ('873411718772215808_0', 'data_image/srilanka_floods/10_6_2017/873411718772215808_0.jpg', 'train', 'd990aa69656ba54cf102fcea2e5137555326fef7f21a74848f979895bb148ffb'), ('871649571058978817_2', 'data_image/srilanka_floods/5_6_2017/871649571058978817_2.jpg', 'train', 'c32c3a7db40dc6204a6e447f1310b43dd343f7ab1d59063350ea45b236309d49'), ('920352459297558529_0', 'data_image/california_wildfires/17_10_2017/920352459297558529_0.jpg', 'test', '56ab2ccffa2f984256e1ceeecfff6ef5f65640cb20b78f973b94e3134f46f0b5'), ('920421696527413248_0', 'data_image/california_wildfires/17_10_2017/920421696527413248_0.jpg', 'test', '42a2cefd928c4f30b71e9dec151d05f73748625d55340c6c6a693a98bffa13ff'), ('918315972976631808_3', 'data_image/california_wildfires/12_10_2017/918315972976631808_3.jpg', 'train', 'cab8bc6fa0bf21f0d811b9c16f21023c2022a976422329c10a17f7e0e8aef1f6'), ('920020172823515137_0', 'data_image/california_wildfires/16_10_2017/920020172823515137_0.jpg', 'test', '69bee673f201d9c2755e385415dfed6c154ed90706158f4fa86fbd30e5a1377f'), ('918242189607444480_0', 'data_image/california_wildfires/11_10_2017/918242189607444480_0.jpg', 'test', 'fe8b0f70eb9bedb8153ef1387bb178b054df79a69e7e827991a2bdd20073ce68'), ('920047820899434496_0', 'data_image/california_wildfires/16_10_2017/920047820899434496_0.jpg', 'dev', 'a407c910e182018bbb61d343c240bed819c6f90456706aa0e36bbbc8ad5f4ab8'), ('919583506891714560_0', 'data_image/california_wildfires/15_10_2017/919583506891714560_0.jpg', 'train', '5ebed3c6ab191f5914b69bad563d2f082a57209404c2b9ec37acfabb1eb89a25'), ('917822562259931137_0', 'data_image/california_wildfires/10_10_2017/917822562259931137_0.jpg', 'test', 'ace7d95ce9220c1167a9a9d583a44dcc6f1aa7cb46d0ff70e05147d64739d583'), ('921623155835224064_1', 'data_image/california_wildfires/21_10_2017/921623155835224064_1.jpg', 'train', '76331cc71b0afb52ed24780db07f7d6e86b648fb3faf13c6e34b4163269dcaee'), ('917856156059623424_0', 'data_image/california_wildfires/10_10_2017/917856156059623424_0.jpg', 'train', 'c8a6c562984e5f124e0847d13ed0810b77a7d436615de218371d50bdcb6f153d'), ('919649495268208640_0', 'data_image/california_wildfires/15_10_2017/919649495268208640_0.jpg', 'test', '8720fee1eca51af9d461278303e531c2f2f7edc8971a25113207e466785f6129'), ('917807529479901184_0', 'data_image/california_wildfires/10_10_2017/917807529479901184_0.jpg', 'train', '7b094e67a02270c168716792c24a3c7b0832b600f00aea600940142500c5a149'), ('920688180713664513_0', 'data_image/california_wildfires/18_10_2017/920688180713664513_0.jpg', 'test', '0e8f199a3ca9df78c3dabf70ef3d9c2211a5d6a57cee5c4c96c4e6f6d4e88d79'), ('917960931715330048_0', 'data_image/california_wildfires/11_10_2017/917960931715330048_0.jpg', 'train', '9cd5b417fa78c94a171ad51cc539ccc4544216b9be13648cea75d575cfe914ce'), ('917869868946296832_0', 'data_image/california_wildfires/10_10_2017/917869868946296832_0.jpg', 'test', '3d71c95d3a97c270903c8a6a63f4a35d00ae2daafe65c54daae7392ba04ec656'), ('908087000401670145_0', 'data_image/hurricane_harvey/13_9_2017/908087000401670145_0.jpg', 'dev', '294f3170985de018be6b46201b71f9be0a192f955cb2988ceb221a9d25d9a119'), ('906817848902868992_0', 'data_image/hurricane_harvey/10_9_2017/906817848902868992_0.jpg', 'train', '08de25310fb54a3a40ad7927c5c24c8e539bc3691790ec635a9a90d61300b3f4'), ('906183470153650176_0', 'data_image/hurricane_harvey/8_9_2017/906183470153650176_0.jpg', 'train', '58291a75135c732fd1537df7316be7e646de007cbc28684f20464a3bb472880a'), ('904919183762219008_0', 'data_image/hurricane_harvey/5_9_2017/904919183762219008_0.jpg', 'train', '220860adda01fe6dbd5dcef6a20e94342c1dc2ab426185fd4b5934bceb856319'), ('904449285831368704_0', 'data_image/hurricane_harvey/3_9_2017/904449285831368704_0.jpg', 'test', 'fb54aff719c20fa6264b9eec6d9b67005603f2c13c74bc95883c5a80c8b0ae09'), ('905883574653652993_0', 'data_image/hurricane_harvey/7_9_2017/905883574653652993_0.jpg', 'dev', '33ff7c8eab7ea57b6716794289e85752143bac8633ec5d1afea13052db97c08a'), ('907265058656407553_0', 'data_image/hurricane_harvey/11_9_2017/907265058656407553_0.jpg', 'test', 'ec4bfeed01cb606477aac422c57ac4dac29957f2e97a8c9b8708f1db04e2a6d0'), ('909892154893447169_0', 'data_image/hurricane_harvey/18_9_2017/909892154893447169_0.jpg', 'test', '2bba1f005cd81533d1e6ce058f81c7708eb39439c36da55f01b93b96e0bbba8a'), ('905562808166776832_0', 'data_image/hurricane_harvey/6_9_2017/905562808166776832_0.jpg', 'test', '5b2276635d5fc6de7364bb5f10d0e43aa7aa64c1b69ad4f709c45fdc21a71690'), ('901747052752674817_0', 'data_image/hurricane_harvey/27_8_2017/901747052752674817_0.jpg', 'dev', '28a6acde97c2e8a18921006ae186cdce4159eb22395a017c1f204ec6aab0b23c'), ('906491347078569984_0', 'data_image/hurricane_harvey/9_9_2017/906491347078569984_0.jpg', 'dev', '1084db4d590b2877831afffdae14763c2f7d862f1be3db4af4970df173232134'), ('908517505819389953_0', 'data_image/hurricane_harvey/15_9_2017/908517505819389953_0.jpg', 'test', '2db4109141344cf1483180d997ef02076cc577d40ebc325f62f82ace583fbb7e'), ('905014427451244544_0', 'data_image/hurricane_harvey/5_9_2017/905014427451244544_0.jpg', 'dev', '6b76b364c78b4fa928ee93773d25d211e057dab471c20ed32e5c782237568254'), ('904993803911790592_0', 'data_image/hurricane_harvey/5_9_2017/904993803911790592_0.jpg', 'train', '26c4f85bea8a6799247f1e388884b3ec3640ceb3922252d3926dc9bf8739dd6e'), ('907748189117304834_0', 'data_image/hurricane_harvey/12_9_2017/907748189117304834_0.jpg', 'dev', '8b28e8b71ac3a678a5dd0fac599b82b7ff1f2c6fdd627224cc07e1ca99785fc8'), ('908221252502003712_0', 'data_image/hurricane_harvey/14_9_2017/908221252502003712_0.jpg', 'test', '7a8ed775126f4b1e06e273bc87f1d4f32a9f5935b62d225e8846a4e17b0932ed'), ('905360560656498688_0', 'data_image/hurricane_harvey/6_9_2017/905360560656498688_0.jpg', 'test', 'a0f41e06ed595bf88e7529f6d503d82419a003c93ede57a6a6cad335cc0c25f3'), ('905035038575788032_0', 'data_image/hurricane_harvey/5_9_2017/905035038575788032_0.jpg', 'dev', 'c0f982d7238f0ab648c656755f1662332e84a176685563ff95b8cf906425ebf2'), ('904397952420118528_0', 'data_image/hurricane_harvey/3_9_2017/904397952420118528_0.jpg', 'train', '87237586188afc590f427fb91cba57e6eb5b0e396d867ff37d00d7bb3f5afaf2'), ('904375834680721408_0', 'data_image/hurricane_harvey/3_9_2017/904375834680721408_0.jpg', 'train', '44e416989c289bec51624a0f8917c7663543cab612153c8162d527c01801b974'), ('907343314067836929_0', 'data_image/hurricane_harvey/11_9_2017/907343314067836929_0.jpg', 'train', '2fc4224a211679bb74ce4268368d0cbe2e45e2006591fe1881720938ed669410'), ('907664850431741953_0', 'data_image/hurricane_harvey/12_9_2017/907664850431741953_0.jpg', 'dev', '0bc134abee1adc19f97a553551a28b97788375f4db92f6ffef495a7c6964d828'), ('909406889368276993_0', 'data_image/hurricane_irma/17_9_2017/909406889368276993_0.jpg', 'train', 'e0d44e4a2c1359ac38e94b569a73ad83954cde1ccb49be412dad2205e6b35c6f'), ('909421137590915073_0', 'data_image/hurricane_irma/17_9_2017/909421137590915073_0.jpg', 'test', 'f6d1a47d5054ad257315b9d66661d27945625f326c897cfe539e7467c0f208d1'), ('905664172809117697_0', 'data_image/hurricane_irma/7_9_2017/905664172809117697_0.jpg', 'dev', '08c2e5647b6655bdd1108af413911864827189a51f13f49d025a4c857039d8a5'), ('909418648996999168_2', 'data_image/hurricane_irma/17_9_2017/909418648996999168_2.jpg', 'test', 'b5398f947d529bd69b81eafd7341bcd4a046f10397c913d3acc2e14e8552c3e1'), ('905636875905507328_0', 'data_image/hurricane_irma/7_9_2017/905636875905507328_0.jpg', 'dev', 'b515665785dad8cf210ad61a88602287be8407336b8e11a7b2590f03b51b1089'), ('909452115218718720_0', 'data_image/hurricane_irma/17_9_2017/909452115218718720_0.jpg', 'train', 'ef6e6b4234f6832d6916108115efc67ec837939dfd99932ad8343dff58e3d301'), ('910255209041022977_0', 'data_image/hurricane_irma/19_9_2017/910255209041022977_0.jpg', 'train', '2dd41bc6be56300e90f0c6bb9ce5b8053eec5c9afe74538a4c1bbd3c923eaa92'), ('909882615829860352_0', 'data_image/hurricane_irma/18_9_2017/909882615829860352_0.jpg', 'train', 'c17a2c875776a425d13a34d8bb04d6cd5bbc42009165ccced820eb076d9fcd3d'), ('909759111713173504_0', 'data_image/hurricane_irma/18_9_2017/909759111713173504_0.jpg', 'test', '76f376471c93e9a0572bfd5f76abfa5d923a45b797af3f2cbffdf2ed9f562d59'), ('910143352120758272_0', 'data_image/hurricane_irma/19_9_2017/910143352120758272_0.jpg', 'dev', '3d2c2c9c057cee21dadc531f057bb26f6a5abe1d63c194e1be76add828fd0a83'), ('909862843947868160_0', 'data_image/hurricane_irma/18_9_2017/909862843947868160_0.jpg', 'train', '383e5e5c9a420d4a290c6dd8d6864ceff07ba188d0b802730f14cdbcae324a61'), ('909808487101878272_0', 'data_image/hurricane_irma/18_9_2017/909808487101878272_0.jpg', 'dev', '03386ad8fc220a8c03fd3e32c2a3873ea53833b85e0c537bb5868d95920f75fa'), ('906927979091152896_0', 'data_image/hurricane_irma/10_9_2017/906927979091152896_0.jpg', 'test', '1424936d555263ec994f88e38c784eb19c34f6350438ee2da4c8fc716398cb6c'), ('909394335992631296_0', 'data_image/hurricane_irma/17_9_2017/909394335992631296_0.jpg', 'test', '12fcee666e0b7562d503956b8588a538a46df5b6be4a108f2e9023b43ff25d0a'), ('910123853954846720_0', 'data_image/hurricane_irma/19_9_2017/910123853954846720_0.jpg', 'test', '5d20922f481167284fe579ddacad17773a7d890fe98a713a83d192494f3aec7b'), ('909839895077834752_1', 'data_image/hurricane_irma/18_9_2017/909839895077834752_1.jpg', 'train', 'a21bfe52d8b416a53efc913c2a7aa3e75885085dc410620974ff06724b2f80cd'), ('909477228668424194_0', 'data_image/hurricane_irma/17_9_2017/909477228668424194_0.jpg', 'train', '56700c810dc02b4e19a73ab982f74a5fc2d838d020d0eba4631010a1d81f9db9'), ('910091744917770241_0', 'data_image/hurricane_irma/19_9_2017/910091744917770241_0.jpg', 'train', '5858aa65c2d4bb9cee3d4738929848ec6d9ca3694235516599bcda2668c20e36'), ('909869095239274497_0', 'data_image/hurricane_irma/18_9_2017/909869095239274497_0.jpg', 'dev', 'fe4ff12c24a58a1ef7aa3ec5d7ac3c6660fc4a3697eabe214e402e7b0450929f'), ('910186325013344257_0', 'data_image/hurricane_irma/19_9_2017/910186325013344257_0.jpg', 'test', 'b232880c5f31356fe842edfaee7f402c74387e0589507cb2a936cb223f30937f'), ('914533356678860800_0', 'data_image/hurricane_maria/1_10_2017/914533356678860800_0.jpg', 'test', 'cf6b20f05a0f264e2339b5041448a6a5e2c965030491a0aee707e40e60d510df'), ('921403448632807424_0', 'data_image/hurricane_maria/20_10_2017/921403448632807424_0.jpg', 'train', '48101b7a9cd785d6af3fd398668ee9fc6a1073f9094d56c4f0d76deccab54ac1'), ('923285999777669120_0', 'data_image/hurricane_maria/25_10_2017/923285999777669120_0.jpg', 'test', '85bdf51990be7359eef7127d2e27b033da001159d573f2724bb76b9ea9c4f943'), ('914663754117386240_0', 'data_image/hurricane_maria/2_10_2017/914663754117386240_0.jpg', 'train', '80b61cde2e617f9557cd1731c3cab38d02ed4ecef601ef9decd0d546d5a4be11'), ('929408546122424320_0', 'data_image/hurricane_maria/11_11_2017/929408546122424320_0.jpg', 'test', '9e850ee447095903d08ebe57400ec73058c3a7be839a7fd18e44a8cf5a7e8103'), ('913939214747742208_0', 'data_image/hurricane_maria/30_9_2017/913939214747742208_0.jpg', 'test', '6687c2ed7c88df3f40f10894f4b63dedd636d77b761eeabd24d59f679726799c'), ('914977057859362816_0', 'data_image/hurricane_maria/2_10_2017/914977057859362816_0.jpg', 'test', '8d360397594f2cde79a6e28a29dd6c8c5a9257089c2186bc3f1d1882ce0bb496'), ('922588370215112704_0', 'data_image/hurricane_maria/23_10_2017/922588370215112704_0.jpg', 'test', '11777093a3de2cd603df7ba2b8e7b212833451a962511be46cca1a0079e58c9a'), ('923605681910849536_0', 'data_image/hurricane_maria/26_10_2017/923605681910849536_0.jpg', 'test', '6a371bd8da530c7b86f8f24493be6d3de8d0e2dd05ef76d66d4050531ccc158c'), ('921304553840152576_1', 'data_image/hurricane_maria/20_10_2017/921304553840152576_1.jpg', 'test', 'd8f27921a291c1c8b5bbb92b740a8654c6cca98689144e72dc13e2e92b463bef'), ('913047030225915904_1', 'data_image/hurricane_maria/27_9_2017/913047030225915904_1.jpg', 'train', 'f5b1082509ada6b038bff171682f9acfe4df7ade7dddda1fae324b2c5ad5a109'), ('914113249934811136_0', 'data_image/hurricane_maria/30_9_2017/914113249934811136_0.jpg', 'train', '009009ab557eb3fd13e53cd2152ad3a94f561aca0d032521d67a4929718867f4'), ('927221367543607297_0', 'data_image/hurricane_maria/5_11_2017/927221367543607297_0.jpg', 'train', '4398f5059bddae1b3aa122031a0d48bea22cbe1eccd9b1c2ab389e94ea3107ed'), ('912325883683377157_0', 'data_image/hurricane_maria/25_9_2017/912325883683377157_0.jpg', 'dev', '3a62bfdf4ead48731fd56f543c9dcbb9979bf1ac50ef8fef81244c2ad1c38136'), ('913397093850603520_0', 'data_image/hurricane_maria/28_9_2017/913397093850603520_0.jpg', 'test', '2a94330e292982e34e4095f13dc41d1346e9f43a67eb2e11dd08ec54dc615225'), ('912968163553366016_0', 'data_image/hurricane_maria/27_9_2017/912968163553366016_0.jpg', 'train', 'c890cec060c405466bdd2e32fbbf9e0c26088ec8eb9f22e4b4cdc093f331a673'), ('914175986916261889_0', 'data_image/hurricane_maria/30_9_2017/914175986916261889_0.jpg', 'train', '75e4d61eb785f5bcd1f2b5c1f433f7077958948bb200bbf24d504ec480c5a57e'), ('921370720931684357_0', 'data_image/hurricane_maria/20_10_2017/921370720931684357_0.jpg', 'dev', '81e43e8c2261b0d7a378f970d63a594c0de082d7f8ba4d535de9a6eeea4f93ca'), ('912154071905263618_0', 'data_image/hurricane_maria/25_9_2017/912154071905263618_0.jpg', 'test', 'e0c43b50ed69c0ef85dd5ccc0d4d82c0d587e6263be20367385bcc538dc78e86'), ('922677168533934080_0', 'data_image/hurricane_maria/24_10_2017/922677168533934080_0.jpg', 'test', 'e82304692560648a8a3878849d156a28622e9dc34c704502879f9821fc361d19'), ('923213239508488194_0', 'data_image/hurricane_maria/25_10_2017/923213239508488194_0.jpg', 'train', 'a76e99a42a58939190a5692f9826b78c68f0f1c14a485d3d0795b2862404cfe9'), ('931925410224529410_0', 'data_image/iraq_iran_earthquake/18_11_2017/931925410224529410_0.jpg', 'train', '1cf183bf29e3b9f22ef6852e0a3cb200fb54374585d403fd4e60fbfd80ed530c'), ('930050908582563840_0', 'data_image/iraq_iran_earthquake/13_11_2017/930050908582563840_0.jpg', 'test', 'abd0ecffbbe4957558528b299194a79ce658b852ea4f6b742b57a15a69b0c958'), ('929994774819479552_0', 'data_image/iraq_iran_earthquake/13_11_2017/929994774819479552_0.jpg', 'test', '7a07b380a45855dae952b5ad084b491ba10acd2fe350a66beedc3a4fce5cee34'), ('930303907825373184_0', 'data_image/iraq_iran_earthquake/14_11_2017/930303907825373184_0.jpg', 'dev', '32c8592400db68da053b14f0ab2c3a0e0a78dd2b495c8cb732cc131b5788eefb'), ('930139458569203712_0', 'data_image/iraq_iran_earthquake/13_11_2017/930139458569203712_0.jpg', 'dev', 'e9e0d9351258ec62f79987876f0609f6aa2e94b00d844a4551b1d123b73f9bbd'), ('930331158604328960_0', 'data_image/iraq_iran_earthquake/14_11_2017/930331158604328960_0.jpg', 'dev', '4dac9ed91f6d277fe0ac5262117c364e1fe966d0dc2d1e7e918ece6d525cd4c8'), ('930268238075674624_0', 'data_image/iraq_iran_earthquake/14_11_2017/930268238075674624_0.jpg', 'train', '06c8b4edf1c905a01b46cbf4aab34d5f4f50c368aa6f71a95355d321bf543d5e'), ('930076476569018368_1', 'data_image/iraq_iran_earthquake/13_11_2017/930076476569018368_1.jpg', 'train', '3b5e09caad6042b648dd8e57a745e284e1049f95e8f26a10eba4236f37afd125'), ('930088679141527552_0', 'data_image/iraq_iran_earthquake/13_11_2017/930088679141527552_0.jpg', 'train', '187c8110fe98ffedc87b48f482db2000679f3944df67738063c91f9257c33191'), ('931771549685899264_1', 'data_image/iraq_iran_earthquake/18_11_2017/931771549685899264_1.jpg', 'train', '4a2b552b6acc9899b45063f0000159c711d68f56776da104deba11354c8cb540'), ('911715482537005057_0', 'data_image/mexico_earthquake/23_9_2017/911715482537005057_0.jpg', 'train', '2cd7c135ec3751b70f9362ccc5fcb3f92521988fc5b9ba27f0cb04988b0a35c6'), ('910700764808564736_0', 'data_image/mexico_earthquake/21_9_2017/910700764808564736_0.jpg', 'test', '1113391d1e05d19632a922f02376a09fbdd832c6ca21c679fc9762eb67636f02'), ('910524100367413249_0', 'data_image/mexico_earthquake/20_9_2017/910524100367413249_0.jpg', 'test', '9d83a32607d23c99088648124760a3459e83ba20e343c97712c93d5ee537089b'), ('910537123060764673_0', 'data_image/mexico_earthquake/20_9_2017/910537123060764673_0.jpg', 'dev', '3956c8e264cc850326d30d7bfbc2bddfe5e03572b067242cdf01b1abcd1d81bb'), ('913786483550638080_0', 'data_image/mexico_earthquake/29_9_2017/913786483550638080_0.jpg', 'train', 'f0cbdb495cef57c5ffff03e27118c6578201cda0b009aa725aa02c162c48251f'), ('911769343586979841_0', 'data_image/mexico_earthquake/24_9_2017/911769343586979841_0.jpg', 'train', 'ac12761d93ac369d6aa9bfaf39704e6711ae57589b48001b5854d11f18ad747d'), ('910745969163161600_0', 'data_image/mexico_earthquake/21_9_2017/910745969163161600_0.jpg', 'dev', '767f7c80a1b758472c70a2c2811316ecd01569a8b821d756ccf40c2d9022244b'), ('910541823981916160_1', 'data_image/mexico_earthquake/20_9_2017/910541823981916160_1.jpg', 'train', 'c9177ba454014c20723903c26aa7dc911ddcfef2ac0e610a3006d7f766436ebe'), ('913920723579629568_0', 'data_image/mexico_earthquake/30_9_2017/913920723579629568_0.jpg', 'train', 'a90de9b9c14e4bb6bcbe6b2a24439b8a11b4fbd3fc5080faba820a537d0002ff'), ('912221649637986304_0', 'data_image/mexico_earthquake/25_9_2017/912221649637986304_0.jpg', 'train', '2c364e2ffa80dc29fd6dc95bbdd9a589ab7fd7b38146cb468639dbe973b6700a'), ('911632341197606914_0', 'data_image/mexico_earthquake/23_9_2017/911632341197606914_0.jpg', 'test', '32018733be1c560e4493b45f041df50b04fe257aedbd0ae5342e19db20e2b431'), ('869988473419309059_0', 'data_image/srilanka_floods/31_5_2017/869988473419309059_0.jpg', 'dev', '6c565e4fc4fab6baedc913fee96e9ed68bb9098bf4a01ba15459b9bf5e15d435'), ('869957461993635840_0', 'data_image/srilanka_floods/31_5_2017/869957461993635840_0.jpg', 'dev', '5193dfaa74f7c1a80585f964641bc5874127084b35a7a97fcd5deb997374d987'), ('870146384904761344_0', 'data_image/srilanka_floods/1_6_2017/870146384904761344_0.jpg', 'test', '336f5f6e44ebfb365d649ad8fc835a0963814036fc47982288b79b16d8c0cd31'), ('871195505241587712_0', 'data_image/srilanka_floods/4_6_2017/871195505241587712_0.jpg', 'train', 'eff6ba32c0ca7dc5d9cb252c8dacd2f2f28916ed4126cd5b0a37ef2701553c1c'), ('877528637653696512_3', 'data_image/srilanka_floods/21_6_2017/877528637653696512_3.jpg', 'train', '2c6ab6bbf70827850c95af004d58d137843a40090b3f97a6b291d6ff20f720ff'), ('873258014953938945_0', 'data_image/srilanka_floods/9_6_2017/873258014953938945_0.jpg', 'train', '11131ffe2ee2e0c868cf0b8c4d1414892703636b99ed96a0fa93442fadac85d7'), ('876640890663522304_0', 'data_image/srilanka_floods/19_6_2017/876640890663522304_0.jpg', 'test', '04118c9e3d9a516823e28c11989bec5bdd0f6922b9059099506db29bb2960787'), ('922144749632479232_2', 'data_image/california_wildfires/22_10_2017/922144749632479232_2.jpg', 'train', '8a3988f0cdeedbc075992ea265068cce9132ba0b30f8a6eca5869bd499ed065a'), ('918077527226118144_0', 'data_image/california_wildfires/11_10_2017/918077527226118144_0.jpg', 'test', '6200ad4f2456aae0331d9f52b3a80209379913462256d0131d4f425ffabded63'), ('919655873001136128_0', 'data_image/california_wildfires/15_10_2017/919655873001136128_0.jpg', 'test', '08265449bc81c6f4ac4914df351c0e75d2d8ca50492a92ed96d88a26d2e130e7'), ('918354932977324033_0', 'data_image/california_wildfires/12_10_2017/918354932977324033_0.jpg', 'dev', 'c0dbc0c3ed9914b023cd9e571a5402f194313f4328b29d605f6ef9f894790704'), ('917987028423331841_0', 'data_image/california_wildfires/11_10_2017/917987028423331841_0.jpg', 'dev', 'e0d4558dbedb3686f085943d8583f6be82062169380911f7dffe463bd8bb4570'), ('919614623405559808_0', 'data_image/california_wildfires/15_10_2017/919614623405559808_0.jpg', 'train', 'd95409f6745e022e48c9fc5fb371d78785557f1b484566aabe70ab9383a6cd2f'), ('921468751463288832_0', 'data_image/california_wildfires/20_10_2017/921468751463288832_0.jpg', 'train', '970952c24d6fccc738ed2e3d08f0fddcf15318b8668f3e62e7030aacaeb499f7'), ('921486417267757056_0', 'data_image/california_wildfires/20_10_2017/921486417267757056_0.jpg', 'train', '15d3aab953b909c72f17df53a453ecb59c1b57b6cbc1daec53fa1a3d23354824'), ('918354529153880066_0', 'data_image/california_wildfires/12_10_2017/918354529153880066_0.jpg', 'test', '17520d4350ec4d8eab1bb2aec528399c4fcd92e21c89e81c595c65f7447c4a25'), ('920368722832916486_0', 'data_image/california_wildfires/17_10_2017/920368722832916486_0.jpg', 'train', '9cd44f53624fcf7d27d65bf5023c414951f31a33c351058db51cd002b0947479'), ('922852900837253120_0', 'data_image/california_wildfires/24_10_2017/922852900837253120_0.jpg', 'train', 'afe4c5738cbb9cac0f936f97602af02fbd359f56eeb866525c310238cfec3298'), ('919679214260953088_0', 'data_image/california_wildfires/15_10_2017/919679214260953088_0.jpg', 'dev', '06d07a84070d3c31099cbdad9b9a4119662383502704dc0a6c4a49a1ef64693d'), ('921099713490903040_0', 'data_image/california_wildfires/19_10_2017/921099713490903040_0.jpg', 'train', '860f67a4e2f4aa99fddabf981ff27f246862bc1bf01f9bc8c03b7f0d9d32c797'), ('920831780629782528_0', 'data_image/california_wildfires/19_10_2017/920831780629782528_0.jpg', 'train', '61ab6f5d2b32ec982ee36e9b24b4667e997840f01af1add48846918807697ecc'), ('917925914914295808_0', 'data_image/california_wildfires/11_10_2017/917925914914295808_0.jpg', 'dev', '641b12312e74cf6f8c9730b7eb95fd0988560565dea71f272bdfe84653822501'), ('909357422132809729_0', 'data_image/hurricane_harvey/17_9_2017/909357422132809729_0.jpg', 'train', '5ebe00c2e9280faf56db7c79621e1e10627e04527f697f38468f47a117236270'), ('907933851300847616_0', 'data_image/hurricane_harvey/13_9_2017/907933851300847616_0.jpg', 'train', 'a19e440c422fe27d12e33cbc915dcf269860a5dbb9e864d6a2c7a0a09f377401'), ('905844167774879744_0', 'data_image/hurricane_harvey/7_9_2017/905844167774879744_0.jpg', 'train', 'c03f8602e5209a10ccc2e683ebf670023eb8a5c9040735dcc69d77bb823defe3'), ('906744961072291840_3', 'data_image/hurricane_harvey/10_9_2017/906744961072291840_3.jpg', 'train', '433cf47c974a9168537fcea6c4f3afafeb9d0a9dbd3db9e8b38a3f6bf194a09f'), ('910160788849008640_0', 'data_image/hurricane_harvey/19_9_2017/910160788849008640_0.jpg', 'train', '2090fbd852409237c9c117ff136edc7fc3e0575305f284c0dee4809538a9d47c'), ('907779904254734341_0', 'data_image/hurricane_harvey/13_9_2017/907779904254734341_0.jpg', 'train', 'd81d2a2cd0ce5d5471774fd429ab6edf3860b4df6866dde26b16ac6767253dbf'), ('905793790081945601_0', 'data_image/hurricane_harvey/7_9_2017/905793790081945601_0.jpg', 'train', '06838bbceb6636a1543407cd5b6245d15221d769dd57dda100c73cce4963b62d'), ('905160883122331648_0', 'data_image/hurricane_harvey/5_9_2017/905160883122331648_0.jpg', 'train', '62d691821bb9ef920eb9af391efb4edafd8c8e2ff759d614b2af167679bd08ca'), ('904221556938825729_0', 'data_image/hurricane_harvey/3_9_2017/904221556938825729_0.jpg', 'train', 'c5780fed8e973b68e11d5efdd37bc40d53e169c71115d51cbb5aa654d2fce8a8'), ('905128208449757184_0', 'data_image/hurricane_harvey/5_9_2017/905128208449757184_0.png', 'train', 'ae1b5f7c0fd7290f0ee8bd0b997b4e5a06a5b2c276c51ce3db40cfd935dcced2'), ('907260858606260224_0', 'data_image/hurricane_harvey/11_9_2017/907260858606260224_0.jpg', 'train', '3b759224ad91d915619f63a038b171bfba7925455289f3e418b5e8b6f1458db6'), ('901731017358139392_0', 'data_image/hurricane_harvey/27_8_2017/901731017358139392_0.jpg', 'train', 'afd3d775c52c8fec38fda6928b4e6387e3e90f368de1c97b249fe8e0c4eafce4'), ('904248225703616512_0', 'data_image/hurricane_harvey/3_9_2017/904248225703616512_0.jpg', 'dev', 'f95f3788b7564fb2cbe6cb15d0b2d1dfb4c1d1f7b506ebe43bfe5d392ac718be'), ('909399962923995136_0', 'data_image/hurricane_harvey/17_9_2017/909399962923995136_0.jpg', 'train', '94c2bc01d4b9588900f6916d301062ff463fd3a0569ff664df06d1e90dc205cf'), ('907700633297412097_0', 'data_image/hurricane_harvey/12_9_2017/907700633297412097_0.jpg', 'dev', 'f9105361392a5d337d3c905ce3355fe413b22a15f40318528fd8ae0c098a676a'), ('905470345175556096_0', 'data_image/hurricane_harvey/6_9_2017/905470345175556096_0.jpg', 'train', 'e5be497e0a614ae75773332e9ab93b36e7be4c7508af791c7deb2db1aacfcca0'), ('906107025876287488_0', 'data_image/hurricane_harvey/8_9_2017/906107025876287488_0.jpg', 'test', '89d0f2fe48a0dbec1b0711a432586b410852d016a256009932007bbfaaf645d2'), ('905927183193042944_0', 'data_image/hurricane_harvey/7_9_2017/905927183193042944_0.jpg', 'dev', '64d6619d6ece5dee23cbbd663c040a0df7d547acf479eab0c773959d5f788542'), ('904285079857487872_0', 'data_image/hurricane_harvey/3_9_2017/904285079857487872_0.jpg', 'dev', '19dbc0c8acabeddc5078f92781b9296e013cb953d31d49051eacffe19250e257'), ('904311772294807552_0', 'data_image/hurricane_harvey/3_9_2017/904311772294807552_0.jpg', 'train', 'fe73be44aea8dd1035a1c3964d078e2df5aa6eb88fbdd8ccfa02f2dc95f8ff42'), ('909787366843445248_0', 'data_image/hurricane_irma/18_9_2017/909787366843445248_0.jpg', 'train', 'cc7220e371ee5a2782cdbed6d487b121a412ec2ac96a9d88203fd6c1aecd0747'), ('909486244689825792_0', 'data_image/hurricane_irma/17_9_2017/909486244689825792_0.jpg', 'dev', 'fe77d70e42ad0c28a20e68ec8cd8f7fc8221028fddd5aa4bf915977330030d98'), ('909890256299745280_0', 'data_image/hurricane_irma/18_9_2017/909890256299745280_0.jpg', 'test', '6925f22c4498cbbf8364804acff0c695a5c9d13822d1602e17e58e491d598b94'), ('909865900706279424_0', 'data_image/hurricane_irma/18_9_2017/909865900706279424_0.jpg', 'train', '44bef549aaea96c906b542ede3f0465d297e635b54d8ebb8ca7f92e098023aa2'), ('910201924087107585_0', 'data_image/hurricane_irma/19_9_2017/910201924087107585_0.jpg', 'train', '0efb4583411472eb99763146aafe756d766f8bbd6bcd6dd8fd921d3b7b293bee'), ('909490898706329600_0', 'data_image/hurricane_irma/17_9_2017/909490898706329600_0.jpg', 'train', '7d49286b2ad76dac0286c2913433ae06ca81aff1783e4e09194b0ad667e24400'), ('909383722742554625_0', 'data_image/hurricane_irma/17_9_2017/909383722742554625_0.jpg', 'train', '069138c0d460f0e32180dc0c0f5461cbdcef46960b147ec694678222fa31f00b'), ('909689379500806145_0', 'data_image/hurricane_irma/18_9_2017/909689379500806145_0.jpg', 'train', '664a01c54bcbbdc5566c80b96cd6806fab97905dc6a212a4534b3f6a1712bf0a'), ('909377668671565825_0', 'data_image/hurricane_irma/17_9_2017/909377668671565825_0.jpg', 'train', '32890209a74a5f47e0bc13970463f483d18621e3e0d14e25c091901a277c7bd1'), ('910218504380538881_0', 'data_image/hurricane_irma/19_9_2017/910218504380538881_0.jpg', 'test', 'b212fafce9714421d1e304fde668accae6f7324ecaad5cb547b88989ddee9166'), ('909774325305225216_0', 'data_image/hurricane_irma/18_9_2017/909774325305225216_0.jpg', 'train', '49c9cab826ac257e3c71b871beb40c0cd9d1a1bc9a3653b1cd9f526463df5ba9'), ('909846121169801219_0', 'data_image/hurricane_irma/18_9_2017/909846121169801219_0.jpg', 'test', 'cf2480664252ba9c6384e760cb5ce1dd4028ec8e7e4674adba5836a113353702'), ('905709911757160449_0', 'data_image/hurricane_irma/7_9_2017/905709911757160449_0.jpg', 'dev', '473a77f629ce104fbc67a0ac1674cb484fae78a9ce9e3a90c888f2ffc43b2473'), ('909837541205008384_0', 'data_image/hurricane_irma/18_9_2017/909837541205008384_0.jpg', 'train', '50a54db4177f6e5eb1ebd7dfe79583415fc51f3e7c02c5707c8c027d87c1b99c'), ('910207622896287744_3', 'data_image/hurricane_irma/19_9_2017/910207622896287744_3.jpg', 'train', '4cbc76fe4061b387a55ef856646d1e7d2e0fc551209b5091f0ab244e3572263d'), ('910151391842205696_0', 'data_image/hurricane_irma/19_9_2017/910151391842205696_0.jpg', 'dev', 'a99230faab189466ce303b3fdb8880956ddf06d857289682928250dd8753eec4'), ('910199813727297537_0', 'data_image/hurricane_irma/19_9_2017/910199813727297537_0.jpg', 'test', 'dc349d87ff6eaa1f90490d316ad49e36644490b969c802fd7c1c87e0bf15d878'), ('909998020003057664_0', 'data_image/hurricane_irma/19_9_2017/909998020003057664_0.jpg', 'test', '003a9d328a4261eb0423cbf75ec76c6504f29670887d05a04b9c64ec7ba8ecca'), ('909828132407930881_0', 'data_image/hurricane_irma/18_9_2017/909828132407930881_0.jpg', 'dev', '8c1225f1f7acd3b0a6086c5bf4075e1663f496c74be6d29c1d5913547a7618ec'), ('913406753324376064_1', 'data_image/hurricane_maria/28_9_2017/913406753324376064_1.jpg', 'train', 'efb519adcad9eb645e2b21c10ad34088c6cc53cdd4b5c9a3bcfafc77c0de023b'), ('915020169554128902_0', 'data_image/hurricane_maria/3_10_2017/915020169554128902_0.jpg', 'train', '3f9c300483a908eab49245150b50c27a75876eae5ee30886c786a3cbba96cff1'), ('913992284986855424_0', 'data_image/hurricane_maria/30_9_2017/913992284986855424_0.jpg', 'train', 'cf9c7a83ee7e2c14ff97d669d6f23b48d813cf7b32f15eb484cc3544702b3c2c'), ('917920718603673601_0', 'data_image/hurricane_maria/11_10_2017/917920718603673601_0.jpg', 'test', '26b4efb9684833744abff5e829d2aed24013dc2c61420a38e04b188a3bea5f8b'), ('923262758992465920_1', 'data_image/hurricane_maria/25_10_2017/923262758992465920_1.jpg', 'test', 'f0f04f4d471bbc98db15d5c4c5aba53c6297a9193a5bd1e88525f8b9771fa04c'), ('914353354503069696_0', 'data_image/hurricane_maria/1_10_2017/914353354503069696_0.jpg', 'dev', 'acb1aa14b96426c391c16d01006fa632fa7c82d9a85180d0ebfe3aa6c73206db'), ('927261500418482177_0', 'data_image/hurricane_maria/5_11_2017/927261500418482177_0.jpg', 'train', '9c62831fda88f9c7fc81f45ddf876c1dd9c9986749dae73bd28265ec11b51c80'), ('928103565859713024_0', 'data_image/hurricane_maria/8_11_2017/928103565859713024_0.jpg', 'train', '06b616c691c7cc12f8b12d3cde001f93eaadb53220be2970dca16c9447172ec1'), ('912075698617102340_3', 'data_image/hurricane_maria/24_9_2017/912075698617102340_3.jpg', 'train', 'eb54c51b3d84091c7a9c2468d3ddba36e64dc04af790ab59c799ce5b1be60644'), ('916291860162859009_0', 'data_image/hurricane_maria/6_10_2017/916291860162859009_0.jpg', 'test', 'eb069efbc70860bf94e2843533b4fe7798983a8bc1cf4d241bb9fe0b71fed33b'), ('914261767144050688_0', 'data_image/hurricane_maria/30_9_2017/914261767144050688_0.jpg', 'train', '45c6a85f0cee9cb372471693ad974d5e68ba6ce9db52e2dbc737a1beea33cfb6'), ('922108041738293248_0', 'data_image/hurricane_maria/22_10_2017/922108041738293248_0.jpg', 'train', 'b1d365ca1e37cda18ff95e61d3533eeb3f94894b85ba5086837adb13e37434bd'), ('928393757761982465_0', 'data_image/hurricane_maria/8_11_2017/928393757761982465_0.jpg', 'train', 'dce246225397e308aa7fa6fef08b2a1448c9deac673e668f9679ad1d815c6da8'), ('914871738722390016_3', 'data_image/hurricane_maria/2_10_2017/914871738722390016_3.jpg', 'train', '558359f159a9fd3234a57d5c6c10bffe3bbe43999ec0025ce94e17d79946d081'), ('910584414496743424_0', 'data_image/hurricane_maria/20_9_2017/910584414496743424_0.jpg', 'test', 'e7248298c265ca26a27ac6a69d25ac70fa326a91e60ef6e292c5a18f4b392968'), ('914444286548115462_0', 'data_image/hurricane_maria/1_10_2017/914444286548115462_0.jpg', 'train', 'ef89d00fcd3750d3ac44a7b31a73adaab89e21f60b8d224fe462f7764947b5a8'), ('912249082097356800_0', 'data_image/hurricane_maria/25_9_2017/912249082097356800_0.jpg', 'dev', 'e9dc5b15a4c764e28e54c6eb7bb4e6ebfc0ca02481fbdc88c0226612bd26eb8b'), ('913176642889109504_0', 'data_image/hurricane_maria/27_9_2017/913176642889109504_0.jpg', 'dev', '11344112191bb4bd7bea496f01d0381608dd5a4606c038ba79241ff761c45c9d'), ('914240939974991872_0', 'data_image/hurricane_maria/30_9_2017/914240939974991872_0.jpg', 'train', '71f43f0c73c020cf80311742029fb2a569b9c98cff94bf07fa875f974424e3b7'), ('916160678553104384_0', 'data_image/hurricane_maria/6_10_2017/916160678553104384_0.jpg', 'train', '5262e577c79d493e3005ed4723f5bdc40e8aa90cfff0dbe10cc656317ce9c225'), ('916061560933638145_0', 'data_image/hurricane_maria/5_10_2017/916061560933638145_0.jpg', 'train', '656e6dd29ab850c3b081653be76d80a6a9472d5895748d6d641d6b1a66d2b1d4'), ('930058600160088065_0', 'data_image/iraq_iran_earthquake/13_11_2017/930058600160088065_0.jpg', 'dev', 'b7738ad2cbcf25432b6bad9ff47419cd3ce2cd047d207487a96b1fba743a3402'), ('929992723146399745_0', 'data_image/iraq_iran_earthquake/13_11_2017/929992723146399745_0.jpg', 'dev', '8330a394f4cb3941745bea4f26f3894dd5a8cb7cd91a011ebe4f4f65f1cfbd32'), ('930493617537454080_0', 'data_image/iraq_iran_earthquake/14_11_2017/930493617537454080_0.jpg', 'train', '1afb9d3a34e4d62c2121623b5ef28fa1a0d57a2ffa9e6416da8a8fa5c7fd7a69'), ('930022107278381057_0', 'data_image/iraq_iran_earthquake/13_11_2017/930022107278381057_0.jpg', 'train', '36b459292322ff8df9ead993b48f2bca04cad1195a0143b8ad6188dc2e8c8c02'), ('930832085375504384_1', 'data_image/iraq_iran_earthquake/15_11_2017/930832085375504384_1.jpg', 'train', 'e40c62f33c7ffebe07c9994e5d815c6adc819d9cf88eb2f0659dc5e0bea2b712'), ('931486388104679424_0', 'data_image/iraq_iran_earthquake/17_11_2017/931486388104679424_0.jpg', 'dev', '10f6eb98a1ab58f560e4bce93d6a5fccaa4447dea92f9bf2cae590fce41f159b'), ('930109831595417600_0', 'data_image/iraq_iran_earthquake/13_11_2017/930109831595417600_0.jpg', 'train', '2745bf810684a32aef4bf8093bfcf44d3307a1f49d45a7b9a5725de93af95800'), ('930332327967969285_0', 'data_image/iraq_iran_earthquake/14_11_2017/930332327967969285_0.jpg', 'test', '5a3bdb37c939b1e8e4be284404d9f1029766fdb1c56897b618d730f6f0f26a24'), ('930058600160088065_3', 'data_image/iraq_iran_earthquake/13_11_2017/930058600160088065_3.jpg', 'dev', 'fecb064672311eee6338d168b6c70889c40622e6bed06d9e33cc33156c5fa4ab'), ('911734518897311744_0', 'data_image/mexico_earthquake/23_9_2017/911734518897311744_0.jpg', 'train', '8dd889ea52e3f4eda0725078139380336888825b1e0e5f7a2b23ac028cd1dd13'), ('910567262834139136_0', 'data_image/mexico_earthquake/20_9_2017/910567262834139136_0.jpg', 'dev', 'a53bbe883ebbc73049f7dae48454253b817be3a65eeee5ef0aa389a72ab5e555'), ('911854728954486784_0', 'data_image/mexico_earthquake/24_9_2017/911854728954486784_0.jpg', 'train', 'adc05c4f9de1467f37713e3054eaac5b17969602bbae6690f56f53707248e8c0'), ('910739720065384449_0', 'data_image/mexico_earthquake/21_9_2017/910739720065384449_0.jpg', 'dev', '8e7e08eabfca63e11de539362ca16832f1b85356c1edcedebe58be63480a5dc3'), ('910695323273641984_1', 'data_image/mexico_earthquake/21_9_2017/910695323273641984_1.jpg', 'train', '30cd2c8f1453ecf9164cf966096703be3667a9ffc43f8b703226ebb23263e373'), ('913522000890441728_0', 'data_image/mexico_earthquake/28_9_2017/913522000890441728_0.jpg', 'train', '3a133e4e354ea0d7144a398f2d684cffa15606f1187992efb9bd95139f210c99'), ('910682398907617280_2', 'data_image/mexico_earthquake/21_9_2017/910682398907617280_2.jpg', 'train', 'bd6ec334f18cb4010e7e21febb817ee454691c155c9ab49b6f594667c653b4dc'), ('911987214858498048_0', 'data_image/mexico_earthquake/24_9_2017/911987214858498048_0.jpg', 'train', '31ba3c747ed47b49ad09c5789f038d798200545403673aa724b672367f8162e6'), ('914993292290469888_0', 'data_image/mexico_earthquake/2_10_2017/914993292290469888_0.jpg', 'train', '6313faeec6e695150dd13b4d9c640943fe0a79480f6b08b2a9fe4846bb1c1a02'), ('912998714121302016_0', 'data_image/mexico_earthquake/27_9_2017/912998714121302016_0.jpg', 'train', '82260327093fa766ee10eff56d932211c157aefea7a4ecbc9eb6b939f5acbb01'), ('911622991288700933_1', 'data_image/mexico_earthquake/23_9_2017/911622991288700933_1.jpg', 'test', '2d4f4e190a6ec23b4a7eb9693485a57e2d947071137f26c684ac502a09f77f7a'), ('870168489759199233_0', 'data_image/srilanka_floods/1_6_2017/870168489759199233_0.jpg', 'train', '232e49c427355cc0999797e048626bc752c79fb98fbca3d549a9b3e3448690b6'), ('870248079043158017_0', 'data_image/srilanka_floods/1_6_2017/870248079043158017_0.jpg', 'train', '9f3a646eeadfc3b0a6060125c42e0f2b0f39eb75f9d38eb8931fedc36767bfbc'), ('870331956415909890_0', 'data_image/srilanka_floods/1_6_2017/870331956415909890_0.jpg', 'train', 'a89ddb158386e3c801c410f89f65ea7f2d7f743030cad87803f43ee3b58db553'), ('870261972603772931_0', 'data_image/srilanka_floods/1_6_2017/870261972603772931_0.jpg', 'dev', '6620cf28dc25ae1f66902508fd9add70849322279f59f5894039c9d8db127bcb'), ('877278276896423936_1', 'data_image/srilanka_floods/20_6_2017/877278276896423936_1.jpg', 'train', '99aa000689a45e80b2f43f8d039f43529c8ea81fe4eb1b266fc22bc400987747'), ('870117818687160321_0', 'data_image/srilanka_floods/1_6_2017/870117818687160321_0.jpg', 'dev', '066bd35038dd09de94a77207d8e9f11d3fc3dd8fc0dd09d215891955e2e989b0'), ('874121599003185152_0', 'data_image/srilanka_floods/12_6_2017/874121599003185152_0.jpg', 'test', '0faa8d388d6b063c365e144491bedc6be567c3d6424cdf03668cab69d6e47447'), ('918273286248599552_0', 'data_image/california_wildfires/12_10_2017/918273286248599552_0.jpg', 'dev', '5ae533107bc37ca14826799ce4e3bc62497ced4eaef39a0f3526ba457c5c45ac'), ('917994406732615681_0', 'data_image/california_wildfires/11_10_2017/917994406732615681_0.jpg', 'test', '27398a4cc4f247994e8e4bac17b2bc2b0367433ae2338cdb0139dd4711ef22bd'), ('920380207625670656_0', 'data_image/california_wildfires/17_10_2017/920380207625670656_0.jpg', 'test', 'e1716e464e2e2659393eb8c2aa546177b1203427cf0cda78302c89ef5be8356d'), ('917871941205164032_0', 'data_image/california_wildfires/10_10_2017/917871941205164032_0.jpg', 'test', '3a8171d2837435e17aad559867c8aafa412e3a4f26d79bddd7ba0b8547f572e4'), ('923727709641084928_0', 'data_image/california_wildfires/27_10_2017/923727709641084928_0.jpg', 'test', '316c71585fd31ad31b18c2204583288aacbef80fc7898621724911db0e1a816e'), ('917848191051206656_0', 'data_image/california_wildfires/10_10_2017/917848191051206656_0.jpg', 'test', 'ac32e9243279acdb711a51ab3eeac80c7dd370a3193db692bf119115c30ddbcb'), ('919575390028468225_0', 'data_image/california_wildfires/15_10_2017/919575390028468225_0.jpg', 'train', '9687cc676b533f7befd8d9e4f5917ae56f3e1271f3141d92b8bb7c88937d838c'), ('919949345914421249_0', 'data_image/california_wildfires/16_10_2017/919949345914421249_0.jpg', 'test', 'c9833d872db8daed319f3568f46845dcf881c16770ba55c2b69e3e06c8d1b772'), ('918236795950813185_0', 'data_image/california_wildfires/11_10_2017/918236795950813185_0.jpg', 'test', '66ea81d8a44cc4868494563be702ef106fe990b3eca49f2857ec01255a1163c7'), ('918010136072945664_1', 'data_image/california_wildfires/11_10_2017/918010136072945664_1.jpg', 'train', '2ee20b835a4944edd8dd4480528c75235587d170ab70bdf198175b70dfda9f21'), ('920397507959562240_0', 'data_image/california_wildfires/17_10_2017/920397507959562240_0.jpg', 'dev', 'd679ddccd4bbce60e1138b38aed08a27b6a54b860851418e8261d63ab3c5a806'), ('920382328618278912_0', 'data_image/california_wildfires/17_10_2017/920382328618278912_0.jpg', 'test', '1f918600f72599df6ce2818e37c55a08c1e97fd734bde8417c9d1557a8a6fd98'), ('919730693285986304_1', 'data_image/california_wildfires/16_10_2017/919730693285986304_1.jpg', 'train', '853536586813f1aa34860e170084637a9e086a4869ff0c93957ed4d33fcbc44f'), ('921059698064867328_0', 'data_image/california_wildfires/19_10_2017/921059698064867328_0.jpg', 'train', '62b37c880583cf0208d85cb4a8d6bf80034cf683650130befca045923949fba8'), ('919694407514902528_0', 'data_image/california_wildfires/15_10_2017/919694407514902528_0.jpg', 'train', 'cdac4409b2799565da357e71dc610649bac3e7d13c6beffda66214e747bb8192'), ('905050833687064577_0', 'data_image/hurricane_harvey/5_9_2017/905050833687064577_0.jpg', 'train', '6e7035a2f9ce9c87aeaec5c47ea418e678fbe343cbbf75821973f69921163f6b'), ('904537294106791936_0', 'data_image/hurricane_harvey/4_9_2017/904537294106791936_0.jpg', 'test', '82e7a2e27231cef07654e9ba240e2e657fc08623af61b3398e6e04d773fbc95c'), ('906983382290194433_1', 'data_image/hurricane_harvey/10_9_2017/906983382290194433_1.jpg', 'train', '1548793676d16154db9d976af86dcac3f41abec07ccdfb51e9351af7a5860b48'), ('908169364716781568_0', 'data_image/hurricane_harvey/14_9_2017/908169364716781568_0.jpg', 'train', 'a0b2b55d5d345a1dbd37b2a672019ddbffbbd3b0c53e2475ab0d17d37f1d0001'), ('905844147227041793_0', 'data_image/hurricane_harvey/7_9_2017/905844147227041793_0.jpg', 'dev', 'ec22428fc5500ae5f4ae8ab40cef3fc3f8f855338d683230729a6c42298c1848'), ('906965327111737345_0', 'data_image/hurricane_harvey/10_9_2017/906965327111737345_0.jpg', 'test', '211fa7464a9e24e3ff328258ecd8ef832ceb532d7b6d147dca6a9c6a6f04cfe3'), ('905239945958178816_0', 'data_image/hurricane_harvey/6_9_2017/905239945958178816_0.jpg', 'train', 'c5ce4a3f35aacdb5ce9867d5df3176b92b78fcc5a6c6c3f9309258a52d03231d'), ('906223191496019968_0', 'data_image/hurricane_harvey/8_9_2017/906223191496019968_0.jpg', 'test', '012ccad2a2cb9178b0433ac570eb04f9fa566a5cd42251267f516740122b815a'), ('906938176337580033_0', 'data_image/hurricane_harvey/10_9_2017/906938176337580033_0.jpg', 'test', '654ffd05e41088fe03b79484fbe0c3cc89fdcea03032c7adede896f0f6eac4b5'), ('901705136782716928_0', 'data_image/hurricane_harvey/27_8_2017/901705136782716928_0.jpg', 'dev', 'edfab4f4762ac44603018f76984c030673606d7f73a855a83758d74ccce39306'), ('907086785599213568_0', 'data_image/hurricane_harvey/11_9_2017/907086785599213568_0.jpg', 'dev', '194a52894015580c1ddfc2b456bb2967a72c7dbc6c48b5fd01f94fb376ad4077'), ('904857217266016256_0', 'data_image/hurricane_harvey/5_9_2017/904857217266016256_0.jpg', 'train', 'a062e412cbce6f72ee57c5e2bf38857f1774fd9d7ddc862621f52d90769a1e7f'), ('904619989923020800_0', 'data_image/hurricane_harvey/4_9_2017/904619989923020800_0.jpg', 'train', '1cc46f83922b7e28ce9b026ca2f9d86569f5dec84cfb6fe02d43709b8e2b3ec0'), ('904666134556819456_0', 'data_image/hurricane_harvey/4_9_2017/904666134556819456_0.jpg', 'train', 'd81a8d552cfebb8fd4928dd7fa3f32dfbbf4a93bd26a26e469f35d3a11c874dc'), ('906664753602457601_0', 'data_image/hurricane_harvey/9_9_2017/906664753602457601_0.jpg', 'train', '616d86cf45e0ddba0cf3ad6cd71ba60af8b5078508575932b9c82b0d62f4b301'), ('904425923025522688_0', 'data_image/hurricane_harvey/3_9_2017/904425923025522688_0.jpg', 'test', 'db66dee44c6fd0719af01fe38db3f36ba6508902c20b0ebaf656b5e2d7ee78fe'), ('906586772573904896_0', 'data_image/hurricane_harvey/9_9_2017/906586772573904896_0.jpg', 'train', '131267c1424ed51123a2d074ae26a10f185f22d6ee9d376aa0b4aed47faa427b'), ('907982457336680448_0', 'data_image/hurricane_harvey/13_9_2017/907982457336680448_0.jpg', 'test', '4035ef14233cdb07e7c418ab5b896edb9242374c8f5d24f7a26e632129be90df'), ('904650635622191104_0', 'data_image/hurricane_harvey/4_9_2017/904650635622191104_0.jpg', 'train', '23cbb3d20f90647c2df0215826b5f6e06a10094e972fb8e6c128e201234b80d2'), ('909820899951595524_0', 'data_image/hurricane_irma/18_9_2017/909820899951595524_0.jpg', 'dev', 'f3378d26b35512db00b340363c228bbae2df1c3d5f7693b4bc81a47f87dc8e7d'), ('910201716934574081_0', 'data_image/hurricane_irma/19_9_2017/910201716934574081_0.jpg', 'train', '8e97ab21961f558d36f9cad67cc20838318e207ec5f294c2cb1b0c38ae9cca7a'), ('909820592941031424_0', 'data_image/hurricane_irma/18_9_2017/909820592941031424_0.jpg', 'test', 'ab4fe0e7b9a9cf0de89758a29b0de5409857dbdc0f82d161e738388206c995ea'), ('909396357718343680_0', 'data_image/hurricane_irma/17_9_2017/909396357718343680_0.jpg', 'test', '918fbd25701f87c86104278934b1385484764a32b17e2007da4a67ddc7286adc'), ('909490898706329600_1', 'data_image/hurricane_irma/17_9_2017/909490898706329600_1.jpg', 'train', '1531cefeb1d47125326f6e6c29b2648ef18cae6cf879a710f77af988d2afe502'), ('909461957555687424_0', 'data_image/hurricane_irma/17_9_2017/909461957555687424_0.jpg', 'train', '36ad1c9144fa17672c47afebf1d3f046b929648c5576e11406f9508868f0cd12'), ('909495717798060032_0', 'data_image/hurricane_irma/17_9_2017/909495717798060032_0.jpg', 'dev', 'b9ce3134da68904a589e5fb88b2d83ea650688d0adbeda1e975946c7ce917946'), ('909480562884198401_1', 'data_image/hurricane_irma/17_9_2017/909480562884198401_1.jpg', 'train', None), ('905733850923458560_0', 'data_image/hurricane_irma/7_9_2017/905733850923458560_0.jpg', 'test', None), ('910095089489383430_0', 'data_image/hurricane_irma/19_9_2017/910095089489383430_0.jpg', 'train', None), ('910124932775989248_0', 'data_image/hurricane_irma/19_9_2017/910124932775989248_0.jpg', 'test', None), ('905714574128644096_0', 'data_image/hurricane_irma/7_9_2017/905714574128644096_0.jpg', 'train', None), ('905755785564741632_0', 'data_image/hurricane_irma/7_9_2017/905755785564741632_0.jpg', 'train', None), ('909793318481137665_0', 'data_image/hurricane_irma/18_9_2017/909793318481137665_0.png', 'test', None), ('909839832066686978_0', 'data_image/hurricane_irma/18_9_2017/909839832066686978_0.jpg', 'train', None), ('909872135686017026_0', 'data_image/hurricane_irma/18_9_2017/909872135686017026_0.jpg', 'train', None), ('909901096243646465_0', 'data_image/hurricane_irma/18_9_2017/909901096243646465_0.jpg', 'dev', None), ('909855425402634247_0', 'data_image/hurricane_irma/18_9_2017/909855425402634247_0.jpg', 'train', None), ('910554153583230977_0', 'data_image/hurricane_maria/20_9_2017/910554153583230977_0.jpg', 'train', None), ('928838939544375296_0', 'data_image/hurricane_maria/10_11_2017/928838939544375296_0.jpg', 'train', None), ('929033551885959169_0', 'data_image/hurricane_maria/10_11_2017/929033551885959169_0.jpg', 'train', None), ('910645943678021632_0', 'data_image/hurricane_maria/20_9_2017/910645943678021632_0.jpg', 'dev', None), ('921364937661976576_0', 'data_image/hurricane_maria/20_10_2017/921364937661976576_0.jpg', 'train', None), ('913149334962573312_0', 'data_image/hurricane_maria/27_9_2017/913149334962573312_0.jpg', 'train', None), ('928468070733111296_0', 'data_image/hurricane_maria/9_11_2017/928468070733111296_0.jpg', 'test', None), ('916052748524597248_0', 'data_image/hurricane_maria/5_10_2017/916052748524597248_0.jpg', 'train', None), ('911983010576306178_0', 'data_image/hurricane_maria/24_9_2017/911983010576306178_0.jpg', 'test', None), ('923648872207081473_0', 'data_image/hurricane_maria/26_10_2017/923648872207081473_0.jpg', 'test', None), ('923618747004354560_0', 'data_image/hurricane_maria/26_10_2017/923618747004354560_0.jpg', 'dev', None), ('923652116383756290_0', 'data_image/hurricane_maria/26_10_2017/923652116383756290_0.jpg', 'train', None), ('922653434024812544_0', 'data_image/hurricane_maria/24_10_2017/922653434024812544_0.jpg', 'train', None), ('928378477866340353_0', 'data_image/hurricane_maria/8_11_2017/928378477866340353_0.jpg', 'dev', None), ('911655561237843970_0', 'data_image/hurricane_maria/23_9_2017/911655561237843970_0.jpg', 'train', None), ('910702638605496321_0', 'data_image/hurricane_maria/21_9_2017/910702638605496321_0.jpg', 'test', None), ('913572743760752640_0', 'data_image/hurricane_maria/29_9_2017/913572743760752640_0.jpg', 'train', None), ('911927354934611968_1', 'data_image/hurricane_maria/24_9_2017/911927354934611968_1.jpg', 'train', None), ('912421828618080258_0', 'data_image/hurricane_maria/25_9_2017/912421828618080258_0.jpg', 'dev', None), ('912048272092520449_0', 'data_image/hurricane_maria/24_9_2017/912048272092520449_0.jpg', 'train', None), ('922489129753108490_0', 'data_image/hurricane_maria/23_10_2017/922489129753108490_0.jpg', 'test', None), ('930022107278381057_1', 'data_image/iraq_iran_earthquake/13_11_2017/930022107278381057_1.jpg', 'train', None), ('930726547073376256_0', 'data_image/iraq_iran_earthquake/15_11_2017/930726547073376256_0.jpg', 'test', None), ('930123336201498625_0', 'data_image/iraq_iran_earthquake/13_11_2017/930123336201498625_0.jpg', 'train', None), ('930191869388251137_0', 'data_image/iraq_iran_earthquake/13_11_2017/930191869388251137_0.jpg', 'test', None), ('930491532766953472_0', 'data_image/iraq_iran_earthquake/14_11_2017/930491532766953472_0.jpg', 'test', None), ('931630430041681920_0', 'data_image/iraq_iran_earthquake/17_11_2017/931630430041681920_0.jpg', 'test', None), ('929989549895438336_0', 'data_image/iraq_iran_earthquake/13_11_2017/929989549895438336_0.jpg', 'test', None), ('930537551093518337_0', 'data_image/iraq_iran_earthquake/14_11_2017/930537551093518337_0.jpg', 'train', None), ('910814149852020736_0', 'data_image/mexico_earthquake/21_9_2017/910814149852020736_0.jpg', 'dev', None), ('910659915298988035_0', 'data_image/mexico_earthquake/21_9_2017/910659915298988035_0.jpg', 'train', None), ('910566780145246208_0', 'data_image/mexico_earthquake/20_9_2017/910566780145246208_0.jpg', 'dev', None), ('910718830107873280_0', 'data_image/mexico_earthquake/21_9_2017/910718830107873280_0.jpg', 'train', None), ('914993292290469888_2', 'data_image/mexico_earthquake/2_10_2017/914993292290469888_2.jpg', 'train', None), ('910532931361185792_0', 'data_image/mexico_earthquake/20_9_2017/910532931361185792_0.jpg', 'train', None), ('910818346152710144_0', 'data_image/mexico_earthquake/21_9_2017/910818346152710144_0.jpg', 'dev', None), ('910658744232456192_0', 'data_image/mexico_earthquake/21_9_2017/910658744232456192_0.jpg', 'train', None), ('910614775498924032_0', 'data_image/mexico_earthquake/20_9_2017/910614775498924032_0.jpg', 'test', None), ('911817491021737984_0', 'data_image/mexico_earthquake/24_9_2017/911817491021737984_0.jpg', 'train', None), ('910735341396557824_0', 'data_image/mexico_earthquake/21_9_2017/910735341396557824_0.jpg', 'dev', None), ('870389707418599426_0', 'data_image/srilanka_floods/1_6_2017/870389707418599426_0.jpg', 'train', None), ('870248079043158017_2', 'data_image/srilanka_floods/1_6_2017/870248079043158017_2.jpg', 'train', None), ('881807298930368513_0', 'data_image/srilanka_floods/3_7_2017/881807298930368513_0.jpg', 'dev', None), ('876496690299240449_0', 'data_image/srilanka_floods/18_6_2017/876496690299240449_0.jpg', 'test', None), ('877877084613300225_0', 'data_image/srilanka_floods/22_6_2017/877877084613300225_0.jpg', 'dev', None), ('870252543703629824_0', 'data_image/srilanka_floods/1_6_2017/870252543703629824_0.jpg', 'train', None), ('917822878124568577_0', 'data_image/california_wildfires/10_10_2017/917822878124568577_0.jpg', 'test', None), ('921528721563095041_0', 'data_image/california_wildfires/21_10_2017/921528721563095041_0.jpg', 'test', None), ('920844048881147906_0', 'data_image/california_wildfires/19_10_2017/920844048881147906_0.jpg', 'train', None), ('920279231887634433_0', 'data_image/california_wildfires/17_10_2017/920279231887634433_0.jpg', 'dev', None), ('918106428434300929_0', 'data_image/california_wildfires/11_10_2017/918106428434300929_0.jpg', 'test', None), ('920249445979181056_0', 'data_image/california_wildfires/17_10_2017/920249445979181056_0.jpg', 'dev', None), ('917830920907890688_0', 'data_image/california_wildfires/10_10_2017/917830920907890688_0.jpg', 'train', None), ('917825152636907523_0', 'data_image/california_wildfires/10_10_2017/917825152636907523_0.jpg', 'dev', None), ('918060571517480960_1', 'data_image/california_wildfires/11_10_2017/918060571517480960_1.jpg', 'dev', None), ('920281158377754625_0', 'data_image/california_wildfires/17_10_2017/920281158377754625_0.jpg', 'test', None), ('918001984153833472_0', 'data_image/california_wildfires/11_10_2017/918001984153833472_0.jpg', 'train', None), ('919648994661486593_0', 'data_image/california_wildfires/15_10_2017/919648994661486593_0.jpg', 'test', None), ('917991257594572800_0', 'data_image/california_wildfires/11_10_2017/917991257594572800_0.jpg', 'train', None), ('920678687204433920_0', 'data_image/california_wildfires/18_10_2017/920678687204433920_0.jpg', 'test', None), ('920774586794496001_0', 'data_image/california_wildfires/18_10_2017/920774586794496001_0.jpg', 'train', None), ('908388748869632002_0', 'data_image/hurricane_harvey/14_9_2017/908388748869632002_0.jpg', 'train', None), ('904445080374833153_0', 'data_image/hurricane_harvey/3_9_2017/904445080374833153_0.jpg', 'train', None), ('906372211216408576_0', 'data_image/hurricane_harvey/9_9_2017/906372211216408576_0.jpg', 'test', None), ('901659390268329988_0', 'data_image/hurricane_harvey/27_8_2017/901659390268329988_0.jpg', 'test', None), ('908566528898895872_3', 'data_image/hurricane_harvey/15_9_2017/908566528898895872_3.jpg', 'test', None), ('904324002176532481_2', 'data_image/hurricane_harvey/3_9_2017/904324002176532481_2.jpg', 'train', None), ('901766764039680001_2', 'data_image/hurricane_harvey/27_8_2017/901766764039680001_2.jpg', 'train', None), ('908678484159074305_0', 'data_image/hurricane_harvey/15_9_2017/908678484159074305_0.jpg', 'train', None), ('907383711246770176_0', 'data_image/hurricane_harvey/11_9_2017/907383711246770176_0.jpg', 'test', None), ('905137064282378240_0', 'data_image/hurricane_harvey/5_9_2017/905137064282378240_0.jpg', 'train', None), ('907020688095154176_0', 'data_image/hurricane_harvey/10_9_2017/907020688095154176_0.jpg', 'train', None), ('906790326215614466_0', 'data_image/hurricane_harvey/10_9_2017/906790326215614466_0.jpg', 'test', None), ('907650014645772288_0', 'data_image/hurricane_harvey/12_9_2017/907650014645772288_0.jpg', 'train', None), ('905133260526714880_2', 'data_image/hurricane_harvey/5_9_2017/905133260526714880_2.jpg', 'test', None), ('908525619037732865_3', 'data_image/hurricane_harvey/15_9_2017/908525619037732865_3.jpg', 'train', None), ('905255318141235200_1', 'data_image/hurricane_harvey/6_9_2017/905255318141235200_1.jpg', 'train', None), ('905801917279940608_0', 'data_image/hurricane_harvey/7_9_2017/905801917279940608_0.jpg', 'train', None), ('908775427833044992_0', 'data_image/hurricane_harvey/15_9_2017/908775427833044992_0.jpg', 'train', None), ('905749776687390721_0', 'data_image/hurricane_irma/7_9_2017/905749776687390721_0.jpg', 'train', None), ('910051877982756864_0', 'data_image/hurricane_irma/19_9_2017/910051877982756864_0.jpg', 'dev', None), ('909988214240305152_1', 'data_image/hurricane_irma/19_9_2017/909988214240305152_1.jpg', 'train', None), ('909936938718760960_0', 'data_image/hurricane_irma/19_9_2017/909936938718760960_0.jpg', 'dev', None), ('910101043253411840_0', 'data_image/hurricane_irma/19_9_2017/910101043253411840_0.jpg', 'train', None), ('909766223201579008_0', 'data_image/hurricane_irma/18_9_2017/909766223201579008_0.jpg', 'train', None), ('906927853471715329_2', 'data_image/hurricane_irma/10_9_2017/906927853471715329_2.jpg', 'train', None), ('909962867553824768_1', 'data_image/hurricane_irma/19_9_2017/909962867553824768_1.jpg', 'train', None), ('909843612309102592_0', 'data_image/hurricane_irma/18_9_2017/909843612309102592_0.jpg', 'train', None), ('909994032436191232_0', 'data_image/hurricane_irma/19_9_2017/909994032436191232_0.jpg', 'test', None), ('910121621897334784_0', 'data_image/hurricane_irma/19_9_2017/910121621897334784_0.jpg', 'dev', None), ('909901099636797442_0', 'data_image/hurricane_irma/18_9_2017/909901099636797442_0.jpg', 'train', None), ('909382965435789313_0', 'data_image/hurricane_irma/17_9_2017/909382965435789313_0.jpg', 'dev', None), ('909920437479624704_1', 'data_image/hurricane_irma/18_9_2017/909920437479624704_1.jpg', 'train', None), ('909798839380406272_2', 'data_image/hurricane_irma/18_9_2017/909798839380406272_2.jpg', 'train', None), ('909433136035041280_0', 'data_image/hurricane_irma/17_9_2017/909433136035041280_0.jpg', 'train', None), ('909928551629115394_0', 'data_image/hurricane_irma/18_9_2017/909928551629115394_0.jpg', 'test', None), ('912437292450631680_0', 'data_image/hurricane_maria/25_9_2017/912437292450631680_0.jpg', 'test', None), ('916311255689973760_0', 'data_image/hurricane_maria/6_10_2017/916311255689973760_0.jpg', 'train', None), ('910627521732972544_0', 'data_image/hurricane_maria/20_9_2017/910627521732972544_0.jpg', 'test', None), ('911650692368158721_3', 'data_image/hurricane_maria/23_9_2017/911650692368158721_3.jpg', 'train', None), ('928088811497803778_0', 'data_image/hurricane_maria/8_11_2017/928088811497803778_0.jpg', 'test', None), ('923557481430835201_0', 'data_image/hurricane_maria/26_10_2017/923557481430835201_0.jpg', 'test', None), ('914180662365736964_0', 'data_image/hurricane_maria/30_9_2017/914180662365736964_0.jpg', 'train', None), ('923579145673519105_0', 'data_image/hurricane_maria/26_10_2017/923579145673519105_0.jpg', 'test', None), ('913373637130022914_0', 'data_image/hurricane_maria/28_9_2017/913373637130022914_0.jpg', 'test', None), ('927696898269630464_0', 'data_image/hurricane_maria/7_11_2017/927696898269630464_0.jpg', 'test', None), ('916027951002550272_0', 'data_image/hurricane_maria/5_10_2017/916027951002550272_0.jpg', 'test', None), ('913588950224265216_0', 'data_image/hurricane_maria/29_9_2017/913588950224265216_0.jpg', 'train', None), ('922819431692201984_0', 'data_image/hurricane_maria/24_10_2017/922819431692201984_0.jpg', 'dev', None), ('910608268799516672_0', 'data_image/hurricane_maria/20_9_2017/910608268799516672_0.jpg', 'test', None), ('912310862270144512_0', 'data_image/hurricane_maria/25_9_2017/912310862270144512_0.jpg', 'train', None), ('913404044999356416_1', 'data_image/hurricane_maria/28_9_2017/913404044999356416_1.jpg', 'train', None), ('911837730413441025_2', 'data_image/hurricane_maria/24_9_2017/911837730413441025_2.jpg', 'train', None), ('911574331758665728_0', 'data_image/hurricane_maria/23_9_2017/911574331758665728_0.jpg', 'dev', None), ('927556458593636353_0', 'data_image/hurricane_maria/6_11_2017/927556458593636353_0.jpg', 'train', None), ('922516226060021761_0', 'data_image/hurricane_maria/23_10_2017/922516226060021761_0.jpg', 'dev', None), ('930002146593869824_0', 'data_image/iraq_iran_earthquake/13_11_2017/930002146593869824_0.jpg', 'train', None), ('931304735029846017_0', 'data_image/iraq_iran_earthquake/16_11_2017/931304735029846017_0.jpg', 'dev', None), ('930058600160088065_1', 'data_image/iraq_iran_earthquake/13_11_2017/930058600160088065_1.jpg', 'dev', None), ('931305936227991552_0', 'data_image/iraq_iran_earthquake/16_11_2017/931305936227991552_0.jpg', 'test', None), ('930508546713767936_0', 'data_image/iraq_iran_earthquake/14_11_2017/930508546713767936_0.jpg', 'train', None), ('930193943408586752_3', 'data_image/iraq_iran_earthquake/13_11_2017/930193943408586752_3.jpg', 'train', None), ('929989459453595648_0', 'data_image/iraq_iran_earthquake/13_11_2017/929989459453595648_0.jpg', 'dev', None), ('930482499427241984_0', 'data_image/iraq_iran_earthquake/14_11_2017/930482499427241984_0.jpg', 'train', None), ('910649364204412928_0', 'data_image/mexico_earthquake/20_9_2017/910649364204412928_0.jpg', 'train', None), ('910572320338485248_0', 'data_image/mexico_earthquake/20_9_2017/910572320338485248_0.jpg', 'train', None), ('915320731826868225_0', 'data_image/mexico_earthquake/3_10_2017/915320731826868225_0.jpg', 'dev', None), ('912430414781808640_1', 'data_image/mexico_earthquake/25_9_2017/912430414781808640_1.jpg', 'train', None), ('910707238007984128_0', 'data_image/mexico_earthquake/21_9_2017/910707238007984128_0.jpg', 'train', None), ('913863201162514432_0', 'data_image/mexico_earthquake/29_9_2017/913863201162514432_0.jpg', 'train', None), ('916075589131436032_0', 'data_image/mexico_earthquake/5_10_2017/916075589131436032_0.jpg', 'dev', None), ('911842861695602693_0', 'data_image/mexico_earthquake/24_9_2017/911842861695602693_0.jpg', 'dev', None), ('910554199548600321_0', 'data_image/mexico_earthquake/20_9_2017/910554199548600321_0.jpg', 'test', None), ('910701059420622854_3', 'data_image/mexico_earthquake/21_9_2017/910701059420622854_3.jpg', 'train', None), ('911717117602316288_2', 'data_image/mexico_earthquake/23_9_2017/911717117602316288_2.jpg', 'train', None), ('878136759959343108_0', 'data_image/srilanka_floods/23_6_2017/878136759959343108_0.jpg', 'dev', None), ('869984832364756992_0', 'data_image/srilanka_floods/31_5_2017/869984832364756992_0.jpg', 'train', None), ('870518669213708293_0', 'data_image/srilanka_floods/2_6_2017/870518669213708293_0.jpg', 'train', None), ('871847555231080448_0', 'data_image/srilanka_floods/5_6_2017/871847555231080448_0.jpg', 'train', None), ('870386986758156290_0', 'data_image/srilanka_floods/1_6_2017/870386986758156290_0.jpg', 'train', None), ('872238834628665344_0', 'data_image/srilanka_floods/6_6_2017/872238834628665344_0.jpg', 'dev', None), ('920746714637721600_0', 'data_image/california_wildfires/18_10_2017/920746714637721600_0.jpg', 'train', None), ('921452372861313025_0', 'data_image/california_wildfires/20_10_2017/921452372861313025_0.jpg', 'test', None), ('917999457014226944_0', 'data_image/california_wildfires/11_10_2017/917999457014226944_0.jpg', 'test', None), ('919949856004767745_0', 'data_image/california_wildfires/16_10_2017/919949856004767745_0.jpg', 'train', None), ('918105765319028736_0', 'data_image/california_wildfires/11_10_2017/918105765319028736_0.jpg', 'dev', None), ('919570320381505536_0', 'data_image/california_wildfires/15_10_2017/919570320381505536_0.jpg', 'dev', None), ('918045067293716480_0', 'data_image/california_wildfires/11_10_2017/918045067293716480_0.jpg', 'train', None), ('923612570078064640_0', 'data_image/california_wildfires/26_10_2017/923612570078064640_0.jpg', 'dev', None), ('917791291823591425_0', 'data_image/california_wildfires/10_10_2017/917791291823591425_0.jpg', 'train', None), ('920066633862750208_0', 'data_image/california_wildfires/16_10_2017/920066633862750208_0.jpg', 'dev', None), ('919566012604874754_0', 'data_image/california_wildfires/15_10_2017/919566012604874754_0.jpg', 'train', None), ('920662274183913472_0', 'data_image/california_wildfires/18_10_2017/920662274183913472_0.jpg', 'dev', None), ('917991257594572800_2', 'data_image/california_wildfires/11_10_2017/917991257594572800_2.jpg', 'train', None), ('918082568322408449_0', 'data_image/california_wildfires/11_10_2017/918082568322408449_0.jpg', 'train', None), ('905188876968501249_0', 'data_image/hurricane_harvey/5_9_2017/905188876968501249_0.jpg', 'test', None), ('907664730936020994_0', 'data_image/hurricane_harvey/12_9_2017/907664730936020994_0.jpg', 'dev', None), ('904830090386735105_0', 'data_image/hurricane_harvey/4_9_2017/904830090386735105_0.jpg', 'test', None), ('906650124302131201_0', 'data_image/hurricane_harvey/9_9_2017/906650124302131201_0.jpg', 'test', None), ('908078784800161792_0', 'data_image/hurricane_harvey/13_9_2017/908078784800161792_0.jpg', 'test', None), ('904742476459655170_0', 'data_image/hurricane_harvey/4_9_2017/904742476459655170_0.jpg', 'train', None), ('908842412776333312_0', 'data_image/hurricane_harvey/15_9_2017/908842412776333312_0.jpg', 'test', None), ('908710215624904705_0', 'data_image/hurricane_harvey/15_9_2017/908710215624904705_0.jpg', 'train', None), ('901835409440768000_0', 'data_image/hurricane_harvey/27_8_2017/901835409440768000_0.jpg', 'test', None), ('904488825178779649_0', 'data_image/hurricane_harvey/3_9_2017/904488825178779649_0.jpg', 'dev', None), ('907274021275885569_0', 'data_image/hurricane_harvey/11_9_2017/907274021275885569_0.jpg', 'dev', None), ('907800232007790594_0', 'data_image/hurricane_harvey/13_9_2017/907800232007790594_0.jpg', 'dev', None), ('908424732554522624_0', 'data_image/hurricane_harvey/14_9_2017/908424732554522624_0.jpg', 'train', None), ('904746247847862278_0', 'data_image/hurricane_harvey/4_9_2017/904746247847862278_0.jpg', 'test', None), ('905200250796687360_0', 'data_image/hurricane_harvey/5_9_2017/905200250796687360_0.jpg', 'test', None), ('904947226232856576_0', 'data_image/hurricane_harvey/5_9_2017/904947226232856576_0.jpg', 'train', None), ('905034039064715264_0', 'data_image/hurricane_harvey/5_9_2017/905034039064715264_0.jpg', 'train', None), ('904677645836804096_0', 'data_image/hurricane_harvey/4_9_2017/904677645836804096_0.jpg', 'dev', None), ('909809252260122624_0', 'data_image/hurricane_irma/18_9_2017/909809252260122624_0.jpg', 'train', None), ('909400394773811200_0', 'data_image/hurricane_irma/17_9_2017/909400394773811200_0.jpg', 'test', None), ('909850724137410560_0', 'data_image/hurricane_irma/18_9_2017/909850724137410560_0.jpg', 'train', None), ('905696714253676544_0', 'data_image/hurricane_irma/7_9_2017/905696714253676544_0.jpg', 'test', None), ('910242983676608512_0', 'data_image/hurricane_irma/19_9_2017/910242983676608512_0.jpg', 'dev', None), ('909810646472118273_0', 'data_image/hurricane_irma/18_9_2017/909810646472118273_0.jpg', 'train', None), ('910092846635548672_2', 'data_image/hurricane_irma/19_9_2017/910092846635548672_2.jpg', 'train', None), ('909830188963389440_1', 'data_image/hurricane_irma/18_9_2017/909830188963389440_1.jpg', 'train', None), ('909871727932473345_0', 'data_image/hurricane_irma/18_9_2017/909871727932473345_0.jpg', 'test', None), ('909377896996892672_0', 'data_image/hurricane_irma/17_9_2017/909377896996892672_0.jpg', 'test', None), ('909930819619233793_0', 'data_image/hurricane_irma/19_9_2017/909930819619233793_0.jpg', 'dev', None), ('910119285623857152_0', 'data_image/hurricane_irma/19_9_2017/910119285623857152_0.jpg', 'test', None), ('910243618409107456_0', 'data_image/hurricane_irma/19_9_2017/910243618409107456_0.jpg', 'train', None), ('910171405790044160_0', 'data_image/hurricane_irma/19_9_2017/910171405790044160_0.jpg', 'test', None), ('909774976609382400_0', 'data_image/hurricane_irma/18_9_2017/909774976609382400_0.jpg', 'test', None), ('909378277776687105_0', 'data_image/hurricane_irma/17_9_2017/909378277776687105_0.jpg', 'test', None), ('909477271206952960_0', 'data_image/hurricane_irma/17_9_2017/909477271206952960_0.jpg', 'test', None), ('910714001709027328_0', 'data_image/hurricane_maria/21_9_2017/910714001709027328_0.jpg', 'test', None), ('915282034628730884_0', 'data_image/hurricane_maria/3_10_2017/915282034628730884_0.jpg', 'dev', None), ('921443803457781766_1', 'data_image/hurricane_maria/20_10_2017/921443803457781766_1.jpg', 'test', None), ('911650692368158721_1', 'data_image/hurricane_maria/23_9_2017/911650692368158721_1.jpg', 'train', None), ('913062398004776960_0', 'data_image/hurricane_maria/27_9_2017/913062398004776960_0.jpg', 'test', None), ('915282034628730884_3', 'data_image/hurricane_maria/3_10_2017/915282034628730884_3.jpg', 'dev', None), ('922451919637008384_0', 'data_image/hurricane_maria/23_10_2017/922451919637008384_0.jpg', 'test', None), ('922610290281402368_3', 'data_image/hurricane_maria/23_10_2017/922610290281402368_3.jpg', 'train', None), ('912075698617102340_1', 'data_image/hurricane_maria/24_9_2017/912075698617102340_1.jpg', 'train', None), ('921868154581737472_0', 'data_image/hurricane_maria/21_10_2017/921868154581737472_0.jpg', 'dev', None), ('923847805390729216_0', 'data_image/hurricane_maria/27_10_2017/923847805390729216_0.jpg', 'train', None), ('922157782677245952_0', 'data_image/hurricane_maria/22_10_2017/922157782677245952_0.jpg', 'dev', None), ('923212089774063616_0', 'data_image/hurricane_maria/25_10_2017/923212089774063616_0.jpg', 'train', None), ('923262758992465920_2', 'data_image/hurricane_maria/25_10_2017/923262758992465920_2.jpg', 'test', None), ('910574893531406336_0', 'data_image/hurricane_maria/20_9_2017/910574893531406336_0.jpg', 'train', None), ('914871738722390016_0', 'data_image/hurricane_maria/2_10_2017/914871738722390016_0.jpg', 'train', None), ('923670505391042561_0', 'data_image/hurricane_maria/26_10_2017/923670505391042561_0.jpg', 'train', None), ('916342141953200128_0', 'data_image/hurricane_maria/6_10_2017/916342141953200128_0.jpg', 'dev', None), ('911648744675860480_1', 'data_image/hurricane_maria/23_9_2017/911648744675860480_1.jpg', 'train', None), ('922935520056950784_0', 'data_image/hurricane_maria/24_10_2017/922935520056950784_0.jpg', 'train', None), ('930767353658437634_1', 'data_image/iraq_iran_earthquake/15_11_2017/930767353658437634_1.jpg', 'train', None), ('930180496507752448_0', 'data_image/iraq_iran_earthquake/13_11_2017/930180496507752448_0.jpg', 'train', None), ('929993504662720512_0', 'data_image/iraq_iran_earthquake/13_11_2017/929993504662720512_0.jpg', 'train', None), ('930101240339542016_0', 'data_image/iraq_iran_earthquake/13_11_2017/930101240339542016_0.jpg', 'train', None), ('930001895288029184_0', 'data_image/iraq_iran_earthquake/13_11_2017/930001895288029184_0.jpg', 'train', None), ('931159934154223618_0', 'data_image/iraq_iran_earthquake/16_11_2017/931159934154223618_0.jpg', 'test', None), ('930837330436542464_0', 'data_image/iraq_iran_earthquake/15_11_2017/930837330436542464_0.jpg', 'train', None), ('930190280384532480_0', 'data_image/iraq_iran_earthquake/13_11_2017/930190280384532480_0.jpg', 'test', None), ('912191568496533504_0', 'data_image/mexico_earthquake/25_9_2017/912191568496533504_0.jpg', 'train', None), ('914045395759767552_0', 'data_image/mexico_earthquake/30_9_2017/914045395759767552_0.jpg', 'dev', None), ('910632006144352256_0', 'data_image/mexico_earthquake/20_9_2017/910632006144352256_0.png', 'train', None), ('910525344532303875_0', 'data_image/mexico_earthquake/20_9_2017/910525344532303875_0.jpg', 'test', None), ('912436581847375872_0', 'data_image/mexico_earthquake/25_9_2017/912436581847375872_0.jpg', 'train', None), ('910765741993005058_0', 'data_image/mexico_earthquake/21_9_2017/910765741993005058_0.jpg', 'test', None), ('910537940346032128_0', 'data_image/mexico_earthquake/20_9_2017/910537940346032128_0.jpg', 'test', None), ('915193216231653376_0', 'data_image/mexico_earthquake/3_10_2017/915193216231653376_0.jpg', 'dev', None), ('911951556781662209_0', 'data_image/mexico_earthquake/24_9_2017/911951556781662209_0.jpg', 'test', None), ('910572982044635138_0', 'data_image/mexico_earthquake/20_9_2017/910572982044635138_0.jpg', 'dev', None), ('912403630610726913_0', 'data_image/mexico_earthquake/25_9_2017/912403630610726913_0.jpg', 'train', None), ('870630443963473921_0', 'data_image/srilanka_floods/2_6_2017/870630443963473921_0.jpg', 'test', None), ('870559283880112128_0', 'data_image/srilanka_floods/2_6_2017/870559283880112128_0.jpg', 'test', None), ('879930732327694337_0', 'data_image/srilanka_floods/28_6_2017/879930732327694337_0.jpg', 'test', None), ('873919197004410882_0', 'data_image/srilanka_floods/11_6_2017/873919197004410882_0.jpg', 'dev', None), ('872085153216163844_0', 'data_image/srilanka_floods/6_6_2017/872085153216163844_0.jpg', 'dev', None), ('870272043538886657_0', 'data_image/srilanka_floods/1_6_2017/870272043538886657_0.jpg', 'train', None), ('919602405439815681_0', 'data_image/california_wildfires/15_10_2017/919602405439815681_0.jpg', 'train', None), ('920110234433146880_0', 'data_image/california_wildfires/17_10_2017/920110234433146880_0.jpg', 'test', None), ('920022506211127296_0', 'data_image/california_wildfires/16_10_2017/920022506211127296_0.jpg', 'test', None), ('919912051685826561_0', 'data_image/california_wildfires/16_10_2017/919912051685826561_0.jpg', 'test', None), ('921483682757664775_0', 'data_image/california_wildfires/20_10_2017/921483682757664775_0.jpg', 'dev', None), ('919643312968974338_0', 'data_image/california_wildfires/15_10_2017/919643312968974338_0.jpg', 'train', None), ('918390119412924416_0', 'data_image/california_wildfires/12_10_2017/918390119412924416_0.jpg', 'train', None), ('920888649348145152_0', 'data_image/california_wildfires/19_10_2017/920888649348145152_0.jpg', 'test', None), ('917856015219154945_0', 'data_image/california_wildfires/10_10_2017/917856015219154945_0.jpg', 'test', None), ('919871082894643200_0', 'data_image/california_wildfires/16_10_2017/919871082894643200_0.jpg', 'train', None), ('923403586540900352_0', 'data_image/california_wildfires/26_10_2017/923403586540900352_0.jpg', 'train', None), ('920356076230467584_0', 'data_image/california_wildfires/17_10_2017/920356076230467584_0.jpg', 'train', None), ('917817114471346177_0', 'data_image/california_wildfires/10_10_2017/917817114471346177_0.jpg', 'train', None), ('905463049061990400_0', 'data_image/hurricane_harvey/6_9_2017/905463049061990400_0.jpg', 'train', None), ('904405775141019649_1', 'data_image/hurricane_harvey/3_9_2017/904405775141019649_1.jpg', 'train', None), ('908772392159387648_2', 'data_image/hurricane_harvey/15_9_2017/908772392159387648_2.jpg', 'train', None), ('905889752968306689_0', 'data_image/hurricane_harvey/7_9_2017/905889752968306689_0.jpg', 'dev', None), ('904651672873570304_0', 'data_image/hurricane_harvey/4_9_2017/904651672873570304_0.jpg', 'dev', None), ('908115249286586369_0', 'data_image/hurricane_harvey/13_9_2017/908115249286586369_0.jpg', 'dev', None), ('905221114376454144_0', 'data_image/hurricane_harvey/6_9_2017/905221114376454144_0.jpg', 'dev', None), ('904230007681273856_0', 'data_image/hurricane_harvey/3_9_2017/904230007681273856_0.jpg', 'train', None), ('907370190182199297_0', 'data_image/hurricane_harvey/11_9_2017/907370190182199297_0.jpg', 'train', None), ('901650495122456576_0', 'data_image/hurricane_harvey/27_8_2017/901650495122456576_0.jpg', 'test', None), ('905001603345862656_0', 'data_image/hurricane_harvey/5_9_2017/905001603345862656_0.jpg', 'test', None), ('906204627393277952_0', 'data_image/hurricane_harvey/8_9_2017/906204627393277952_0.jpg', 'test', None), ('901817961610174464_2', 'data_image/hurricane_harvey/27_8_2017/901817961610174464_2.jpg', 'dev', None), ('906758959977127937_2', 'data_image/hurricane_harvey/10_9_2017/906758959977127937_2.jpg', 'train', None), ('904316018524291073_0', 'data_image/hurricane_harvey/3_9_2017/904316018524291073_0.jpg', 'dev', None), ('901650435391541250_0', 'data_image/hurricane_harvey/27_8_2017/901650435391541250_0.jpg', 'dev', None), ('909890980027011073_0', 'data_image/hurricane_irma/18_9_2017/909890980027011073_0.jpg', 'train', None), ('909912459640700930_2', 'data_image/hurricane_irma/18_9_2017/909912459640700930_2.jpg', 'test', None), ('909436630267760641_3', 'data_image/hurricane_irma/17_9_2017/909436630267760641_3.jpg', 'train', None), ('910205368394964992_0', 'data_image/hurricane_irma/19_9_2017/910205368394964992_0.jpg', 'test', None), ('910062175657631744_0', 'data_image/hurricane_irma/19_9_2017/910062175657631744_0.jpg', 'train', None), ('905663970219929601_0', 'data_image/hurricane_irma/7_9_2017/905663970219929601_0.jpg', 'test', None), ('909776579584196610_0', 'data_image/hurricane_irma/18_9_2017/909776579584196610_0.png', 'test', None), ('909839145597571072_0', 'data_image/hurricane_irma/18_9_2017/909839145597571072_0.jpg', 'train', None), ('909914772816179201_3', 'data_image/hurricane_irma/18_9_2017/909914772816179201_3.jpg', 'train', None), ('909814933285752832_0', 'data_image/hurricane_irma/18_9_2017/909814933285752832_0.jpg', 'test', None), ('909929395053195269_0', 'data_image/hurricane_irma/18_9_2017/909929395053195269_0.jpg', 'dev', None), ('909380504201367553_0', 'data_image/hurricane_irma/17_9_2017/909380504201367553_0.jpg', 'test', None), ('909748935392137217_0', 'data_image/hurricane_irma/18_9_2017/909748935392137217_0.png', 'train', None), ('909447146268585985_0', 'data_image/hurricane_irma/17_9_2017/909447146268585985_0.jpg', 'dev', None), ('910158610776383489_0', 'data_image/hurricane_irma/19_9_2017/910158610776383489_0.jpg', 'test', None), ('909827606912040965_0', 'data_image/hurricane_irma/18_9_2017/909827606912040965_0.jpg', 'dev', None), ('909894624898101248_0', 'data_image/hurricane_irma/18_9_2017/909894624898101248_0.jpg', 'train', None), ('923009219955249153_0', 'data_image/hurricane_maria/25_10_2017/923009219955249153_0.jpg', 'train', None), ('914159348267540482_0', 'data_image/hurricane_maria/30_9_2017/914159348267540482_0.jpg', 'dev', None), ('912077714227593216_0', 'data_image/hurricane_maria/24_9_2017/912077714227593216_0.jpg', 'dev', None), ('927518834201899011_0', 'data_image/hurricane_maria/6_11_2017/927518834201899011_0.jpg', 'train', None), ('927340512817926144_0', 'data_image/hurricane_maria/6_11_2017/927340512817926144_0.jpg', 'dev', None), ('912204795427123200_0', 'data_image/hurricane_maria/25_9_2017/912204795427123200_0.jpg', 'train', None), ('927349857202958338_0', 'data_image/hurricane_maria/6_11_2017/927349857202958338_0.jpg', 'train', None), ('912346586910425088_3', 'data_image/hurricane_maria/25_9_2017/912346586910425088_3.jpg', 'train', None), ('914113249934811136_2', 'data_image/hurricane_maria/30_9_2017/914113249934811136_2.jpg', 'train', None), ('922930213922652160_0', 'data_image/hurricane_maria/24_10_2017/922930213922652160_0.jpg', 'train', None), ('921780671760171008_0', 'data_image/hurricane_maria/21_10_2017/921780671760171008_0.jpg', 'test', None), ('910639429664026624_0', 'data_image/hurricane_maria/20_9_2017/910639429664026624_0.jpg', 'dev', None), ('921440313771671552_0', 'data_image/hurricane_maria/20_10_2017/921440313771671552_0.jpg', 'train', None), ('927984655093047296_0', 'data_image/hurricane_maria/7_11_2017/927984655093047296_0.jpg', 'dev', None), ('922907836052422656_0', 'data_image/hurricane_maria/24_10_2017/922907836052422656_0.jpg', 'test', None), ('922878599232974848_0', 'data_image/hurricane_maria/24_10_2017/922878599232974848_0.jpg', 'train', None), ('914555402737668097_2', 'data_image/hurricane_maria/1_10_2017/914555402737668097_2.jpg', 'dev', None), ('916045347377512449_0', 'data_image/hurricane_maria/5_10_2017/916045347377512449_0.jpg', 'test', None), ('930445781424844800_1', 'data_image/iraq_iran_earthquake/14_11_2017/930445781424844800_1.jpg', 'train', None), ('930091312665243651_0', 'data_image/iraq_iran_earthquake/13_11_2017/930091312665243651_0.jpg', 'train', None), ('930096683119243264_0', 'data_image/iraq_iran_earthquake/13_11_2017/930096683119243264_0.jpg', 'dev', None), ('930028516837642240_0', 'data_image/iraq_iran_earthquake/13_11_2017/930028516837642240_0.jpg', 'train', None), ('930387954299736065_0', 'data_image/iraq_iran_earthquake/14_11_2017/930387954299736065_0.jpg', 'train', None), ('930028600992165888_1', 'data_image/iraq_iran_earthquake/13_11_2017/930028600992165888_1.jpg', 'train', None), ('930010024700760064_0', 'data_image/iraq_iran_earthquake/13_11_2017/930010024700760064_0.jpg', 'train', None), ('930006201068347392_0', 'data_image/iraq_iran_earthquake/13_11_2017/930006201068347392_0.jpg', 'test', None), ('912430414781808640_2', 'data_image/mexico_earthquake/25_9_2017/912430414781808640_2.jpg', 'train', None), ('910807746236186624_0', 'data_image/mexico_earthquake/21_9_2017/910807746236186624_0.png', 'test', None), ('912301115257053184_1', 'data_image/mexico_earthquake/25_9_2017/912301115257053184_1.jpg', 'dev', None), ('912111560679505920_0', 'data_image/mexico_earthquake/25_9_2017/912111560679505920_0.jpg', 'test', None), ('913802201813979136_0', 'data_image/mexico_earthquake/29_9_2017/913802201813979136_0.jpg', 'train', None), ('910586004784009216_0', 'data_image/mexico_earthquake/20_9_2017/910586004784009216_0.jpg', 'test', None), ('912466758602313728_0', 'data_image/mexico_earthquake/26_9_2017/912466758602313728_0.jpg', 'dev', None), ('910549291940352001_0', 'data_image/mexico_earthquake/20_9_2017/910549291940352001_0.jpg', 'dev', None), ('915028671475310592_0', 'data_image/mexico_earthquake/3_10_2017/915028671475310592_0.jpg', 'train', None), ('915254814145175552_0', 'data_image/mexico_earthquake/3_10_2017/915254814145175552_0.jpg', 'dev', None), ('915288925945647104_0', 'data_image/mexico_earthquake/3_10_2017/915288925945647104_0.jpg', 'train', None), ('870248079043158017_1', 'data_image/srilanka_floods/1_6_2017/870248079043158017_1.jpg', 'train', None), ('873663464774017024_0', 'data_image/srilanka_floods/10_6_2017/873663464774017024_0.jpg', 'test', None), ('873258014953938945_1', 'data_image/srilanka_floods/9_6_2017/873258014953938945_1.jpg', 'train', None), ('875745988983619584_0', 'data_image/srilanka_floods/16_6_2017/875745988983619584_0.jpg', 'dev', None), ('870938078629609472_2', 'data_image/srilanka_floods/3_6_2017/870938078629609472_2.jpg', 'train', None), ('870113671699193857_0', 'data_image/srilanka_floods/1_6_2017/870113671699193857_0.jpg', 'train', None), ('919700705866342400_0', 'data_image/california_wildfires/15_10_2017/919700705866342400_0.jpg', 'dev', None), ('921159449708060672_0', 'data_image/california_wildfires/19_10_2017/921159449708060672_0.jpg', 'train', None), ('918062929634770944_0', 'data_image/california_wildfires/11_10_2017/918062929634770944_0.jpg', 'train', None), ('920379138627817473_0', 'data_image/california_wildfires/17_10_2017/920379138627817473_0.jpg', 'train', None), ('917947880035450880_0', 'data_image/california_wildfires/11_10_2017/917947880035450880_0.jpg', 'train', None), ('919721898065272832_0', 'data_image/california_wildfires/16_10_2017/919721898065272832_0.jpg', 'train', None), ('921113224568885248_0', 'data_image/california_wildfires/19_10_2017/921113224568885248_0.jpg', 'dev', None), ('917827102308671488_0', 'data_image/california_wildfires/10_10_2017/917827102308671488_0.jpg', 'train', None), ('920045440594268160_0', 'data_image/california_wildfires/16_10_2017/920045440594268160_0.jpg', 'test', None), ('920092174846451712_0', 'data_image/california_wildfires/17_10_2017/920092174846451712_0.jpg', 'test', None), ('920952107007692801_0', 'data_image/california_wildfires/19_10_2017/920952107007692801_0.jpg', 'test', None), ('920386617025748992_0', 'data_image/california_wildfires/17_10_2017/920386617025748992_0.jpg', 'train', None), ('922663032555278336_0', 'data_image/california_wildfires/24_10_2017/922663032555278336_0.jpg', 'train', None), ('905897516612313088_0', 'data_image/hurricane_harvey/7_9_2017/905897516612313088_0.jpg', 'dev', None), ('909938076423675905_0', 'data_image/hurricane_harvey/19_9_2017/909938076423675905_0.jpg', 'train', None), ('904257943805333505_0', 'data_image/hurricane_harvey/3_9_2017/904257943805333505_0.jpg', 'test', None), ('904556150510596096_0', 'data_image/hurricane_harvey/4_9_2017/904556150510596096_0.jpg', 'test', None), ('901809544401170433_0', 'data_image/hurricane_harvey/27_8_2017/901809544401170433_0.jpg', 'test', None), ('905491643243143174_0', 'data_image/hurricane_harvey/6_9_2017/905491643243143174_0.jpg', 'train', None), ('905157741525729280_0', 'data_image/hurricane_harvey/5_9_2017/905157741525729280_0.jpg', 'train', None), ('907855146721443840_0', 'data_image/hurricane_harvey/13_9_2017/907855146721443840_0.jpg', 'dev', None), ('904597048158064640_0', 'data_image/hurricane_harvey/4_9_2017/904597048158064640_0.jpg', 'test', None), ('906149216376037376_0', 'data_image/hurricane_harvey/8_9_2017/906149216376037376_0.png', 'dev', None), ('908854428123455489_0', 'data_image/hurricane_harvey/16_9_2017/908854428123455489_0.jpg', 'train', None), ('910187021477437440_0', 'data_image/hurricane_harvey/19_9_2017/910187021477437440_0.jpg', 'test', None), ('907260601298247680_0', 'data_image/hurricane_harvey/11_9_2017/907260601298247680_0.jpg', 'train', None), ('904856788922720256_1', 'data_image/hurricane_harvey/5_9_2017/904856788922720256_1.jpg', 'train', None), ('907234865409675265_0', 'data_image/hurricane_harvey/11_9_2017/907234865409675265_0.jpg', 'train', None), ('909730737401716736_0', 'data_image/hurricane_irma/18_9_2017/909730737401716736_0.jpg', 'test', None), ('910156250368888832_1', 'data_image/hurricane_irma/19_9_2017/910156250368888832_1.jpg', 'train', None), ('910145127376703488_0', 'data_image/hurricane_irma/19_9_2017/910145127376703488_0.jpg', 'dev', None), ('909793412525826048_0', 'data_image/hurricane_irma/18_9_2017/909793412525826048_0.jpg', 'test', None), ('910102466204770304_1', 'data_image/hurricane_irma/19_9_2017/910102466204770304_1.jpg', 'train', None), ('909758147560181760_0', 'data_image/hurricane_irma/18_9_2017/909758147560181760_0.jpg', 'dev', None), ('910126207836069888_0', 'data_image/hurricane_irma/19_9_2017/910126207836069888_0.png', 'train', None), ('910241915874004992_1', 'data_image/hurricane_irma/19_9_2017/910241915874004992_1.jpg', 'train', None), ('905711015572250632_0', 'data_image/hurricane_irma/7_9_2017/905711015572250632_0.jpg', 'dev', None), ('909481301274587136_0', 'data_image/hurricane_irma/17_9_2017/909481301274587136_0.jpg', 'train', None), ('905631022540357632_0', 'data_image/hurricane_irma/7_9_2017/905631022540357632_0.jpg', 'test', None), ('910109472231653376_0', 'data_image/hurricane_irma/19_9_2017/910109472231653376_0.jpg', 'dev', None), ('909775809170337792_0', 'data_image/hurricane_irma/18_9_2017/909775809170337792_0.jpg', 'train', None), ('909774149698179072_2', 'data_image/hurricane_irma/18_9_2017/909774149698179072_2.jpg', 'train', None), ('909418672913096704_0', 'data_image/hurricane_irma/17_9_2017/909418672913096704_0.jpg', 'dev', None), ('909404734460743680_0', 'data_image/hurricane_irma/17_9_2017/909404734460743680_0.jpg', 'train', None), ('909454440574390272_0', 'data_image/hurricane_irma/17_9_2017/909454440574390272_0.jpg', 'dev', None), ('911566354016325632_1', 'data_image/hurricane_maria/23_9_2017/911566354016325632_1.jpg', 'train', None), ('916027872212656129_2', 'data_image/hurricane_maria/5_10_2017/916027872212656129_2.jpg', 'dev', None), ('913903407718690816_0', 'data_image/hurricane_maria/29_9_2017/913903407718690816_0.jpg', 'train', None), ('913116036676882435_0', 'data_image/hurricane_maria/27_9_2017/913116036676882435_0.jpg', 'dev', None), ('914353354503069696_2', 'data_image/hurricane_maria/1_10_2017/914353354503069696_2.jpg', 'dev', None), ('913047529494892546_0', 'data_image/hurricane_maria/27_9_2017/913047529494892546_0.jpg', 'train', None), ('922934077019316225_0', 'data_image/hurricane_maria/24_10_2017/922934077019316225_0.jpg', 'train', None), ('914125586494365696_0', 'data_image/hurricane_maria/30_9_2017/914125586494365696_0.jpg', 'train', None), ('929457105022615552_0', 'data_image/hurricane_maria/11_11_2017/929457105022615552_0.jpg', 'train', None), ('910670814847967233_0', 'data_image/hurricane_maria/21_9_2017/910670814847967233_0.jpg', 'train', None), ('913901444662808576_0', 'data_image/hurricane_maria/29_9_2017/913901444662808576_0.jpg', 'test', None), ('913445362660233217_0', 'data_image/hurricane_maria/28_9_2017/913445362660233217_0.jpg', 'dev', None), ('913127232377380865_0', 'data_image/hurricane_maria/27_9_2017/913127232377380865_0.jpg', 'train', None), ('911634445035278337_0', 'data_image/hurricane_maria/23_9_2017/911634445035278337_0.jpg', 'test', None), ('917920843430334464_0', 'data_image/hurricane_maria/11_10_2017/917920843430334464_0.jpg', 'dev', None), ('913846340265873409_0', 'data_image/hurricane_maria/29_9_2017/913846340265873409_0.jpg', 'train', None), ('923365901138833409_2', 'data_image/hurricane_maria/26_10_2017/923365901138833409_2.jpg', 'dev', None), ('931017749375184897_0', 'data_image/iraq_iran_earthquake/16_11_2017/931017749375184897_0.jpg', 'dev', None), ('930019166907719681_0', 'data_image/iraq_iran_earthquake/13_11_2017/930019166907719681_0.jpg', 'test', None), ('930070681080999936_0', 'data_image/iraq_iran_earthquake/13_11_2017/930070681080999936_0.jpg', 'train', None), ('930135734056640512_0', 'data_image/iraq_iran_earthquake/13_11_2017/930135734056640512_0.jpg', 'test', None), ('930018939962306560_0', 'data_image/iraq_iran_earthquake/13_11_2017/930018939962306560_0.jpg', 'dev', None), ('931789781087981568_0', 'data_image/iraq_iran_earthquake/18_11_2017/931789781087981568_0.jpg', 'train', None), ('931174405945704448_0', 'data_image/iraq_iran_earthquake/16_11_2017/931174405945704448_0.jpg', 'dev', None), ('911962200402784259_0', 'data_image/mexico_earthquake/24_9_2017/911962200402784259_0.jpg', 'train', None), ('914952139209891852_0', 'data_image/mexico_earthquake/2_10_2017/914952139209891852_0.jpg', 'dev', None), ('910525541454934019_0', 'data_image/mexico_earthquake/20_9_2017/910525541454934019_0.jpg', 'dev', None), ('910610131641085952_0', 'data_image/mexico_earthquake/20_9_2017/910610131641085952_0.jpg', 'train', None), ('911861625895911424_1', 'data_image/mexico_earthquake/24_9_2017/911861625895911424_1.jpg', 'train', None), ('913682539985825792_0', 'data_image/mexico_earthquake/29_9_2017/913682539985825792_0.jpg', 'dev', None), ('910719199785472001_0', 'data_image/mexico_earthquake/21_9_2017/910719199785472001_0.jpg', 'test', None), ('911723538863734784_0', 'data_image/mexico_earthquake/23_9_2017/911723538863734784_0.jpg', 'train', None), ('910670466356072448_3', 'data_image/mexico_earthquake/21_9_2017/910670466356072448_3.jpg', 'train', None), ('910749888320868353_0', 'data_image/mexico_earthquake/21_9_2017/910749888320868353_0.jpg', 'train', None), ('911623548061474816_0', 'data_image/mexico_earthquake/23_9_2017/911623548061474816_0.png', 'train', None), ('870165289576521730_0', 'data_image/srilanka_floods/1_6_2017/870165289576521730_0.jpg', 'dev', None), ('869957051547435008_0', 'data_image/srilanka_floods/31_5_2017/869957051547435008_0.jpg', 'train', None), ('871479454119247872_0', 'data_image/srilanka_floods/4_6_2017/871479454119247872_0.jpg', 'train', None), ('879810253310877697_0', 'data_image/srilanka_floods/27_6_2017/879810253310877697_0.jpg', 'train', None), ('879142017330282497_0', 'data_image/srilanka_floods/26_6_2017/879142017330282497_0.jpg', 'test', None), ('872175891929063426_0', 'data_image/srilanka_floods/6_6_2017/872175891929063426_0.jpg', 'test', None), ('922171312516288513_0', 'data_image/california_wildfires/22_10_2017/922171312516288513_0.jpg', 'train', None), ('917921510731452416_0', 'data_image/california_wildfires/11_10_2017/917921510731452416_0.jpg', 'dev', None), ('917804456422473734_0', 'data_image/california_wildfires/10_10_2017/917804456422473734_0.jpg', 'test', None), ('923508260128329729_0', 'data_image/california_wildfires/26_10_2017/923508260128329729_0.jpg', 'dev', None), ('918235406512074752_0', 'data_image/california_wildfires/11_10_2017/918235406512074752_0.jpg', 'test', None), ('917962781646819328_0', 'data_image/california_wildfires/11_10_2017/917962781646819328_0.jpg', 'train', None), ('923185030804967424_0', 'data_image/california_wildfires/25_10_2017/923185030804967424_0.jpg', 'train', None), ('921058369170964480_0', 'data_image/california_wildfires/19_10_2017/921058369170964480_0.jpg', 'train', None), ('918052949078761473_0', 'data_image/california_wildfires/11_10_2017/918052949078761473_0.jpg', 'dev', None), ('920184377501470720_0', 'data_image/california_wildfires/17_10_2017/920184377501470720_0.jpg', 'dev', None), ('919718726890545152_1', 'data_image/california_wildfires/16_10_2017/919718726890545152_1.jpg', 'train', None), ('917819230732603397_0', 'data_image/california_wildfires/10_10_2017/917819230732603397_0.jpg', 'test', None), ('920143388590080001_0', 'data_image/california_wildfires/17_10_2017/920143388590080001_0.jpg', 'test', None), ('907491611222745088_0', 'data_image/hurricane_harvey/12_9_2017/907491611222745088_0.jpg', 'train', None), ('904670957247373312_0', 'data_image/hurricane_harvey/4_9_2017/904670957247373312_0.jpg', 'dev', None), ('908804979686641664_0', 'data_image/hurricane_harvey/15_9_2017/908804979686641664_0.jpg', 'train', None), ('906744961072291840_0', 'data_image/hurricane_harvey/10_9_2017/906744961072291840_0.jpg', 'train', None), ('901822661080244224_0', 'data_image/hurricane_harvey/27_8_2017/901822661080244224_0.jpg', 'train', None), ('910129169417199616_0', 'data_image/hurricane_harvey/19_9_2017/910129169417199616_0.jpg', 'dev', None), ('907214672809656320_0', 'data_image/hurricane_harvey/11_9_2017/907214672809656320_0.jpg', 'train', None), ('904669405585457152_0', 'data_image/hurricane_harvey/4_9_2017/904669405585457152_0.jpg', 'train', None), ('905797452095143936_0', 'data_image/hurricane_harvey/7_9_2017/905797452095143936_0.jpg', 'train', None), ('906972555537473536_0', 'data_image/hurricane_harvey/10_9_2017/906972555537473536_0.jpg', 'train', None), ('908846423680049152_0', 'data_image/hurricane_harvey/16_9_2017/908846423680049152_0.jpg', 'train', None), ('907413959711567873_0', 'data_image/hurricane_harvey/12_9_2017/907413959711567873_0.jpg', 'test', None), ('906758959977127937_0', 'data_image/hurricane_harvey/10_9_2017/906758959977127937_0.jpg', 'train', None), ('905137837665869825_0', 'data_image/hurricane_harvey/5_9_2017/905137837665869825_0.jpg', 'test', None), ('906081532141600768_0', 'data_image/hurricane_harvey/8_9_2017/906081532141600768_0.jpg', 'dev', None), ('909880473790369792_0', 'data_image/hurricane_irma/18_9_2017/909880473790369792_0.jpg', 'train', None), ('910107780467617792_0', 'data_image/hurricane_irma/19_9_2017/910107780467617792_0.jpg', 'dev', None), ('909971209290440706_1', 'data_image/hurricane_irma/19_9_2017/909971209290440706_1.jpg', 'train', None), ('909452163801219072_0', 'data_image/hurricane_irma/17_9_2017/909452163801219072_0.jpg', 'test', None), ('909888420905930754_0', 'data_image/hurricane_irma/18_9_2017/909888420905930754_0.jpg', 'train', None), ('909780742787432452_2', 'data_image/hurricane_irma/18_9_2017/909780742787432452_2.jpg', 'train', None), ('909829523339964417_0', 'data_image/hurricane_irma/18_9_2017/909829523339964417_0.jpg', 'dev', None), ('905625434955673601_0', 'data_image/hurricane_irma/7_9_2017/905625434955673601_0.jpg', 'dev', None), ('906927885226737671_0', 'data_image/hurricane_irma/10_9_2017/906927885226737671_0.jpg', 'dev', None), ('909945133520510976_0', 'data_image/hurricane_irma/19_9_2017/909945133520510976_0.jpg', 'train', None), ('909479261748305921_0', 'data_image/hurricane_irma/17_9_2017/909479261748305921_0.jpg', 'test', None), ('905700072259690496_0', 'data_image/hurricane_irma/7_9_2017/905700072259690496_0.jpg', 'test', None), ('910112984315416576_0', 'data_image/hurricane_irma/19_9_2017/910112984315416576_0.jpg', 'train', None), ('910171693636833280_0', 'data_image/hurricane_irma/19_9_2017/910171693636833280_0.jpg', 'train', None), ('910158619622199296_0', 'data_image/hurricane_irma/19_9_2017/910158619622199296_0.jpg', 'train', None), ('909469971469733889_0', 'data_image/hurricane_irma/17_9_2017/909469971469733889_0.jpg', 'train', None), ('912454288596709376_0', 'data_image/hurricane_maria/25_9_2017/912454288596709376_0.jpg', 'train', None), ('913549726083141632_2', 'data_image/hurricane_maria/28_9_2017/913549726083141632_2.jpg', 'test', None), ('913236041397669888_0', 'data_image/hurricane_maria/28_9_2017/913236041397669888_0.jpg', 'train', None), ('922911846457364481_0', 'data_image/hurricane_maria/24_10_2017/922911846457364481_0.jpg', 'train', None), ('913029358444564480_0', 'data_image/hurricane_maria/27_9_2017/913029358444564480_0.jpg', 'train', None), ('912313952096407552_0', 'data_image/hurricane_maria/25_9_2017/912313952096407552_0.jpg', 'dev', None), ('915300171872919552_0', 'data_image/hurricane_maria/3_10_2017/915300171872919552_0.jpg', 'train', None), ('921473876781162496_0', 'data_image/hurricane_maria/20_10_2017/921473876781162496_0.jpg', 'train', None), ('914540213875200000_3', 'data_image/hurricane_maria/1_10_2017/914540213875200000_3.jpg', 'train', None), ('911984028630306816_3', 'data_image/hurricane_maria/24_9_2017/911984028630306816_3.jpg', 'dev', None), ('912114897869209600_0', 'data_image/hurricane_maria/25_9_2017/912114897869209600_0.jpg', 'dev', None), ('910711979974680576_0', 'data_image/hurricane_maria/21_9_2017/910711979974680576_0.jpg', 'dev', None), ('929020545194188800_0', 'data_image/hurricane_maria/10_11_2017/929020545194188800_0.jpg', 'train', None), ('912324981702500352_0', 'data_image/hurricane_maria/25_9_2017/912324981702500352_0.jpg', 'dev', None), ('910704333041303552_0', 'data_image/hurricane_maria/21_9_2017/910704333041303552_0.jpg', 'train', None), ('922865890844016641_0', 'data_image/hurricane_maria/24_10_2017/922865890844016641_0.jpg', 'train', None), ('931046661274701825_0', 'data_image/iraq_iran_earthquake/16_11_2017/931046661274701825_0.jpg', 'train', None), ('930209629589721088_0', 'data_image/iraq_iran_earthquake/13_11_2017/930209629589721088_0.jpg', 'test', None), ('929996293245718528_1', 'data_image/iraq_iran_earthquake/13_11_2017/929996293245718528_1.jpg', 'train', None), ('930035325480722432_0', 'data_image/iraq_iran_earthquake/13_11_2017/930035325480722432_0.jpg', 'train', None), ('930015299017105408_0', 'data_image/iraq_iran_earthquake/13_11_2017/930015299017105408_0.jpg', 'train', None), ('930016900943314944_0', 'data_image/iraq_iran_earthquake/13_11_2017/930016900943314944_0.jpg', 'dev', None), ('931531506564157440_0', 'data_image/iraq_iran_earthquake/17_11_2017/931531506564157440_0.jpg', 'train', None), ('911642846507053056_0', 'data_image/mexico_earthquake/23_9_2017/911642846507053056_0.jpg', 'test', None), ('910545977668505601_0', 'data_image/mexico_earthquake/20_9_2017/910545977668505601_0.jpg', 'train', None), ('911654962916143104_0', 'data_image/mexico_earthquake/23_9_2017/911654962916143104_0.jpg', 'test', None), ('910696335812243458_0', 'data_image/mexico_earthquake/21_9_2017/910696335812243458_0.jpg', 'test', None), ('914181646383042562_0', 'data_image/mexico_earthquake/30_9_2017/914181646383042562_0.jpg', 'train', None), ('910541591496011776_0', 'data_image/mexico_earthquake/20_9_2017/910541591496011776_0.jpg', 'train', None), ('910709821392879616_0', 'data_image/mexico_earthquake/21_9_2017/910709821392879616_0.jpg', 'train', None), ('912277770918653952_0', 'data_image/mexico_earthquake/25_9_2017/912277770918653952_0.jpg', 'train', None), ('910650995415195648_0', 'data_image/mexico_earthquake/20_9_2017/910650995415195648_0.jpg', 'dev', None), ('914184287062073344_0', 'data_image/mexico_earthquake/30_9_2017/914184287062073344_0.jpg', 'train', None), ('910725304909389824_0', 'data_image/mexico_earthquake/21_9_2017/910725304909389824_0.jpg', 'train', None), ('870244871629451264_0', 'data_image/srilanka_floods/1_6_2017/870244871629451264_0.jpg', 'train', None), ('875502695922139137_0', 'data_image/srilanka_floods/15_6_2017/875502695922139137_0.jpg', 'test', None), ('875350400278441985_0', 'data_image/srilanka_floods/15_6_2017/875350400278441985_0.jpg', 'dev', None), ('873327683014471682_0', 'data_image/srilanka_floods/9_6_2017/873327683014471682_0.jpg', 'dev', None), ('870295153872908288_0', 'data_image/srilanka_floods/1_6_2017/870295153872908288_0.jpg', 'test', None), ('871880350238203904_0', 'data_image/srilanka_floods/6_6_2017/871880350238203904_0.jpg', 'dev', None), ('919907904634916864_0', 'data_image/california_wildfires/16_10_2017/919907904634916864_0.jpg', 'test', None), ('921089830074929153_0', 'data_image/california_wildfires/19_10_2017/921089830074929153_0.jpg', 'train', None), ('922568378367086593_0', 'data_image/california_wildfires/23_10_2017/922568378367086593_0.jpg', 'train', None), ('920420446037839872_0', 'data_image/california_wildfires/17_10_2017/920420446037839872_0.jpg', 'train', None), ('920753988068356096_0', 'data_image/california_wildfires/18_10_2017/920753988068356096_0.jpg', 'train', None), ('917794024173563904_0', 'data_image/california_wildfires/10_10_2017/917794024173563904_0.jpg', 'train', None), ('919594534488731653_0', 'data_image/california_wildfires/15_10_2017/919594534488731653_0.jpg', 'train', None), ('921537209043369984_0', 'data_image/california_wildfires/21_10_2017/921537209043369984_0.jpg', 'dev', None), ('921051431368515584_0', 'data_image/california_wildfires/19_10_2017/921051431368515584_0.png', 'train', None), ('917797601763889160_0', 'data_image/california_wildfires/10_10_2017/917797601763889160_0.jpg', 'dev', None), ('920474622205628416_0', 'data_image/california_wildfires/18_10_2017/920474622205628416_0.jpg', 'train', None), ('917914955118272513_0', 'data_image/california_wildfires/11_10_2017/917914955118272513_0.jpg', 'test', None), ('919966039756111873_0', 'data_image/california_wildfires/16_10_2017/919966039756111873_0.jpg', 'test', None), ('901788499430518784_0', 'data_image/hurricane_harvey/27_8_2017/901788499430518784_0.jpg', 'test', None), ('904947206607818753_0', 'data_image/hurricane_harvey/5_9_2017/904947206607818753_0.jpg', 'test', None), ('905862761263104001_0', 'data_image/hurricane_harvey/7_9_2017/905862761263104001_0.jpg', 'dev', None), ('905028821858082816_0', 'data_image/hurricane_harvey/5_9_2017/905028821858082816_0.jpg', 'test', None), ('909841771286339584_0', 'data_image/hurricane_harvey/18_9_2017/909841771286339584_0.jpg', 'train', None), ('904503986149056513_0', 'data_image/hurricane_harvey/4_9_2017/904503986149056513_0.jpg', 'train', None), ('905947898155864064_0', 'data_image/hurricane_harvey/8_9_2017/905947898155864064_0.jpg', 'test', None), ('905855941484171264_0', 'data_image/hurricane_harvey/7_9_2017/905855941484171264_0.jpg', 'test', None), ('906199066966667264_0', 'data_image/hurricane_harvey/8_9_2017/906199066966667264_0.jpg', 'test', None), ('905549199974031362_0', 'data_image/hurricane_harvey/6_9_2017/905549199974031362_0.jpg', 'train', None), ('906347707748380672_0', 'data_image/hurricane_harvey/9_9_2017/906347707748380672_0.jpg', 'test', None), ('906306393501712384_0', 'data_image/hurricane_harvey/9_9_2017/906306393501712384_0.jpg', 'test', None), ('905796606796197888_0', 'data_image/hurricane_harvey/7_9_2017/905796606796197888_0.jpg', 'test', None), ('904357513914716160_0', 'data_image/hurricane_harvey/3_9_2017/904357513914716160_0.jpg', 'test', None), ('909087043258642432_0', 'data_image/hurricane_harvey/16_9_2017/909087043258642432_0.jpg', 'train', None), ('909779008145018880_0', 'data_image/hurricane_irma/18_9_2017/909779008145018880_0.jpg', 'test', None), ('910140800658497537_0', 'data_image/hurricane_irma/19_9_2017/910140800658497537_0.jpg', 'dev', None), ('909497199100399616_0', 'data_image/hurricane_irma/17_9_2017/909497199100399616_0.jpg', 'test', None), ('910183007637561344_0', 'data_image/hurricane_irma/19_9_2017/910183007637561344_0.jpg', 'test', None), ('909455928851910656_0', 'data_image/hurricane_irma/17_9_2017/909455928851910656_0.jpg', 'dev', None), ('910207153817894912_0', 'data_image/hurricane_irma/19_9_2017/910207153817894912_0.jpg', 'dev', None), ('905720563435438080_0', 'data_image/hurricane_irma/7_9_2017/905720563435438080_0.jpg', 'dev', None), ('910235206308831232_0', 'data_image/hurricane_irma/19_9_2017/910235206308831232_0.jpg', 'dev', None), ('909946459692392448_0', 'data_image/hurricane_irma/19_9_2017/909946459692392448_0.jpg', 'train', None), ('909854852603367424_0', 'data_image/hurricane_irma/18_9_2017/909854852603367424_0.jpg', 'test', None), ('905760814413598721_0', 'data_image/hurricane_irma/7_9_2017/905760814413598721_0.jpg', 'dev', None), ('909377238532943872_0', 'data_image/hurricane_irma/17_9_2017/909377238532943872_0.jpg', 'test', None), ('909487005779841025_0', 'data_image/hurricane_irma/17_9_2017/909487005779841025_0.jpg', 'train', None), ('909793408352440321_0', 'data_image/hurricane_irma/18_9_2017/909793408352440321_0.jpg', 'test', None), ('909717907147337729_0', 'data_image/hurricane_irma/18_9_2017/909717907147337729_0.jpg', 'train', None), ('905625440093687809_0', 'data_image/hurricane_irma/7_9_2017/905625440093687809_0.jpg', 'train', None), ('916019382857420800_0', 'data_image/hurricane_maria/5_10_2017/916019382857420800_0.jpg', 'train', None), ('922153859354677254_0', 'data_image/hurricane_maria/22_10_2017/922153859354677254_0.jpg', 'dev', None), ('927131866523676673_0', 'data_image/hurricane_maria/5_11_2017/927131866523676673_0.jpg', 'test', None), ('914154101600112640_0', 'data_image/hurricane_maria/30_9_2017/914154101600112640_0.jpg', 'test', None), ('921891995420889088_0', 'data_image/hurricane_maria/22_10_2017/921891995420889088_0.jpg', 'train', None), ('914111723824705536_0', 'data_image/hurricane_maria/30_9_2017/914111723824705536_0.jpg', 'dev', None), ('913229700771844096_0', 'data_image/hurricane_maria/28_9_2017/913229700771844096_0.jpg', 'train', None), ('910646555673104385_0', 'data_image/hurricane_maria/20_9_2017/910646555673104385_0.jpg', 'train', None), ('915995949662576645_1', 'data_image/hurricane_maria/5_10_2017/915995949662576645_1.jpg', 'train', None), ('913934298243305473_2', 'data_image/hurricane_maria/30_9_2017/913934298243305473_2.jpg', 'train', None), ('911634025844011008_0', 'data_image/hurricane_maria/23_9_2017/911634025844011008_0.jpg', 'dev', None), ('910640322522251264_0', 'data_image/hurricane_maria/20_9_2017/910640322522251264_0.jpg', 'dev', None), ('913735106371735553_0', 'data_image/hurricane_maria/29_9_2017/913735106371735553_0.jpg', 'train', None), ('923188843712454656_0', 'data_image/hurricane_maria/25_10_2017/923188843712454656_0.jpg', 'train', None), ('928616273226944512_2', 'data_image/hurricane_maria/9_11_2017/928616273226944512_2.jpg', 'train', None), ('913916479002828800_2', 'data_image/hurricane_maria/30_9_2017/913916479002828800_2.jpg', 'train', None), ('930898405399855104_0', 'data_image/iraq_iran_earthquake/15_11_2017/930898405399855104_0.jpg', 'dev', None), ('930012355009941505_0', 'data_image/iraq_iran_earthquake/13_11_2017/930012355009941505_0.jpg', 'test', None), ('930496036216496129_0', 'data_image/iraq_iran_earthquake/14_11_2017/930496036216496129_0.jpg', 'test', None), ('930202675727216640_0', 'data_image/iraq_iran_earthquake/13_11_2017/930202675727216640_0.jpg', 'dev', None), ('930193943408586752_2', 'data_image/iraq_iran_earthquake/13_11_2017/930193943408586752_2.jpg', 'train', None), ('930185231960653824_0', 'data_image/iraq_iran_earthquake/13_11_2017/930185231960653824_0.jpg', 'dev', None), ('929989743810576384_0', 'data_image/iraq_iran_earthquake/13_11_2017/929989743810576384_0.jpg', 'dev', None), ('914882705074122754_0', 'data_image/mexico_earthquake/2_10_2017/914882705074122754_0.jpg', 'train', None), ('912427774576644096_0', 'data_image/mexico_earthquake/25_9_2017/912427774576644096_0.jpg', 'dev', None), ('912048104651788289_1', 'data_image/mexico_earthquake/24_9_2017/912048104651788289_1.jpg', 'train', None), ('910541788024360960_0', 'data_image/mexico_earthquake/20_9_2017/910541788024360960_0.jpg', 'train', None), ('911928966868484097_0', 'data_image/mexico_earthquake/24_9_2017/911928966868484097_0.jpg', 'train', None), ('911846702558138368_0', 'data_image/mexico_earthquake/24_9_2017/911846702558138368_0.jpg', 'test', None), ('910708925548384256_0', 'data_image/mexico_earthquake/21_9_2017/910708925548384256_0.jpg', 'test', None), ('910682398907617280_0', 'data_image/mexico_earthquake/21_9_2017/910682398907617280_0.jpg', 'train', None), ('910655575238651905_0', 'data_image/mexico_earthquake/21_9_2017/910655575238651905_0.jpg', 'train', None), ('912947444996067328_0', 'data_image/mexico_earthquake/27_9_2017/912947444996067328_0.jpg', 'train', None), ('910688565109039105_0', 'data_image/mexico_earthquake/21_9_2017/910688565109039105_0.jpg', 'train', None), ('871269851108708357_1', 'data_image/srilanka_floods/4_6_2017/871269851108708357_1.jpg', 'train', None), ('869954479524691968_0', 'data_image/srilanka_floods/31_5_2017/869954479524691968_0.jpg', 'train', None), ('870938078629609472_0', 'data_image/srilanka_floods/3_6_2017/870938078629609472_0.jpg', 'train', None), ('873653821121212417_0', 'data_image/srilanka_floods/10_6_2017/873653821121212417_0.jpg', 'train', None), ('877964234826555392_0', 'data_image/srilanka_floods/22_6_2017/877964234826555392_0.jpg', 'test', None), ('878109177213722626_0', 'data_image/srilanka_floods/23_6_2017/878109177213722626_0.jpg', 'train', None), ('919935865631129600_0', 'data_image/california_wildfires/16_10_2017/919935865631129600_0.jpg', 'train', None), ('919752558926802944_0', 'data_image/california_wildfires/16_10_2017/919752558926802944_0.jpg', 'test', None), ('920202029741383680_0', 'data_image/california_wildfires/17_10_2017/920202029741383680_0.jpg', 'dev', None), ('920081216837115904_0', 'data_image/california_wildfires/17_10_2017/920081216837115904_0.jpg', 'train', None), ('917990093876539398_0', 'data_image/california_wildfires/11_10_2017/917990093876539398_0.jpg', 'test', None), ('920589043712196609_0', 'data_image/california_wildfires/18_10_2017/920589043712196609_0.jpg', 'test', None), ('921739965981122561_0', 'data_image/california_wildfires/21_10_2017/921739965981122561_0.png', 'dev', None), ('922948121939927040_2', 'data_image/california_wildfires/24_10_2017/922948121939927040_2.jpg', 'train', None), ('922235973752279041_0', 'data_image/california_wildfires/22_10_2017/922235973752279041_0.jpg', 'dev', None), ('921623155835224064_3', 'data_image/california_wildfires/21_10_2017/921623155835224064_3.jpg', 'train', None), ('918031205551034368_0', 'data_image/california_wildfires/11_10_2017/918031205551034368_0.jpg', 'dev', None), ('919573605700083713_0', 'data_image/california_wildfires/15_10_2017/919573605700083713_0.jpg', 'train', None), ('907292764164866049_0', 'data_image/hurricane_harvey/11_9_2017/907292764164866049_0.jpg', 'test', None), ('904464789811863552_0', 'data_image/hurricane_harvey/3_9_2017/904464789811863552_0.jpg', 'dev', None), ('908631770538610688_0', 'data_image/hurricane_harvey/15_9_2017/908631770538610688_0.jpg', 'test', None), ('905968597905833988_0', 'data_image/hurricane_harvey/8_9_2017/905968597905833988_0.jpg', 'dev', None), ('904689705954013184_0', 'data_image/hurricane_harvey/4_9_2017/904689705954013184_0.jpg', 'test', None), ('909054527151706112_0', 'data_image/hurricane_harvey/16_9_2017/909054527151706112_0.jpg', 'train', None), ('908399918297047042_0', 'data_image/hurricane_harvey/14_9_2017/908399918297047042_0.jpg', 'train', None), ('907345772823408642_0', 'data_image/hurricane_harvey/11_9_2017/907345772823408642_0.jpg', 'train', None), ('905960553830522880_0', 'data_image/hurricane_harvey/8_9_2017/905960553830522880_0.jpg', 'test', None), ('906563869702856706_0', 'data_image/hurricane_harvey/9_9_2017/906563869702856706_0.jpg', 'train', None), ('906381678465241088_0', 'data_image/hurricane_harvey/9_9_2017/906381678465241088_0.jpg', 'train', None), ('908031796092252162_0', 'data_image/hurricane_harvey/13_9_2017/908031796092252162_0.jpg', 'train', None), ('905922126779228161_0', 'data_image/hurricane_harvey/7_9_2017/905922126779228161_0.jpg', 'train', None), ('907473268684161024_0', 'data_image/hurricane_harvey/12_9_2017/907473268684161024_0.jpg', 'train', None), ('910160558011318279_0', 'data_image/hurricane_harvey/19_9_2017/910160558011318279_0.jpg', 'train', None), ('909458363850862592_0', 'data_image/hurricane_irma/17_9_2017/909458363850862592_0.jpg', 'test', None), ('909879652306575360_0', 'data_image/hurricane_irma/18_9_2017/909879652306575360_0.jpg', 'test', None), ('909484767497900032_0', 'data_image/hurricane_irma/17_9_2017/909484767497900032_0.jpg', 'dev', None), ('909714816943697920_0', 'data_image/hurricane_irma/18_9_2017/909714816943697920_0.jpg', 'test', None), ('910209322038284288_0', 'data_image/hurricane_irma/19_9_2017/910209322038284288_0.jpg', 'dev', None), ('910159144535158784_0', 'data_image/hurricane_irma/19_9_2017/910159144535158784_0.jpg', 'test', None), ('910261997660459013_1', 'data_image/hurricane_irma/19_9_2017/910261997660459013_1.jpg', 'train', None), ('909798840471080961_0', 'data_image/hurricane_irma/18_9_2017/909798840471080961_0.jpg', 'test', None), ('909380841276678144_0', 'data_image/hurricane_irma/17_9_2017/909380841276678144_0.jpg', 'train', None), ('910100808049451008_0', 'data_image/hurricane_irma/19_9_2017/910100808049451008_0.jpg', 'train', None), ('910028182560047104_0', 'data_image/hurricane_irma/19_9_2017/910028182560047104_0.jpg', 'test', None), ('909942371554594816_0', 'data_image/hurricane_irma/19_9_2017/909942371554594816_0.jpg', 'test', None), ('910193034104414209_0', 'data_image/hurricane_irma/19_9_2017/910193034104414209_0.jpg', 'test', None), ('909415252705542144_0', 'data_image/hurricane_irma/17_9_2017/909415252705542144_0.jpg', 'test', None), ('909757693749075968_0', 'data_image/hurricane_irma/18_9_2017/909757693749075968_0.jpg', 'train', None), ('914776007873200128_0', 'data_image/hurricane_maria/2_10_2017/914776007873200128_0.jpg', 'train', None), ('913051514419269632_0', 'data_image/hurricane_maria/27_9_2017/913051514419269632_0.jpg', 'dev', None), ('910616491799834628_0', 'data_image/hurricane_maria/20_9_2017/910616491799834628_0.jpg', 'dev', None), ('911551093083131905_0', 'data_image/hurricane_maria/23_9_2017/911551093083131905_0.jpg', 'train', None), ('928789771463340034_0', 'data_image/hurricane_maria/10_11_2017/928789771463340034_0.jpg', 'test', None), ('912097936200355841_0', 'data_image/hurricane_maria/24_9_2017/912097936200355841_0.jpg', 'train', None), ('916248174196658178_0', 'data_image/hurricane_maria/6_10_2017/916248174196658178_0.jpg', 'train', None), ('927146873269317632_0', 'data_image/hurricane_maria/5_11_2017/927146873269317632_0.jpg', 'train', None), ('911935529905684481_0', 'data_image/hurricane_maria/24_9_2017/911935529905684481_0.jpg', 'train', None), ('923767779488141312_0', 'data_image/hurricane_maria/27_10_2017/923767779488141312_0.jpg', 'dev', None), ('911735538436231168_0', 'data_image/hurricane_maria/23_9_2017/911735538436231168_0.jpg', 'dev', None), ('911726520951066624_0', 'data_image/hurricane_maria/23_9_2017/911726520951066624_0.jpg', 'test', None), ('915294577476689921_0', 'data_image/hurricane_maria/3_10_2017/915294577476689921_0.jpg', 'test', None), ('927192806401171456_1', 'data_image/hurricane_maria/5_11_2017/927192806401171456_1.jpg', 'train', None), ('922676531536846848_0', 'data_image/hurricane_maria/24_10_2017/922676531536846848_0.jpg', 'train', None), ('931046661274701825_2', 'data_image/iraq_iran_earthquake/16_11_2017/931046661274701825_2.jpg', 'train', None), ('930028600992165888_2', 'data_image/iraq_iran_earthquake/13_11_2017/930028600992165888_2.jpg', 'train', None), ('930334010248593409_0', 'data_image/iraq_iran_earthquake/14_11_2017/930334010248593409_0.jpg', 'test', None), ('930028106668179458_0', 'data_image/iraq_iran_earthquake/13_11_2017/930028106668179458_0.jpg', 'train', None), ('930114382268334080_3', 'data_image/iraq_iran_earthquake/13_11_2017/930114382268334080_3.jpg', 'train', None), ('930596309886799872_0', 'data_image/iraq_iran_earthquake/15_11_2017/930596309886799872_0.jpg', 'train', None), ('931490440305692672_0', 'data_image/iraq_iran_earthquake/17_11_2017/931490440305692672_0.jpg', 'test', None), ('910614165076750337_3', 'data_image/mexico_earthquake/20_9_2017/910614165076750337_3.jpg', 'train', None), ('910709499660398592_0', 'data_image/mexico_earthquake/21_9_2017/910709499660398592_0.jpg', 'dev', None), ('910566301306732544_0', 'data_image/mexico_earthquake/20_9_2017/910566301306732544_0.jpg', 'train', None), ('911634888998055936_1', 'data_image/mexico_earthquake/23_9_2017/911634888998055936_1.jpg', 'train', None), ('914993292290469888_3', 'data_image/mexico_earthquake/2_10_2017/914993292290469888_3.jpg', 'train', None), ('910545578001600512_0', 'data_image/mexico_earthquake/20_9_2017/910545578001600512_0.jpg', 'dev', None), ('910765490334814208_0', 'data_image/mexico_earthquake/21_9_2017/910765490334814208_0.jpg', 'train', None), ('910551115049168896_0', 'data_image/mexico_earthquake/20_9_2017/910551115049168896_0.jpg', 'train', None), ('913560085837766657_0', 'data_image/mexico_earthquake/29_9_2017/913560085837766657_0.jpg', 'train', None), ('910635649501323269_0', 'data_image/mexico_earthquake/20_9_2017/910635649501323269_0.jpg', 'test', None), ('911690588260462592_2', 'data_image/mexico_earthquake/23_9_2017/911690588260462592_2.jpg', 'train', None), ('877761736002027520_0', 'data_image/srilanka_floods/22_6_2017/877761736002027520_0.jpg', 'dev', None), ('871328241956597761_0', 'data_image/srilanka_floods/4_6_2017/871328241956597761_0.jpg', 'test', None), ('871514121799213058_0', 'data_image/srilanka_floods/4_6_2017/871514121799213058_0.jpg', 'train', None), ('875073044510679040_3', 'data_image/srilanka_floods/14_6_2017/875073044510679040_3.jpg', 'train', None), ('874772578971394048_0', 'data_image/srilanka_floods/13_6_2017/874772578971394048_0.jpg', 'train', None), ('870605397538856960_0', 'data_image/srilanka_floods/2_6_2017/870605397538856960_0.jpg', 'train', None), ('921630495221473281_0', 'data_image/california_wildfires/21_10_2017/921630495221473281_0.jpg', 'test', None), ('918009550409752576_0', 'data_image/california_wildfires/11_10_2017/918009550409752576_0.jpg', 'train', None), ('918394689107226624_0', 'data_image/california_wildfires/12_10_2017/918394689107226624_0.jpg', 'train', None), ('917844223021293569_0', 'data_image/california_wildfires/10_10_2017/917844223021293569_0.jpg', 'test', None), ('919985668692103169_0', 'data_image/california_wildfires/16_10_2017/919985668692103169_0.jpg', 'train', None), ('919634966321000448_0', 'data_image/california_wildfires/15_10_2017/919634966321000448_0.jpg', 'train', None), ('920463405508677633_0', 'data_image/california_wildfires/18_10_2017/920463405508677633_0.jpg', 'dev', None), ('919639781432164352_0', 'data_image/california_wildfires/15_10_2017/919639781432164352_0.jpg', 'dev', None), ('920071861211357185_0', 'data_image/california_wildfires/16_10_2017/920071861211357185_0.jpg', 'train', None), ('922629061033197568_0', 'data_image/california_wildfires/24_10_2017/922629061033197568_0.jpg', 'test', None), ('920281575748788224_1', 'data_image/california_wildfires/17_10_2017/920281575748788224_1.jpg', 'dev', None), ('917803905681121280_0', 'data_image/california_wildfires/10_10_2017/917803905681121280_0.jpg', 'test', None), ('905060535674298370_3', 'data_image/hurricane_harvey/5_9_2017/905060535674298370_3.jpg', 'train', None), ('908168392716234753_0', 'data_image/hurricane_harvey/14_9_2017/908168392716234753_0.jpg', 'test', None), ('904302986826014720_0', 'data_image/hurricane_harvey/3_9_2017/904302986826014720_0.jpg', 'test', None), ('904441419523293188_0', 'data_image/hurricane_harvey/3_9_2017/904441419523293188_0.jpg', 'test', None), ('905762330755158017_0', 'data_image/hurricane_harvey/7_9_2017/905762330755158017_0.jpg', 'train', None), ('906255309123018752_0', 'data_image/hurricane_harvey/8_9_2017/906255309123018752_0.jpg', 'train', None), ('906576414362755072_1', 'data_image/hurricane_harvey/9_9_2017/906576414362755072_1.jpg', 'train', None), ('904434023660888064_0', 'data_image/hurricane_harvey/3_9_2017/904434023660888064_0.jpg', 'train', None), ('901809624474628096_0', 'data_image/hurricane_harvey/27_8_2017/901809624474628096_0.jpg', 'train', None), ('907297469964447744_0', 'data_image/hurricane_harvey/11_9_2017/907297469964447744_0.jpg', 'train', None), ('901646123080830976_2', 'data_image/hurricane_harvey/27_8_2017/901646123080830976_2.jpg', 'train', None), ('904947742224650241_0', 'data_image/hurricane_harvey/5_9_2017/904947742224650241_0.jpg', 'train', None), ('904564707872227328_0', 'data_image/hurricane_harvey/4_9_2017/904564707872227328_0.jpg', 'test', None), ('905999213611405313_0', 'data_image/hurricane_harvey/8_9_2017/905999213611405313_0.jpg', 'dev', None), ('904261454949437440_0', 'data_image/hurricane_harvey/3_9_2017/904261454949437440_0.jpg', 'train', None), ('909428145908064258_0', 'data_image/hurricane_irma/17_9_2017/909428145908064258_0.jpg', 'test', None), ('909796226169819140_1', 'data_image/hurricane_irma/18_9_2017/909796226169819140_1.jpg', 'test', None), ('909919169319260160_0', 'data_image/hurricane_irma/18_9_2017/909919169319260160_0.jpg', 'test', None), ('909447240959262720_2', 'data_image/hurricane_irma/17_9_2017/909447240959262720_2.jpg', 'train', None), ('909985685125324800_0', 'data_image/hurricane_irma/19_9_2017/909985685125324800_0.jpg', 'train', None), ('909447649291464705_0', 'data_image/hurricane_irma/17_9_2017/909447649291464705_0.jpg', 'train', None), ('909976839006949376_0', 'data_image/hurricane_irma/19_9_2017/909976839006949376_0.jpg', 'train', None), ('909820472258383872_0', 'data_image/hurricane_irma/18_9_2017/909820472258383872_0.jpg', 'train', None), ('909845244648284160_0', 'data_image/hurricane_irma/18_9_2017/909845244648284160_0.jpg', 'test', None), ('910129632812371968_0', 'data_image/hurricane_irma/19_9_2017/910129632812371968_0.jpg', 'test', None), ('910216702687940608_2', 'data_image/hurricane_irma/19_9_2017/910216702687940608_2.jpg', 'train', None), ('909772037224361984_0', 'data_image/hurricane_irma/18_9_2017/909772037224361984_0.jpg', 'train', None), ('909395274610696192_0', 'data_image/hurricane_irma/17_9_2017/909395274610696192_0.jpg', 'test', None), ('909433483331809281_0', 'data_image/hurricane_irma/17_9_2017/909433483331809281_0.jpg', 'train', None), ('909802524198051841_0', 'data_image/hurricane_irma/18_9_2017/909802524198051841_0.jpg', 'dev', None), ('911507373059080192_0', 'data_image/hurricane_maria/23_9_2017/911507373059080192_0.jpg', 'train', None), ('922675551835508736_0', 'data_image/hurricane_maria/24_10_2017/922675551835508736_0.jpg', 'train', None), ('911729595207917568_0', 'data_image/hurricane_maria/23_9_2017/911729595207917568_0.jpg', 'dev', None), ('911965850198036480_2', 'data_image/hurricane_maria/24_9_2017/911965850198036480_2.jpg', 'train', None), ('910570598362882048_0', 'data_image/hurricane_maria/20_9_2017/910570598362882048_0.jpg', 'train', None), ('911944150697807872_0', 'data_image/hurricane_maria/24_9_2017/911944150697807872_0.jpg', 'test', None), ('911694704034680833_0', 'data_image/hurricane_maria/23_9_2017/911694704034680833_0.jpg', 'train', None), ('914007121087418368_0', 'data_image/hurricane_maria/30_9_2017/914007121087418368_0.jpg', 'train', None), ('921414977507995648_0', 'data_image/hurricane_maria/20_10_2017/921414977507995648_0.jpg', 'train', None), ('911524172664360961_0', 'data_image/hurricane_maria/23_9_2017/911524172664360961_0.jpg', 'train', None), ('922802649640075264_0', 'data_image/hurricane_maria/24_10_2017/922802649640075264_0.jpg', 'test', None), ('910585154900516864_1', 'data_image/hurricane_maria/20_9_2017/910585154900516864_1.jpg', 'train', None), ('914188098228989952_0', 'data_image/hurricane_maria/30_9_2017/914188098228989952_0.jpg', 'test', None), ('921477771305484288_0', 'data_image/hurricane_maria/20_10_2017/921477771305484288_0.jpg', 'train', None), ('915986289320714245_0', 'data_image/hurricane_maria/5_10_2017/915986289320714245_0.jpg', 'train', None), ('931046661274701825_3', 'data_image/iraq_iran_earthquake/16_11_2017/931046661274701825_3.jpg', 'train', None), ('930002028365049856_0', 'data_image/iraq_iran_earthquake/13_11_2017/930002028365049856_0.jpg', 'test', None), ('929992868156108800_0', 'data_image/iraq_iran_earthquake/13_11_2017/929992868156108800_0.jpg', 'train', None), ('930159055435419648_0', 'data_image/iraq_iran_earthquake/13_11_2017/930159055435419648_0.png', 'train', None), ('930021249807417344_0', 'data_image/iraq_iran_earthquake/13_11_2017/930021249807417344_0.jpg', 'test', None), ('930151784655544320_0', 'data_image/iraq_iran_earthquake/13_11_2017/930151784655544320_0.jpg', 'dev', None), ('930151625032740865_0', 'data_image/iraq_iran_earthquake/13_11_2017/930151625032740865_0.jpg', 'train', None), ('912045277170806786_0', 'data_image/mexico_earthquake/24_9_2017/912045277170806786_0.jpg', 'train', None), ('913806954023325697_0', 'data_image/mexico_earthquake/29_9_2017/913806954023325697_0.jpg', 'dev', None), ('911729011616862208_0', 'data_image/mexico_earthquake/23_9_2017/911729011616862208_0.jpg', 'dev', None), ('910684543815892993_0', 'data_image/mexico_earthquake/21_9_2017/910684543815892993_0.jpg', 'train', None), ('910725708959404033_0', 'data_image/mexico_earthquake/21_9_2017/910725708959404033_0.jpg', 'train', None), ('910584722778071040_0', 'data_image/mexico_earthquake/20_9_2017/910584722778071040_0.jpg', 'train', None), ('910585414808854528_0', 'data_image/mexico_earthquake/20_9_2017/910585414808854528_0.jpg', 'train', None), ('911697447822462978_0', 'data_image/mexico_earthquake/23_9_2017/911697447822462978_0.jpg', 'test', None), ('910558494020952065_0', 'data_image/mexico_earthquake/20_9_2017/910558494020952065_0.jpg', 'train', None), ('870679026041409540_0', 'data_image/srilanka_floods/2_6_2017/870679026041409540_0.jpg', 'train', None), ('870015964795908097_0', 'data_image/srilanka_floods/31_5_2017/870015964795908097_0.jpg', 'test', None), ('870047567912075265_0', 'data_image/srilanka_floods/31_5_2017/870047567912075265_0.jpg', 'dev', None), ('874622437605986304_0', 'data_image/srilanka_floods/13_6_2017/874622437605986304_0.jpg', 'test', None), ('878103823146860544_0', 'data_image/srilanka_floods/23_6_2017/878103823146860544_0.jpg', 'test', None), ('870109024036864000_1', 'data_image/srilanka_floods/1_6_2017/870109024036864000_1.jpg', 'train', None), ('922765182794895360_0', 'data_image/california_wildfires/24_10_2017/922765182794895360_0.jpg', 'dev', None), ('917944458523566080_0', 'data_image/california_wildfires/11_10_2017/917944458523566080_0.jpg', 'test', None), ('922565688689958912_0', 'data_image/california_wildfires/23_10_2017/922565688689958912_0.jpg', 'test', None), ('920281602218868736_0', 'data_image/california_wildfires/17_10_2017/920281602218868736_0.jpg', 'train', None), ('919579754583478273_0', 'data_image/california_wildfires/15_10_2017/919579754583478273_0.jpg', 'dev', None), ('919578094398144513_1', 'data_image/california_wildfires/15_10_2017/919578094398144513_1.jpg', 'train', None), ('920661807936823297_0', 'data_image/california_wildfires/18_10_2017/920661807936823297_0.jpg', 'test', None), ('922436953433366531_0', 'data_image/california_wildfires/23_10_2017/922436953433366531_0.jpg', 'test', None), ('920291650106200065_0', 'data_image/california_wildfires/17_10_2017/920291650106200065_0.jpg', 'test', None), ('921220735523270656_0', 'data_image/california_wildfires/20_10_2017/921220735523270656_0.jpg', 'test', None), ('917815040962695168_1', 'data_image/california_wildfires/10_10_2017/917815040962695168_1.jpg', 'test', None), ('906293811541925894_0', 'data_image/hurricane_harvey/8_9_2017/906293811541925894_0.jpg', 'train', None), ('901688029521334272_3', 'data_image/hurricane_harvey/27_8_2017/901688029521334272_3.jpg', 'train', None), ('904572612520660992_0', 'data_image/hurricane_harvey/4_9_2017/904572612520660992_0.jpg', 'test', None), ('905804734929481728_0', 'data_image/hurricane_harvey/7_9_2017/905804734929481728_0.jpg', 'test', None), ('901759813788602368_0', 'data_image/hurricane_harvey/27_8_2017/901759813788602368_0.jpg', 'train', None), ('909490653733715969_0', 'data_image/hurricane_harvey/17_9_2017/909490653733715969_0.jpg', 'test', None), ('905757477467299840_1', 'data_image/hurricane_harvey/7_9_2017/905757477467299840_1.jpg', 'train', None), ('905570748303007744_0', 'data_image/hurricane_harvey/6_9_2017/905570748303007744_0.jpg', 'dev', None), ('906850898147729408_0', 'data_image/hurricane_harvey/10_9_2017/906850898147729408_0.jpg', 'test', None), ('907286466568613888_0', 'data_image/hurricane_harvey/11_9_2017/907286466568613888_0.jpg', 'dev', None), ('907717423192788992_0', 'data_image/hurricane_harvey/12_9_2017/907717423192788992_0.jpg', 'test', None), ('904221225072836608_0', 'data_image/hurricane_harvey/3_9_2017/904221225072836608_0.jpg', 'train', None), ('905074370367950848_0', 'data_image/hurricane_harvey/5_9_2017/905074370367950848_0.jpg', 'test', None), ('907723614820200448_0', 'data_image/hurricane_harvey/12_9_2017/907723614820200448_0.jpg', 'train', None), ('901804807303954432_0', 'data_image/hurricane_harvey/27_8_2017/901804807303954432_0.jpg', 'train', None), ('905653236765302784_0', 'data_image/hurricane_irma/7_9_2017/905653236765302784_0.jpg', 'train', None), ('909436630267760641_2', 'data_image/hurricane_irma/17_9_2017/909436630267760641_2.jpg', 'train', None), ('910247377105297413_0', 'data_image/hurricane_irma/19_9_2017/910247377105297413_0.jpg', 'test', None), ('909797688278908933_0', 'data_image/hurricane_irma/18_9_2017/909797688278908933_0.jpg', 'test', None), ('909875385688616961_0', 'data_image/hurricane_irma/18_9_2017/909875385688616961_0.jpg', 'dev', None), ('909459186119008262_0', 'data_image/hurricane_irma/17_9_2017/909459186119008262_0.jpg', 'train', None), ('910189349425295365_0', 'data_image/hurricane_irma/19_9_2017/910189349425295365_0.jpg', 'test', None), ('905745716575838208_0', 'data_image/hurricane_irma/7_9_2017/905745716575838208_0.jpg', 'train', None), ('909962301683453952_0', 'data_image/hurricane_irma/19_9_2017/909962301683453952_0.jpg', 'train', None), ('909819401657970688_0', 'data_image/hurricane_irma/18_9_2017/909819401657970688_0.jpg', 'dev', None), ('909799814656856064_0', 'data_image/hurricane_irma/18_9_2017/909799814656856064_0.jpg', 'train', None), ('909410917233897472_0', 'data_image/hurricane_irma/17_9_2017/909410917233897472_0.jpg', 'train', None), ('909838084786786304_0', 'data_image/hurricane_irma/18_9_2017/909838084786786304_0.jpg', 'dev', None), ('909702840838373376_0', 'data_image/hurricane_irma/18_9_2017/909702840838373376_0.jpg', 'train', None), ('909831928618061826_0', 'data_image/hurricane_irma/18_9_2017/909831928618061826_0.jpg', 'test', None), ('913014958971027457_0', 'data_image/hurricane_maria/27_9_2017/913014958971027457_0.jpg', 'test', None), ('915294511579897857_0', 'data_image/hurricane_maria/3_10_2017/915294511579897857_0.jpg', 'train', None), ('912065374929264640_0', 'data_image/hurricane_maria/24_9_2017/912065374929264640_0.jpg', 'test', None), ('922840948358316032_0', 'data_image/hurricane_maria/24_10_2017/922840948358316032_0.jpg', 'train', None), ('910566158855684096_1', 'data_image/hurricane_maria/20_9_2017/910566158855684096_1.jpg', 'train', None), ('927182978379284486_0', 'data_image/hurricane_maria/5_11_2017/927182978379284486_0.jpg', 'dev', None), ('916341429680640000_0', 'data_image/hurricane_maria/6_10_2017/916341429680640000_0.jpg', 'dev', None), ('922090013092966401_0', 'data_image/hurricane_maria/22_10_2017/922090013092966401_0.jpg', 'dev', None), ('921403426717569025_0', 'data_image/hurricane_maria/20_10_2017/921403426717569025_0.jpg', 'test', None), ('910614037884522496_0', 'data_image/hurricane_maria/20_9_2017/910614037884522496_0.jpg', 'train', None), ('914617195065430016_0', 'data_image/hurricane_maria/1_10_2017/914617195065430016_0.jpg', 'dev', None), ('923433079087255552_0', 'data_image/hurricane_maria/26_10_2017/923433079087255552_0.jpg', 'train', None), ('913478597331050496_0', 'data_image/hurricane_maria/28_9_2017/913478597331050496_0.jpg', 'test', None), ('910534351112359936_0', 'data_image/hurricane_maria/20_9_2017/910534351112359936_0.jpg', 'train', None), ('922904689624133634_0', 'data_image/hurricane_maria/24_10_2017/922904689624133634_0.jpg', 'train', None), ('930466587576020993_0', 'data_image/iraq_iran_earthquake/14_11_2017/930466587576020993_0.jpg', 'train', None), ('930112300174008321_0', 'data_image/iraq_iran_earthquake/13_11_2017/930112300174008321_0.jpg', 'dev', None), ('930074418944110593_0', 'data_image/iraq_iran_earthquake/13_11_2017/930074418944110593_0.jpg', 'dev', None), ('930057284599853057_0', 'data_image/iraq_iran_earthquake/13_11_2017/930057284599853057_0.jpg', 'train', None), ('930116130496167936_0', 'data_image/iraq_iran_earthquake/13_11_2017/930116130496167936_0.jpg', 'dev', None), ('930316631951147008_0', 'data_image/iraq_iran_earthquake/14_11_2017/930316631951147008_0.jpg', 'dev', None), ('930007000632713217_3', 'data_image/iraq_iran_earthquake/13_11_2017/930007000632713217_3.jpg', 'train', None), ('910764046516764672_0', 'data_image/mexico_earthquake/21_9_2017/910764046516764672_0.jpg', 'train', None), ('911623875565375488_0', 'data_image/mexico_earthquake/23_9_2017/911623875565375488_0.jpg', 'train', None), ('913740142678069248_0', 'data_image/mexico_earthquake/29_9_2017/913740142678069248_0.jpg', 'dev', None), ('910523633956683778_0', 'data_image/mexico_earthquake/20_9_2017/910523633956683778_0.jpg', 'train', None), ('910674003211173888_0', 'data_image/mexico_earthquake/21_9_2017/910674003211173888_0.jpg', 'dev', None), ('912184177071620096_0', 'data_image/mexico_earthquake/25_9_2017/912184177071620096_0.jpg', 'train', None), ('911808511948058629_0', 'data_image/mexico_earthquake/24_9_2017/911808511948058629_0.jpg', 'train', None), ('910809743060381697_0', 'data_image/mexico_earthquake/21_9_2017/910809743060381697_0.jpg', 'train', None), ('912379955257810944_0', 'data_image/mexico_earthquake/25_9_2017/912379955257810944_0.jpg', 'train', None), ('870156017392287744_0', 'data_image/srilanka_floods/1_6_2017/870156017392287744_0.jpg', 'train', None), ('872076035726204929_0', 'data_image/srilanka_floods/6_6_2017/872076035726204929_0.jpg', 'test', None), ('874631666219732994_0', 'data_image/srilanka_floods/13_6_2017/874631666219732994_0.jpg', 'dev', None), ('872457085572829185_0', 'data_image/srilanka_floods/7_6_2017/872457085572829185_0.jpg', 'train', None), ('870190470466596865_0', 'data_image/srilanka_floods/1_6_2017/870190470466596865_0.jpg', 'test', None), ('870233018341797888_0', 'data_image/srilanka_floods/1_6_2017/870233018341797888_0.jpg', 'dev', None), ('921097679345733634_0', 'data_image/california_wildfires/19_10_2017/921097679345733634_0.jpg', 'train', None), ('919611976220737536_0', 'data_image/california_wildfires/15_10_2017/919611976220737536_0.jpg', 'train', None), ('918065827047407616_0', 'data_image/california_wildfires/11_10_2017/918065827047407616_0.jpg', 'test', None), ('920405181329797120_0', 'data_image/california_wildfires/17_10_2017/920405181329797120_0.jpg', 'train', None), ('920443591423680518_0', 'data_image/california_wildfires/18_10_2017/920443591423680518_0.jpg', 'dev', None), ('919974245106626561_1', 'data_image/california_wildfires/16_10_2017/919974245106626561_1.jpg', 'test', None), ('919625515409002498_0', 'data_image/california_wildfires/15_10_2017/919625515409002498_0.jpg', 'train', None), ('921693689902256128_0', 'data_image/california_wildfires/21_10_2017/921693689902256128_0.jpg', 'train', None), ('920221872356253697_0', 'data_image/california_wildfires/17_10_2017/920221872356253697_0.jpg', 'dev', None), ('921895226905477120_0', 'data_image/california_wildfires/22_10_2017/921895226905477120_0.jpg', 'train', None), ('907551935796039681_3', 'data_image/hurricane_harvey/12_9_2017/907551935796039681_3.jpg', 'train', None), ('904576825690554372_0', 'data_image/hurricane_harvey/4_9_2017/904576825690554372_0.jpg', 'dev', None), ('905033648268652544_0', 'data_image/hurricane_harvey/5_9_2017/905033648268652544_0.jpg', 'train', None), ('904289302946906113_0', 'data_image/hurricane_harvey/3_9_2017/904289302946906113_0.jpg', 'dev', None), ('901663058388504576_1', 'data_image/hurricane_harvey/27_8_2017/901663058388504576_1.jpg', 'train', None), ('901705055144955904_0', 'data_image/hurricane_harvey/27_8_2017/901705055144955904_0.jpg', 'train', None), ('906253859181473792_0', 'data_image/hurricane_harvey/8_9_2017/906253859181473792_0.jpg', 'dev', None), ('904552138897932288_0', 'data_image/hurricane_harvey/4_9_2017/904552138897932288_0.jpg', 'test', None), ('910236569629609984_0', 'data_image/hurricane_harvey/19_9_2017/910236569629609984_0.jpg', 'test', None), ('904402558877302784_0', 'data_image/hurricane_harvey/3_9_2017/904402558877302784_0.jpg', 'train', None), ('908377679489105926_0', 'data_image/hurricane_harvey/14_9_2017/908377679489105926_0.jpg', 'train', None), ('906115454321664000_0', 'data_image/hurricane_harvey/8_9_2017/906115454321664000_0.jpg', 'dev', None), ('908827775066394625_0', 'data_image/hurricane_harvey/15_9_2017/908827775066394625_0.jpg', 'test', None), ('910145294595223553_0', 'data_image/hurricane_harvey/19_9_2017/910145294595223553_0.jpg', 'train', None), ('904644969985478656_0', 'data_image/hurricane_harvey/4_9_2017/904644969985478656_0.jpg', 'train', None), ('905669201133584385_0', 'data_image/hurricane_irma/7_9_2017/905669201133584385_0.jpg', 'train', None), ('909421699778715648_0', 'data_image/hurricane_irma/17_9_2017/909421699778715648_0.jpg', 'dev', None), ('909779079213305864_0', 'data_image/hurricane_irma/18_9_2017/909779079213305864_0.jpg', 'test', None), ('909751167063801857_0', 'data_image/hurricane_irma/18_9_2017/909751167063801857_0.jpg', 'train', None), ('909472971647012864_0', 'data_image/hurricane_irma/17_9_2017/909472971647012864_0.jpg', 'dev', None), ('909984408479916032_0', 'data_image/hurricane_irma/19_9_2017/909984408479916032_0.jpg', 'dev', None), ('909962867553824768_0', 'data_image/hurricane_irma/19_9_2017/909962867553824768_0.jpg', 'train', None), ('910191453019758592_0', 'data_image/hurricane_irma/19_9_2017/910191453019758592_0.jpg', 'test', None), ('909733938616250369_0', 'data_image/hurricane_irma/18_9_2017/909733938616250369_0.jpg', 'test', None), ('909796661756653569_0', 'data_image/hurricane_irma/18_9_2017/909796661756653569_0.jpg', 'train', None), ('909873972501676033_0', 'data_image/hurricane_irma/18_9_2017/909873972501676033_0.jpg', 'test', None), ('909809381159636993_0', 'data_image/hurricane_irma/18_9_2017/909809381159636993_0.jpg', 'test', None), ('909876188100915201_0', 'data_image/hurricane_irma/18_9_2017/909876188100915201_0.jpg', 'test', None), ('909745313816219648_2', 'data_image/hurricane_irma/18_9_2017/909745313816219648_2.jpg', 'train', None), ('909971209290440706_0', 'data_image/hurricane_irma/19_9_2017/909971209290440706_0.jpg', 'train', None), ('922906810981302305_0', 'data_image/hurricane_maria/24_10_2017/922906810981302305_0.jpg', 'test', None), ('915173788119109633_0', 'data_image/hurricane_maria/3_10_2017/915173788119109633_0.jpg', 'dev', None), ('927964130488848384_0', 'data_image/hurricane_maria/7_11_2017/927964130488848384_0.jpg', 'dev', None), ('913411649918390273_0', 'data_image/hurricane_maria/28_9_2017/913411649918390273_0.jpg', 'train', None), ('927913545538433024_0', 'data_image/hurricane_maria/7_11_2017/927913545538433024_0.jpg', 'train', None), ('912075698617102340_0', 'data_image/hurricane_maria/24_9_2017/912075698617102340_0.jpg', 'train', None), ('923636376393928704_0', 'data_image/hurricane_maria/26_10_2017/923636376393928704_0.jpg', 'test', None), ('923696810366865408_0', 'data_image/hurricane_maria/26_10_2017/923696810366865408_0.jpg', 'dev', None), ('921382178625335296_0', 'data_image/hurricane_maria/20_10_2017/921382178625335296_0.jpg', 'dev', None), ('911554774008500224_0', 'data_image/hurricane_maria/23_9_2017/911554774008500224_0.jpg', 'test', None), ('912065273380798465_0', 'data_image/hurricane_maria/24_9_2017/912065273380798465_0.jpg', 'test', None), ('911983033477140480_3', 'data_image/hurricane_maria/24_9_2017/911983033477140480_3.jpg', 'train', None), ('911855222921859072_0', 'data_image/hurricane_maria/24_9_2017/911855222921859072_0.jpg', 'dev', None), ('923255510517862402_0', 'data_image/hurricane_maria/25_10_2017/923255510517862402_0.jpg', 'test', None), ('921488926392676352_0', 'data_image/hurricane_maria/20_10_2017/921488926392676352_0.jpg', 'train', None), ('930671304470073344_0', 'data_image/iraq_iran_earthquake/15_11_2017/930671304470073344_0.jpg', 'train', None), ('929989752882806785_0', 'data_image/iraq_iran_earthquake/13_11_2017/929989752882806785_0.jpg', 'train', None), ('930014337552367617_0', 'data_image/iraq_iran_earthquake/13_11_2017/930014337552367617_0.jpg', 'train', None), ('930804684155219968_0', 'data_image/iraq_iran_earthquake/15_11_2017/930804684155219968_0.jpg', 'test', None), ('930015299017105408_1', 'data_image/iraq_iran_earthquake/13_11_2017/930015299017105408_1.jpg', 'train', None), ('930191651187802113_0', 'data_image/iraq_iran_earthquake/13_11_2017/930191651187802113_0.jpg', 'dev', None), ('931243132783652864_0', 'data_image/iraq_iran_earthquake/16_11_2017/931243132783652864_0.jpg', 'test', None), ('912326845827371008_0', 'data_image/mexico_earthquake/25_9_2017/912326845827371008_0.jpg', 'dev', None), ('910696337234120704_0', 'data_image/mexico_earthquake/21_9_2017/910696337234120704_0.jpg', 'dev', None), ('910658484257030144_2', 'data_image/mexico_earthquake/21_9_2017/910658484257030144_2.jpg', 'train', None), ('910542539450667009_0', 'data_image/mexico_earthquake/20_9_2017/910542539450667009_0.jpg', 'test', None), ('910711041516036096_0', 'data_image/mexico_earthquake/21_9_2017/910711041516036096_0.jpg', 'train', None), ('910685618778161152_1', 'data_image/mexico_earthquake/21_9_2017/910685618778161152_1.jpg', 'train', None), ('912346493809577984_1', 'data_image/mexico_earthquake/25_9_2017/912346493809577984_1.jpg', 'train', None), ('910813377492127745_0', 'data_image/mexico_earthquake/21_9_2017/910813377492127745_0.jpg', 'train', None), ('910611284617555968_0', 'data_image/mexico_earthquake/20_9_2017/910611284617555968_0.jpg', 'train', None), ('870942570704019456_0', 'data_image/srilanka_floods/3_6_2017/870942570704019456_0.jpg', 'test', None), ('879204877917986816_0', 'data_image/srilanka_floods/26_6_2017/879204877917986816_0.jpg', 'train', None), ('877663582489423872_0', 'data_image/srilanka_floods/21_6_2017/877663582489423872_0.jpg', 'train', None), ('871735126635544578_0', 'data_image/srilanka_floods/5_6_2017/871735126635544578_0.jpg', 'train', None), ('869972354004393987_0', 'data_image/srilanka_floods/31_5_2017/869972354004393987_0.jpg', 'train', None), ('920547023350153217_0', 'data_image/california_wildfires/18_10_2017/920547023350153217_0.jpg', 'train', None), ('920703535141994496_0', 'data_image/california_wildfires/18_10_2017/920703535141994496_0.jpg', 'test', None), ('918316317152890881_0', 'data_image/california_wildfires/12_10_2017/918316317152890881_0.jpg', 'test', None), ('920053496245866496_0', 'data_image/california_wildfires/16_10_2017/920053496245866496_0.jpg', 'dev', None), ('920310501451509762_0', 'data_image/california_wildfires/17_10_2017/920310501451509762_0.jpg', 'train', None), ('921101661745098756_0', 'data_image/california_wildfires/19_10_2017/921101661745098756_0.jpg', 'test', None), ('917987134518321154_0', 'data_image/california_wildfires/11_10_2017/917987134518321154_0.jpg', 'train', None), ('919587173061443587_0', 'data_image/california_wildfires/15_10_2017/919587173061443587_0.jpg', 'train', None), ('922709469246709760_0', 'data_image/california_wildfires/24_10_2017/922709469246709760_0.jpg', 'train', None), ('908740288314855426_0', 'data_image/hurricane_harvey/15_9_2017/908740288314855426_0.jpg', 'test', None), ('905863374122209281_0', 'data_image/hurricane_harvey/7_9_2017/905863374122209281_0.jpg', 'dev', None), ('908007709399756801_0', 'data_image/hurricane_harvey/13_9_2017/908007709399756801_0.jpg', 'dev', None), ('901692349289566210_2', 'data_image/hurricane_harvey/27_8_2017/901692349289566210_2.jpg', 'test', None), ('908019661543002112_0', 'data_image/hurricane_harvey/13_9_2017/908019661543002112_0.jpg', 'train', None), ('904897655230259201_0', 'data_image/hurricane_harvey/5_9_2017/904897655230259201_0.jpg', 'dev', None), ('905776703141761024_0', 'data_image/hurricane_harvey/7_9_2017/905776703141761024_0.jpg', 'train', None), ('907025098024071168_0', 'data_image/hurricane_harvey/10_9_2017/907025098024071168_0.jpg', 'test', None), ('905878921664761858_0', 'data_image/hurricane_harvey/7_9_2017/905878921664761858_0.jpg', 'test', None), ('905327512162533377_0', 'data_image/hurricane_harvey/6_9_2017/905327512162533377_0.jpg', 'test', None), ('905863389511081984_0', 'data_image/hurricane_harvey/7_9_2017/905863389511081984_0.jpg', 'test', None), ('909830129538490368_0', 'data_image/hurricane_harvey/18_9_2017/909830129538490368_0.jpg', 'dev', None), ('905444979387990016_0', 'data_image/hurricane_harvey/6_9_2017/905444979387990016_0.jpg', 'test', None), ('906464976906461202_0', 'data_image/hurricane_harvey/9_9_2017/906464976906461202_0.jpg', 'dev', None), ('904289769319890944_0', 'data_image/hurricane_harvey/3_9_2017/904289769319890944_0.jpg', 'dev', None), ('909932614785609728_0', 'data_image/hurricane_irma/19_9_2017/909932614785609728_0.jpg', 'test', None), ('909839453010702338_0', 'data_image/hurricane_irma/18_9_2017/909839453010702338_0.jpg', 'train', None), ('909900730735190016_1', 'data_image/hurricane_irma/18_9_2017/909900730735190016_1.jpg', 'train', None), ('909873221188702209_0', 'data_image/hurricane_irma/18_9_2017/909873221188702209_0.jpg', 'dev', None), ('909776832555360256_0', 'data_image/hurricane_irma/18_9_2017/909776832555360256_0.jpg', 'train', None), ('909936161384214528_0', 'data_image/hurricane_irma/19_9_2017/909936161384214528_0.jpg', 'train', None), ('910178653052555264_0', 'data_image/hurricane_irma/19_9_2017/910178653052555264_0.jpg', 'train', None), ('910118839081472000_0', 'data_image/hurricane_irma/19_9_2017/910118839081472000_0.jpg', 'dev', None), ('910213017681281024_0', 'data_image/hurricane_irma/19_9_2017/910213017681281024_0.jpg', 'train', None), ('905714887531388929_0', 'data_image/hurricane_irma/7_9_2017/905714887531388929_0.jpg', 'train', None), ('910227503960281088_0', 'data_image/hurricane_irma/19_9_2017/910227503960281088_0.jpg', 'test', None), ('910114782371442689_0', 'data_image/hurricane_irma/19_9_2017/910114782371442689_0.jpg', 'train', None), ('909965848235741187_0', 'data_image/hurricane_irma/19_9_2017/909965848235741187_0.jpg', 'train', None), ('909697106079862784_0', 'data_image/hurricane_irma/18_9_2017/909697106079862784_0.jpg', 'dev', None), ('909469987001241601_2', 'data_image/hurricane_irma/17_9_2017/909469987001241601_2.jpg', 'train', None), ('916309677721473026_0', 'data_image/hurricane_maria/6_10_2017/916309677721473026_0.jpg', 'train', None), ('915215052176789504_0', 'data_image/hurricane_maria/3_10_2017/915215052176789504_0.jpg', 'dev', None), ('923583826189729793_0', 'data_image/hurricane_maria/26_10_2017/923583826189729793_0.jpg', 'dev', None), ('913070687501344768_0', 'data_image/hurricane_maria/27_9_2017/913070687501344768_0.jpg', 'train', None), ('923286011651661825_0', 'data_image/hurricane_maria/25_10_2017/923286011651661825_0.jpg', 'train', None), ('914880414736699393_0', 'data_image/hurricane_maria/2_10_2017/914880414736699393_0.jpg', 'train', None), ('913936743556173825_0', 'data_image/hurricane_maria/30_9_2017/913936743556173825_0.jpg', 'test', None), ('921461351381393408_1', 'data_image/hurricane_maria/20_10_2017/921461351381393408_1.jpg', 'train', None), ('912295059315404800_0', 'data_image/hurricane_maria/25_9_2017/912295059315404800_0.jpg', 'train', None), ('911750614631243776_0', 'data_image/hurricane_maria/24_9_2017/911750614631243776_0.jpg', 'test', None), ('913783977340784641_2', 'data_image/hurricane_maria/29_9_2017/913783977340784641_2.jpg', 'train', None), ('928053068964552705_0', 'data_image/hurricane_maria/8_11_2017/928053068964552705_0.jpg', 'dev', None), ('913257006710034434_0', 'data_image/hurricane_maria/28_9_2017/913257006710034434_0.jpg', 'train', None), ('914545657460994049_0', 'data_image/hurricane_maria/1_10_2017/914545657460994049_0.jpg', 'test', None), ('921529194730983424_2', 'data_image/hurricane_maria/21_10_2017/921529194730983424_2.jpg', 'train', None), ('930360981259141120_0', 'data_image/iraq_iran_earthquake/14_11_2017/930360981259141120_0.jpg', 'train', None), ('930127736898752513_0', 'data_image/iraq_iran_earthquake/13_11_2017/930127736898752513_0.jpg', 'train', None), ('930085434360778754_0', 'data_image/iraq_iran_earthquake/13_11_2017/930085434360778754_0.jpg', 'train', None), ('931371221052219393_0', 'data_image/iraq_iran_earthquake/17_11_2017/931371221052219393_0.jpg', 'dev', None), ('930023048782086146_0', 'data_image/iraq_iran_earthquake/13_11_2017/930023048782086146_0.jpg', 'train', None), ('930007000632713217_2', 'data_image/iraq_iran_earthquake/13_11_2017/930007000632713217_2.jpg', 'train', None), ('911769884434010114_0', 'data_image/mexico_earthquake/24_9_2017/911769884434010114_0.jpg', 'train', None), ('910670548191010817_0', 'data_image/mexico_earthquake/21_9_2017/910670548191010817_0.jpg', 'dev', None), ('911794471804784641_0', 'data_image/mexico_earthquake/24_9_2017/911794471804784641_0.jpg', 'dev', None), ('910614165076750337_1', 'data_image/mexico_earthquake/20_9_2017/910614165076750337_1.jpg', 'train', None), ('910555658746863616_0', 'data_image/mexico_earthquake/20_9_2017/910555658746863616_0.jpg', 'test', None), ('911711785882755077_0', 'data_image/mexico_earthquake/23_9_2017/911711785882755077_0.jpg', 'train', None), ('910541543961964544_0', 'data_image/mexico_earthquake/20_9_2017/910541543961964544_0.jpg', 'train', None), ('912123204994560000_0', 'data_image/mexico_earthquake/25_9_2017/912123204994560000_0.jpg', 'train', None), ('877815334408921088_0', 'data_image/srilanka_floods/22_6_2017/877815334408921088_0.jpg', 'test', None), ('875461354584289281_0', 'data_image/srilanka_floods/15_6_2017/875461354584289281_0.jpg', 'train', None), ('876386158309236737_0', 'data_image/srilanka_floods/18_6_2017/876386158309236737_0.jpg', 'test', None), ('871637244695650304_0', 'data_image/srilanka_floods/5_6_2017/871637244695650304_0.jpg', 'test', None), ('870528490184908800_2', 'data_image/srilanka_floods/2_6_2017/870528490184908800_2.jpg', 'train', None), ('922144749632479232_1', 'data_image/california_wildfires/22_10_2017/922144749632479232_1.jpg', 'train', None), ('922740491187380224_0', 'data_image/california_wildfires/24_10_2017/922740491187380224_0.jpg', 'test', None), ('923245527608807425_0', 'data_image/california_wildfires/25_10_2017/923245527608807425_0.jpg', 'test', None), ('921973226510733313_0', 'data_image/california_wildfires/22_10_2017/921973226510733313_0.jpg', 'test', None), ('917861480673902592_0', 'data_image/california_wildfires/10_10_2017/917861480673902592_0.jpg', 'train', None), ('922260115079692288_0', 'data_image/california_wildfires/23_10_2017/922260115079692288_0.jpg', 'dev', None), ('917828283047260161_0', 'data_image/california_wildfires/10_10_2017/917828283047260161_0.jpg', 'train', None), ('922282490357829632_0', 'data_image/california_wildfires/23_10_2017/922282490357829632_0.jpg', 'dev', None), ('918015310992674816_0', 'data_image/california_wildfires/11_10_2017/918015310992674816_0.jpg', 'train', None), ('904958031363899392_0', 'data_image/hurricane_harvey/5_9_2017/904958031363899392_0.jpg', 'train', None), ('910261798569472003_0', 'data_image/hurricane_harvey/19_9_2017/910261798569472003_0.jpg', 'test', None), ('901776373177561088_0', 'data_image/hurricane_harvey/27_8_2017/901776373177561088_0.jpg', 'dev', None), ('907041310518935552_1', 'data_image/hurricane_harvey/11_9_2017/907041310518935552_1.jpg', 'train', None), ('908037995344814080_0', 'data_image/hurricane_harvey/13_9_2017/908037995344814080_0.jpg', 'test', None), ('908221170184531969_0', 'data_image/hurricane_harvey/14_9_2017/908221170184531969_0.jpg', 'train', None), ('907346349598834688_1', 'data_image/hurricane_harvey/11_9_2017/907346349598834688_1.jpg', 'train', None), ('907343282442776581_0', 'data_image/hurricane_harvey/11_9_2017/907343282442776581_0.jpg', 'train', None), ('905697698484977664_0', 'data_image/hurricane_harvey/7_9_2017/905697698484977664_0.jpg', 'test', None), ('906661514903306240_0', 'data_image/hurricane_harvey/9_9_2017/906661514903306240_0.jpg', 'train', None), ('905115024829636609_1', 'data_image/hurricane_harvey/5_9_2017/905115024829636609_1.jpg', 'train', None), ('904803081124175872_0', 'data_image/hurricane_harvey/4_9_2017/904803081124175872_0.jpg', 'train', None), ('908842874309095424_0', 'data_image/hurricane_harvey/16_9_2017/908842874309095424_0.jpg', 'test', None), ('904869820478488576_0', 'data_image/hurricane_harvey/5_9_2017/904869820478488576_0.jpg', 'test', None), ('901779402446528513_0', 'data_image/hurricane_harvey/27_8_2017/901779402446528513_0.jpg', 'train', None), ('909720414271807488_0', 'data_image/hurricane_irma/18_9_2017/909720414271807488_0.jpg', 'dev', None), ('909470884532047874_0', 'data_image/hurricane_irma/17_9_2017/909470884532047874_0.jpg', 'test', None), ('905695796422684673_0', 'data_image/hurricane_irma/7_9_2017/905695796422684673_0.jpg', 'train', None), ('909763338514653184_0', 'data_image/hurricane_irma/18_9_2017/909763338514653184_0.jpg', 'test', None), ('905637033858740228_0', 'data_image/hurricane_irma/7_9_2017/905637033858740228_0.jpg', 'dev', None), ('909824681493938177_0', 'data_image/hurricane_irma/18_9_2017/909824681493938177_0.jpg', 'train', None), ('909722471812796416_0', 'data_image/hurricane_irma/18_9_2017/909722471812796416_0.jpg', 'dev', None), ('909780895137193984_0', 'data_image/hurricane_irma/18_9_2017/909780895137193984_0.jpg', 'dev', None), ('910094563922055170_0', 'data_image/hurricane_irma/19_9_2017/910094563922055170_0.jpg', 'test', None), ('910222049569673216_0', 'data_image/hurricane_irma/19_9_2017/910222049569673216_0.jpg', 'train', None), ('909437598170574848_0', 'data_image/hurricane_irma/17_9_2017/909437598170574848_0.jpg', 'train', None), ('905728545862942720_0', 'data_image/hurricane_irma/7_9_2017/905728545862942720_0.jpg', 'train', None), ('910173880437817344_0', 'data_image/hurricane_irma/19_9_2017/910173880437817344_0.jpg', 'dev', None), ('910187502375456768_0', 'data_image/hurricane_irma/19_9_2017/910187502375456768_0.jpg', 'dev', None), ('909380939561697280_0', 'data_image/hurricane_irma/17_9_2017/909380939561697280_0.jpg', 'train', None), ('923807511592910848_0', 'data_image/hurricane_maria/27_10_2017/923807511592910848_0.jpg', 'train', None), ('910566731273433091_0', 'data_image/hurricane_maria/20_9_2017/910566731273433091_0.jpg', 'test', None), ('912283309870157824_0', 'data_image/hurricane_maria/25_9_2017/912283309870157824_0.jpg', 'test', None), ('910587509964464128_0', 'data_image/hurricane_maria/20_9_2017/910587509964464128_0.jpg', 'test', None), ('921854452465917953_0', 'data_image/hurricane_maria/21_10_2017/921854452465917953_0.jpg', 'test', None), ('927769335489208321_0', 'data_image/hurricane_maria/7_11_2017/927769335489208321_0.jpg', 'dev', None), ('911753059994357760_0', 'data_image/hurricane_maria/24_9_2017/911753059994357760_0.jpg', 'train', None), ('916352428714414080_0', 'data_image/hurricane_maria/6_10_2017/916352428714414080_0.jpg', 'train', None), ('922915653887049729_0', 'data_image/hurricane_maria/24_10_2017/922915653887049729_0.jpg', 'test', None), ('911671137511276547_0', 'data_image/hurricane_maria/23_9_2017/911671137511276547_0.jpg', 'test', None), ('913029119008313344_0', 'data_image/hurricane_maria/27_9_2017/913029119008313344_0.jpg', 'test', None), ('927299174302875649_0', 'data_image/hurricane_maria/5_11_2017/927299174302875649_0.jpg', 'dev', None), ('923193683196887040_0', 'data_image/hurricane_maria/25_10_2017/923193683196887040_0.jpg', 'test', None), ('921808746959048704_0', 'data_image/hurricane_maria/21_10_2017/921808746959048704_0.jpg', 'train', None), ('913750333280661506_0', 'data_image/hurricane_maria/29_9_2017/913750333280661506_0.jpg', 'test', None), ('932000082072948736_0', 'data_image/iraq_iran_earthquake/18_11_2017/932000082072948736_0.jpg', 'test', None), ('930407396698218498_0', 'data_image/iraq_iran_earthquake/14_11_2017/930407396698218498_0.jpg', 'dev', None), ('930072924752670720_0', 'data_image/iraq_iran_earthquake/13_11_2017/930072924752670720_0.jpg', 'train', None), ('930036642282397696_0', 'data_image/iraq_iran_earthquake/13_11_2017/930036642282397696_0.jpg', 'dev', None), ('930192063626440705_0', 'data_image/iraq_iran_earthquake/13_11_2017/930192063626440705_0.jpg', 'dev', None), ('930864332673609728_0', 'data_image/iraq_iran_earthquake/15_11_2017/930864332673609728_0.jpg', 'train', None), ('910635936396046337_0', 'data_image/mexico_earthquake/20_9_2017/910635936396046337_0.jpg', 'dev', None), ('912232092997767168_0', 'data_image/mexico_earthquake/25_9_2017/912232092997767168_0.jpg', 'train', None), ('910532720870137856_0', 'data_image/mexico_earthquake/20_9_2017/910532720870137856_0.jpg', 'test', None), ('914198248256102400_0', 'data_image/mexico_earthquake/30_9_2017/914198248256102400_0.jpg', 'train', None), ('910538578131914752_0', 'data_image/mexico_earthquake/20_9_2017/910538578131914752_0.jpg', 'test', None), ('910563637445750792_0', 'data_image/mexico_earthquake/20_9_2017/910563637445750792_0.jpg', 'test', None), ('910622872455938048_0', 'data_image/mexico_earthquake/20_9_2017/910622872455938048_0.jpg', 'dev', None), ('911642673404088320_0', 'data_image/mexico_earthquake/23_9_2017/911642673404088320_0.jpg', 'dev', None), ('870553882992164864_0', 'data_image/srilanka_floods/2_6_2017/870553882992164864_0.jpg', 'train', None), ('870561029071044608_0', 'data_image/srilanka_floods/2_6_2017/870561029071044608_0.jpg', 'train', None), ('877614604183928832_0', 'data_image/srilanka_floods/21_6_2017/877614604183928832_0.jpg', 'train', None), ('871703713722880000_0', 'data_image/srilanka_floods/5_6_2017/871703713722880000_0.jpg', 'train', None), ('871287130709925888_0', 'data_image/srilanka_floods/4_6_2017/871287130709925888_0.jpg', 'test', None), ('918315180085276672_0', 'data_image/california_wildfires/12_10_2017/918315180085276672_0.jpg', 'train', None), ('917793137925459968_1', 'data_image/california_wildfires/10_10_2017/917793137925459968_1.jpg', 'dev', None), ('923698200090853376_0', 'data_image/california_wildfires/26_10_2017/923698200090853376_0.jpg', 'train', None), ('920474622205628416_2', 'data_image/california_wildfires/18_10_2017/920474622205628416_2.jpg', 'train', None), ('920387925187194881_0', 'data_image/california_wildfires/17_10_2017/920387925187194881_0.jpg', 'train', None), ('921692663195111424_0', 'data_image/california_wildfires/21_10_2017/921692663195111424_0.jpg', 'test', None), ('919609029701201921_0', 'data_image/california_wildfires/15_10_2017/919609029701201921_0.jpg', 'dev', None), ('921088764021112832_0', 'data_image/california_wildfires/19_10_2017/921088764021112832_0.jpg', 'test', None), ('922391403103178753_0', 'data_image/california_wildfires/23_10_2017/922391403103178753_0.jpg', 'dev', None), ('901822579362627584_0', 'data_image/hurricane_harvey/27_8_2017/901822579362627584_0.jpg', 'train', None), ('905534245178880001_0', 'data_image/hurricane_harvey/6_9_2017/905534245178880001_0.jpg', 'train', None), ('901701789279547392_0', 'data_image/hurricane_harvey/27_8_2017/901701789279547392_0.jpg', 'train', None), ('904248609641603072_0', 'data_image/hurricane_harvey/3_9_2017/904248609641603072_0.jpg', 'train', None), ('901726532569268224_0', 'data_image/hurricane_harvey/27_8_2017/901726532569268224_0.jpg', 'test', None), ('905019431818027008_0', 'data_image/hurricane_harvey/5_9_2017/905019431818027008_0.jpg', 'dev', None), ('904267784565252096_0', 'data_image/hurricane_harvey/3_9_2017/904267784565252096_0.jpg', 'test', None), ('908677187359850497_0', 'data_image/hurricane_harvey/15_9_2017/908677187359850497_0.jpg', 'test', None), ('905065364601667585_0', 'data_image/hurricane_harvey/5_9_2017/905065364601667585_0.jpg', 'test', None), ('906888945413357568_0', 'data_image/hurricane_harvey/10_9_2017/906888945413357568_0.jpg', 'test', None), ('907690425481355264_0', 'data_image/hurricane_harvey/12_9_2017/907690425481355264_0.jpg', 'dev', None), ('904240121922322432_0', 'data_image/hurricane_harvey/3_9_2017/904240121922322432_0.jpg', 'train', None), ('907335391103111168_0', 'data_image/hurricane_harvey/11_9_2017/907335391103111168_0.jpg', 'dev', None), ('908742171154665472_0', 'data_image/hurricane_harvey/15_9_2017/908742171154665472_0.jpg', 'train', None), ('905750817709404160_0', 'data_image/hurricane_harvey/7_9_2017/905750817709404160_0.jpg', 'test', None), ('909750386340249601_0', 'data_image/hurricane_irma/18_9_2017/909750386340249601_0.jpg', 'test', None), ('909446946535936000_1', 'data_image/hurricane_irma/17_9_2017/909446946535936000_1.jpg', 'train', None), ('910129616353992704_0', 'data_image/hurricane_irma/19_9_2017/910129616353992704_0.jpg', 'dev', None), ('910211896451592193_0', 'data_image/hurricane_irma/19_9_2017/910211896451592193_0.jpg', 'test', None), ('909490898706329600_2', 'data_image/hurricane_irma/17_9_2017/909490898706329600_2.jpg', 'train', None), ('909914794550992896_0', 'data_image/hurricane_irma/18_9_2017/909914794550992896_0.jpg', 'train', None), ('909800638330150913_0', 'data_image/hurricane_irma/18_9_2017/909800638330150913_0.jpg', 'train', None), ('909822717574238208_0', 'data_image/hurricane_irma/18_9_2017/909822717574238208_0.jpg', 'test', None), ('910246138564943872_0', 'data_image/hurricane_irma/19_9_2017/910246138564943872_0.jpg', 'test', None), ('909863262371663872_0', 'data_image/hurricane_irma/18_9_2017/909863262371663872_0.jpg', 'test', None), ('910220719904063488_0', 'data_image/hurricane_irma/19_9_2017/910220719904063488_0.jpg', 'dev', None), ('910218125446205440_0', 'data_image/hurricane_irma/19_9_2017/910218125446205440_0.jpg', 'train', None), ('909383764723331072_0', 'data_image/hurricane_irma/17_9_2017/909383764723331072_0.jpg', 'train', None), ('909751620514148354_0', 'data_image/hurricane_irma/18_9_2017/909751620514148354_0.jpg', 'test', None), ('909800810221105154_0', 'data_image/hurricane_irma/18_9_2017/909800810221105154_0.jpg', 'train', None), ('921597509025370112_0', 'data_image/hurricane_maria/21_10_2017/921597509025370112_0.jpg', 'dev', None), ('912463510461992960_1', 'data_image/hurricane_maria/25_9_2017/912463510461992960_1.jpg', 'train', None), ('912959167417192448_0', 'data_image/hurricane_maria/27_9_2017/912959167417192448_0.jpg', 'dev', None), ('913157619455426560_0', 'data_image/hurricane_maria/27_9_2017/913157619455426560_0.jpg', 'dev', None), ('911931707892236289_0', 'data_image/hurricane_maria/24_9_2017/911931707892236289_0.jpg', 'dev', None), ('913546885067165696_0', 'data_image/hurricane_maria/28_9_2017/913546885067165696_0.jpg', 'test', None), ('921875031533203457_0', 'data_image/hurricane_maria/21_10_2017/921875031533203457_0.jpg', 'train', None), ('916062214016131072_0', 'data_image/hurricane_maria/5_10_2017/916062214016131072_0.jpg', 'test', None), ('912288130354511873_0', 'data_image/hurricane_maria/25_9_2017/912288130354511873_0.jpg', 'train', None), ('915995090316750848_0', 'data_image/hurricane_maria/5_10_2017/915995090316750848_0.jpg', 'test', None), ('910584503931895814_0', 'data_image/hurricane_maria/20_9_2017/910584503931895814_0.jpg', 'test', None), ('914336165314035713_0', 'data_image/hurricane_maria/1_10_2017/914336165314035713_0.jpg', 'train', None), ('923657354196987904_0', 'data_image/hurricane_maria/26_10_2017/923657354196987904_0.jpg', 'train', None), ('929309114605494272_0', 'data_image/hurricane_maria/11_11_2017/929309114605494272_0.jpg', 'train', None), ('921565502857711616_0', 'data_image/hurricane_maria/21_10_2017/921565502857711616_0.jpg', 'test', None), ('930000656525680640_0', 'data_image/iraq_iran_earthquake/13_11_2017/930000656525680640_0.jpg', 'test', None), ('930122283913490433_0', 'data_image/iraq_iran_earthquake/13_11_2017/930122283913490433_0.jpg', 'dev', None), ('930374412708335616_0', 'data_image/iraq_iran_earthquake/14_11_2017/930374412708335616_0.jpg', 'train', None), ('929992506158149635_0', 'data_image/iraq_iran_earthquake/13_11_2017/929992506158149635_0.jpg', 'train', None), ('930421814525939712_0', 'data_image/iraq_iran_earthquake/14_11_2017/930421814525939712_0.jpg', 'dev', None), ('913419277830520833_0', 'data_image/mexico_earthquake/28_9_2017/913419277830520833_0.jpg', 'test', None), ('912331848478609409_0', 'data_image/mexico_earthquake/25_9_2017/912331848478609409_0.jpg', 'test', None), ('911881648097992714_0', 'data_image/mexico_earthquake/24_9_2017/911881648097992714_0.jpg', 'test', None), ('910524779269984258_0', 'data_image/mexico_earthquake/20_9_2017/910524779269984258_0.jpg', 'dev', None), ('912200897769431041_0', 'data_image/mexico_earthquake/25_9_2017/912200897769431041_0.jpg', 'test', None), ('910749869014384640_0', 'data_image/mexico_earthquake/21_9_2017/910749869014384640_0.jpg', 'train', None), ('910622132236718081_0', 'data_image/mexico_earthquake/20_9_2017/910622132236718081_0.jpg', 'test', None), ('910813678739558400_0', 'data_image/mexico_earthquake/21_9_2017/910813678739558400_0.jpg', 'train', None), ('870209884452552705_0', 'data_image/srilanka_floods/1_6_2017/870209884452552705_0.jpg', 'train', None), ('870265400440758272_0', 'data_image/srilanka_floods/1_6_2017/870265400440758272_0.jpg', 'train', None), ('870135631308931073_0', 'data_image/srilanka_floods/1_6_2017/870135631308931073_0.jpg', 'test', None), ('875589229341843456_0', 'data_image/srilanka_floods/16_6_2017/875589229341843456_0.jpg', 'train', None), ('870849885553045504_0', 'data_image/srilanka_floods/3_6_2017/870849885553045504_0.jpg', 'test', None), ('919702207351500800_0', 'data_image/california_wildfires/15_10_2017/919702207351500800_0.jpg', 'dev', None), ('919719914881142784_0', 'data_image/california_wildfires/16_10_2017/919719914881142784_0.jpg', 'train', None), ('920101016384794625_0', 'data_image/california_wildfires/17_10_2017/920101016384794625_0.jpg', 'train', None), ('922377902708183040_0', 'data_image/california_wildfires/23_10_2017/922377902708183040_0.jpg', 'test', None), ('923195494565781505_0', 'data_image/california_wildfires/25_10_2017/923195494565781505_0.jpg', 'train', None), ('919649109782421504_0', 'data_image/california_wildfires/15_10_2017/919649109782421504_0.jpg', 'test', None), ('917991662512758784_0', 'data_image/california_wildfires/11_10_2017/917991662512758784_0.jpg', 'test', None), ('920140720689827841_0', 'data_image/california_wildfires/17_10_2017/920140720689827841_0.jpg', 'train', None), ('917827701431570432_0', 'data_image/california_wildfires/10_10_2017/917827701431570432_0.jpg', 'train', None), ('905467245265956866_0', 'data_image/hurricane_harvey/6_9_2017/905467245265956866_0.jpg', 'test', None), ('907296821357289476_0', 'data_image/hurricane_harvey/11_9_2017/907296821357289476_0.jpg', 'test', None), ('904362108531466241_0', 'data_image/hurricane_harvey/3_9_2017/904362108531466241_0.jpg', 'train', None), ('904327280612306944_0', 'data_image/hurricane_harvey/3_9_2017/904327280612306944_0.jpg', 'dev', None), ('907964950739607552_1', 'data_image/hurricane_harvey/13_9_2017/907964950739607552_1.jpg', 'dev', None), ('907807323082518528_0', 'data_image/hurricane_harvey/13_9_2017/907807323082518528_0.jpg', 'dev', None), ('901831048358760453_0', 'data_image/hurricane_harvey/27_8_2017/901831048358760453_0.jpg', 'train', None), ('905491119705702400_1', 'data_image/hurricane_harvey/6_9_2017/905491119705702400_1.jpg', 'test', None), ('908657931880685569_0', 'data_image/hurricane_harvey/15_9_2017/908657931880685569_0.jpg', 'test', None), ('905461810828070913_0', 'data_image/hurricane_harvey/6_9_2017/905461810828070913_0.jpg', 'dev', None), ('905819206108684288_0', 'data_image/hurricane_harvey/7_9_2017/905819206108684288_0.jpg', 'train', None), ('905552718088900608_0', 'data_image/hurricane_harvey/6_9_2017/905552718088900608_0.jpg', 'dev', None), ('905252739109724160_0', 'data_image/hurricane_harvey/6_9_2017/905252739109724160_0.jpg', 'test', None), ('905185032352010242_0', 'data_image/hurricane_harvey/5_9_2017/905185032352010242_0.png', 'dev', None), ('907551935796039681_1', 'data_image/hurricane_harvey/12_9_2017/907551935796039681_1.jpg', 'train', None), ('909719313963274245_0', 'data_image/hurricane_irma/18_9_2017/909719313963274245_0.jpg', 'dev', None), ('910156856869433344_0', 'data_image/hurricane_irma/19_9_2017/910156856869433344_0.jpg', 'dev', None), ('909946398300409856_0', 'data_image/hurricane_irma/19_9_2017/909946398300409856_0.jpg', 'test', None), ('909455492438564864_0', 'data_image/hurricane_irma/17_9_2017/909455492438564864_0.jpg', 'train', None), ('910242977762734081_0', 'data_image/hurricane_irma/19_9_2017/910242977762734081_0.jpg', 'test', None), ('910171888894214145_0', 'data_image/hurricane_irma/19_9_2017/910171888894214145_0.jpg', 'train', None), ('909890311941484544_0', 'data_image/hurricane_irma/18_9_2017/909890311941484544_0.jpg', 'train', None), ('909382127199821824_0', 'data_image/hurricane_irma/17_9_2017/909382127199821824_0.jpg', 'test', None), ('909411047039229953_0', 'data_image/hurricane_irma/17_9_2017/909411047039229953_0.jpg', 'dev', None), ('910082408883658758_0', 'data_image/hurricane_irma/19_9_2017/910082408883658758_0.jpg', 'train', None), ('909775762869428225_0', 'data_image/hurricane_irma/18_9_2017/909775762869428225_0.jpg', 'test', None), ('905658136169652224_0', 'data_image/hurricane_irma/7_9_2017/905658136169652224_0.jpg', 'train', None), ('909757379620872193_0', 'data_image/hurricane_irma/18_9_2017/909757379620872193_0.jpg', 'test', None), ('909805701140242434_1', 'data_image/hurricane_irma/18_9_2017/909805701140242434_1.jpg', 'train', None), ('909894624898101248_1', 'data_image/hurricane_irma/18_9_2017/909894624898101248_1.jpg', 'train', None), ('923149294017875968_0', 'data_image/hurricane_maria/25_10_2017/923149294017875968_0.jpg', 'train', None), ('910549146360311808_0', 'data_image/hurricane_maria/20_9_2017/910549146360311808_0.jpg', 'train', None), ('911705355700310016_0', 'data_image/hurricane_maria/23_9_2017/911705355700310016_0.jpg', 'dev', None), ('912306806021591041_0', 'data_image/hurricane_maria/25_9_2017/912306806021591041_0.jpg', 'test', None), ('928345500763459586_0', 'data_image/hurricane_maria/8_11_2017/928345500763459586_0.jpg', 'test', None), ('923524261276504070_0', 'data_image/hurricane_maria/26_10_2017/923524261276504070_0.jpg', 'train', None), ('910585965906944000_0', 'data_image/hurricane_maria/20_9_2017/910585965906944000_0.jpg', 'dev', None), ('923264541743566851_1', 'data_image/hurricane_maria/25_10_2017/923264541743566851_1.jpg', 'train', None), ('911789292116217856_0', 'data_image/hurricane_maria/24_9_2017/911789292116217856_0.jpg', 'train', None), ('928053068964552705_1', 'data_image/hurricane_maria/8_11_2017/928053068964552705_1.jpg', 'dev', None), ('913516169247600640_0', 'data_image/hurricane_maria/28_9_2017/913516169247600640_0.jpg', 'train', None), ('921486256479068160_0', 'data_image/hurricane_maria/20_10_2017/921486256479068160_0.jpg', 'dev', None), ('911921934027362304_0', 'data_image/hurricane_maria/24_9_2017/911921934027362304_0.jpg', 'dev', None), ('929320152373645312_0', 'data_image/hurricane_maria/11_11_2017/929320152373645312_0.jpg', 'train', None), ('912389558016323584_2', 'data_image/hurricane_maria/25_9_2017/912389558016323584_2.jpg', 'train', None), ('931577845780025348_0', 'data_image/iraq_iran_earthquake/17_11_2017/931577845780025348_0.jpg', 'train', None), ('930059219306508289_0', 'data_image/iraq_iran_earthquake/13_11_2017/930059219306508289_0.jpg', 'train', None), ('930205140287741953_2', 'data_image/iraq_iran_earthquake/13_11_2017/930205140287741953_2.jpg', 'train', None), ('930327732688490497_0', 'data_image/iraq_iran_earthquake/14_11_2017/930327732688490497_0.jpg', 'train', None), ('930426908361314304_0', 'data_image/iraq_iran_earthquake/14_11_2017/930426908361314304_0.jpg', 'train', None), ('910524793740513280_1', 'data_image/mexico_earthquake/20_9_2017/910524793740513280_1.jpg', 'train', None), ('910708399377080322_0', 'data_image/mexico_earthquake/21_9_2017/910708399377080322_0.jpg', 'train', None), ('910533089809575938_0', 'data_image/mexico_earthquake/20_9_2017/910533089809575938_0.jpg', 'train', None), ('910525564641009666_0', 'data_image/mexico_earthquake/20_9_2017/910525564641009666_0.jpg', 'dev', None), ('910716949843337216_0', 'data_image/mexico_earthquake/21_9_2017/910716949843337216_0.jpg', 'train', None), ('910558121570926593_0', 'data_image/mexico_earthquake/20_9_2017/910558121570926593_0.jpg', 'dev', None), ('910627061722697728_0', 'data_image/mexico_earthquake/20_9_2017/910627061722697728_0.jpg', 'dev', None), ('911704186261311488_0', 'data_image/mexico_earthquake/23_9_2017/911704186261311488_0.jpg', 'dev', None), ('870667100368637953_0', 'data_image/srilanka_floods/2_6_2017/870667100368637953_0.jpg', 'train', None), ('870619854104961025_1', 'data_image/srilanka_floods/2_6_2017/870619854104961025_1.jpg', 'train', None), ('873673345929183237_1', 'data_image/srilanka_floods/10_6_2017/873673345929183237_1.jpg', 'train', None), ('870319064501104640_0', 'data_image/srilanka_floods/1_6_2017/870319064501104640_0.jpg', 'train', None), ('872051180549152769_0', 'data_image/srilanka_floods/6_6_2017/872051180549152769_0.jpg', 'train', None), ('918094880072663041_0', 'data_image/california_wildfires/11_10_2017/918094880072663041_0.jpg', 'test', None), ('919839993438326785_0', 'data_image/california_wildfires/16_10_2017/919839993438326785_0.jpg', 'dev', None), ('923577783367946240_0', 'data_image/california_wildfires/26_10_2017/923577783367946240_0.jpg', 'train', None), ('920358036761231360_0', 'data_image/california_wildfires/17_10_2017/920358036761231360_0.jpg', 'dev', None), ('917961969122017282_0', 'data_image/california_wildfires/11_10_2017/917961969122017282_0.jpg', 'test', None), ('920916661519007745_3', 'data_image/california_wildfires/19_10_2017/920916661519007745_3.jpg', 'train', None), ('917815168825995264_0', 'data_image/california_wildfires/10_10_2017/917815168825995264_0.jpg', 'train', None), ('919730693285986304_0', 'data_image/california_wildfires/16_10_2017/919730693285986304_0.jpg', 'train', None), ('919806240183631872_1', 'data_image/california_wildfires/16_10_2017/919806240183631872_1.jpg', 'train', None), ('901646511016116225_0', 'data_image/hurricane_harvey/27_8_2017/901646511016116225_0.jpg', 'train', None), ('908073044341673986_0', 'data_image/hurricane_harvey/13_9_2017/908073044341673986_0.jpg', 'dev', None), ('904516911391805440_0', 'data_image/hurricane_harvey/4_9_2017/904516911391805440_0.jpg', 'test', None), ('901683735837462528_0', 'data_image/hurricane_harvey/27_8_2017/901683735837462528_0.jpg', 'test', None), ('905354545122988032_0', 'data_image/hurricane_harvey/6_9_2017/905354545122988032_0.jpg', 'train', None), ('908805175514558464_0', 'data_image/hurricane_harvey/15_9_2017/908805175514558464_0.jpg', 'train', None), ('909121211409592320_0', 'data_image/hurricane_harvey/16_9_2017/909121211409592320_0.jpg', 'train', None), ('908537559093039105_0', 'data_image/hurricane_harvey/15_9_2017/908537559093039105_0.jpg', 'dev', None), ('905251531615764484_0', 'data_image/hurricane_harvey/6_9_2017/905251531615764484_0.jpg', 'test', None), ('907071126815219712_0', 'data_image/hurricane_harvey/11_9_2017/907071126815219712_0.jpg', 'train', None), ('906015507693703168_0', 'data_image/hurricane_harvey/8_9_2017/906015507693703168_0.jpg', 'dev', None), ('905056852383113216_0', 'data_image/hurricane_harvey/5_9_2017/905056852383113216_0.jpg', 'test', None), ('906226780385705984_0', 'data_image/hurricane_harvey/8_9_2017/906226780385705984_0.jpg', 'train', None), ('904406703046762496_0', 'data_image/hurricane_harvey/3_9_2017/904406703046762496_0.jpg', 'test', None), ('904379734745866240_0', 'data_image/hurricane_harvey/3_9_2017/904379734745866240_0.jpg', 'train', None), ('909904324360359936_2', 'data_image/hurricane_irma/18_9_2017/909904324360359936_2.jpg', 'train', None), ('909715026872762368_0', 'data_image/hurricane_irma/18_9_2017/909715026872762368_0.jpg', 'test', None), ('909846121169801219_1', 'data_image/hurricane_irma/18_9_2017/909846121169801219_1.jpg', 'test', None), ('910246898497458180_0', 'data_image/hurricane_irma/19_9_2017/910246898497458180_0.jpg', 'test', None), ('910259542763634688_0', 'data_image/hurricane_irma/19_9_2017/910259542763634688_0.jpg', 'train', None), ('909999286607376385_0', 'data_image/hurricane_irma/19_9_2017/909999286607376385_0.jpg', 'train', None), ('909777210491510784_1', 'data_image/hurricane_irma/18_9_2017/909777210491510784_1.jpg', 'train', None), ('910260829324324864_0', 'data_image/hurricane_irma/19_9_2017/910260829324324864_0.jpg', 'train', None), ('909846121169801219_2', 'data_image/hurricane_irma/18_9_2017/909846121169801219_2.jpg', 'test', None), ('909739383481360385_0', 'data_image/hurricane_irma/18_9_2017/909739383481360385_0.jpg', 'test', None), ('909479804797423616_0', 'data_image/hurricane_irma/17_9_2017/909479804797423616_0.jpg', 'train', None), ('909925880851517445_0', 'data_image/hurricane_irma/18_9_2017/909925880851517445_0.jpg', 'test', None), ('910118870001881088_0', 'data_image/hurricane_irma/19_9_2017/910118870001881088_0.jpg', 'train', None), ('909395364133863424_0', 'data_image/hurricane_irma/17_9_2017/909395364133863424_0.jpg', 'dev', None), ('909709694658334721_0', 'data_image/hurricane_irma/18_9_2017/909709694658334721_0.jpg', 'dev', None), ('914142837112999936_0', 'data_image/hurricane_maria/30_9_2017/914142837112999936_0.jpg', 'dev', None), ('916134570759802880_0', 'data_image/hurricane_maria/6_10_2017/916134570759802880_0.jpg', 'train', None), ('913101371246878726_0', 'data_image/hurricane_maria/27_9_2017/913101371246878726_0.jpg', 'test', None), ('910523398253461504_2', 'data_image/hurricane_maria/20_9_2017/910523398253461504_2.jpg', 'train', None), ('922728894112976896_0', 'data_image/hurricane_maria/24_10_2017/922728894112976896_0.jpg', 'train', None), ('922225811721142274_3', 'data_image/hurricane_maria/22_10_2017/922225811721142274_3.jpg', 'train', None), ('921304553840152576_0', 'data_image/hurricane_maria/20_10_2017/921304553840152576_0.jpg', 'test', None), ('910545522133528576_0', 'data_image/hurricane_maria/20_9_2017/910545522133528576_0.jpg', 'train', None), ('911753146757795842_2', 'data_image/hurricane_maria/24_9_2017/911753146757795842_2.jpg', 'dev', None), ('913514800189251584_0', 'data_image/hurricane_maria/28_9_2017/913514800189251584_0.jpg', 'train', None), ('921533628965900288_0', 'data_image/hurricane_maria/21_10_2017/921533628965900288_0.jpg', 'train', None), ('913049726852796416_0', 'data_image/hurricane_maria/27_9_2017/913049726852796416_0.jpg', 'dev', None), ('913616129481875456_0', 'data_image/hurricane_maria/29_9_2017/913616129481875456_0.jpg', 'dev', None), ('911683671358365698_1', 'data_image/hurricane_maria/23_9_2017/911683671358365698_1.jpg', 'train', None), ('930373217159712768_0', 'data_image/iraq_iran_earthquake/14_11_2017/930373217159712768_0.jpg', 'test', None), ('929990524798492672_0', 'data_image/iraq_iran_earthquake/13_11_2017/929990524798492672_0.jpg', 'dev', None), ('930001895288029184_1', 'data_image/iraq_iran_earthquake/13_11_2017/930001895288029184_1.jpg', 'train', None), ('931691151236780037_0', 'data_image/iraq_iran_earthquake/18_11_2017/931691151236780037_0.jpg', 'test', None), ('930181046791962626_0', 'data_image/iraq_iran_earthquake/13_11_2017/930181046791962626_0.jpg', 'train', None), ('913068210882322432_0', 'data_image/mexico_earthquake/27_9_2017/913068210882322432_0.jpg', 'train', None), ('913044211611652097_0', 'data_image/mexico_earthquake/27_9_2017/913044211611652097_0.jpg', 'test', None), ('911656099463467008_0', 'data_image/mexico_earthquake/23_9_2017/911656099463467008_0.jpg', 'train', None), ('911634888998055936_0', 'data_image/mexico_earthquake/23_9_2017/911634888998055936_0.jpg', 'train', None), ('913257481320718336_0', 'data_image/mexico_earthquake/28_9_2017/913257481320718336_0.jpg', 'test', None), ('910536593664090112_0', 'data_image/mexico_earthquake/20_9_2017/910536593664090112_0.jpg', 'train', None), ('910523398144397312_2', 'data_image/mexico_earthquake/20_9_2017/910523398144397312_2.jpg', 'train', None), ('911661864156725248_0', 'data_image/mexico_earthquake/23_9_2017/911661864156725248_0.jpg', 'train', None), ('873178065395961857_0', 'data_image/srilanka_floods/9_6_2017/873178065395961857_0.jpg', 'dev', None), ('870169438556897281_0', 'data_image/srilanka_floods/1_6_2017/870169438556897281_0.jpg', 'test', None), ('870389549116948480_1', 'data_image/srilanka_floods/1_6_2017/870389549116948480_1.jpg', 'train', None), ('871763298286305280_0', 'data_image/srilanka_floods/5_6_2017/871763298286305280_0.jpg', 'test', None), ('869950122972635136_0', 'data_image/srilanka_floods/31_5_2017/869950122972635136_0.jpg', 'train', None), ('920428996029947905_0', 'data_image/california_wildfires/17_10_2017/920428996029947905_0.png', 'dev', None), ('921396519386927105_0', 'data_image/california_wildfires/20_10_2017/921396519386927105_0.jpg', 'test', None), ('917815040962695168_0', 'data_image/california_wildfires/10_10_2017/917815040962695168_0.jpg', 'test', None), ('917863129161252865_0', 'data_image/california_wildfires/10_10_2017/917863129161252865_0.jpg', 'test', None), ('921883056146927616_0', 'data_image/california_wildfires/21_10_2017/921883056146927616_0.jpg', 'train', None), ('923263948518055936_0', 'data_image/california_wildfires/25_10_2017/923263948518055936_0.jpg', 'dev', None), ('920683918457819136_0', 'data_image/california_wildfires/18_10_2017/920683918457819136_0.jpg', 'dev', None), ('918101737835319297_0', 'data_image/california_wildfires/11_10_2017/918101737835319297_0.jpg', 'dev', None), ('919924193977712640_0', 'data_image/california_wildfires/16_10_2017/919924193977712640_0.jpg', 'test', None), ('904842215620583425_1', 'data_image/hurricane_harvey/4_9_2017/904842215620583425_1.jpg', 'train', None), ('904324002176532481_3', 'data_image/hurricane_harvey/3_9_2017/904324002176532481_3.jpg', 'train', None), ('908344663698001922_3', 'data_image/hurricane_harvey/14_9_2017/908344663698001922_3.jpg', 'train', None), ('905433786166513664_0', 'data_image/hurricane_harvey/6_9_2017/905433786166513664_0.jpg', 'test', None), ('901752316205953024_0', 'data_image/hurricane_harvey/27_8_2017/901752316205953024_0.jpg', 'dev', None), ('906192288426196992_0', 'data_image/hurricane_harvey/8_9_2017/906192288426196992_0.jpg', 'test', None), ('904907562679767040_0', 'data_image/hurricane_harvey/5_9_2017/904907562679767040_0.jpg', 'train', None), ('904814892095598592_0', 'data_image/hurricane_harvey/4_9_2017/904814892095598592_0.jpg', 'dev', None), ('907238863030145024_1', 'data_image/hurricane_harvey/11_9_2017/907238863030145024_1.jpg', 'train', None), ('904303802639056896_0', 'data_image/hurricane_harvey/3_9_2017/904303802639056896_0.jpg', 'test', None), ('901658895671840768_0', 'data_image/hurricane_harvey/27_8_2017/901658895671840768_0.jpg', 'train', None), ('909892429716774917_0', 'data_image/hurricane_harvey/18_9_2017/909892429716774917_0.jpg', 'dev', None), ('908359696406466560_0', 'data_image/hurricane_harvey/14_9_2017/908359696406466560_0.jpg', 'train', None), ('908482791712673792_0', 'data_image/hurricane_harvey/15_9_2017/908482791712673792_0.jpg', 'train', None), ('901663546701905920_2', 'data_image/hurricane_harvey/27_8_2017/901663546701905920_2.jpg', 'train', None), ('909897178574761984_0', 'data_image/hurricane_irma/18_9_2017/909897178574761984_0.jpg', 'dev', None), ('910192971492012032_0', 'data_image/hurricane_irma/19_9_2017/910192971492012032_0.jpg', 'dev', None), ('909890699600044032_0', 'data_image/hurricane_irma/18_9_2017/909890699600044032_0.jpg', 'train', None), ('909779240689848320_0', 'data_image/hurricane_irma/18_9_2017/909779240689848320_0.jpg', 'dev', None), ('909411653699108866_0', 'data_image/hurricane_irma/17_9_2017/909411653699108866_0.jpg', 'test', None), ('909974841272487936_0', 'data_image/hurricane_irma/19_9_2017/909974841272487936_0.jpg', 'dev', None), ('909870032540327936_0', 'data_image/hurricane_irma/18_9_2017/909870032540327936_0.jpg', 'test', None), ('909768841718325249_0', 'data_image/hurricane_irma/18_9_2017/909768841718325249_0.jpg', 'test', None), ('910184448469164032_0', 'data_image/hurricane_irma/19_9_2017/910184448469164032_0.jpg', 'train', None), ('909791620857581568_1', 'data_image/hurricane_irma/18_9_2017/909791620857581568_1.jpg', 'test', None), ('910160153055498241_1', 'data_image/hurricane_irma/19_9_2017/910160153055498241_1.jpg', 'train', None), ('910156548843991040_0', 'data_image/hurricane_irma/19_9_2017/910156548843991040_0.jpg', 'test', None), ('910045690558005248_0', 'data_image/hurricane_irma/19_9_2017/910045690558005248_0.jpg', 'train', None), ('909753534484221953_0', 'data_image/hurricane_irma/18_9_2017/909753534484221953_0.jpg', 'train', None), ('910145634967199744_0', 'data_image/hurricane_irma/19_9_2017/910145634967199744_0.jpg', 'train', None), ('922722347085316096_0', 'data_image/hurricane_maria/24_10_2017/922722347085316096_0.jpg', 'test', None), ('916262585552556032_0', 'data_image/hurricane_maria/6_10_2017/916262585552556032_0.jpg', 'train', None), ('915251009227776001_0', 'data_image/hurricane_maria/3_10_2017/915251009227776001_0.jpg', 'train', None), ('912346202716467200_0', 'data_image/hurricane_maria/25_9_2017/912346202716467200_0.jpg', 'test', None), ('911571811263041536_0', 'data_image/hurricane_maria/23_9_2017/911571811263041536_0.jpg', 'dev', None), ('911493372866551808_0', 'data_image/hurricane_maria/23_9_2017/911493372866551808_0.jpg', 'dev', None), ('910567805531164673_0', 'data_image/hurricane_maria/20_9_2017/910567805531164673_0.jpg', 'dev', None), ('910785863197626368_0', 'data_image/hurricane_maria/21_9_2017/910785863197626368_0.jpg', 'dev', None), ('910559723770040326_1', 'data_image/hurricane_maria/20_9_2017/910559723770040326_1.jpg', 'train', None), ('910714596381663233_0', 'data_image/hurricane_maria/21_9_2017/910714596381663233_0.jpg', 'dev', None), ('921693401602580480_0', 'data_image/hurricane_maria/21_10_2017/921693401602580480_0.jpg', 'dev', None), ('922220710541561857_0', 'data_image/hurricane_maria/22_10_2017/922220710541561857_0.jpg', 'train', None), ('913015748901163008_0', 'data_image/hurricane_maria/27_9_2017/913015748901163008_0.jpg', 'train', None), ('916283550357491713_0', 'data_image/hurricane_maria/6_10_2017/916283550357491713_0.jpg', 'train', None), ('930133906107617280_0', 'data_image/iraq_iran_earthquake/13_11_2017/930133906107617280_0.jpg', 'dev', None), ('930274091738845185_0', 'data_image/iraq_iran_earthquake/14_11_2017/930274091738845185_0.jpg', 'test', None), ('930121838763626496_0', 'data_image/iraq_iran_earthquake/13_11_2017/930121838763626496_0.jpg', 'train', None), ('930193943408586752_1', 'data_image/iraq_iran_earthquake/13_11_2017/930193943408586752_1.jpg', 'train', None), ('930044431037861888_0', 'data_image/iraq_iran_earthquake/13_11_2017/930044431037861888_0.jpg', 'train', None), ('911624557311672321_0', 'data_image/mexico_earthquake/23_9_2017/911624557311672321_0.jpg', 'test', None), ('910524557483798529_0', 'data_image/mexico_earthquake/20_9_2017/910524557483798529_0.jpg', 'test', None), ('913089638583152641_3', 'data_image/mexico_earthquake/27_9_2017/913089638583152641_3.jpg', 'train', None), ('912268009120681984_0', 'data_image/mexico_earthquake/25_9_2017/912268009120681984_0.jpg', 'train', None), ('911932765062311936_0', 'data_image/mexico_earthquake/24_9_2017/911932765062311936_0.jpg', 'dev', None), ('910677217528762368_0', 'data_image/mexico_earthquake/21_9_2017/910677217528762368_0.jpg', 'dev', None), ('914376430359908352_0', 'data_image/mexico_earthquake/1_10_2017/914376430359908352_0.jpg', 'test', None), ('870182420582199296_0', 'data_image/srilanka_floods/1_6_2017/870182420582199296_0.png', 'train', None), ('880988166219497476_0', 'data_image/srilanka_floods/1_7_2017/880988166219497476_0.jpg', 'test', None), ('869977622377320448_2', 'data_image/srilanka_floods/31_5_2017/869977622377320448_2.jpg', 'train', None), ('869963334401052673_0', 'data_image/srilanka_floods/31_5_2017/869963334401052673_0.jpg', 'test', None), ('917976747139846144_0', 'data_image/california_wildfires/11_10_2017/917976747139846144_0.jpg', 'train', None), ('920104629094010880_0', 'data_image/california_wildfires/17_10_2017/920104629094010880_0.jpg', 'train', None), ('917944403481714688_0', 'data_image/california_wildfires/11_10_2017/917944403481714688_0.jpg', 'train', None), ('920029485423980544_0', 'data_image/california_wildfires/16_10_2017/920029485423980544_0.jpg', 'dev', None), ('922977684728061952_0', 'data_image/california_wildfires/25_10_2017/922977684728061952_0.jpg', 'train', None), ('919665020899704832_0', 'data_image/california_wildfires/15_10_2017/919665020899704832_0.jpg', 'train', None), ('917832175881412608_0', 'data_image/california_wildfires/10_10_2017/917832175881412608_0.jpg', 'train', None), ('917853524301623298_0', 'data_image/california_wildfires/10_10_2017/917853524301623298_0.jpg', 'train', None), ('921066672882114560_0', 'data_image/california_wildfires/19_10_2017/921066672882114560_0.jpg', 'train', None), ('904556415527813120_0', 'data_image/hurricane_harvey/4_9_2017/904556415527813120_0.jpg', 'train', None), ('906468629419581440_0', 'data_image/hurricane_harvey/9_9_2017/906468629419581440_0.jpg', 'test', None), ('904343743075160069_2', 'data_image/hurricane_harvey/3_9_2017/904343743075160069_2.jpg', 'test', None), ('907429078562521090_0', 'data_image/hurricane_harvey/12_9_2017/907429078562521090_0.jpg', 'test', None), ('907689277408075777_0', 'data_image/hurricane_harvey/12_9_2017/907689277408075777_0.jpg', 'train', None), ('906122462311714817_0', 'data_image/hurricane_harvey/8_9_2017/906122462311714817_0.jpg', 'train', None), ('906257689742901248_0', 'data_image/hurricane_harvey/8_9_2017/906257689742901248_0.jpg', 'train', None), ('905164194970050561_0', 'data_image/hurricane_harvey/5_9_2017/905164194970050561_0.jpg', 'train', None), ('904881590005850117_0', 'data_image/hurricane_harvey/5_9_2017/904881590005850117_0.jpg', 'test', None), ('905362011139313664_0', 'data_image/hurricane_harvey/6_9_2017/905362011139313664_0.jpg', 'train', None), ('906911422747160578_0', 'data_image/hurricane_harvey/10_9_2017/906911422747160578_0.jpg', 'train', None), ('907724023076966401_0', 'data_image/hurricane_harvey/12_9_2017/907724023076966401_0.jpg', 'dev', None), ('908113634898870272_0', 'data_image/hurricane_harvey/13_9_2017/908113634898870272_0.jpg', 'train', None), ('906258753707790336_0', 'data_image/hurricane_harvey/8_9_2017/906258753707790336_0.jpg', 'test', None), ('908078869026091013_0', 'data_image/hurricane_harvey/13_9_2017/908078869026091013_0.jpg', 'train', None), ('910063271058608128_0', 'data_image/hurricane_irma/19_9_2017/910063271058608128_0.jpg', 'dev', None), ('909791620857581568_2', 'data_image/hurricane_irma/18_9_2017/909791620857581568_2.jpg', 'test', None), ('909490457603919872_1', 'data_image/hurricane_irma/17_9_2017/909490457603919872_1.jpg', 'train', None), ('905636901306163200_1', 'data_image/hurricane_irma/7_9_2017/905636901306163200_1.jpg', 'train', None), ('909803016621928448_0', 'data_image/hurricane_irma/18_9_2017/909803016621928448_0.jpg', 'test', None), ('909705256078016513_1', 'data_image/hurricane_irma/18_9_2017/909705256078016513_1.jpg', 'train', None), ('910032178825895936_0', 'data_image/hurricane_irma/19_9_2017/910032178825895936_0.jpg', 'train', None), ('905658298082267140_0', 'data_image/hurricane_irma/7_9_2017/905658298082267140_0.jpg', 'train', None), ('910247213900730368_0', 'data_image/hurricane_irma/19_9_2017/910247213900730368_0.jpg', 'train', None), ('909396901254090752_0', 'data_image/hurricane_irma/17_9_2017/909396901254090752_0.jpg', 'train', None), ('910262035954520071_0', 'data_image/hurricane_irma/19_9_2017/910262035954520071_0.jpg', 'dev', None), ('906928056790433792_0', 'data_image/hurricane_irma/10_9_2017/906928056790433792_0.jpg', 'train', None), ('909449133039030272_0', 'data_image/hurricane_irma/17_9_2017/909449133039030272_0.jpg', 'train', None), ('910178328694403073_1', 'data_image/hurricane_irma/19_9_2017/910178328694403073_1.jpg', 'train', None), ('909839736734404609_0', 'data_image/hurricane_irma/18_9_2017/909839736734404609_0.jpg', 'train', None), ('915319640145911809_3', 'data_image/hurricane_maria/3_10_2017/915319640145911809_3.jpg', 'test', None), ('921560430803382272_0', 'data_image/hurricane_maria/21_10_2017/921560430803382272_0.jpg', 'test', None), ('910534972699811843_0', 'data_image/hurricane_maria/20_9_2017/910534972699811843_0.jpg', 'test', None), ('912042078745485312_0', 'data_image/hurricane_maria/24_9_2017/912042078745485312_0.jpg', 'test', None), ('912333428430901249_0', 'data_image/hurricane_maria/25_9_2017/912333428430901249_0.jpg', 'dev', None), ('916301851464740864_0', 'data_image/hurricane_maria/6_10_2017/916301851464740864_0.jpg', 'test', None), ('928819998495313920_0', 'data_image/hurricane_maria/10_11_2017/928819998495313920_0.jpg', 'dev', None), ('913934298243305473_1', 'data_image/hurricane_maria/30_9_2017/913934298243305473_1.jpg', 'train', None), ('912314148762955776_0', 'data_image/hurricane_maria/25_9_2017/912314148762955776_0.jpg', 'dev', None), ('916020944187076609_0', 'data_image/hurricane_maria/5_10_2017/916020944187076609_0.jpg', 'dev', None), ('910628645043089408_0', 'data_image/hurricane_maria/20_9_2017/910628645043089408_0.jpg', 'train', None), ('911551604196872192_0', 'data_image/hurricane_maria/23_9_2017/911551604196872192_0.jpg', 'dev', None), ('912098852752281600_0', 'data_image/hurricane_maria/24_9_2017/912098852752281600_0.jpg', 'dev', None), ('916002828711333888_0', 'data_image/hurricane_maria/5_10_2017/916002828711333888_0.jpg', 'dev', None), ('930064975304966145_0', 'data_image/iraq_iran_earthquake/13_11_2017/930064975304966145_0.jpg', 'dev', None), ('929992374293495808_0', 'data_image/iraq_iran_earthquake/13_11_2017/929992374293495808_0.jpg', 'train', None), ('930174446383460352_0', 'data_image/iraq_iran_earthquake/13_11_2017/930174446383460352_0.jpg', 'train', None), ('930039379996938242_1', 'data_image/iraq_iran_earthquake/13_11_2017/930039379996938242_1.jpg', 'train', None), ('930407656224968704_0', 'data_image/iraq_iran_earthquake/14_11_2017/930407656224968704_0.jpg', 'train', None), ('913191492868440071_0', 'data_image/mexico_earthquake/27_9_2017/913191492868440071_0.jpg', 'train', None), ('910746110511210496_0', 'data_image/mexico_earthquake/21_9_2017/910746110511210496_0.jpg', 'test', None), ('910563191700185088_1', 'data_image/mexico_earthquake/20_9_2017/910563191700185088_1.jpg', 'train', None), ('910523949213138944_0', 'data_image/mexico_earthquake/20_9_2017/910523949213138944_0.jpg', 'dev', None), ('910654273729462273_0', 'data_image/mexico_earthquake/20_9_2017/910654273729462273_0.jpg', 'test', None), ('910564980369567744_1', 'data_image/mexico_earthquake/20_9_2017/910564980369567744_1.jpg', 'train', None), ('912355031591948288_0', 'data_image/mexico_earthquake/25_9_2017/912355031591948288_0.jpg', 'dev', None), ('871271580923547648_0', 'data_image/srilanka_floods/4_6_2017/871271580923547648_0.jpg', 'train', None), ('876221636948746242_0', 'data_image/srilanka_floods/17_6_2017/876221636948746242_0.jpg', 'dev', None), ('878417733766873088_0', 'data_image/srilanka_floods/24_6_2017/878417733766873088_0.jpg', 'train', None), ('874131454514429952_0', 'data_image/srilanka_floods/12_6_2017/874131454514429952_0.jpg', 'train', None), ('918102877981364226_0', 'data_image/california_wildfires/11_10_2017/918102877981364226_0.jpg', 'train', None), ('917856722315874306_0', 'data_image/california_wildfires/10_10_2017/917856722315874306_0.jpg', 'test', None), ('921095476455321600_0', 'data_image/california_wildfires/19_10_2017/921095476455321600_0.jpg', 'train', None), ('923070534266613760_0', 'data_image/california_wildfires/25_10_2017/923070534266613760_0.jpg', 'test', None), ('919949040233496576_0', 'data_image/california_wildfires/16_10_2017/919949040233496576_0.jpg', 'train', None), ('922549392397103110_0', 'data_image/california_wildfires/23_10_2017/922549392397103110_0.jpg', 'dev', None), ('919715738104995840_1', 'data_image/california_wildfires/16_10_2017/919715738104995840_1.jpg', 'train', None), ('917850510400589824_0', 'data_image/california_wildfires/10_10_2017/917850510400589824_0.jpg', 'dev', None), ('919600773507383296_0', 'data_image/california_wildfires/15_10_2017/919600773507383296_0.jpg', 'train', None), ('905733959211823104_0', 'data_image/hurricane_harvey/7_9_2017/905733959211823104_0.jpg', 'train', None), ('904925838839169028_0', 'data_image/hurricane_harvey/5_9_2017/904925838839169028_0.jpg', 'train', None), ('905315238769299456_0', 'data_image/hurricane_harvey/6_9_2017/905315238769299456_0.jpg', 'test', None), ('906584717054566400_0', 'data_image/hurricane_harvey/9_9_2017/906584717054566400_0.jpg', 'test', None), ('910119881718112262_0', 'data_image/hurricane_harvey/19_9_2017/910119881718112262_0.jpg', 'train', None), ('907363126143631360_0', 'data_image/hurricane_harvey/11_9_2017/907363126143631360_0.jpg', 'train', None), ('909113302000824323_0', 'data_image/hurricane_harvey/16_9_2017/909113302000824323_0.jpg', 'test', None), ('908705573570478080_0', 'data_image/hurricane_harvey/15_9_2017/908705573570478080_0.jpg', 'test', None), ('904602540041310209_0', 'data_image/hurricane_harvey/4_9_2017/904602540041310209_0.jpg', 'test', None), ('906810549387890688_0', 'data_image/hurricane_harvey/10_9_2017/906810549387890688_0.jpg', 'dev', None), ('904609010762096640_0', 'data_image/hurricane_harvey/4_9_2017/904609010762096640_0.jpg', 'train', None), ('904500325834342400_0', 'data_image/hurricane_harvey/4_9_2017/904500325834342400_0.jpg', 'test', None), ('908386514555547651_0', 'data_image/hurricane_harvey/14_9_2017/908386514555547651_0.jpg', 'test', None), ('908763237293760513_0', 'data_image/hurricane_harvey/15_9_2017/908763237293760513_0.jpg', 'train', None), ('904291309502038018_0', 'data_image/hurricane_harvey/3_9_2017/904291309502038018_0.jpg', 'train', None), ('909729724443873280_0', 'data_image/hurricane_irma/18_9_2017/909729724443873280_0.jpg', 'train', None), ('909835088044580864_1', 'data_image/hurricane_irma/18_9_2017/909835088044580864_1.jpg', 'train', None), ('910237972141891585_0', 'data_image/hurricane_irma/19_9_2017/910237972141891585_0.jpg', 'dev', None), ('909769085721956352_0', 'data_image/hurricane_irma/18_9_2017/909769085721956352_0.jpg', 'test', None), ('905625064451833856_0', 'data_image/hurricane_irma/7_9_2017/905625064451833856_0.jpg', 'train', None), ('909868400352120834_0', 'data_image/hurricane_irma/18_9_2017/909868400352120834_0.jpg', 'dev', None), ('909708564691914752_0', 'data_image/hurricane_irma/18_9_2017/909708564691914752_0.jpg', 'dev', None), ('910128100083724288_0', 'data_image/hurricane_irma/19_9_2017/910128100083724288_0.jpg', 'test', None), ('905756573280198656_0', 'data_image/hurricane_irma/7_9_2017/905756573280198656_0.jpg', 'test', None), ('909454515765743617_0', 'data_image/hurricane_irma/17_9_2017/909454515765743617_0.jpg', 'test', None), ('909452604576604165_0', 'data_image/hurricane_irma/17_9_2017/909452604576604165_0.jpg', 'dev', None), ('905652959861587971_0', 'data_image/hurricane_irma/7_9_2017/905652959861587971_0.jpg', 'train', None), ('910246479456956416_2', 'data_image/hurricane_irma/19_9_2017/910246479456956416_2.jpg', 'train', None), ('909709562537811968_0', 'data_image/hurricane_irma/18_9_2017/909709562537811968_0.jpg', 'train', None), ('912461779657162752_0', 'data_image/hurricane_maria/25_9_2017/912461779657162752_0.jpg', 'dev', None), ('910647323260342272_0', 'data_image/hurricane_maria/20_9_2017/910647323260342272_0.jpg', 'test', None), ('912093260880936961_2', 'data_image/hurricane_maria/24_9_2017/912093260880936961_2.jpg', 'train', None), ('910534109553979392_3', 'data_image/hurricane_maria/20_9_2017/910534109553979392_3.jpg', 'test', None), ('921304745561804800_0', 'data_image/hurricane_maria/20_10_2017/921304745561804800_0.jpg', 'train', None), ('915972065399799809_0', 'data_image/hurricane_maria/5_10_2017/915972065399799809_0.jpg', 'test', None), ('921662384976154624_0', 'data_image/hurricane_maria/21_10_2017/921662384976154624_0.jpg', 'test', None), ('911750614631243776_3', 'data_image/hurricane_maria/24_9_2017/911750614631243776_3.jpg', 'test', None), ('921425813970579457_0', 'data_image/hurricane_maria/20_10_2017/921425813970579457_0.jpg', 'train', None), ('912245314559541248_0', 'data_image/hurricane_maria/25_9_2017/912245314559541248_0.jpg', 'train', None), ('910546529877979137_0', 'data_image/hurricane_maria/20_9_2017/910546529877979137_0.jpg', 'train', None), ('911837730413441025_3', 'data_image/hurricane_maria/24_9_2017/911837730413441025_3.jpg', 'train', None), ('913783977340784641_0', 'data_image/hurricane_maria/29_9_2017/913783977340784641_0.jpg', 'train', None), ('916139079032541184_0', 'data_image/hurricane_maria/6_10_2017/916139079032541184_0.jpg', 'train', None), ('930652773116534784_0', 'data_image/iraq_iran_earthquake/15_11_2017/930652773116534784_0.jpg', 'train', None), ('930230964067930112_0', 'data_image/iraq_iran_earthquake/14_11_2017/930230964067930112_0.jpg', 'train', None), ('930000550657064960_0', 'data_image/iraq_iran_earthquake/13_11_2017/930000550657064960_0.jpg', 'train', None), ('930252808607105024_0', 'data_image/iraq_iran_earthquake/14_11_2017/930252808607105024_0.png', 'train', None), ('930934280326385664_0', 'data_image/iraq_iran_earthquake/15_11_2017/930934280326385664_0.jpg', 'test', None), ('912130800748859393_0', 'data_image/mexico_earthquake/25_9_2017/912130800748859393_0.jpg', 'dev', None)]
ROWS = [{'image_id': i, 'image_path': path, 'split': split,
         'image_sha256': sha, 'group': 'tweet:'+i.rsplit('_',1)[0]}
        for i, path, split, sha in COHORT_ROWS]
with stage(RUN, '03_inputs'):
    assert len(ROWS) == 4000 and len({r['image_id'] for r in ROWS}) == 4000
    write_json(RUN/'input_manifest.json', ROWS)
    print('INPUTS', len(ROWS), 'known image hashes', sum(bool(r['image_sha256']) for r in ROWS), flush=True)


## 4. Numerical contract checks and isolated fixture mode

Fixtures test software behavior, never detector accuracy. They do not enter the real image cohort.
Checks include count/area semantics, quantization, failure versus empty output, and a rejected registry mismatch.


In [ ]:
class FixtureDetector:
    signature = {'fixture': True, 'registry_sha256': REGISTRY_SHA256}
    producer_sha256 = digest(signature)
    device = 'cpu'
    def sync(self): pass
    def predict(self, image):
        return [{'label': 'person', 'box': [0, 0, 8, 8], 'score': .9},
                {'label': 'water', 'box': [0, 16, 32, 32], 'score': .8}]

with stage(RUN, '04_numerical_checks'):
    test_hash = '1'*64
    array, audit = summarize([{'label': 'person', 'box': [0,0,20,20], 'score': .9},
                              {'label': 'water', 'box': [0,50,100,100], 'score': .8}],
                             100, 100, test_hash)
    assert array.counts[0] == 1 and array.largest_area[0] == .04
    assert array.count_available[25] is False and array.largest_area[25] == .5
    decoded = SceneArray.deserialize(array.serialize(), test_hash)
    assert len(array.serialize()) == 170 and len(decoded.features(test_hash)) == 130
    assert max(abs(a-b) for a,b in zip(array.largest_area, decoded.largest_area)) <= .5/255
    empty, _ = summarize([], 100, 100, test_hash)
    assert empty.detector_ok and empty.count_available[0] and empty.counts[0] == 0
    assert not unavailable(test_hash).count_available[0]
    if MODE == 'fixture':
        from PIL import Image
        folder = RUN/'fixture_inputs'; folder.mkdir(exist_ok=True)
        ROWS = []
        for number in range(12):
            path = folder/(str(number)+'.png')
            Image.new('RGB', (32,32), (number*10, 20, 30)).save(path)
            ROWS.append({'image_id': 'fixture_'+str(number), 'image_path': str(path),
                         'image_sha256': file_hash(path), 'split': 'train'})
    print('Numerical checks passed; workload', len(ROWS), MODE, flush=True)


## 5. Model identity and real-image preflight

One YOLO-World small checkpoint, FP32, one image at a time. Class text encoding is a one-time setup
cost and its actual feature hash is part of the producer identity. Retain the managed CUDA runtime.
Warmup calls are reported separately and excluded from the final per-image cache measurements.


In [ ]:
with stage(RUN, '05_detector_preflight'):
    if MODE == 'fixture':
        DETECTOR = FixtureDetector(); WEIGHTS = None
        print('Fixture provider: no model inference or detector accuracy validation', flush=True)
    else:
        import torch
        DEVICE = os.environ.get('SCENE_ARRAY_DEVICE', 'cuda:0' if IS_KAGGLE else 'cpu')
        if IS_KAGGLE and not torch.cuda.is_available(): raise RuntimeError('Kaggle GPU unavailable')
        if IS_KAGGLE:
            from ultralytics.utils.downloads import attempt_download_asset
            WEIGHTS = Path(attempt_download_asset(str(RUN/'yolov8s-worldv2.pt')))
        else:
            WEIGHTS = Path(os.environ.get('SCENE_ARRAY_WEIGHTS', str(PROJECT/'models/yolov8s-worldv2.pt')))
        DETECTOR = Detector(WEIGHTS, DEVICE)
        write_json(RUN/'producer.json', {**DETECTOR.signature,
            'producer_sha256': DETECTOR.producer_sha256,
            'load_seconds': DETECTOR.load_seconds,
            'vocabulary_setup_seconds': DETECTOR.vocabulary_setup_seconds})
        real_checks = []
        for row in [r for r in ROWS if r['split']=='train'][:10]:
            path, sha = get_image(row, RUN/'images', IS_KAGGLE)
            result, _ = extract(path, row['image_id'], sha, DETECTOR)
            real_checks.append({'image_id': row['image_id'], 'status': result['status'],
                                'reason': result['failure_reason']})
        write_json(RUN/'preflight.json', real_checks)
        if len(real_checks) != 10 or any(r['status'] != 'ok' for r in real_checks):
            raise RuntimeError('real-image preflight failed; inspect saved evidence')
        first = next(r for r in ROWS if r['split']=='train')
        path, _ = get_image(first, RUN/'images', IS_KAGGLE)
        image = read_image(path)
        for _ in range(10): DETECTOR.predict(image)
        DETECTOR.sync()
        print('Real preflight and 10 warmups completed', flush=True)


## 6. Cache every image, commit each record, export every 25

Use `SCENE_ARRAY_RESUME_DIR` to resume the same run. Source, input list, vocabulary, weights,
text features and detector configuration must match exactly. Failed rows stay explicit; success is
not manufactured by dropping them. The notebook reports partial coverage and failures separately.
SQLite commits survive ordinary process failure; durable Kaggle output retrieval must still be verified.


In [ ]:
with stage(RUN, '06_cache'):
    CACHE_PROGRESS = cache_all(ROWS, DETECTOR, RUN, allow_download=IS_KAGGLE,
                              seconds_limit=int(os.environ.get('SCENE_ARRAY_SECONDS', '14400')))
    print('CACHE RESULT', CACHE_PROGRESS, flush=True)
    if not CACHE_PROGRESS['coverage_complete']:
        raise RuntimeError('bounded session checkpointed; resume required before declaring completion')


## 7. Separate CPU/GPU cost passes

Release the bulk producer before profiling. For each backend: fresh model object and vocabulary setup,
ten warmups, ten fixed development images, three CPU calls/image or ten GPU calls/image. Synchronize GPU
work. Measure memory separately. Throughput is sequential batch-one throughput, not a batch-ten claim.
Supported forward FLOPs use two FLOPs/MAC; uncounted operators remain listed and totals stay **partial**.
No training or VLM runs occur in this scene-only notebook. Network/download time is separate from inference.


In [ ]:
with stage(RUN, '07_cost_profiles'):
    import gc
    del DETECTOR
    gc.collect()
    if MODE == 'real':
        import torch
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        devices = ('cpu', 'cuda:0') if torch.cuda.is_available() else ('cpu',)
        profile(ROWS, WEIGHTS, RUN, devices)
        if 'cuda:0' not in devices:
            write_json(RUN/'profiles/cuda_0.json', {'status': 'unavailable', 'reason': 'no GPU assigned'})
    else:
        write_json(RUN/'profiles/fixture.json', {'status': 'not executed',
            'reason': 'fixture contract checks are not model latency, FLOPs or memory measurements'})


## 8. Coverage, failures, quantization and scheduler handoff

The NPZ has ID-aligned float and quantized-wire feature matrices. Use wire features for deployment-matched
training and preserve the vocabulary/producer contracts. This notebook exports neither SPI labels nor
mission/history values; join those separately by verified identity and enforce your grouped splits.


In [ ]:
with stage(RUN, '08_results'):
    SUMMARY = report(RUN)
    print(json.dumps(SUMMARY, indent=2), flush=True)
    import numpy as np
    with np.load(RUN/'scheduler_arrays.npz', allow_pickle=False) as data:
        assert data['float_features'].shape == (len(ROWS), 130)
        assert len(set(data['image_ids'].tolist())) == len(ROWS)
        wire_error = float(np.abs(data['float_features'][:,32:64] - data['wire_features'][:,32:64]).max())
        assert wire_error <= .5/255 + 1e-7
        print('Actual maximum area quantization error', wire_error, flush=True)
    write_json(RUN/'validation.json', {'mode': MODE, 'records': len(ROWS),
        'software_contract_checks': 'passed', 'area_quantization_max_error': wire_error,
        'real_detector_accuracy': 'not measured', 'scheduler_advantage': 'not evaluated',
        'cache_progress': CACHE_PROGRESS})


## 9. Inspect detections and save diagnostics

These plots expose category coverage, coordinate/count failures and example detections. They are not
accuracy plots. Region counts remain masked and are never displayed as numbers of floods or fires.


In [ ]:
with stage(RUN, '09_diagnostics'):
    import matplotlib.pyplot as plt
    from PIL import ImageDraw
    (RUN/'figures').mkdir(exist_ok=True)
    counts = SUMMARY['per_class_detected_images']
    fig, ax = plt.subplots(figsize=(12,4))
    ax.bar(range(len(CLASSES)), [counts[c] for c in CLASSES])
    ax.set_xticks(range(len(CLASSES)), CLASSES, rotation=70, ha='right')
    ax.set_ylabel('Images with retained detection')
    ax.set_title('Detector coverage by configured class — accuracy not established')
    fig.tight_layout(); fig.savefig(RUN/'figures/class_coverage.png', dpi=140); plt.show()
    cached = [json.loads(s) for s in (RUN/'scene_records.jsonl').read_text().splitlines()]
    lookup = {r['image_id']: r for r in ROWS}
    for number, record in enumerate([r for r in cached if r['status']=='ok'][:4]):
        source = lookup[record['image_id']]
        path, _ = get_image(source, RUN/'images', False)
        image = read_image(path); draw = ImageDraw.Draw(image)
        for detected in record['audit']['detections']:
            draw.rectangle(detected['box'], outline='red', width=2)
            draw.text(tuple(detected['box'][:2]), detected['label'], fill='red')
        image.thumbnail((700,700)); image.save(RUN/'figures'/('example_'+str(number)+'.png'))
        display(image)
    print('Figures saved', RUN/'figures', flush=True)


## 10. Final artifact inventory and acceptance

Retrieve `scene_cache.sqlite`, JSONL, NPZ, registry, source/environment identities, profiles, figures and
validation receipts. Avoid pulling the `images/` cache or model weights onto the laptop by default.
Every record must be inspected through its success/mask fields. Full coverage with failures is incomplete
extraction, even when the notebook itself reached the end. The old 35-byte header is not modified here.


In [ ]:
with stage(RUN, '10_finalize'):
    report(RUN)
    write_json(RUN/'ACCEPTANCE.json', {'mode': MODE, 'coverage_complete': CACHE_PROGRESS['coverage_complete'],
        'all_images_successful': CACHE_PROGRESS['all_successful'], 'status': 'passed' if CACHE_PROGRESS['all_successful'] else 'incomplete',
        'claims': 'software and cache status only; semantic detection accuracy and ranking benefit unmeasured'})
    print('OUTPUT', RUN, 'scene bytes', WIRE_BYTES, 'features', 130, flush=True)
    if not CACHE_PROGRESS['all_successful']:
        raise RuntimeError('extraction has explicit failed records; inspect failures.json')
